# Text → STL: end-to-end pipeline
Type one plain-English description of a **pipe gasket, washer/spacer, L-bracket, rectangular container, or cylindrical container**, including any holes, rounded/chamfered edges, dividers or cutouts → review one checked table → **Build** → download an STL.

**How to run:** Runtime ▸ Change runtime type ▸ **T4 GPU** → run the cells top to bottom (Runtime ▸ Run all). When asked, upload `24679_Project1_Data.xlsx`. Near the top, Run all stops and shows a Hugging Face link and code: open the link, check the code and click Authorize, and the notebook carries on by itself. The first run trains the models and publishes them to public Hugging Face repos with model cards. Later runs, even after Colab disconnects, download them and skip training, and reuse the spreadsheet from the dataset repo instead of asking for the upload. A saved model is only reused if the data and settings it was trained with are unchanged; otherwise it retrains automatically. Set `REUSE_SAVED_MODELS = False` to force retraining. The last-but-one cell starts the app and shows it right in the notebook (plain HTML page + a small Flask server, no Gradio).

| Your pipeline diagram box | Where it lives |
|---|---|
| Input text description → GUI text box (the part AND its features, one box) | GUI cell (§10) |
| Preprocess to numeric decimal form | §2 `normalize_text` (+ `normalize_feature_text`: screw size → clearance-hole diameter) |
| Split the request: part description vs feature description | §9 `split_request` (rules, self-tested) |
| Trained-from-scratch model → part type | §6 (PyTorch classifier, random init) |
| Fine-tuned model → base-part parameters | §7 (LoRA on Qwen2.5) |
| JSON with standard schema for all parts | §3 `SCHEMA`, `params_to_json` |
| Validation (negatives, missing, range) | §5 rules (always on) |
| Table for the user to check/edit → Build button | GUI cell (§10) |
| JSON → CAD script → CAD file → 3D preview, download, start over | §5 CAD + GUI cell |
| Features: holes (single, rows, grids), chamfers/radii, dividers, cutouts — **off-the-shelf model** | §5 feature schema + checks + CAD, §5c feature requests, §7b editing model, GUI "Features" tables |
| User edits loop back | edited values are re-checked on every Build; changing the part-type dropdown re-runs the extractor; every build is logged to `work/feedback.jsonl` |

The deterministic parts (preprocessing, validation, CAD) run and self-test **before** the slow training cells, so a problem there shows up in seconds.

In [ ]:
# Runtime > Change runtime type > T4 GPU (needed for the fine-tuning cell). Then run this cell once.
# If Colab asks you to restart the runtime after installing, do that and continue from the next cell.
!pip -q install cadquery trimesh flask "transformers>=4.44" peft accelerate openpyxl scikit-learn
# Colab ships an old torchao (0.10) that newer peft refuses to import; we don't use it, so remove it.
!pip -q uninstall -y torchao

## 0. Setup & settings
Everything you might want to tune is at the top of this cell.

In [ ]:
import os, re, json, math, random, time, zipfile, copy
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # must be set before torch touches the GPU; reduces memory fragmentation
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import pandas as pd

# ------------------------------------------------------------------ CONFIG
SEED = 42
DATA_PATH = None                            # None = Colab asks you to upload the .xlsx every run (so an old copy is never used by accident). Locally: set a path.
WORKDIR = Path("work"); WORKDIR.mkdir(exist_ok=True)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # fine-tuned model. Use "Qwen/Qwen2.5-0.5B-Instruct" for a faster/lighter run
LORA_EPOCHS = 5                             # fewer epochs than before: with augmentation there is ~2.5x more training data per epoch
LORA_LR = 2e-4
BATCH_SIZE = 8                              # examples per optimizer step (effective batch)
MICRO_BATCH = 2                             # examples per forward/backward pass; BATCH_SIZE is reached by gradient accumulation. Lower to 1 if you still run out of GPU memory
GRAD_CHECKPOINTING = True                   # recompute activations instead of storing them: ~30% slower but needs far less GPU memory (needed for the 1.5B model on a T4)

LOAD_FEATURE_DATA = True                    # feature requests (built-in hand-written + generated) = the editing model's example bank and test set. False = skip
FEATURE_GENERATED_PER_TYPE = 150            # generated feature requests per part type (3 types) for the example bank. 0 = only the hand-written ones
FEATURE_MODEL_NAME = None                   # OFF-THE-SHELF editing model. None = reuse the base model of the fine-tuned extractor with its adapter switched off
                                            # (no extra memory). Try "Qwen/Qwen2.5-3B-Instruct" for better accuracy; it falls back to the base model if it can't load
FEATURE_K = 6                               # how many similar examples are retrieved and shown to the editing model for each request
AUGMENT_PER_TYPE = 100                      # synthetic training samples per part type (5 types x 100 = 500). Set 0 to turn augmentation off
AUG_SEED = 7
TARGET_MANUAL_SAMPLES = 500                 # assignment goal, only used for a progress line
EXCLUDE_AUDIT_FLAGGED = False               # True = drop rows whose label numbers don't appear in their text
GROUP_SPLIT = True                          # True = a given pipe size / screw size never appears in both train and test
TEST_FRAC, VAL_FRAC = 0.15, 0.15

NUM_TOL = 0.0075                            # inches: tolerance for "value appears in text" and for eval correctness
SNAP_TO_STANDARD = True                     # snap values within tolerance of a known catalog size to that size...
SNAP_PART_TYPES = {"pipe_gasket", "washer", "bracket"}   # ...but ONLY for catalog parts. Container sizes are free-form (3.58 vs 3.57 are both real)
RECT_DIMS_ARE_OUTER = True                  # rectangular container: length/width are OUTSIDE dimensions (walls + floor are cut inward from them)
USE_LLM_VERIFIER = False                    # optional off-the-shelf LLM check (weak with small models, see Stage 3)

# ---- Hugging Face: trained models are saved to public repos (with model cards) and reloaded from them instead of retraining
HF_USER = "yennik16"                        # your Hugging Face username
CLF_REPO = "text-to-stl-part-classifier"    # Stage 1 model repo
EXT_REPO = "text-to-stl-parameter-extractor"  # Stage 2 model repo (LoRA adapter)
FEAT_REPO = "text-to-stl-feature-editor"    # Stage 3 repo (off-the-shelf model: prompts + example bank, no weights)
DATA_REPO = "text-to-stl-parts"             # dataset repo (includes a copy of your spreadsheet)
REUSE_SAVED_MODELS = True                   # load the models from those repos when they were trained on this run's data and settings
REUSE_SAVED_DATA = True                     # use the spreadsheet saved in the dataset repo instead of asking for an upload
PUBLISH_TO_HF = True                        # upload newly trained models, and in the last cell the model cards, example bank and dataset
                                            # (Run all stops near the top to have you sign in to Hugging Face: open the link and approve the code)
MODEL_LICENSE, DATA_LICENSE = "apache-2.0", "cc-by-4.0"
COLAB_LINK = ""                             # optional: this notebook's share link (File > Share), shown in the cards

random.seed(SEED); np.random.seed(SEED)

def show_df(df):
    """pretty table in Colab/Jupyter, plain text elsewhere"""
    try: display(df)
    except NameError: print(df.to_string())

**Hugging Face Hub:** trained models are saved to public repos with model cards and reloaded from there on later runs instead of retraining (see the Hugging Face settings above).

In [ ]:
# ------------------------------------------------------------------ HUGGING FACE HUB: reload trained models instead of retraining
# Each trained model is uploaded with a fingerprint of the data and settings it was trained on. A later run (even after Colab disconnects)
# downloads it and skips training if the fingerprint matches; if anything changed, it retrains and uploads the new model.
import hashlib
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from safetensors.torch import save_file

def repo_id(name): return f"{HF_USER}/{name}"

import os, time, requests
HF_ENDPOINT = os.environ.get("HF_ENDPOINT", "https://huggingface.co").rstrip("/")

def hf_sign_in():
    """Sign in to Hugging Face in your browser: shows a link and a code, and waits here until you approve it on Hugging Face."""
    client_id = os.environ.get("HF_DEVICE_CODE_OAUTH_CLIENT_ID", "26be6b09-91c5-47da-9861-d2d2bb7a7e36")   # huggingface_hub's own sign-in app
    r = requests.post(f"{HF_ENDPOINT}/oauth/device", data={"client_id": client_id}, timeout=30); r.raise_for_status(); info = r.json()
    url, code = info.get("verification_uri_complete") or info["verification_uri"], info["user_code"]
    try:
        from IPython.display import HTML, display
        display(HTML(f'<div style="font-size:15px;line-height:1.6"><b>Sign in to Hugging Face to save and publish the models.</b><br>'
                     f'1. Open <a href="{url}" target="_blank">{url}</a><br>2. Check the code matches '
                     f'<b style="font-family:monospace;font-size:20px;letter-spacing:3px">{code}</b> and click Authorize.<br>'
                     f'<i>This cell waits until you do (up to {info.get("expires_in", 900) // 60} minutes).</i></div>'))
    except ImportError:
        print(f"Sign in to Hugging Face: open {url} and confirm the code {code}. Waiting...")
    interval, deadline = info.get("interval", 5), time.time() + info.get("expires_in", 900)
    while time.time() < deadline:
        time.sleep(interval)
        try:
            t = requests.post(f"{HF_ENDPOINT}/oauth/token", timeout=30, data={"grant_type": "urn:ietf:params:oauth:grant-type:device_code",
                                                                              "device_code": info["device_code"], "client_id": client_id})
            data = t.json() if t.status_code < 500 else None
        except Exception:
            data = None                                           # network blip: keep waiting
        if not data or data.get("error") in (None, "authorization_pending"):
            if data and "access_token" in data: return data["access_token"]
            continue
        if data["error"] == "slow_down": interval += 5; continue
        raise RuntimeError({"access_denied": "you declined the sign-in", "expired_token": "the code expired"}.get(data["error"], data["error"]))
    raise RuntimeError("the code expired before it was approved")

_HF_TOKEN = None
if PUBLISH_TO_HF:
    try:
        from google.colab import userdata; _HF_TOKEN = userdata.get("HF_TOKEN")       # a saved Colab secret skips the sign-in
    except Exception:
        pass
    if not _HF_TOKEN:
        try:
            _HF_TOKEN = hf_sign_in()
        except Exception as e:
            print(f"Hugging Face sign-in didn't complete ({e}). Continuing WITHOUT saving to Hugging Face: models are reused from the repos "
                  "if they're there, otherwise trained for this session only. Re-run this cell to sign in again.")
    if _HF_TOKEN:
        try:
            _me = requests.get(f"{HF_ENDPOINT}/api/whoami-v2", headers={"Authorization": f"Bearer {_HF_TOKEN}"}, timeout=30).json()["name"]
            if _me != HF_USER:
                print(f"Signed in as {_me}, so {_me}'s repos are used (HF_USER was {HF_USER}).")
                HF_USER = _me
            print(f"Signed in to Hugging Face as {_me}.")
        except Exception:
            pass

def hf_token():
    return _HF_TOKEN

def fingerprint(*parts):
    """short hash of everything a trained model depends on (data and settings)"""
    h = hashlib.sha256()
    for p in parts:
        h.update((p.to_csv(index=False) if isinstance(p, pd.DataFrame) else json.dumps(p, sort_keys=True, default=str)).encode())
    return h.hexdigest()[:16]

def load_from_hub(name, fp):
    """local copy of a public model repo if it holds a model trained on exactly this data and these settings, else None"""
    if not REUSE_SAVED_MODELS: return None
    try:
        saved_fp = Path(hf_hub_download(repo_id(name), "fingerprint.txt")).read_text().strip()
    except Exception:
        return None                                          # no repo yet, or offline: train
    if saved_fp != fp:
        print(f"{repo_id(name)} holds a model trained on different data or settings - retraining.")
        return None
    try:
        return Path(snapshot_download(repo_id(name)))
    except Exception as e:
        print(f"Couldn't download {repo_id(name)} ({type(e).__name__}) - retraining."); return None

def push_to_hub(name, folder, repo_type="model", what="model"):
    """create the public repo if needed and upload a folder; never stops the notebook"""
    if not (PUBLISH_TO_HF and hf_token()): return            # publishing off, or not signed in
    try:
        api = HfApi(token=hf_token())
        api.create_repo(repo_id(name), repo_type=repo_type, exist_ok=True)
        api.upload_folder(folder_path=str(folder), repo_id=repo_id(name), repo_type=repo_type, commit_message=f"Upload {what} from the notebook")
        print(f"uploaded {what}: https://huggingface.co/{'datasets/' if repo_type == 'dataset' else ''}{repo_id(name)}")
    except Exception as e:
        print(f"Couldn't upload {what} to {repo_id(name)} ({type(e).__name__}: {str(e)[:160]}). Carrying on; the model is still used in this session.")

def write_classifier(folder):
    """the classifier in the published format: safetensors weights, vocabulary, config and fingerprint"""
    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)
    sd = {k: v.detach().cpu().contiguous() for k, v in clf.state_dict().items()}
    save_file(sd, str(folder / "model.safetensors"))
    (folder / "vocab.json").write_text(json.dumps(VOCAB))
    (folder / "config.json").write_text(json.dumps({"part_types": list(PART_TYPES), "vocab_size": len(VOCAB), "embedding_dim": int(sd["emb.weight"].shape[1]),
        "hidden": int(sd["net.1.weight"].shape[0]), "dropout": 0.3, "best_val_acc": float(best_acc),
        "architecture": "EmbeddingBag(mean) -> Dropout -> Linear -> ReLU -> Dropout -> Linear"}, indent=1))
    (folder / "fingerprint.txt").write_text(CLF_FP)

def write_adapter(folder):
    """the LoRA adapter, its tokenizer and fingerprint"""
    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(folder); tok.save_pretrained(folder)
    (folder / "README.md").unlink(missing_ok=True)            # peft's blank template; the real model card is written by the last cell
    (folder / "fingerprint.txt").write_text(EXT_FP)

print("Hugging Face repos:", ", ".join(f"https://huggingface.co/{repo_id(n)}" for n in (CLF_REPO, EXT_REPO, FEAT_REPO)),
      f"https://huggingface.co/datasets/{repo_id(DATA_REPO)}")


## 1. Load and clean the spreadsheet
All sheets are combined. The `L_brackets` sheet's text column is headed `n`; it is renamed to `text`. Rows without text stay available for the catalog tables but can't be used to train the language models.

In [ ]:
# ------------------------------------------------------------------ LOAD + CLEAN THE SPREADSHEET
if (DATA_PATH is None or not Path(DATA_PATH).exists()) and REUSE_SAVED_DATA:
    try:
        DATA_PATH = hf_hub_download(repo_id(DATA_REPO), "raw/data.xlsx", repo_type="dataset")
        print(f"Using the spreadsheet saved in https://huggingface.co/datasets/{repo_id(DATA_REPO)} (set REUSE_SAVED_DATA = False to upload a different one).")
    except Exception:
        pass                                                  # not published yet: ask for the upload
if DATA_PATH is None or not Path(DATA_PATH).exists():
    try:
        from google.colab import files            # only exists inside Colab
        print("Upload your main dataset spreadsheet (.xlsx).")
        DATA_PATH = next(iter(files.upload()))
    except ImportError:
        raise FileNotFoundError("Set DATA_PATH to your .xlsx file (the notebook is not running in Colab).")
print("Using file:", DATA_PATH)

xl = pd.ExcelFile(DATA_PATH)
frames = []
for sheet in xl.sheet_names:
    d = xl.parse(sheet)
    if "text" not in d.columns and "n" in d.columns:   # the L_brackets sheet's text column is headed "n"
        d = d.rename(columns={"n": "text"})
    d["sheet"] = sheet
    d["sheet_row"] = np.arange(len(d)) + 2              # Excel row number (header = row 1)
    frames.append(d)
raw_all = pd.concat(frames, ignore_index=True)
raw_all["text"] = raw_all["text"].astype("string").str.strip()

has_text = raw_all["text"].notna() & (raw_all["text"] != "")
summary = raw_all.assign(has_text=has_text).groupby("sheet").agg(rows=("has_text", "size"), with_text=("has_text", "sum"))
summary["missing_text"] = summary["rows"] - summary["with_text"]
print("sheets found:", xl.sheet_names)
print(summary, "\n")

df_all = raw_all[has_text].copy().reset_index(drop=True)
print(f"Rows with real text (used for ML): {len(df_all)}  |  goal: {TARGET_MANUAL_SAMPLES}  |  still needed: {max(0, TARGET_MANUAL_SAMPLES - len(df_all))}")
print("Rows without text are kept for lookup tables / CAD standards, but not used to train the language models.")

## 2. Preprocessing — everything to decimal numbers
Fractions (`1 1/16`), mixed numbers, word fractions (`a sixteenth`, `three and a half`), number words (`eight bolt holes`, `a dozen`), leading-dot decimals (`.0625`), glued units (`4.62OD`), and screw-size phrasing (`triple aught`, `quarter twenty`, `#8`, `metric 3.9`).  
The **same function** runs on training text and on what a user types, so the models never see two different formats.

In [ ]:
# ------------------------------------------------------------------ PREPROCESSING: everything -> decimal numbers
# Pipeline box: "Data is preprocessed to make all dimensions and numbers numeric decimal form".
# The SAME function is used for training text and for live user input (no train/serve skew).

NUM_WORDS = {"zero":0,"one":1,"two":2,"three":3,"four":4,"five":5,"six":6,"seven":7,"eight":8,"nine":9,"ten":10,
             "eleven":11,"twelve":12,"thirteen":13,"fourteen":14,"fifteen":15,"sixteen":16,"seventeen":17,
             "eighteen":18,"nineteen":19,"twenty":20,"thirty":30,"forty":40,"fifty":50}
_UNITS = "one|two|three|four|five|six|seven|eight|nine"
_WORDNUM = "|".join(sorted(NUM_WORDS, key=len, reverse=True))
WORD_FRACS = {"sixteenth": 1/16, "eighth": 1/8, "quarter": 1/4, "thirty-second": 1/32, "sixty-fourth": 1/64}

def dec(x, places=4):
    """0.0625 -> '0.0625', 3.5 -> '3.5', 4.0 -> '4'  (max 4 decimals, no trailing zeros)"""
    return f"{x:.{places}f}".rstrip("0").rstrip(".")

def _word_or_digit(s):
    s = s.lower()
    return float(NUM_WORDS[s]) if s in NUM_WORDS else float(s)

_AUGHT   = [(re.compile(r"\btriple[- ]aught\b", re.I), "No. 000"),
            (re.compile(r"\bdouble[- ]aught\b", re.I), "No. 00"),
            (re.compile(r"\bsingle[- ]aught\b", re.I), "No. 0")]
_QTR20   = re.compile(r"\bquarter[- ]twenty\b", re.I)
_FRACWORD = {"half": 0.5, "quarter": 0.25, "eighth": 0.125, "sixteenth": 0.0625, "three quarters": 0.75, "three-quarters": 0.75}
_AND_FRAC = re.compile(rf"\b(an|{_WORDNUM}|\d+(?:\.\d+)?)(?:\s+inch(?:es)?)?\s+and\s+(?:(?:a|an)\s+(half|quarter|eighth|sixteenth)|(three[- ]quarters))\b(?:\s+of\s+an)?(?:\s+inch(?:es)?)?", re.I)
_DOZEN   = re.compile(r"\b(?:a\s+)?dozen\b", re.I)
_HALF_IN = re.compile(r"\b(?:a\s+)?half(?:[- ]an?)?[- ]inch(?:es)?\b", re.I)
_AN_INCH = re.compile(r"\ban\s+inch\b", re.I)
_WORDFRAC= re.compile(r"\b(?:an?\s+)?(sixteenth|eighth|quarter|thirty[- ]second|sixty[- ]fourth)\b(?:\s+of\s+an)?(?:\s+inch(?:es)?)?", re.I)
_COUNT_CTX = r'(?=\s*(?:inch|inches|in\b|"|bolts?\b|holes?\b|mounting\b|mount\b|rivets?\b|points?\b|hardware\b|screws?\b|feet\b|foot\b|ft\b|millimet\w*|mm\b|centimet\w*|cm\b|meters?\b|metres?\b))'
_TENS_UNITS = re.compile(rf"\b(twenty|thirty|forty|fifty)[- ]({_UNITS})\b{_COUNT_CTX}", re.I)
_WORDNUM_CTX= re.compile(rf"\b({_WORDNUM})\b{_COUNT_CTX}", re.I)
_NO_SIZE = re.compile(r"(?<!\w)(?:no\.?|number|#)\s*(\d+)\b", re.I)
_SIZE_N  = re.compile(r"\bsize\s+(\d+)\b(?=\s+(?:screw|bolt|hardware))", re.I)
_METRIC  = re.compile(r"\bmetric\s+(\d+(?:\.\d+)?)\b", re.I)
_MIXED   = re.compile(r"(?<![\d.])(\d+)\s+(\d+)/(\d+)(?!\d)")
_FRAC    = re.compile(r"(?<![\d.])(\d+)/(\d+)(?!\d)")
_LEADDOT = re.compile(r"(?<!\d)\.(\d+)")
_DIMX    = re.compile(r"(?<=\d)\s*[xX\u00d7]\s*(?=\d)")          # 3x11 / 3 X 11 / 2x4x6  ->  "3 x 11"
_NUMRE   = r"(\d+(?:\.\d+)?)"
# ---- units -> inches (the whole pipeline works in inches, so the models only ever see inches)
_FT_AND_HALF = re.compile(rf"\b({_WORDNUM}|\d+(?:\.\d+)?)\s+and\s+a\s+half\s+(?:feet|foot|ft)\b", re.I)
_FOOT_WORDS  = [(re.compile(r"\b(?:a|one)\s+foot\s+and\s+a\s+half\b", re.I), "18 inch"),
                (re.compile(r"\bhalf\s+(?:a\s+)?foot\b", re.I), "6 inch"),
                (re.compile(r"\b(?:a|one)\s+foot\b", re.I), "12 inch")]
_FT_IN   = re.compile(_NUMRE + r"\s*(?:feet|foot|ft\b\.?|')\s*(?:and\s+)?" + _NUMRE + r"\s*(?:inches|inch|in\b|\")", re.I)     # 5 ft 6 in / 5'6"
_FT      = re.compile(_NUMRE + r"\s*(?:feet\b|foot\b|ft\b\.?)|" + _NUMRE + r"\s*'(?![A-Za-z0-9])", re.I)                        # 2 feet / 2 ft / 2'
_MM_SCREW= re.compile(_NUMRE + r"\s*mm\s*(?=(?:machine\s+)?(?:screws?|bolts?|fasteners?)\b)", re.I)                          # "6 mm screw" is the size M6, not a length
_MM      = re.compile(_NUMRE + r"\s*(?:millimet(?:er|re)s?|mm)\b", re.I)
_CM      = re.compile(_NUMRE + r"\s*(?:centimet(?:er|re)s?|cm)\b", re.I)
_METERS  = re.compile(_NUMRE + r"\s*(?:meters?|metres?)\b", re.I)
_M_ABBR  = re.compile(r"(?<![A-Za-z])" + _NUMRE + r"\s*m\b(?!\s*\d)")                                                          # bare lowercase 'm' after a number (M5 is a screw, so no capital M)
_GLUE    = re.compile(r"(\d)([A-Za-z])")

def normalize_text(text: str) -> str:
    t = str(text)
    for pat, rep in _AUGHT: t = pat.sub(rep, t)
    t = _QTR20.sub("0.25", t)
    t = _DOZEN.sub("12", t)
    t = _FT_AND_HALF.sub(lambda m: dec((_word_or_digit(m.group(1)) + 0.5) * 12) + " inch", t)
    for pat, rep in _FOOT_WORDS: t = pat.sub(rep, t)
    t = _AND_FRAC.sub(lambda m: dec((1.0 if m.group(1).lower() == "an" else _word_or_digit(m.group(1)))
                                    + (0.75 if m.group(3) else _FRACWORD[m.group(2).lower()])) + " inch", t)
    t = _HALF_IN.sub("0.5 inch", t)
    t = _WORDFRAC.sub(lambda m: dec(WORD_FRACS[re.sub(r"\s", "-", m.group(1).lower())]) + " inch", t)
    t = _AN_INCH.sub("1 inch", t)                               # after the word fractions: "a quarter of an inch" must not leave a stray "1 inch"
    t = _TENS_UNITS.sub(lambda m: str(NUM_WORDS[m.group(1).lower()] + NUM_WORDS[m.group(2).lower()]), t)
    t = _WORDNUM_CTX.sub(lambda m: str(NUM_WORDS[m.group(1).lower()]), t)
    t = _NO_SIZE.sub(lambda m: f"No. {m.group(1)}", t)
    t = _SIZE_N.sub(lambda m: f"No. {m.group(1)}", t)
    t = _METRIC.sub(lambda m: f"M{m.group(1)}", t)
    t = _MIXED.sub(lambda m: dec(int(m.group(1)) + int(m.group(2)) / int(m.group(3))), t)
    t = _FRAC.sub(lambda m: dec(int(m.group(1)) / int(m.group(2))), t)
    t = _LEADDOT.sub(r"0.\1", t)
    t = _DIMX.sub(" x ", t)
    t = _FT_IN.sub(lambda m: dec(float(m.group(1)) * 12 + float(m.group(2))) + " inch", t)
    t = _FT.sub(lambda m: dec(float(m.group(1) or m.group(2)) * 12) + " inch", t)
    t = _MM_SCREW.sub(lambda m: "M" + dec(float(m.group(1))) + " ", t)
    t = _MM.sub(lambda m: dec(float(m.group(1)) / 25.4) + " inch", t)
    t = _CM.sub(lambda m: dec(float(m.group(1)) / 2.54) + " inch", t)
    t = _METERS.sub(lambda m: dec(float(m.group(1)) * 39.3701) + " inch", t)
    t = _M_ABBR.sub(lambda m: dec(float(m.group(1)) * 39.3701) + " inch", t)
    t = _GLUE.sub(r"\1 \2", t)
    return re.sub(r"\s+", " ", t).strip()

# Clearance holes (normal fit) for common screws, in inches: ISO 273 medium for metric, standard clearance drills for inch screws
CLEARANCE_IN = {"M2": 0.0945, "M2.5": 0.1142, "M3": 0.1339, "M4": 0.1772, "M5": 0.2165, "M6": 0.2598, "M8": 0.3543, "M10": 0.4331, "M12": 0.5315, "M16": 0.6890,
                "No. 2": 0.096, "No. 4": 0.1285, "No. 6": 0.1495, "No. 8": 0.177, "No. 10": 0.201, "No. 12": 0.228,
                "0.25": 0.266, "0.3125": 0.332, "0.375": 0.397, "0.5": 0.531}
_SIZE = r"(M\d+(?:\.\d+)?|No\. \d+|0\.25|0\.3125|0\.375|0\.5)"
_FIT_SCREW = re.compile(r"\b(?:fits?|for|clears?|takes?|accepts?|clearance for)\s+(?:an?\s+)?" + _SIZE + r"(?:\s*inch)?\s*(?:bolts?|screws?|fasteners?)\b", re.I)
_SIZE_HOLES = re.compile(r"\b(M\d+(?:\.\d+)?|No\. \d+)\s+(?:clearance\s+)?holes?\b", re.I)    # "M5 holes", "#8 holes" - NOT "1/4 hole" (that is a diameter)

def _canon_size(s):
    s = s.strip(); return ("M" + s[1:]) if s[:1] in "mM" else ("No. " + s.split()[-1]) if s.lower().startswith("no") else s

def normalize_feature_text(text: str) -> str:
    """normalize_text + turn a screw size into the clearance-hole diameter it needs, so the editing model sees a number.
    ONLY for feature requests: in a base-part description 'washer for an M8 bolt' means the screw size, not a hole."""
    t = normalize_text(text)
    def add(m):
        d = CLEARANCE_IN.get(_canon_size(m.group(1)))
        return m.group(0) + (f" ({dec(d)} inch clearance hole)" if d else "")
    t = _FIT_SCREW.sub(add, t)
    return _SIZE_HOLES.sub(add, t)

def numbers_in(text_norm: str):
    """All standalone numbers in already-normalized text (used for the 'is this value in the text?' check)."""
    return [float(x) for x in re.findall(r"(?<![\w.])\d+(?:\.\d+)?", text_norm)]

def tol_for(v: float) -> float:
    """How close counts as 'the same number'. Absolute 0.0075 in for normal sizes, but much tighter for tiny values
    (a 0.012" washer must not be confused with a 0.015" one)."""
    return min(NUM_TOL, max(0.0006, 0.05 * abs(v)))

# ---- quick self-tests (they run every time so a broken edit is caught immediately)
_tests = {
    'make a pipe gasket 1 1/16" ID and 3 7/8" OD. 1/16" thick': ['1.0625"', '3.875"', '0.0625"'],
    "They should be a sixteenth inch thick": ["0.0625 inch thick"],
    "3 and a half inch pipe with a seven and a half inch flange": ["3.5 inch pipe", "7.5 inch flange"],
    "fastened at four mount points, eight bolt holes": ["4 mount points", "8 bolt holes"],
    "4.62OD and 1.66ID, 3.75od, 3.5in": ["4.62 OD", "1.66 ID", "3.75 od", "3.5 in"],
    "washer for a triple aught screw": ["No. 000 screw"],
    "a quarter twenty screw": ["0.25 screw"],
    "for a no1 screw": ["No. 1 screw"],
    "spacer for a #8 screw": ["No. 8 screw"],
    "washer for a size ten screw": ["No. 10 screw"],
    "for a metric 3.9 bolt": ["M3.9 bolt"],
    'it is .0625" tall': ['0.0625" tall'],
    "id should be half an inch, make it an inch wide": ["0.5 inch", "1 inch wide"],
    "square base 3x11 inches, a 2x4x6 box": ["3 x 11 inches", "2 x 4 x 6 box"],
    "a nine inch pipe on a four inch flange with twenty four bolts": ["9 inch pipe", "4 inch flange", "24 bolts"],
    "one leg measuring 1 inch and one measuring 2": ["one leg"],           # 'one' must NOT be converted here
    "12 3/4 pipe with a 20 1/2 flange": ["12.75 pipe", "20.5 flange"],
    "washer for an inch and a half screw": ["1.5 inch screw"],
    "an inch and a quarter wide": ["1.25 inch wide"],
    "a quarter of an inch hole, an eighth of an inch thick": ["0.25 inch hole", "0.125 inch thick"],
    "make it an inch wide": ["1 inch wide"],
    "make a box thats 40inches by 12.73 inches and 1/2\" thick. i want it to be two feet tall": ["40 inches by 12.73 inches", "0.5\" thick", "24 inch tall"],
    "Box 4 x 8 in. 1/2\" thick 1.75\" tall": ["4 x 8 in.", "0.5\" thick", "1.75\" tall"],
    "2 ft long, 5' x 3', a foot and a half, half a foot, two and a half feet": ["24 inch long", "60 inch x 36 inch", "18 inch", "6 inch", "30 inch"],
    "5 feet 6 inches and 5'6\" and 1 1/2 feet": ["66 inch and 66 inch and 18 inch"],
    "300 mm wide, 2.5 cm thick, 1 meter tall, 2m long, 25.4 mm hole": ["11.811 inch wide", "0.9843 inch thick", "39.3701 inch tall", "78.7402 inch long", "1 inch hole"],
    "washer for a 6 mm screw, M5 washer, no. 8 screw": ["M6 screw", "M5 washer", "No. 8 screw"],
    "gasket for 4 and a quarter flange": ["4.25 inch flange"],
    "it should have a dozen holes": ["12 holes"],
    "a 12 and a half inch wide gasket": ["12.5 inch wide"],
}
for src, must in _tests.items():
    out = normalize_text(src)
    assert all(m in out for m in must), f"normalizer test failed:\n  in : {src}\n  out: {out}\n  need: {must}"
assert tol_for(0.84) == NUM_TOL and tol_for(0.0118) < 0.001 and tol_for(0.0625) < NUM_TOL
assert "0.2165 inch clearance hole" in normalize_feature_text("holes should fit an m5 bolt") and "0.177 inch clearance hole" in normalize_feature_text("4 #8 holes")
assert "0.266 inch clearance hole" in normalize_feature_text("holes for a 1/4 bolt") and "clearance" not in normalize_text("washer for an M8 bolt")
assert all("clearance" not in normalize_feature_text(t) for t in ("a 1/4 hole in the left wall", "two 0.5 holes", "a 3/8 inch hole", "1/4 inch holes in the floor"))
assert "0.1772 inch clearance hole" in normalize_feature_text("an M4 hole in the floor") and "0.2165 inch clearance hole" in normalize_feature_text("M5 clearance holes")
print("normalizer self-tests passed")

## 3. Standard schema, catalog tables, train/val/test split
* One JSON shape for every part (5 part types) (same key names wherever a concept is shared, e.g. `thickness_in`).
* **Catalog tables** are built from every sheet row: known sizes (used to snap `0.84` → `0.8438` for gaskets, washers and brackets only; container sizes are free-form so they are never snapped) and the washer size lookup (`M7` → its dimensions).
* With `GROUP_SPLIT = True`, a given pipe size / screw size never appears in both train and test, so test scores reflect genuinely unseen sizes.

In [ ]:
# ------------------------------------------------------------------ ONE STANDARD JSON SCHEMA FOR ALL PARTS
PART_TYPES = ["pipe_gasket", "washer", "bracket", "rect_container", "cyl_container"]
SCHEMA = {                                             # key order here = key order in the JSON the model learns to write
    "pipe_gasket": ["od_in", "id_in", "thickness_in", "hole_count"],
    "washer":      ["screw_size", "od_in", "id_in", "thickness_in"],
    "bracket":     ["length_a_in", "length_b_in", "height_in", "thickness_in"],   # height_in = bracket width (your column name)
    "rect_container": ["length_a_in", "length_b_in", "height_in", "thickness_in"],  # open-top box; thickness = wall (and floor) thickness
    "cyl_container":  ["id_in", "height_in", "thickness_in"],                       # open-top cup; id = INSIDE diameter
}
INT_FIELDS, STR_FIELDS = {"hole_count"}, {"screw_size"}
NUM_FIELDS = {pt: [f for f in fs if f not in STR_FIELDS] for pt, fs in SCHEMA.items()}

def canon_screw_size(s):
    """'No. 10'->'No. 10', 'M2.5'->'M2.5', '1 1/8"'->'1.125'. Returns None if it is not a recognisable screw size
    (the fine-tuned model can emit anything, e.g. a copy of the user's text, so this must never raise)."""
    s2 = normalize_text(str(s).replace('"', "").strip())
    m = re.fullmatch(r"(?i)m\s*(\d+(?:\.\d+)?)", s2)
    if m: return "M" + m.group(1)
    m = re.fullmatch(r"(?i)no\.?\s*(\d+)", s2)
    if m: return "No. " + m.group(1)
    if re.fullmatch(r"\d+(?:\.\d+)?", s2): return s2
    return None

def screw_in_text(sz, text_norm) -> bool:
    if not sz: return False
    if sz.startswith("M"):  return re.search(rf"\b{re.escape(sz)}\b", text_norm, re.I) is not None
    if sz.startswith("No"): return re.search(rf"\bno\.? ?{sz[4:]}\b", text_norm, re.I) is not None
    try: return any(abs(float(sz) - n) <= 1e-3 for n in numbers_in(text_norm))
    except ValueError: return False

def coerce_value(field, v):
    """Turn whatever came back (str/float/None/junk) into a clean typed value, or None."""
    if v is None or (isinstance(v, float) and math.isnan(v)) or (isinstance(v, str) and not v.strip()):
        return None
    try:
        if field in STR_FIELDS: return canon_screw_size(v)
        x = float(normalize_text(v)) if isinstance(v, str) else float(v)
        if field in INT_FIELDS:
            return int(round(x)) if abs(x - round(x)) < 1e-9 else None
        return round(x, 4)
    except (ValueError, TypeError):
        return None

def row_to_params(r) -> dict:
    pt = r["part_type"]
    p = {"part_type": pt}
    for f in SCHEMA[pt]:
        p[f] = coerce_value(f, r[f])
    return p

def params_to_json(p: dict) -> str:
    """Canonical compact JSON (fixed key order) - this exact string is what the fine-tuned model learns to output."""
    ordered = {"part_type": p["part_type"], **{f: p.get(f) for f in SCHEMA[p["part_type"]]}}
    return json.dumps(ordered, separators=(",", ":"))

# ---- every part_type in the sheet must be known to the schema
_unknown = set(raw_all.part_type.dropna()) - set(PART_TYPES)
assert not _unknown, f"spreadsheet has part types the notebook doesn't know: {_unknown}"
missing_types = set(PART_TYPES) - set(raw_all.part_type)
if missing_types: print("note: no rows in the sheet for", missing_types)

# ---- build the modelling table
data = df_all.copy()
data["text_norm"] = data["text"].map(normalize_text)
data["params"]    = data.apply(row_to_params, axis=1)
data["target"]    = data["params"].map(params_to_json)
assert data["params"].map(lambda p: all(v is not None for v in p.values())).all(), "some labelled rows have empty label cells"

# ---- catalog tables built from EVERY sheet row (including rows that have no text yet - geometry is still valid catalog data)
def _col(df, c): return pd.to_numeric(df[c], errors="coerce").dropna().round(4)

def _digits(v): return len(f"{v:.4f}".rstrip("0").split(".")[1]) if "." in f"{v:.4f}".rstrip("0") else 0

def canonical_values(values):
    """Your sheet stores some sizes at two precisions (4.62 AND 4.625, 0.84 AND 0.8438). Merge values that are within
    tol_for() of each other and keep the most precise one, so snapping has ONE catalog value per real size."""
    out, cluster = [], []
    for v in sorted(set(values)):
        if cluster and v - cluster[0] > tol_for(cluster[0]):
            out.append(max(cluster, key=lambda x: (_digits(x), x))); cluster = []
        cluster.append(v)
    if cluster: out.append(max(cluster, key=lambda x: (_digits(x), x)))
    return out

def _num_fields(pt): return [f for f in SCHEMA[pt] if f not in STR_FIELDS and f not in INT_FIELDS]
STANDARDS = {}                                          # known sizes per field. Used to snap (catalog parts only) and for range warnings
for _pt in PART_TYPES:
    _sub = raw_all[raw_all.part_type == _pt]
    STANDARDS[_pt] = {f: (canonical_values(_col(_sub, f)) if _pt in SNAP_PART_TYPES else sorted(set(_col(_sub, f))))
                      for f in _num_fields(_pt)}
_dupe_sizes = sum(len(set(_col(raw_all[raw_all.part_type == pt], f))) - len(canonical_values(_col(raw_all[raw_all.part_type == pt], f)))
                  for pt in ("pipe_gasket", "washer") for f in ["od_in", "id_in", "thickness_in"])
print(f"label precision check: {_dupe_sizes} sizes appear at two different precisions in your sheet (e.g. 4.62 vs 4.625)")
WASHER_LOOKUP = {}                                      # 'M7' -> {od_in, id_in, thickness_in}: fills in a washer when only the size is given
for _, r in raw_all[raw_all.part_type == "washer"].iterrows():
    assert canon_screw_size(r["screw_size"]), f"washer row {r['sheet_row']}: screw_size {r['screw_size']!r} is not a recognisable size"
    WASHER_LOOKUP.setdefault(canon_screw_size(r["screw_size"]).lower(), {f: round(float(r[f]), 4) for f in ["od_in", "id_in", "thickness_in"]})
RANGES = {pt: {f: (float(min(v)), float(max(v))) for f, v in fs.items()} for pt, fs in STANDARDS.items()}
print("washer sizes in lookup table:", len(WASHER_LOOKUP), "| example M7 ->", WASHER_LOOKUP.get("m7"))

# ---- train / val / test split
def _group_key(r):
    p = r["params"]
    if r["part_type"] == "pipe_gasket": return ("g", round(p["od_in"], 1), round(p["id_in"], 1))
    if r["part_type"] == "washer":      return ("w", p["screw_size"])
    if r["part_type"] == "bracket":     return ("b", p["length_a_in"], p["length_b_in"])
    if r["part_type"] == "rect_container": return ("r", p["length_a_in"])        # same length never in both train and test
    return ("c", p["id_in"])

def make_split(df):
    rng = random.Random(SEED)
    df = df.copy(); df["group"] = df.apply(_group_key, axis=1) if GROUP_SPLIT else np.arange(len(df))
    df["split"] = "train"
    for pt, sub in df.groupby("part_type"):
        groups = sorted(set(sub["group"]), key=str); rng.shuffle(groups)
        n_test, n_val, got = TEST_FRAC * len(sub), VAL_FRAC * len(sub), {"test": 0, "val": 0}
        for g in groups:
            rows = sub.index[sub["group"] == g]
            if got["test"] < n_test:  s = "test"
            elif got["val"] < n_val:  s = "val"
            else:                     s = "train"
            if s != "train": got[s] += len(rows)
            df.loc[rows, "split"] = s
    return df

flagged_ids = set()    # filled in by the audit cell below
data = make_split(data)
print(data.groupby(["part_type", "split"]).size().unstack(fill_value=0))
assert set(data.loc[data.split == "train", "part_type"]) == set(PART_TYPES), "train split is missing a part type"

## 4. Audit — do the labels agree with their own text?
A label whose numbers don't appear in its text teaches the model to invent values. The list below gives the sheet name and **Excel row** of each suspect row so you can fix it in the spreadsheet (or set `EXCLUDE_AUDIT_FLAGGED = True`).

In [ ]:
# ------------------------------------------------------------------ AUDIT: do the label numbers actually appear in the text?
def audit_row(r):
    nums, bad = numbers_in(r["text_norm"]), []
    for f in NUM_FIELDS[r["part_type"]]:
        v = r["params"][f]
        ok = (int(v) in [int(n) for n in nums if n == int(n)]) if f in INT_FIELDS else any(abs(v - n) <= tol_for(v) for n in nums)
        if not ok: bad.append((f, v))
    if r["part_type"] == "washer":
        sz = r["params"]["screw_size"]
        if not screw_in_text(sz, r["text_norm"]): bad.append(("screw_size", sz))
    return bad

data["audit_bad"] = data.apply(audit_row, axis=1)
audit_report = data[data.audit_bad.map(len) > 0][["sheet", "sheet_row", "text", "audit_bad"]].copy()
audit_report["numbers_in_text"] = data.loc[audit_report.index, "text_norm"].map(numbers_in)
pd.set_option("display.max_colwidth", 120, "display.width", 220)
print(f"{len(audit_report)} of {len(data)} rows have a label value that is NOT in their own text.")
print("Fix these in the spreadsheet (sheet + Excel row shown), or set EXCLUDE_AUDIT_FLAGGED = True.\n")
show_df(audit_report)

dups = data[data.duplicated("text_norm", keep=False)]
print(f"\nduplicate texts after normalization: {len(dups)}")

if EXCLUDE_AUDIT_FLAGGED and len(audit_report):
    data = data.drop(index=audit_report.index).reset_index(drop=True)
    print(f"dropped {len(audit_report)} flagged rows; {len(data)} remain")
    data = make_split(data.drop(columns=["split", "group"]))
train_df, val_df, test_df = (data[data.split == s].reset_index(drop=True) for s in ["train", "val", "test"])
print(f"\ntrain {len(train_df)} | val {len(val_df)} | test {len(test_df)}")

## 5. Validation layer + CAD generator (deterministic — no ML)
**Validation** turns the model's JSON into the table the user sees. Each value gets a status:

| status | meaning |
|---|---|
| ✅ extracted | the model found it and the number really is in your text |
| ⚠️ not found in your text | the model produced it but it isn't in your description (possible hallucination) |
| ❌ missing | not extracted |
| 🔷 snapped / standard size | matched to a catalog size, or filled from the washer size table (`M7` alone is enough) — values you state explicitly are never overwritten |
| 🟨 assumed default | needed for CAD but not in your data (gasket bolt-circle diameter and bolt-hole size) — editable |
| ✏️ edited | changed by the user |

Hard rules (positive values, OD > ID, bolt holes inside the ring, …) block CAD generation; range warnings don't.

In [ ]:
# ------------------------------------------------------------------ STAGE 3: VALIDATION (rules first, optional off-the-shelf LLM second)
# Pipeline box: "Validation checks for negative values, missing data, out of range, etc."
STATUS_LABEL = {
    "extracted":   "✅ extracted",
    "missing":     "❌ missing",
    "not_in_text": "⚠️ not found in your text",
    "snapped":     "🔷 snapped to catalog size",
    "lookup":      "🔷 from standard-size table",
    "default":     "🟨 assumed default",
    "edited":      "✏️ edited by you",
    "llm_flag":    "⚠️ verifier disagrees",
}
DERIVED_GASKET = ["bolt_circle_in", "bolt_hole_in"]      # not in the dataset -> derived defaults the user can override

def fmt_value(v):
    if v is None: return ""
    if isinstance(v, int) and not isinstance(v, bool): return str(v)
    if isinstance(v, float): return dec(v)
    return str(v)

def snap_value(part_type, field, v):
    if not SNAP_TO_STANDARD or part_type not in SNAP_PART_TYPES or field not in STANDARDS[part_type]: return v
    best = min(STANDARDS[part_type][field], key=lambda s: abs(s - v))
    return best if abs(best - v) <= tol_for(v) else v

def default_bolt_geometry(od, idd, n=None):
    if od <= idd: return None, None                               # invalid OD/ID: the rules will report it
    w = (od - idd) / 2                                            # radial width of the gasket ring
    bc = (od + idd) / 2
    hd = min(max(0.4 * w, 0.19), 1.25, 0.6 * w)
    if n and n >= 2: hd = min(hd, 0.6 * bc * math.sin(math.pi / n))   # many bolts -> smaller default holes so neighbours never touch
    return round(bc, 4), round(hd, 4)

def make_table(part_type, text_norm, raw_params):
    """model output (dict or None) -> list of row dicts {field, value, status, note}"""
    nums, raw = numbers_in(text_norm), (raw_params or {})
    rows = [{"field": f, "value": coerce_value(f, raw.get(f)), "status": "extracted", "note": ""} for f in SCHEMA[part_type]]
    R = {r["field"]: r for r in rows}
    for r in rows:
        if r["value"] is None: r.update(status="missing", note="the model did not find this in your text")

    std = WASHER_LOOKUP.get((R["screw_size"]["value"] or "").lower()) if part_type == "washer" else None
    if std:                                                        # fill only what's missing (partial overrides survive)
        for f, sv in std.items():
            if R[f]["value"] is None:
                R[f].update(value=sv, status="lookup", note=f"standard size for {R['screw_size']['value']}")

    for f in SCHEMA[part_type]:                                    # grounding check + snapping for model-extracted values
        r, v = R[f], R[f]["value"]
        if r["status"] != "extracted": continue
        if f in STR_FIELDS: grounded = screw_in_text(v, text_norm)
        elif f in INT_FIELDS: grounded = v in [int(n) for n in nums if n == int(n)]
        else: grounded = any(abs(v - n) <= tol_for(v) for n in nums)
        if not grounded:
            r.update(status="not_in_text", note="this value is not in your description - please check it"); continue
        if f not in STR_FIELDS and f not in INT_FIELDS:
            sv = snap_value(part_type, f, v)
            if sv != v: r.update(value=sv, status="snapped", note=f"you wrote ~{dec(v)}; catalog size is {dec(sv)}")
        if std and f in std and abs(R[f]["value"] - std[f]) > tol_for(std[f]):
            r["note"] = (r["note"] + " | " if r["note"] else "") + f"differs from standard {R['screw_size']['value']} ({dec(std[f])})"

    if part_type == "pipe_gasket" and R["od_in"]["value"] and R["id_in"]["value"]:
        bc, hd = default_bolt_geometry(R["od_in"]["value"], R["id_in"]["value"], R["hole_count"]["value"])
        rows += [{"field": "bolt_circle_in", "value": bc, "status": "default", "note": "assumed: halfway between ID and OD (not in your data)"},
                 {"field": "bolt_hole_in",   "value": hd, "status": "default", "note": "assumed hole size (not in your data)"}]
    return rows

def rows_to_params(part_type, rows):
    p = {"part_type": part_type}
    for r in rows: p[r["field"]] = coerce_value(r["field"], r["value"])
    return p

def apply_edits(part_type, old_rows, ui_values):
    """ui_values: {field: text typed in the table}. Only changed cells get 'edited'; everything else keeps its status."""
    rows, edited = copy.deepcopy(old_rows), set()
    for r in rows:
        if r["field"] not in ui_values: continue
        nv = coerce_value(r["field"], ui_values[r["field"]])
        if nv != r["value"]:
            r.update(value=nv, status="edited" if nv is not None else "missing", note="changed by you" if nv is not None else "cleared")
            edited.add(r["field"])
    R = {r["field"]: r for r in rows}
    if part_type == "washer" and R["screw_size"]["value"]:         # user typed a size -> fill blanks from the standard table
        std = WASHER_LOOKUP.get(R["screw_size"]["value"].lower())
        for f, sv in (std or {}).items():
            if R[f]["value"] is None: R[f].update(value=sv, status="lookup", note=f"standard size for {R['screw_size']['value']}")
    if part_type == "pipe_gasket" and R["od_in"]["value"] and R["id_in"]["value"]:
        bc, hd = default_bolt_geometry(R["od_in"]["value"], R["id_in"]["value"], R["hole_count"]["value"])
        for f, dv in zip(DERIVED_GASKET, (bc, hd)):                # follow OD/ID edits unless the user set them
            if f not in R: rows.append({"field": f, "value": dv, "status": "default", "note": "assumed (not in your data)"}); R[f] = rows[-1]
            elif f not in edited and R[f]["status"] == "default": R[f]["value"] = dv
    return rows, edited

def check_rules(part_type, p):
    """hard errors block CAD generation; warnings are just shown"""
    issues = []
    err  = lambda f, m: issues.append({"level": "error", "field": f, "msg": m})
    warn = lambda f, m: issues.append({"level": "warning", "field": f, "msg": m})
    def check_present(fields):
        for f in fields:
            v = p.get(f)
            if v is None: err(f, "missing - fill it in the table")
            elif f not in STR_FIELDS and v <= 0: err(f, "must be greater than zero")
    check_present(SCHEMA[part_type])
    if issues: return issues
    if part_type in ("pipe_gasket", "washer") and p["od_in"] <= p["id_in"]:
        err("od_in", "outer diameter must be larger than inner diameter"); return issues
    if part_type == "bracket" and p["thickness_in"] >= min(p["length_a_in"], p["length_b_in"]):
        err("thickness_in", "thickness must be smaller than both leg lengths"); return issues
    if part_type == "rect_container":
        inner = min(p["length_a_in"], p["length_b_in"]) - 2 * p["thickness_in"] if RECT_DIMS_ARE_OUTER else min(p["length_a_in"], p["length_b_in"])
        if inner <= 0: err("thickness_in", "walls are too thick: nothing is left inside the container"); return issues
        if p["height_in"] <= p["thickness_in"]: err("height_in", "height must be greater than the floor/wall thickness"); return issues
    if part_type == "cyl_container" and p["height_in"] <= p["thickness_in"]:
        err("height_in", "height must be greater than the floor/wall thickness"); return issues
    if part_type == "pipe_gasket":
        check_present(DERIVED_GASKET)
        if issues: return issues
        n, bc, hd = p["hole_count"], p["bolt_circle_in"], p["bolt_hole_in"]
        if not 1 <= n <= 64: err("hole_count", "expected between 1 and 64 bolt holes"); return issues
        if bc / 2 - hd / 2 <= p["id_in"] / 2 or bc / 2 + hd / 2 >= p["od_in"] / 2:
            err("bolt_circle_in", "bolt holes would fall outside the gasket ring - adjust bolt circle / hole size"); return issues
        if n >= 2 and bc * math.sin(math.pi / n) <= hd:
            err("bolt_hole_in", "neighbouring bolt holes overlap - reduce the hole size or the number of holes"); return issues
        if n >= 2 and bc * math.sin(math.pi / n) < 1.1 * hd:
            warn("hole_count", "bolt holes nearly touch (very thin material between them) - consider a smaller hole size")
    for f, (lo, hi) in RANGES[part_type].items():
        if p[f] < 0.5 * lo or p[f] > 2 * hi: warn(f, f"unusual size (catalog range is {dec(lo)} to {dec(hi)} in)")
    return issues

def llm_verify(part_type, text_norm, params):
    """OPTIONAL off-the-shelf check: ask the UNTUNED base model which values look wrong. Small models are weak verifiers -
    that's why the deterministic rules above do the heavy lifting and this is off by default."""
    shown = {k: params[k] for k in SCHEMA[part_type] if params.get(k) is not None}
    prompt = (f"Description: {text_norm}\nExtracted values: {json.dumps(shown)}\n"
              "List the keys whose value does NOT match the description, as a JSON list ([] if all match).\nAnswer: ")
    out = generate([prompt], use_adapter=False, max_new_tokens=40)[0]
    m = re.search(r"\[[^\]]*\]", out)
    try:    return [k for k in json.loads(m.group(0)) if k in SCHEMA[part_type]] if m else []
    except Exception: return []

def rows_to_df(rows):
    return pd.DataFrame([[r["field"], fmt_value(r["value"]), STATUS_LABEL[r["status"]], r["note"]] for r in rows],
                        columns=["field", "value", "status", "note"])

**Add features (holes):** the schema, position conventions, centring defaults and geometric checks for holes on brackets and containers. A hole must lie fully inside its wall/leg, clear of the bend, rims, floor and its neighbours.

In [ ]:
# ------------------------------------------------------------------ ADD FEATURES: holes at user-specified locations
# Available for L-brackets and both containers (gaskets/washers have no add-on features).
# Everything is in inches. Positions follow the conventions in HOLE_SPEC (shown to the user in the GUI).
MAX_HOLES = 12
HOLE_EPS = 0.02            # in: the cutter starts/ends slightly outside the wall so cuts never leave paper-thin skins

HOLE_SPEC = {
    "bracket": {
        "faces": ["leg_a", "leg_b"],
        "keys": ["face", "from_end_in", "across_in", "diameter_in"],
        "labels": ["Leg", "Distance from the leg's free end (in)", "Height above the bottom face (in)", "Hole diameter (in)"],
        "hint": "leg_a runs left-to-right in the preview and leg_b runs away from you. Distance is measured from the leg's free end to the "
                "hole centre. Height is measured up from the bottom face (the face lying on the print bed). Leave a position blank to centre the hole.",
    },
    "rect_container": {
        "faces": ["front", "back", "left", "right"],
        "keys": ["face", "along_in", "up_in", "diameter_in"],
        "labels": ["Wall", "Distance from the wall's left end (in)", "Height above the bottom (in)", "Hole diameter (in)"],
        "hint": "Walls are named as seen in the preview: front faces you, back is behind, left/right are the sides (length_a runs left-to-right). "
                "Distance is measured from the wall's left end as you look at it from outside. Leave a position blank to centre the hole.",
    },
    "cyl_container": {
        "faces": ["side"],
        "keys": ["face", "angle_deg", "up_in", "diameter_in"],
        "labels": ["Surface", "Angle around the cup (deg)", "Height above the bottom (in)", "Hole diameter (in)"],
        "hint": "Angle is measured counter-clockwise looking down from above: 0 = right, 90 = back, 180 = left, 270 = front. "
                "Leave the height blank to centre the hole vertically.",
    },
}
HOLE_PART_TYPES = set(HOLE_SPEC)
_FACE_ALIAS = {"rear": "back", "a": "leg_a", "b": "leg_b", "leg_1": "leg_a", "leg_2": "leg_b"}
_DIRECTION_WORDS = re.compile(r"\b(front|back|rear|behind|left|right|opposite|north|south|east|west)\b", re.I)

def rect_outer(p):
    a, b, t = p["length_a_in"], p["length_b_in"], p["thickness_in"]
    return (a, b) if RECT_DIMS_ARE_OUTER else (a + 2 * t, b + 2 * t)

def leg_length(p, face): return p["length_a_in"] if face == "leg_a" else p["length_b_in"]
def wall_length(p, face):
    oa, ob = rect_outer(p); return oa if face in ("front", "back") else ob

def hole_defaults(pt, p, face):
    """positions used when the user leaves one blank = centred (angle has no sensible default)"""
    t, k = p["thickness_in"], HOLE_SPEC[pt]["keys"]
    if pt == "bracket":        return {k[1]: round((leg_length(p, face) - t) / 2, 4), k[2]: round(p["height_in"] / 2, 4)}
    if pt == "rect_container": return {k[1]: round(wall_length(p, face) / 2, 4), k[2]: round((p["height_in"] + t) / 2, 4)}
    return {k[2]: round((p["height_in"] + t) / 2, 4)}

def coerce_face(pt, v):
    if v is None: return None
    s = re.sub(r"[\s\-]+", "_", str(v).strip().lower()); s = _FACE_ALIAS.get(s, s)
    return s if s in HOLE_SPEC[pt]["faces"] else None

def _cell(value, status, note=""): return {"value": value, "status": status, "note": note}

def _grounded(pt, key, v, text_norm):
    nums = numbers_in(text_norm)
    if key == "angle_deg":
        return any(abs(((v - n + 180) % 360) - 180) <= 0.5 for n in nums) or bool(_DIRECTION_WORDS.search(text_norm))
    return any(abs(v - n) <= tol_for(v) for n in nums)

def make_hole_rows(pt, p, text_norm, raw_holes):
    """model output (list of dicts, or None) -> table rows. Blank positions become 'centred' defaults; values not in the text are flagged."""
    spec, rows = HOLE_SPEC[pt], []
    for raw in (raw_holes if isinstance(raw_holes, list) else [])[:MAX_HOLES]:
        raw = raw if isinstance(raw, dict) else {}
        k = spec["keys"]; face = coerce_face(pt, raw.get(k[0]))
        if face is None and len(spec["faces"]) == 1: face = spec["faces"][0]
        cells = {"face": _cell(face, "extracted") if face else _cell(None, "missing", "which wall / leg?")}
        defaults = hole_defaults(pt, p, face) if face else {}
        for ck, key in zip(("p1", "p2", "d"), k[1:]):
            v = coerce_value(key, raw.get(key))
            if v is None:
                if key in defaults: cells[ck] = _cell(defaults[key], "default", "centred (you didn't give a position)")
                else:               cells[ck] = _cell(None, "missing", "not found in your description")
            elif not _grounded(pt, key, v, text_norm):
                cells[ck] = _cell(v, "not_in_text", "this number is not in your description - please check it")
            else:
                cells[ck] = _cell(v, "extracted")
        rows.append({"cells": cells})
    return rows

def resolve_hole_rows(pt, p, ui_rows, prev_rows=None):
    """UI rows ({face,p1,p2,d} as typed text) -> (rows with statuses, holes dicts). Blank positions are centred; changed cells become 'edited'."""
    spec, rows, holes = HOLE_SPEC[pt], [], []
    k = spec["keys"]
    for i, ur in enumerate((ui_rows or [])[:MAX_HOLES]):
        prev = prev_rows[i]["cells"] if prev_rows and i < len(prev_rows) else None
        face = coerce_face(pt, ur.get("face"))
        if face is None and len(spec["faces"]) == 1: face = spec["faces"][0]
        face_changed = bool(prev) and prev["face"]["value"] != face
        cells = {"face": (_cell(face, prev["face"]["status"] if prev and not face_changed else "edited") if face else _cell(None, "missing", "choose a wall / leg"))}
        defaults = hole_defaults(pt, p, face) if face else {}
        h = {k[0]: face}
        for ck, key in zip(("p1", "p2", "d"), k[1:]):
            v = coerce_value(key, ur.get(ck)); pc = prev.get(ck) if prev else None
            untouched = pc is not None and pc["value"] is not None and v is not None and abs(v - pc["value"]) <= 1e-9
            if untouched and pc["status"] == "default" and face_changed: v = None                # an UNTOUCHED centred value belonged to the old face -> recompute
            if v is None:
                if key in defaults: cells[ck] = _cell(defaults[key], "default", "centred (blank = centred)")
                else:               cells[ck] = _cell(None, "missing", "required")
            elif pc and pc["value"] is not None and abs(v - pc["value"]) <= 1e-9:
                cells[ck] = _cell(v, pc["status"], pc["note"])                                    # untouched -> keep its original status
            else:
                cells[ck] = _cell(v, "edited", "entered/changed by you")
            h[key] = cells[ck]["value"]
        rows.append({"cells": cells}); holes.append(h)
    return rows, holes

def check_holes(pt, p, holes):
    """hard errors block the build; warnings are shown. Every hole must lie fully inside its wall/leg and clear of its neighbours."""
    issues, k = [], HOLE_SPEC[pt]["keys"]
    err  = lambda i, f, m: issues.append({"level": "error",   "hole": i, "field": f, "msg": m})
    warn = lambda i, f, m: issues.append({"level": "warning", "hole": i, "field": f, "msg": m})
    if len(holes) > MAX_HOLES: err(None, None, f"at most {MAX_HOLES} holes")
    for i, h in enumerate(holes):
        for key in k:
            if h.get(key) is None: err(i, key, "missing - fill it in the table")
    if issues: return issues
    t, hgt = p["thickness_in"], p["height_in"]
    pos = []                                                       # (face, u, v) in a flat 2-D layout of each face, for the overlap test
    for i, h in enumerate(holes):
        d = h[k[3]]; r = d / 2; u, v = h[k[1]], h[k[2]]
        if d <= 0: err(i, k[3], "diameter must be greater than zero"); pos.append(None); continue
        if pt == "bracket":
            free = leg_length(p, h["face"]) - t
            if u - r < 0:      err(i, k[1], "hole breaks through the leg's free end")
            if u + r > free:   err(i, k[1], f"hole runs into the bend (this leg is only {dec(free)} in long past the bend)")
            if v - r < 0 or v + r > hgt: err(i, k[2], f"hole breaks through the bracket's edge (bracket is {dec(hgt)} in wide)")
            edge = min(u - r, free - u - r, v - r, hgt - v - r)
            if edge >= 0 and edge < 0.05: warn(i, k[1], "less than 0.05 in of material between the hole and an edge")
            pos.append((h["face"], u, v))
        else:
            if pt == "rect_container":
                L = wall_length(p, h["face"])
                if u - r < 0 or u + r > L: err(i, k[1], f"hole runs off the wall (wall is {dec(L)} in long)")
                elif u - r < t or u + r > L - t: warn(i, k[1], "hole is within one wall-thickness of a corner")
                x = u
            else:
                if not 0 <= u <= 360: err(i, k[1], "angle must be between 0 and 360 degrees")
                x = u                                              # converted to arc length below
            if v - r < 0:      err(i, k[2], "hole breaks through the bottom")
            elif v + r > hgt:  err(i, k[2], f"hole breaks through the top rim (container is {dec(hgt)} in tall)")
            elif v - r < t:    warn(i, k[2], "hole dips into the floor - raise it to keep the cut inside the wall")
            pos.append((h["face"], x, v))
    for i in range(len(holes)):
        for j in range(i + 1, len(holes)):
            if pos[i] is None or pos[j] is None or pos[i][0] != pos[j][0]: continue
            if pt == "cyl_container":
                da = abs(pos[i][1] - pos[j][1]); da = min(da, 360 - da)
                dist = math.hypot(math.radians(da) * (p["id_in"] / 2 + t), pos[i][2] - pos[j][2])
            else:
                dist = math.hypot(pos[i][1] - pos[j][1], pos[i][2] - pos[j][2])
            if dist <= (holes[i][k[3]] + holes[j][k[3]]) / 2 + 0.03:
                err(j, k[1], f"hole {j + 1} is too close to (or overlaps) hole {i + 1} - leave at least 0.03 in of material between them")
    return issues

def holes_to_json(pt, holes):
    k = HOLE_SPEC[pt]["keys"]
    return json.dumps({"holes": [{key: h.get(key) for key in k} for h in holes]}, separators=(",", ":"))

def audit_holes(pt, holes, text_norm):
    """dataset check: every numeric hole value that is NOT blank must be readable in its own text"""
    k, bad = HOLE_SPEC[pt]["keys"], []
    for i, h in enumerate(holes):
        for key in k[1:]:
            v = h.get(key)
            if v is not None and not _grounded(pt, key, v, text_norm): bad.append((i, key, v))
    return bad
# ---- regression tests for the edit logic (these caught a real bug: a typed value must never be replaced by a default)
_bp = {"length_a_in": 4.0, "length_b_in": 2.0, "height_in": 1.0, "thickness_in": 0.125}
_prev = [{"cells": {"face": _cell("leg_a", "extracted"), "p1": _cell(1.9375, "default"), "p2": _cell(0.5, "default"), "d": _cell(0.25, "extracted")}}]
_r, _h = resolve_hole_rows("bracket", _bp, [{"face": "leg_b", "p1": "1.9375", "p2": "0.95", "d": "0.25"}], _prev)      # face changed, p1 untouched, p2 typed by the user
assert _h[0]["across_in"] == 0.95 and _r[0]["cells"]["p2"]["status"] == "edited", _h                               # typed value survives
assert _h[0]["from_end_in"] == round((2.0 - 0.125) / 2, 4) and _r[0]["cells"]["p1"]["status"] == "default", _h      # untouched centred value is recomputed for the new leg
_r, _h = resolve_hole_rows("bracket", _bp, [{"face": "leg_a", "p1": "", "p2": "", "d": "0.25"}], None)
assert _h[0]["from_end_in"] == 1.9375 and _h[0]["across_in"] == 0.5, _h                                           # blank = centred
_CARDINAL_WORDS = {"front": 270, "back": 90, "rear": 90, "left": 180, "right": 0}      # cup directions -> angle (matches HOLE_SPEC hint)
def semantic_hole_problems(pt, holes, text_norm):
    """beyond 'numbers are present': named walls / directions must agree with the labels (rect walls must be named; cup directions must match angles)"""
    t, probs, k = text_norm.lower(), [], HOLE_SPEC[pt]["keys"]
    if pt == "rect_container":
        for f in {h["face"] for h in holes}:
            if not re.search(rf"\b{'(?:back|rear)' if f == 'back' else f}\b", t): probs.append(f"wall '{f}' is not named in the text")
    if pt == "cyl_container":
        angs = {float(h[k[1]]) for h in holes if h.get(k[1]) is not None}
        named = {w for w in _CARDINAL_WORDS if re.search(rf"\b{w}\b", t)}
        for w in named:
            if _CARDINAL_WORDS[w] not in angs: probs.append(f"text says '{w}' but no hole is at {_CARDINAL_WORDS[w]} degrees")
        nums = numbers_in(text_norm)
        for a in angs & {0.0, 90.0, 180.0, 270.0}:
            if a not in nums and not any(_CARDINAL_WORDS[w] == a for w in named): probs.append(f"angle {a:g} is neither written as a number nor named")
    return probs
print("hole schema ready for:", sorted(HOLE_PART_TYPES))

**Generalised features:** holes, chamfers/radii on named edges, dividers inside boxes and rectangular cutouts share one schema. This cell defines the fields, the defaults, the size limits (the CAD kernel does not reliably reject impossible chamfers/radii, so they are enforced here), the validation rules and the table logic the app uses.

In [ ]:
# ------------------------------------------------------------------ ADD FEATURES (generalised): holes, chamfers/radii, dividers, cutouts
# One schema for every feature the "Add features" step can add. A feature is a dict {"type": ..., <fields>}; the editing model outputs
# {"features": [...]}. Everything is in inches. Blank (None) positions/sizes mean "centred / default" where a default makes sense.
MAX_FEATURES = 16
FEATURE_LABELS = {"hole": "Holes in the walls / legs", "bottom_hole": "Holes in the bottom", "edge": "Chamfers & radii", "divider": "Dividers (inner walls)", "cutout": "Cutouts / slots"}
FEATURE_TYPES = {"bracket": ["hole", "edge", "cutout"], "rect_container": ["hole", "bottom_hole", "edge", "divider", "cutout"], "cyl_container": ["hole", "bottom_hole", "edge"]}
MAX_HOLES_TOTAL = 40
EDGE_OPTIONS = {
    "bracket": ["bend", "outer_bend", "inner_bend", "free_ends"],
    "rect_container": ["corners", "inner_corners", "top_rim", "bottom", "inner_bottom"],
    "cyl_container": ["top_rim", "bottom", "inner_bottom"],
}
EDGE_HELP = {
    "bracket": "bend = the bend with an inside radius/chamfer of the size you give (outside is concentric, one wall thickness larger); outer_bend / inner_bend = only the outside / inside "
               "corner of the bend; free_ends = all edges around the ends of both legs.",
    "rect_container": "corners = the four vertical corners (outside, with a concentric inside); inner_corners = inside corners only; top_rim = the top edges of the walls; "
                      "bottom = the outside bottom edges; inner_bottom = the inside edges where the floor meets the walls.",
    "cyl_container": "top_rim = the top edges of the wall; bottom = the outside bottom edge; inner_bottom = the inside edge where the floor meets the wall.",
}
PATTERN_HELP = ("count + spacing make a row of identical holes: the position is the CENTRE of the row; blank spacing with count > 1 spreads the holes evenly across the whole wall/leg/floor "
                "(around the whole cup for a cup's side, where spacing is in degrees).")
BOTTOM_HELP = {"rect_container": "Holes through the floor. x = distance from the LEFT outside edge, y = distance from the FRONT outside edge (blank = centred). direction = which way a row of holes runs (blank = along the longer side). rows = number of rows, so a grid is count x rows. inset = the grid spans the floor this far in from the outside edges (e.g. '4 holes in a rectangular pattern 1 inch from the sides' = count 2, rows 2, inset 1).",
               "cyl_container": "Holes through the floor. x / y = offset from the centre (x to the right, y toward the back; blank = centre). direction = which way a row of holes runs. rows = number of rows (a grid is count x rows)."}
DIVIDER_HELP = ("crosswise = a wall running front-to-back that separates the left and right sides; lengthwise = a wall running left-to-right that separates front from back "
                "(blank = split the longer side, e.g. 'three sections' on a long box = two crosswise dividers). "
                "position = distance from the inside face of the left (crosswise) or front (lengthwise) wall to the divider's centre; leave blank to centre it (or split evenly when count > 1). "
                "thickness defaults to the wall thickness; height (above the floor) defaults to full height.")
CUTOUT_HELP = ("A rectangular cut through the wall/leg. Position = centre of the cutout, measured like holes; width runs along the wall/leg, height runs up. A cutout that reaches the top rim becomes an open notch.")
_CHAMFER_FILLET = ["chamfer", "fillet"]
_SQ2 = math.sqrt(2.0)

def fdefs(ftype, pt):
    """ordered field definitions for a feature type on a part type: key, label, kind (choice/num/int), options, nullable"""
    if ftype in ("hole", "cutout"):
        k, L, faces = HOLE_SPEC[pt]["keys"], HOLE_SPEC[pt]["labels"], HOLE_SPEC[pt]["faces"]
        base = [dict(key=k[0], label=L[0], kind="choice", options=faces),
                dict(key=k[1], label=L[1], kind="num", nullable=(pt != "cyl_container")),
                dict(key=k[2], label=L[2], kind="num", nullable=True)]
        if ftype == "hole":
            sp = dict(key="spacing_deg", label="Spacing (deg, blank = evenly around)", kind="num", nullable=True) if pt == "cyl_container" else \
                 dict(key="spacing_in", label="Spacing centre-to-centre (in, blank = evenly)", kind="num", nullable=True)
            return base + [dict(key=k[3], label=L[3], kind="num"), dict(key="count", label="How many (blank = 1)", kind="int", nullable=True), sp]
        return base + [dict(key="width_in", label="Width along the wall/leg (in)", kind="num"), dict(key="height_in", label="Height (in)", kind="num")]
    if ftype == "bottom_hole":
        if pt == "rect_container": lx, ly = "x: from the left outside edge (in, blank = centred)", "y: from the front outside edge (in, blank = centred)"
        else: lx, ly = "x: right of centre (in, blank = 0)", "y: toward the back (in, blank = 0)"
        out = [dict(key="x_in", label=lx, kind="num", nullable=True), dict(key="y_in", label=ly, kind="num", nullable=True), dict(key="diameter_in", label="Hole diameter (in)", kind="num"),
               dict(key="count", label="Holes per row (blank = 1)", kind="int", nullable=True), dict(key="rows", label="Rows (blank = 1)", kind="int", nullable=True),
               dict(key="spacing_in", label="Spacing centre-to-centre (in, blank = evenly)", kind="num", nullable=True)]
        if pt == "rect_container": out.append(dict(key="inset_in", label="Inset from the sides (in): pattern spans the floor this far from the outside edges", kind="num", nullable=True))
        return out + [dict(key="direction", label="Row direction (blank = longer side)", kind="choice", options=["left_right", "front_back"], nullable=True)]
    if ftype == "edge":
        return [dict(key="kind", label="Type (blank = fillet)", kind="choice", options=_CHAMFER_FILLET, nullable=True),
                dict(key="edge", label="Where (blank = the main outside edges)", kind="choice", options=EDGE_OPTIONS[pt], nullable=True),
                dict(key="size_in", label="Size / radius (in, blank = a safe default)", kind="num", nullable=True)]
    if ftype == "divider":
        return [dict(key="orientation", label="Orientation (blank = split the longer side)", kind="choice", options=["crosswise", "lengthwise"], nullable=True),
                dict(key="position_in", label="Position from the inside of the left/front wall (in, blank = centred/even)", kind="num", nullable=True),
                dict(key="count", label="Number of dividers (blank = 1)", kind="int", nullable=True),
                dict(key="thickness_in", label="Thickness (in, blank = wall thickness)", kind="num", nullable=True),
                dict(key="height_in", label="Height above the floor (in, blank = full)", kind="num", nullable=True)]
    raise KeyError(ftype)

def coerce_field(fd, v, pt=None):
    if fd["kind"] == "choice":
        if v is None: return None
        s = re.sub(r"[\s\-]+", "_", str(v).strip().lower())
        if fd["key"] == "face": s = _FACE_ALIAS.get(s, s)
        return s if s in fd["options"] else None
    x = coerce_value(fd["key"], v)
    if fd["kind"] == "int": return int(round(x)) if x is not None and abs(x - round(x)) < 1e-9 and x >= 1 else None
    return x

def _longer_side_is_length(p):
    oa, ob = rect_outer(p); return oa >= ob

def feature_default(ftype, pt, p, f, key):
    """value used when a nullable field is left blank (None = 'no specific value': centred / evenly split)"""
    if key in ("count", "rows"): return 1
    if key in ("spacing_in", "spacing_deg", "inset_in"): return None
    if ftype == "edge":
        if key == "kind": return "fillet"
        if key == "edge": return DEFAULT_EDGE[pt]
        if key == "size_in":                                     # 'fillet the edges' with no size: half the safe limit, at most 1/8 in
            lim = edge_limit(pt, p, f.get("edge") or DEFAULT_EDGE[pt], f.get("kind") or "fillet")
            return max(0.005, round(min(0.125, 0.5 * lim), 3))
    if ftype in ("hole", "cutout"): return hole_defaults(pt, p, f.get("face")).get(key) if f.get("face") else None
    if ftype == "bottom_hole":
        if key == "direction": return "left_right" if (pt != "rect_container" or _longer_side_is_length(p)) else "front_back"
        if pt == "cyl_container": return 0.0
        oa, ob = rect_outer(p); return round((oa if key == "x_in" else ob) / 2, 4)
    if ftype == "divider":
        if key == "orientation": return "crosswise" if _longer_side_is_length(p) else "lengthwise"
        if key == "thickness_in": return round(p["thickness_in"], 4)
        if key == "height_in": return round(p["height_in"] - p["thickness_in"], 4)
        if key == "count": return 1
    return None

_DEFAULT_NOTE = {"kind": "rounded (fillet)", "edge": "the main outside edges", "size_in": "a safe default size", "rows": "one row", "inset_in": "not used",
                 "position_in": "centred / evenly spaced", "thickness_in": "wall thickness", "height_in": "full height", "count": "just one", "orientation": "splits the longer side",
                 "direction": "along the longer side", "spacing_in": "spread evenly", "spacing_deg": "evenly around", "x_in": "centred", "y_in": "centred"}
def _default_note(key): return _DEFAULT_NOTE.get(key, "centred (blank = centred)")

def _grounded_f(pt, key, v, text_norm):
    if key == "count": return True                       # 'two dividers' is not a measurement
    if key in ("x_in", "y_in"): v = abs(v)               # cup-floor offsets: the side is a word ('left of centre'), the number is the size
    return _grounded(pt, key, v, text_norm)

def make_feature_rows(pt, p, text_norm, raw_features):
    """editing-model output (list of dicts, each with a 'type') -> table rows; unreadable values become 'missing', numbers not in the text are flagged"""
    rows = []
    for raw in (raw_features if isinstance(raw_features, list) else [])[:MAX_FEATURES]:
        if not isinstance(raw, dict): continue
        ftype = str(raw.get("type", "hole")).strip().lower()
        if ftype not in FEATURE_TYPES[pt]: continue
        cells, f = {}, {}
        for fd in fdefs(ftype, pt):
            key, v = fd["key"], coerce_field(fd, raw.get(fd["key"]), pt)
            if fd["kind"] == "choice" and v is None and len(fd["options"]) == 1: v = fd["options"][0]
            if v is None:
                d = feature_default(ftype, pt, p, f, key) if fd.get("nullable") else None
                if fd.get("nullable"): cells[key] = _cell(d, "default", _default_note(key)); f[key] = d
                else: cells[key] = _cell(None, "missing", "not found in your description"); f[key] = None
            elif fd["kind"] != "choice" and not _grounded_f(pt, key, v, text_norm):
                cells[key] = _cell(v, "not_in_text", "this number is not in your description - please check it"); f[key] = v
            else:
                cells[key] = _cell(v, "extracted"); f[key] = v
        rows.append({"type": ftype, "cells": cells})
    return rows

def resolve_feature_rows(pt, p, ui_rows, prev_rows=None):
    """UI rows ({type, v:{key: text}}) -> (rows with statuses, feature dicts). Blank nullable fields take their default; changed cells become 'edited'."""
    rows, feats, seen = [], [], {}
    prev_by_type = {}
    for r in prev_rows or []: prev_by_type.setdefault(r["type"], []).append(r)
    for ur in (ui_rows or [])[:MAX_FEATURES]:
        ftype = str(ur.get("type", "")).strip().lower()
        if ftype not in FEATURE_TYPES[pt]: continue
        k = seen.get(ftype, 0); seen[ftype] = k + 1               # rows are matched to the previous ones by (type, position within that type)
        vals = ur.get("v") or {}; prev = prev_by_type[ftype][k]["cells"] if k < len(prev_by_type.get(ftype, [])) else None
        cells, f = {}, {"type": ftype}
        for fd in fdefs(ftype, pt):
            key, pc = fd["key"], (prev.get(fd["key"]) if prev else None)
            v = coerce_field(fd, vals.get(key), pt)
            if fd["kind"] == "choice" and v is None and len(fd["options"]) == 1: v = fd["options"][0]
            untouched = pc is not None and pc["value"] is not None and v is not None and (v == pc["value"] if fd["kind"] == "choice" else abs(v - pc["value"]) <= 1e-9)
            if untouched and pc["status"] == "default": v = None        # an untouched default is recomputed for the current part, face and settings
            if v is None:
                if fd.get("nullable"):
                    d = feature_default(ftype, pt, p, f, key); cells[key] = _cell(d, "default", _default_note(key)); f[key] = d
                else:
                    cells[key] = _cell(None, "missing", "required"); f[key] = None
            elif untouched: cells[key] = _cell(v, pc["status"], pc["note"]); f[key] = v
            else: cells[key] = _cell(v, "edited", "entered/changed by you"); f[key] = v
        rows.append({"type": ftype, "cells": cells}); feats.append(f)
    return rows, feats

def edge_limit(pt, p, edge, kind):
    """largest chamfer/radius the geometry can take (the CAD kernel does NOT reliably reject impossible sizes, so we enforce these)"""
    t, fil = p["thickness_in"], (kind == "fillet")
    if pt == "bracket":
        a, b, w = p["length_a_in"], p["length_b_in"], p["height_in"]
        if edge == "outer_bend": return min((3.0 if fil else 1.7) * t, 0.9 * min(a, b))
        if edge == "inner_bend": return 0.9 * min(a - t, b - t)
        if edge == "bend":       return min(0.9 * min(a - t, b - t), 0.9 * min(a, b) - (t if fil else (2 - _SQ2) * t))
        if edge == "free_ends":  return 0.45 * min(t, w)
    h = p["height_in"]
    if pt == "rect_container":
        oa, ob = rect_outer(p); ia, ib = oa - 2 * t, ob - 2 * t
        return {"corners": 0.95 * min(oa, ob) / 2, "inner_corners": 0.95 * min(ia, ib) / 2, "top_rim": 0.45 * t,
                "bottom": min((3.0 if fil else 1.7) * t, 0.9 * min(oa, ob) / 2), "inner_bottom": min(0.9 * min(ia, ib) / 2, 0.9 * (h - t))}[edge]
    R = p["id_in"] / 2 + t
    return {"top_rim": 0.45 * t, "bottom": min((3.0 if fil else 1.7) * t, 0.9 * R), "inner_bottom": min(0.9 * p["id_in"] / 2, 0.9 * (h - t))}[edge]

DEFAULT_EDGE = {"bracket": "bend", "rect_container": "corners", "cyl_container": "top_rim"}      # what "the edges" means when no edge is named
_EDGE_CONFLICT = [{"bend", "outer_bend", "inner_bend"}, {"corners", "inner_corners"}]

def divider_positions(p, f):
    """centre coordinate (in the part's own XY frame) of every divider of a feature, plus (thickness, span axis length)"""
    oa, ob = rect_outer(p); t = p["thickness_in"]; ia, ib = oa - 2 * t, ob - 2 * t
    ori = f.get("orientation") or ("crosswise" if oa >= ob else "lengthwise")
    td = f.get("thickness_in") or t; n = f.get("count") or 1; span = ia if ori == "crosswise" else ib
    if f.get("position_in") is not None and n == 1: return [-span / 2 + f["position_in"]], td, span
    cw = (span - n * td) / (n + 1)
    return [-span / 2 + i * cw + (i - 0.5) * td for i in range(1, n + 1)], td, span

def apply_defaults(pt, p, feats):
    """fill every blank optional field with its default (centred positions, wall thickness, one hole, ...) - what the app actually builds"""
    out = []
    for f in feats:
        g = dict(f)
        for fd in fdefs(f["type"], pt):
            if fd.get("nullable") and g.get(fd["key"]) is None:
                d = feature_default(f["type"], pt, p, g, fd["key"])
                if d is not None: g[fd["key"]] = d
        out.append(g)
    return out

def hole_positions(pt, p, f):
    """centre of every hole a 'hole' / 'bottom_hole' feature makes (rows expanded). Positions must already have their defaults filled."""
    n = f.get("count") or 1
    if f["type"] == "bottom_hole":
        if pt == "rect_container":
            oa, ob = rect_outer(p); t = p["thickness_in"]
            x = f["x_in"] if f.get("x_in") is not None else oa / 2; y = f["y_in"] if f.get("y_in") is not None else ob / 2
            d = f.get("direction") or ("left_right" if oa >= ob else "front_back"); lo, hi = (t, oa - t) if d == "left_right" else (t, ob - t)
        else:
            R = p["id_in"] / 2; x = f.get("x_in") or 0.0; y = f.get("y_in") or 0.0; d = f.get("direction") or "left_right"; lo, hi = -R, R
        m, s, inset = f.get("rows") or 1, f.get("spacing_in"), f.get("inset_in")
        if n == 1 and m == 1: return [(x, y)]
        if pt == "rect_container": oa, ob = rect_outer(p); spans = {"left_right": (t, oa - t), "front_back": (t, ob - t)}; full = {"left_right": oa, "front_back": ob}
        else: spans = {"left_right": (lo, hi), "front_back": (lo, hi)}; full = None
        cross = "front_back" if d == "left_right" else "left_right"
        def axis(k, centre, ax):
            if k == 1: return [centre]
            if inset is not None and full is not None: a, b = inset, full[ax] - inset; return [a + i * (b - a) / (k - 1) for i in range(k)]
            if s is not None: return [centre + (i - (k - 1) / 2) * s for i in range(k)]
            a, b = spans[ax]; return [a + (i + 1) * (b - a) / (k + 1) for i in range(k)]
        along = axis(n, x if d == "left_right" else y, d); across = axis(m, y if d == "left_right" else x, cross)
        return [(u, v) if d == "left_right" else (v, u) for u in along for v in across]
    k = HOLE_SPEC[pt]["keys"]; u, v = f[k[1]], f[k[2]]
    if n == 1: return [(u, v)]
    if pt == "cyl_container":
        s = f.get("spacing_deg"); return [(((u + i * 360 / n) if s is None else (u + (i - (n - 1) / 2) * s)) % 360, v) for i in range(n)]
    s = f.get("spacing_in"); L = (leg_length(p, f["face"]) - p["thickness_in"]) if pt == "bracket" else wall_length(p, f["face"])
    return [((i + 1) * L / (n + 1), v) for i in range(n)] if s is None else [(u + (i - (n - 1) / 2) * s, v) for i in range(n)]

def expand_holes(pt, p, f):
    """a wall/leg/cup-side hole feature -> list of single holes (HOLE_SPEC keys) the CAD and the hole rules understand"""
    k = HOLE_SPEC[pt]["keys"]
    return [{k[0]: f[k[0]], k[1]: round(u, 4), k[2]: v, k[3]: f[k[3]]} for u, v in hole_positions(pt, p, f)]

def check_bottom_holes(pt, p, feats, idx, divs, inner_bottom, inner_corner):
    out, t = [], p["thickness_in"]
    err = lambda i, f, m: out.append({"level": "error", "feature": idx[i], "type": "bottom_hole", "field": f, "msg": m})
    pts = []
    for i, f in enumerate(feats):
        r = f["diameter_in"] / 2
        if f["diameter_in"] <= 0: err(i, "diameter_in", "diameter must be greater than zero"); continue
        if (f.get("count") or 1) > 1 and f.get("spacing_in") is not None and f["spacing_in"] <= f["diameter_in"] + 0.03: err(i, "spacing_in", "spacing must be larger than the hole diameter (plus a little material)"); continue
        m = 0.05 + inner_bottom
        for x, y in hole_positions(pt, p, f):
            if pt == "rect_container":
                oa, ob = rect_outer(p)
                if x - r < t + m or x + r > oa - t - m or y - r < t + m or y + r > ob - t - m:
                    err(i, "x_in", f"a hole at ({dec(round(x, 3))}, {dec(round(y, 3))}) is not fully inside the floor (keep {dec(round(m, 3))} in clear of the walls)"); break
                if inner_corner > 0 and min(x - t, oa - t - x) < inner_corner + r and min(y - t, ob - t - y) < inner_corner + r:
                    err(i, "x_in", "a hole is too close to a rounded inside corner"); break
                for ori, ctr, td in divs:                                  # not under a divider
                    c0 = (x - oa / 2) if ori == "crosswise" else (y - ob / 2)
                    if abs(c0 - ctr) < td / 2 + r + 0.03: err(i, "x_in", "a hole sits under a divider - move it into a compartment"); break
            else:
                if math.hypot(x, y) + r > p["id_in"] / 2 - m: err(i, "x_in", f"a hole is not fully inside the floor (keep {dec(round(m, 3))} in clear of the wall)"); break
            pts.append((i, x, y, r))
    for a in range(len(pts)):
        for b in range(a + 1, len(pts)):
            if math.hypot(pts[a][1] - pts[b][1], pts[a][2] - pts[b][2]) <= pts[a][3] + pts[b][3] + 0.03:
                err(pts[b][0], "x_in", "two holes in the bottom are too close (or overlap) - leave at least 0.03 in between"); break
    return out

def check_cutouts(pt, p, cuts, idx):
    """cutouts: same placement rules as holes but with a width x height rectangle; reaching the top rim is allowed (open notch)"""
    out, k, t, hgt = [], HOLE_SPEC[pt]["keys"], p["thickness_in"], p["height_in"]
    err = lambda i, f, m: out.append({"level": "error", "feature": idx[i], "type": "cutout", "field": f, "msg": m})
    warn = lambda i, f, m: out.append({"level": "warning", "feature": idx[i], "type": "cutout", "field": f, "msg": m})
    for i, c in enumerate(cuts):
        rw, rh, u, v = c["width_in"] / 2, c["height_in"] / 2, c[k[1]], c[k[2]]
        if c["width_in"] <= 0 or c["height_in"] <= 0: err(i, "width_in", "width and height must be greater than zero"); continue
        if pt == "bracket":
            free = leg_length(p, c["face"]) - t
            if u - rw < 0 or u + rw > free: err(i, k[1], f"cutout runs off the leg (usable length {dec(free)} in)")
            if v - rh < 0 or v + rh > hgt: err(i, k[2], f"cutout runs off the bracket's width ({dec(hgt)} in)")
        else:
            L = wall_length(p, c["face"])
            if u - rw < 0 or u + rw > L: err(i, k[1], f"cutout runs off the wall (wall is {dec(L)} in long)")
            elif u - rw < t or u + rw > L - t: warn(i, k[1], "cutout is within one wall-thickness of a corner")
            if v - rh < 0: err(i, k[2], "cutout breaks through the bottom")
            elif v + rh > hgt + 1e-6: err(i, k[2], f"cutout goes above the top rim (container is {dec(hgt)} in tall)")
            elif v - rh < t: warn(i, k[2], "cutout dips into the floor")
    return out

def check_features(pt, p, feats):
    """hard errors block the build; warnings are shown. Returns issues: {level, feature (index), type, field, msg}"""
    issues, t = [], p["thickness_in"]
    err = lambda i, ty, f, m: issues.append({"level": "error", "feature": i, "type": ty, "field": f, "msg": m})
    warn = lambda i, ty, f, m: issues.append({"level": "warning", "feature": i, "type": ty, "field": f, "msg": m})
    if len(feats) > MAX_FEATURES: err(None, None, None, f"at most {MAX_FEATURES} features")
    for i, f in enumerate(feats):
        for fd in fdefs(f["type"], pt):
            if f.get(fd["key"]) is None and not fd.get("nullable"): err(i, f["type"], fd["key"], "missing - fill it in the table")
        if f["type"] == "hole" and f.get(HOLE_SPEC[pt]["keys"][1]) is None and pt == "cyl_container": pass
    if issues: return issues
    singles, owner = [], []                                         # rows of holes are expanded; every hole is checked on its own
    for i, f in enumerate(feats):
        if f["type"] == "hole":
            if (f.get("count") or 1) > 1 and f.get("spacing_in") is not None and f["spacing_in"] <= f[HOLE_SPEC[pt]["keys"][3]] + 0.03:
                err(i, "hole", "spacing_in", "spacing must be larger than the hole diameter (plus a little material)"); continue
            for h in expand_holes(pt, p, f): singles.append(h); owner.append(i)
    n_holes = len(singles) + sum(len(hole_positions(pt, p, f)) for f in feats if f["type"] == "bottom_hole")
    if n_holes > MAX_HOLES_TOTAL: err(None, "hole", "count", f"too many holes ({n_holes}) - at most {MAX_HOLES_TOTAL}"); return issues
    for iss in check_holes(pt, p, singles):
        issues.append({"level": iss["level"], "feature": owner[iss["hole"]] if iss["hole"] is not None else None, "type": "hole", "field": iss["field"], "msg": iss["msg"]})
    cidx = [i for i, f in enumerate(feats) if f["type"] == "cutout"]
    issues += check_cutouts(pt, p, [feats[i] for i in cidx], cidx)
    seen_edges = []
    for i, f in enumerate(feats):
        if f["type"] == "edge":
            lim = edge_limit(pt, p, f["edge"], f["kind"])
            if f["size_in"] <= 0: err(i, "edge", "size_in", "size must be greater than zero")
            elif f["size_in"] > lim: err(i, "edge", "size_in", f"too large for '{f['edge']}' on this part: the {f['kind']} can be at most {dec(round(lim, 4))} in (limited by the wall/leg thickness and size)")
            for e0 in seen_edges:
                if f["edge"] == e0 or any(f["edge"] in g and e0 in g for g in _EDGE_CONFLICT): err(i, "edge", "edge", f"'{f['edge']}' conflicts with another chamfer/radius on '{e0}'")
            seen_edges.append(f["edge"])
        elif f["type"] == "divider":
            pos, td, span = divider_positions(p, f); n = f.get("count") or 1
            hmax, hh = p["height_in"] - t, (f.get("height_in") or p["height_in"] - t)
            if n > 8: err(i, "divider", "count", "at most 8 dividers"); continue
            if td <= 0 or td > span / 3: err(i, "divider", "thickness_in", "divider thickness must be positive and less than a third of the box"); continue
            if hh <= 0 or hh > hmax + 1e-6: err(i, "divider", "height_in", f"divider height must be between 0 and the inside height ({dec(round(hmax, 4))} in)"); continue
            if f.get("position_in") is not None and n == 1:
                if f["position_in"] - td / 2 < 0.02 or f["position_in"] + td / 2 > span - 0.02: err(i, "divider", "position_in", f"divider must sit inside the box (inside length is {dec(round(span, 4))} in)")
            elif f.get("position_in") is not None: warn(i, "divider", "position_in", "a position is ignored when there is more than one divider - they are spaced evenly")
            cw = (span - n * td) / (n + 1)
            if cw < 0.2: err(i, "divider", "count", f"compartments would be only {dec(round(cw, 3))} in wide")
    # ---- edges that are rounded/chamfered change the geometry near them: keep holes/cutouts out of those zones, and keep the edge options compatible
    z = {"bend": 0.0, "free_end": 0.0, "corner": 0.0, "bottom_z": 0.0}; edge_f = [(i, f) for i, f in enumerate(feats) if f["type"] == "edge"]
    for i, f in edge_f:
        s, e, kd = f["size_in"], f["edge"], f["kind"]
        if pt == "bracket":
            if e == "bend": z["bend"] = max(z["bend"], s + t)
            elif e == "outer_bend": z["bend"] = max(z["bend"], s)
            elif e == "inner_bend": z["bend"] = max(z["bend"], t + s)
            elif e == "free_ends": z["free_end"] = max(z["free_end"], s)
        if pt == "rect_container":
            if e == "corners": z["corner"] = max(z["corner"], s + (0.5 * t if kd == "chamfer" else 0.0))
            elif e == "inner_corners": z["corner"] = max(z["corner"], t + s)
        if e == "bottom": z["bottom_z"] = max(z["bottom_z"], s)
        elif e == "inner_bottom": z["bottom_z"] = max(z["bottom_z"], t + s)
    inner_c = [(f["size_in"] - (t if f["kind"] == "fillet" else (2 - _SQ2) * t)) if f["edge"] == "corners" else f["size_in"] for _, f in edge_f if f["edge"] in ("corners", "inner_corners")]
    inner_c = [x for x in inner_c if x > 1e-6]
    for i, f in edge_f:
        if f["edge"] == "inner_bottom" and inner_c and f["size_in"] > 0.9 * min(inner_c):
            err(i, "edge", "size_in", f"the inside-bottom {f['kind']} must be smaller than the inside corner radius/chamfer ({dec(round(min(inner_c), 4))} in) - make it at most {dec(round(0.9 * min(inner_c), 4))} in")
        if f["edge"] == "bottom" and pt == "rect_container":
            outer_c = [g["size_in"] for _, g in edge_f if g["edge"] == "corners"]
            if outer_c and f["size_in"] > 0.9 * min(outer_c): err(i, "edge", "size_in", f"the bottom {f['kind']} must be smaller than the corner radius/chamfer ({dec(round(min(outer_c), 4))} in)")
    dv = [(i, f, *divider_positions(p, f)) for i, f in enumerate(feats) if f["type"] == "divider"]
    for x in range(len(dv)):
        for y in range(x + 1, len(dv)):
            if dv[x][1]["orientation"] != dv[y][1]["orientation"]: continue
            for c1 in dv[x][2]:
                for c2 in dv[y][2]:
                    if abs(c1 - c2) < (dv[x][3] + dv[y][3]) / 2 + 0.2:
                        err(dv[y][0], "divider", "position_in", f"divider {dv[y][0] + 1} is too close to divider {dv[x][0] + 1} (keep at least 0.2 in between them)")
    holes_cuts = []; k = HOLE_SPEC[pt]["keys"]
    for h, i in zip(singles, owner): holes_cuts.append((i, "hole", h["face"], h[k[1]], h[k[2]], h[k[3]] / 2, h[k[3]] / 2))
    for i, f in enumerate(feats):
        if f["type"] == "cutout": holes_cuts.append((i, "cutout", f["face"], f[k[1]], f[k[2]], f["width_in"] / 2, f["height_in"] / 2))
    bidx = [i for i, f in enumerate(feats) if f["type"] == "bottom_hole"]
    if bidx:
        divs_xy = [((f.get("orientation") or ("crosswise" if rect_outer(p)[0] >= rect_outer(p)[1] else "lengthwise")), c, td) for f in feats if f["type"] == "divider" for c, td in [(c, divider_positions(p, f)[1]) for c in divider_positions(p, f)[0]]] if pt == "rect_container" else []
        ib_size = max([f["size_in"] for _, f in edge_f if f["edge"] == "inner_bottom"] or [0.0])
        ic = max([((f["size_in"] - (t if f["kind"] == "fillet" else (2 - _SQ2) * t)) if f["edge"] == "corners" else f["size_in"]) for _, f in edge_f if f["edge"] in ("corners", "inner_corners")] or [0.0])
        issues += check_bottom_holes(pt, p, [feats[i] for i in bidx], bidx, divs_xy, ib_size, max(ic, 0.0))
    k = HOLE_SPEC[pt]["keys"]
    for i, ty, face, u, v, rw, rh in holes_cuts:
        if pt == "bracket":
            dist = leg_length(p, face) - u                         # distance from the outer corner of the bend to the feature centre
            if z["bend"] and dist - rw < z["bend"] + 0.05: err(i, ty, k[1], f"too close to the rounded/chamfered bend - keep it at least {dec(round(z['bend'] + 0.05 + rw, 3))} in from the bend (centre-to-bend)")
            if z["free_end"] and u - rw < z["free_end"] + 0.03: err(i, ty, k[1], "too close to the rounded/chamfered end of the leg")
        elif pt == "rect_container":
            L = wall_length(p, face)
            if z["corner"] and (u - rw < z["corner"] + 0.03 or L - u - rw < z["corner"] + 0.03): err(i, ty, k[1], f"too close to the rounded/chamfered corner - keep it {dec(round(z['corner'] + 0.03, 3))} in clear of the corner")
        if pt != "bracket" and z["bottom_z"] and v - rh < z["bottom_z"] + 0.03: err(i, ty, k[2], f"too close to the rounded/chamfered bottom - keep it at least {dec(round(z['bottom_z'] + 0.03 + rh, 3))} in above the bottom")
    # holes / cutouts that overlap on the same wall (a cutout is involved; hole-vs-hole is checked by check_holes)
    for x in range(len(holes_cuts)):
        for y in range(x + 1, len(holes_cuts)):
            a, b = holes_cuts[x], holes_cuts[y]
            if a[2] != b[2] or "cutout" not in (a[1], b[1]) or pt == "cyl_container": continue
            if abs(a[3] - b[3]) < a[5] + b[5] + 0.03 and abs(a[4] - b[4]) < a[6] + b[6] + 0.03: err(b[0], b[1], None, f"feature {b[0] + 1} is too close to (or overlaps) feature {a[0] + 1} - keep at least 0.03 in between them")
    return issues

def features_to_json(pt, feats):
    """canonical compact JSON for a list of features (fixed key order) - the format the editing model is asked to write"""
    out = []
    for f in feats:
        d = {"type": f["type"]}
        for fd in fdefs(f["type"], pt): d[fd["key"]] = f.get(fd["key"])
        out.append(d)
    return json.dumps({"features": out}, separators=(",", ":"))

_CENTRE_CUE = re.compile(r"cent|middle|dead|halfway|midway|mid-", re.I)
_CROSSWISE_WORDS = re.compile(r"front[ -]to[ -]back|left from (?:the )?right|left and right|left side from", re.I)
_LENGTHWISE_WORDS = re.compile(r"left[ -]to[ -]right|front from (?:the )?back|front and back", re.I)
def audit_features(pt, feats, text_norm):
    """dataset check: every explicit number must be readable in the sentence; words must agree with the labels"""
    bad, t = [], text_norm.lower()
    for i, f in enumerate(feats):
        for fd in fdefs(f["type"], pt):
            v = f.get(fd["key"])
            if fd["kind"] == "num" and v is not None and not _grounded_f(pt, fd["key"], v, text_norm): bad.append(f"feature {i + 1}: {fd['key']}={v} not in text")
        if f["type"] == "edge" and not re.search(r"chamfer|bevel" if f["kind"] == "chamfer" else r"fillet|radius|round|soften", t): bad.append(f"feature {i + 1}: no {f['kind']} word in text")
        if f["type"] == "divider" and not re.search(r"divid|partition|wall|compartment|section|split|separat", t): bad.append(f"feature {i + 1}: no divider word in text")
        if f["type"] == "bottom_hole" and not re.search(r"bottom|base|floor|underside|drain", t): bad.append(f"feature {i + 1}: no bottom/base/floor word in text")
        if f["type"] == "cutout" and not re.search(r"slot|notch|cutout|cut out|cut-out|opening|window|rectang|cut ", t): bad.append(f"feature {i + 1}: no cutout word in text")
    n_centred = sum(1 for f in feats if f["type"] in ("hole", "cutout") and (f.get("count") or 1) == 1 for fd in fdefs(f["type"], pt)[1:3] if fd.get("nullable") and f.get(fd["key"]) is None) + \
                sum(1 for f in feats if f["type"] == "bottom_hole" and (f.get("count") or 1) == 1 and pt == "rect_container" and (f.get("x_in") is None or f.get("y_in") is None))
    if n_centred and not _CENTRE_CUE.search(t): bad.append("a position is blank (centred) but the text never says centred/middle/halfway")
    divs = {f.get("orientation") for f in feats if f["type"] == "divider"}
    t_div = re.sub(r"cent\w*(?: it)?(?: from)? left[ -]to[ -]right|middle left[ -]to[ -]right|cent\w* side[ -]to[ -]side", " ", t)      # 'centered left to right' is a POSITION, not a divider direction
    about_dividers = bool(re.search(r"divid|partition|compartment|section", t))
    if not about_dividers: t_div = ""
    if _CROSSWISE_WORDS.search(t_div) and "crosswise" not in divs: bad.append("text describes a front-to-back divider but none is labelled crosswise")
    if _LENGTHWISE_WORDS.search(t_div) and "lengthwise" not in divs: bad.append("text describes a left-to-right divider but none is labelled lengthwise")
    hs = [{kk: vv for kk, vv in f.items() if kk != "type"} for f in feats if f["type"] == "hole"]
    if hs: bad += semantic_hole_problems(pt, hs, text_norm)
    if pt == "rect_container" and not divs:                      # every named wall must exist among the hole/cutout labels (skipped when dividers also talk about walls)
        faces = {f["face"] for f in feats if f["type"] in ("hole", "cutout")}
        t2 = re.sub(r"left[ -]to[ -]right|front[ -]to[ -]back|toward the back|front and the back|from the (?:left|right|front|back|rear)(?: wall| end| edge| corner| side)?|(?:left|right) end|(?:left|right) as you", " ", t)
        for w in ("front", "back", "left", "right"):
            if re.search(rf"\b{'(?:back|rear)' if w == 'back' else w}\b", t2) and faces and w not in faces: bad.append(f"text names the {w} wall but no feature is on it")
    return bad
print("feature framework ready:", {pt: FEATURE_TYPES[pt] for pt in FEATURE_TYPES})

**CAD:** parameters (inches) → CadQuery solid → scaled ×25.4 → STL in mm. The self-test checks that volumes match the maths, which proves the bolt holes are really cut.

In [ ]:
# ------------------------------------------------------------------ STAGE 4: JSON -> CAD -> STL  (deterministic, no ML)
import cadquery as cq, trimesh
IN_TO_MM = 25.4        # STL has no units and slicers assume mm, so geometry built in inches is scaled once, at export

def build_gasket(p):
    od, idd, t, n = p["od_in"], p["id_in"], p["thickness_in"], int(p["hole_count"])
    bc, hd = p["bolt_circle_in"], p["bolt_hole_in"]
    ring = cq.Workplane("XY").circle(od / 2).circle(idd / 2).extrude(t)
    pts = [((bc / 2) * math.cos(2 * math.pi * i / n), (bc / 2) * math.sin(2 * math.pi * i / n)) for i in range(n)]
    holes = cq.Workplane("XY").workplane(offset=-t).pushPoints(pts).circle(hd / 2).extrude(3 * t)   # oversized so it cuts clean through
    return ring.cut(holes)

def build_washer(p):
    return cq.Workplane("XY").circle(p["od_in"] / 2).circle(p["id_in"] / 2).extrude(p["thickness_in"])

def build_bracket(p):
    a, b, w, t = p["length_a_in"], p["length_b_in"], p["height_in"], p["thickness_in"]
    return cq.Workplane("XY").polyline([(0, 0), (a, 0), (a, t), (t, t), (t, b), (0, b)]).close().extrude(w)   # L profile, extruded by the width

class FeatureBuildError(Exception):
    """a requested feature could not be built; the message is shown to the user"""

def build_rect_container(p, corner_outer=None, corner_inner=None):
    """open-top box. Walls and floor are both `thickness` thick. length_a along X, length_b along Y, height along Z.
    corner_outer / corner_inner = (kind, size) to chamfer or round the four vertical corners of the outside / the inside."""
    h, t = p["height_in"], p["thickness_in"]
    oa, ob = rect_outer(p)
    def rounded(wp, spec):
        if not spec: return wp
        kind, size = spec
        return wp.edges("|Z").chamfer(size) if kind == "chamfer" else wp.edges("|Z").fillet(size)
    outer = rounded(cq.Workplane("XY").box(oa, ob, h, centered=(True, True, False)), corner_outer)
    cavity = rounded(cq.Workplane("XY").workplane(offset=t).box(oa - 2 * t, ob - 2 * t, h, centered=(True, True, False)), corner_inner)   # runs past the top -> open
    return outer.cut(cavity)

def build_cyl_container(p):
    """open-top cup. id_in is the INSIDE diameter, so outside diameter = id + 2*thickness. Floor = wall thickness."""
    idd, h, t = p["id_in"], p["height_in"], p["thickness_in"]
    outer = cq.Workplane("XY").circle(idd / 2 + t).extrude(h)
    cavity = cq.Workplane("XY").workplane(offset=t).circle(idd / 2).extrude(h)
    return outer.cut(cavity)

BUILDERS = {"pipe_gasket": build_gasket, "washer": build_washer, "bracket": build_bracket,
            "rect_container": build_rect_container, "cyl_container": build_cyl_container}

def hole_cutters(pt, p, holes):
    """one cylinder per hole, positioned from the user's numbers (see HOLE_SPEC for the conventions). Each cutter spans only the wall it
    pierces (plus a small overshoot), so it can never cut the opposite wall."""
    k, e, t, out = HOLE_SPEC[pt]["keys"], HOLE_EPS, p["thickness_in"], []
    V = cq.Vector
    for h in holes:
        r, u, v, face = h[k[3]] / 2, h[k[1]], h[k[2]], h["face"]
        if pt == "bracket":                                   # L profile lies flat in XY; leg_a along +X, leg_b along +Y; width is Z
            if face == "leg_a": pnt, d = V(p["length_a_in"] - u, -e, v), V(0, 1, 0)
            else:               pnt, d = V(-e, p["length_b_in"] - u, v), V(1, 0, 0)
            out.append(cq.Solid.makeCylinder(r, t + 2 * e, pnt, d))
        elif pt == "rect_container":                          # centred in XY; front = -Y wall; positions measured from the wall's left end seen from outside
            oa, ob = rect_outer(p)
            pnt, d = {"front": (V(-oa / 2 + u, -ob / 2 - e, v), V(0, 1, 0)),
                      "back":  (V(oa / 2 - u, ob / 2 + e, v),   V(0, -1, 0)),
                      "left":  (V(-oa / 2 - e, ob / 2 - u, v),  V(1, 0, 0)),
                      "right": (V(oa / 2 + e, -ob / 2 + u, v),  V(-1, 0, 0))}[face]
            out.append(cq.Solid.makeCylinder(r, t + 2 * e, pnt, d))
        else:                                                 # cup: radial hole at an angle (0 = +X, counter-clockwise from above)
            R, th = p["id_in"] / 2 + t, math.radians(u)
            out.append(cq.Solid.makeCylinder(r, t + 2 * e, V((R + e) * math.cos(th), (R + e) * math.sin(th), v), V(-math.cos(th), -math.sin(th), 0)))
    return out

def _near(a, b, tol=1e-5): return abs(a - b) < tol

def _edge_pred(pt, p, edge):
    """geometric test that picks out the edges of one named group"""
    t, h = p["thickness_in"], p["height_in"]
    S, E = (lambda e: e.startPoint()), (lambda e: e.endPoint())
    if pt == "bracket":
        a, b = p["length_a_in"], p["length_b_in"]
        vert = lambda x, y: (lambda e: _near(S(e).x, x) and _near(S(e).y, y) and _near(E(e).x, x) and _near(E(e).y, y))     # edge parallel to Z at profile corner (x, y)
        if edge == "outer_bend": return vert(0, 0)
        if edge == "inner_bend": return vert(t, t)
        return lambda e: (_near(S(e).x, a) and _near(E(e).x, a)) or (_near(S(e).y, b) and _near(E(e).y, b))                  # free_ends
    if pt == "cyl_container":
        circ = lambda z: (lambda e: e.geomType() == "CIRCLE" and _near(S(e).z, z))
        return {"top_rim": circ(h), "bottom": circ(0), "inner_bottom": circ(t)}[edge]
    flat = lambda z: (lambda e: _near(S(e).z, z) and _near(E(e).z, z))
    return {"top_rim": flat(h), "bottom": flat(0), "inner_bottom": flat(t)}[edge]

def _edge_op(shape, kind, pred, size, what):
    edges = [e for e in shape.Edges() if pred(e)]
    if not edges: raise FeatureBuildError(f"could not find the '{what}' edges to {kind}")
    try: out = shape.fillet(size, edges) if kind == "fillet" else shape.chamfer(size, None, edges)
    except Exception as ex: raise FeatureBuildError(f"the CAD kernel could not {kind} the '{what}' edges at {dec(size)} in - try a smaller size") from ex
    if not out.isValid(): raise FeatureBuildError(f"a {dec(size)} in {kind} on '{what}' gives an invalid shape - try a smaller size")
    return out

def _apply_edge_features(shape, pt, p, edge_feats):
    t = p["thickness_in"]
    for f in edge_feats:
        kind, edge, size = f["kind"], f["edge"], f["size_in"]
        if edge in ("corners", "inner_corners"): continue                      # built into the box itself (build_part)
        if pt == "bracket" and edge == "bend":                                 # standard convention: size = INSIDE radius/chamfer, outside is concentric
            shape = _edge_op(shape, kind, _edge_pred(pt, p, "inner_bend"), size, "inner bend")
            shape = _edge_op(shape, kind, _edge_pred(pt, p, "outer_bend"), size + (t if kind == "fillet" else (2 - math.sqrt(2)) * t), "outer bend")
        else:
            shape = _edge_op(shape, kind, _edge_pred(pt, p, edge), size, edge)
    return shape

def divider_solids(p, divs):
    out, oa, ob = [], *rect_outer(p); t, h = p["thickness_in"], p["height_in"]; ia, ib = oa - 2 * t, ob - 2 * t
    for f in divs:
        centres, td, span = divider_positions(p, f); hh = f.get("height_in") or (h - t)
        z0, z1 = t / 2, t + hh                                                 # starts inside the floor so it fuses with it
        for ctr in centres:
            if f["orientation"] == "crosswise": out.append(cq.Workplane("XY").box(td, ib + t, z1 - z0, centered=(True, True, False)).translate((ctr, 0, z0)).val())
            else:                               out.append(cq.Workplane("XY").box(ia + t, td, z1 - z0, centered=(True, True, False)).translate((0, ctr, z0)).val())
    return out

def cutout_cutters(pt, p, cuts):
    k, e, t, h = HOLE_SPEC[pt]["keys"], HOLE_EPS, p["thickness_in"], p["height_in"]; out = []
    for c in cuts:
        u, v, w, hh, face = c[k[1]], c[k[2]], c["width_in"], c["height_in"], c["face"]
        z0, z1 = v - hh / 2, v + hh / 2
        if pt != "bracket" and z1 >= h - 1e-6: z1 = h + e                      # reaches the rim -> open notch
        box = lambda sx, sy, cx, cy: cq.Workplane("XY").box(sx, sy, z1 - z0, centered=(True, True, True)).translate((cx, cy, (z0 + z1) / 2)).val()
        if pt == "bracket":
            out.append(box(w, t + 2 * e, p["length_a_in"] - u, t / 2) if face == "leg_a" else box(t + 2 * e, w, t / 2, p["length_b_in"] - u))
        else:
            oa, ob = rect_outer(p)
            out.append({"front": lambda: box(w, t + 2 * e, -oa / 2 + u, -ob / 2 + t / 2), "back": lambda: box(w, t + 2 * e, oa / 2 - u, ob / 2 - t / 2),
                        "left": lambda: box(t + 2 * e, w, -oa / 2 + t / 2, ob / 2 - u), "right": lambda: box(t + 2 * e, w, oa / 2 - t / 2, -ob / 2 + u)}[face]())
    return out

def bottom_hole_cutters(pt, p, feats):
    """vertical cylinders through the floor only (from just below the bottom to just above the floor's top face)"""
    out, t, e = [], p["thickness_in"], HOLE_EPS
    for f in feats:
        r = f["diameter_in"] / 2
        for x, y in hole_positions(pt, p, f):
            if pt == "rect_container": oa, ob = rect_outer(p); X, Y = -oa / 2 + x, -ob / 2 + y
            else: X, Y = x, y
            out.append(cq.Solid.makeCylinder(r, t + 2 * e, cq.Vector(X, Y, -e), cq.Vector(0, 0, 1)))
    return out

def build_part(params, features=None):
    """base part -> (box corners) -> edge chamfers/radii -> dividers -> holes + cutouts. Always returns a valid solid or raises FeatureBuildError."""
    pt = params["part_type"]; feats = [dict(f, type=f.get("type", "hole")) for f in (features or [])]
    edge_f, t = [f for f in feats if f["type"] == "edge"], params.get("thickness_in")
    try:
        if pt == "rect_container":
            co = ci = None
            for f in edge_f:
                if f["edge"] == "corners":
                    co = (f["kind"], f["size_in"]); s_in = f["size_in"] - (t if f["kind"] == "fillet" else (2 - math.sqrt(2)) * t)      # concentric inside
                    ci = (f["kind"], s_in) if s_in > 1e-6 else None
                elif f["edge"] == "inner_corners": ci = (f["kind"], f["size_in"])
            wp = build_rect_container(params, co, ci)
        else: wp = BUILDERS[pt](params)
        if edge_f:
            wp = cq.Workplane("XY").newObject([_apply_edge_features(wp.val(), pt, params, edge_f)])
        divs = [f for f in feats if f["type"] == "divider"]
        for s in (divider_solids(params, divs) if divs else []): wp = wp.union(s)                # one at a time: overlapping dividers must each be fused into the part
        filled = apply_defaults(pt, params, feats) if pt in FEATURE_TYPES else feats
        cut = (hole_cutters(pt, params, [h for f in filled if f["type"] == "hole" for h in expand_holes(pt, params, f)])
               + cutout_cutters(pt, params, [f for f in filled if f["type"] == "cutout"])
               + bottom_hole_cutters(pt, params, [f for f in filled if f["type"] == "bottom_hole"])) if pt in HOLE_SPEC else []
        for s in cut: wp = wp.cut(s)                                                            # one at a time, for the same reason
    except FeatureBuildError: raise
    except Exception as ex:
        raise FeatureBuildError(f"the CAD kernel could not build these features ({type(ex).__name__}) - try different sizes or positions") from ex
    if feats and (not wp.val().isValid() or len(wp.val().Solids()) != 1): raise FeatureBuildError("these features give an invalid or disconnected shape - try different sizes or positions")
    return wp

def build_stl(params: dict, out_path: str, features=None) -> dict:
    wp = build_part(params, features)
    solid = wp.val().scale(IN_TO_MM)
    cq.exporters.export(solid, str(out_path), tolerance=0.01, angularTolerance=0.05)
    m = trimesh.load(str(out_path))
    info = {"watertight": bool(m.is_watertight), "volume_mm3": round(float(m.volume), 2), "size_mm": [round(float(x), 2) for x in m.extents]}
    if features and not info["watertight"]: raise FeatureBuildError("the resulting mesh is not watertight - try different sizes or positions")
    return info

# ---- tests: geometry must match the maths (this is how we know the holes were REALLY cut)
def _cuin(x): return x * IN_TO_MM ** 3
_g = {"part_type": "pipe_gasket", "od_in": 3.5, "id_in": 0.8438, "thickness_in": 0.0625, "hole_count": 4, "bolt_circle_in": 2.1719, "bolt_hole_in": 0.5325}
_i = build_stl(_g, WORKDIR / "test_gasket.stl")
_expect = _cuin(math.pi / 4 * (3.5**2 - 0.8438**2) * 0.0625 - 4 * math.pi / 4 * 0.5325**2 * 0.0625)
assert _i["watertight"] and abs(_i["volume_mm3"] - _expect) / _expect < 0.01, (_i, _expect)
_w = {"part_type": "washer", "od_in": 0.5512, "id_in": 0.2913, "thickness_in": 0.063}
_i = build_stl(_w, WORKDIR / "test_washer.stl")
assert _i["watertight"] and abs(_i["size_mm"][0] - 0.5512 * 25.4) < 0.05, _i
_b = {"part_type": "bracket", "length_a_in": 2.0, "length_b_in": 1.5, "height_in": 0.5, "thickness_in": 0.125}
_i = build_stl(_b, WORKDIR / "test_bracket.stl")
_expect = _cuin((2.0 * 0.125 + (1.5 - 0.125) * 0.125) * 0.5)
assert _i["watertight"] and abs(_i["volume_mm3"] - _expect) / _expect < 0.01 and sorted(_i["size_mm"], reverse=True)[0] == round(2.0 * 25.4, 2), (_i, _expect)
_r = {"part_type": "rect_container", "length_a_in": 4.5, "length_b_in": 2.88, "height_in": 1.67, "thickness_in": 0.092}
_i = build_stl(_r, WORKDIR / "test_rect.stl")
_a, _b, _h, _t = 4.5, 2.88, 1.67, 0.092
_expect = _cuin(_a * _b * _h - (_a - 2 * _t) * (_b - 2 * _t) * (_h - _t)) if RECT_DIMS_ARE_OUTER else None
if _expect: assert _i["watertight"] and abs(_i["volume_mm3"] - _expect) / _expect < 0.01 and abs(_i["size_mm"][2] - _h * 25.4) < 0.05, (_i, _expect)
_cy = {"part_type": "cyl_container", "id_in": 2.75, "height_in": 4.0, "thickness_in": 0.0963}
_i = build_stl(_cy, WORKDIR / "test_cyl.stl")
_expect = _cuin(math.pi * ((2.75 / 2 + 0.0963) ** 2 * 4.0 - (2.75 / 2) ** 2 * (4.0 - 0.0963)))
assert _i["watertight"] and abs(_i["volume_mm3"] - _expect) / _expect < 0.01 and abs(_i["size_mm"][0] - (2.75 + 2 * 0.0963) * 25.4) < 0.05, (_i, _expect)
def _check_holes_cut(pt, p, holes, centre_pts, expect_each):
    """for every hole: the point at its centre (mid-wall) must be EMPTY, a point 2 radii away along the wall must be SOLID, and the
    removed volume must match the maths. This is what proves the hole is where we said, on the face we said."""
    base = _cuin_vol(BUILDERS[pt](p))
    wp = BUILDERS[pt](p).cut(cq.Compound.makeCompound(hole_cutters(pt, p, holes)))
    for (cx, cy, cz), (sx, sy, sz) in centre_pts:
        assert not wp.val().isInside(cq.Vector(cx, cy, cz)), ("hole centre is still solid", pt, (cx, cy, cz))
        assert wp.val().isInside(cq.Vector(sx, sy, sz)), ("material next to the hole is missing", pt, (sx, sy, sz))
    removed = base - _cuin_vol(wp)
    assert abs(removed - expect_each * len(holes)) / (expect_each * len(holes)) < 0.03, (pt, removed, expect_each * len(holes))
def _cuin_vol(w): return w.val().Volume()      # still in cubic inches here (scaled only at export)

_t, _r = 0.125, 0.125
_bp = {"part_type": "bracket", "length_a_in": 4.0, "length_b_in": 2.0, "height_in": 1.0, "thickness_in": _t}
_ha = {"face": "leg_a", "from_end_in": 1.0, "across_in": 0.4, "diameter_in": 0.25}
_hb = {"face": "leg_b", "from_end_in": 0.75, "across_in": 0.6, "diameter_in": 0.25}
_check_holes_cut("bracket", _bp, [_ha, _hb], [((3.0, _t / 2, 0.4), (3.0, _t / 2, 0.4 + 2 * _r)), ((_t / 2, 1.25, 0.6), (_t / 2, 1.25, 0.6 + 2 * _r))], math.pi * _r**2 * _t)
_rp = {"part_type": "rect_container", "length_a_in": 4.5, "length_b_in": 2.88, "height_in": 1.67, "thickness_in": 0.092}
_hs = [{"face": f, "along_in": 1.0, "up_in": 0.9, "diameter_in": 0.25} for f in ("front", "back", "left", "right")]
_oa, _ob, _wt = 4.5, 2.88, 0.092
_pts = [((-_oa / 2 + 1.0, -_ob / 2 + _wt / 2, 0.9), (-_oa / 2 + 1.0 + 0.3, -_ob / 2 + _wt / 2, 0.9)),       # front: 1.0 from the left end seen from outside (-X end)
        ((_oa / 2 - 1.0, _ob / 2 - _wt / 2, 0.9), (_oa / 2 - 1.0 - 0.3, _ob / 2 - _wt / 2, 0.9)),            # back: left end seen from outside is the +X end
        ((-_oa / 2 + _wt / 2, _ob / 2 - 1.0, 0.9), (-_oa / 2 + _wt / 2, _ob / 2 - 1.0 - 0.3, 0.9)),          # left: left end seen from outside is the +Y end
        ((_oa / 2 - _wt / 2, -_ob / 2 + 1.0, 0.9), (_oa / 2 - _wt / 2, -_ob / 2 + 1.0 + 0.3, 0.9))]          # right: left end seen from outside is the -Y end
_check_holes_cut("rect_container", _rp, _hs, _pts, math.pi * 0.125**2 * _wt)
_cp = {"part_type": "cyl_container", "id_in": 2.75, "height_in": 4.0, "thickness_in": 0.0963}
_Rm = 2.75 / 2 + 0.0963 / 2                                                                                    # mid-wall radius
_hc = [{"face": "side", "angle_deg": a, "up_in": 2.0, "diameter_in": 0.5} for a in (0, 90, 180, 270, 45)]
_pts = [((_Rm * math.cos(math.radians(a)), _Rm * math.sin(math.radians(a)), 2.0), (_Rm * math.cos(math.radians(a)), _Rm * math.sin(math.radians(a)), 2.0 + 0.5)) for a in (0, 90, 180, 270, 45)]
_check_holes_cut("cyl_container", _cp, _hc, _pts, math.pi * 0.25**2 * 0.0963)
_i0 = build_stl(_bp, WORKDIR / "t_nohole.stl"); _i1 = build_stl(_bp, WORKDIR / "t_hole.stl", [_ha, _hb])
assert _i1["watertight"] and _i0["volume_mm3"] - _i1["volume_mm3"] > 0 and abs((_i0["volume_mm3"] - _i1["volume_mm3"]) - 2 * math.pi * 0.125**2 * 0.125 * 25.4**3) / (2 * math.pi * 0.125**2 * 0.125 * 25.4**3) < 0.03, (_i0, _i1)
# ---- new features: exact volumes (inches^3) and real point-in-solid probes
def _vol(p, feats=None): return build_part(p, feats).val().Volume()
def _within(x, y, rel): return abs(x - y) <= rel * abs(y)
_bp = {"part_type": "bracket", "length_a_in": 4.0, "length_b_in": 3.0, "height_in": 1.0, "thickness_in": 0.125}; _w, _tb = 1.0, 0.125; _V0 = _vol(_bp)
_k = (1 - math.pi / 4)
_E = lambda kind, edge, size: {"type": "edge", "kind": kind, "edge": edge, "size_in": size}
assert _within(_V0 - _vol(_bp, [_E("fillet", "outer_bend", 0.3)]), 0.3**2 * _k * _w, 0.005), "outer bend fillet volume"
assert _within(_vol(_bp, [_E("chamfer", "inner_bend", 0.2)]) - _V0, 0.2**2 / 2 * _w, 0.005), "inner bend chamfer volume"
assert _within(_vol(_bp, [_E("fillet", "bend", 0.5)]) - _V0, _w * _k * (0.5**2 - (0.5 + _tb)**2), 0.01), "concentric bend volume"
assert _vol(_bp, [_E("fillet", "free_ends", 0.05)]) < _V0 and build_part(_bp, [_E("chamfer", "free_ends", 0.04)]).val().isValid()
_wp = build_part(_bp, [_E("chamfer", "outer_bend", 0.2)]).val()
assert not _wp.isInside(cq.Vector(0.02, 0.02, 0.5)) and _wp.isInside(cq.Vector(0.3, 0.05, 0.5)), "chamfer is in the wrong place"
assert build_part(_bp, [_E("chamfer", "inner_bend", 0.2)]).val().isInside(cq.Vector(0.02, 0.02, 0.5)), "an INSIDE-corner chamfer must leave the outside corner intact"
_rp = {"part_type": "rect_container", "length_a_in": 4.5, "length_b_in": 2.88, "height_in": 1.67, "thickness_in": 0.092}
_oa, _ob, _h, _t = 4.5, 2.88, 1.67, 0.092; _ia, _ib = _oa - 2 * _t, _ob - 2 * _t; _R0 = _vol(_rp)
_R = 0.4; assert _within(_vol(_rp, [_E("fillet", "corners", _R)]), (_oa * _ob - (4 - math.pi) * _R**2) * _h - (_ia * _ib - (4 - math.pi) * (_R - _t)**2) * (_h - _t), 0.002), "rounded box volume"
_c = 0.3; _ci = _c - (2 - math.sqrt(2)) * _t
assert _within(_vol(_rp, [_E("chamfer", "corners", _c)]), (_oa * _ob - 2 * _c**2) * _h - (_ia * _ib - 2 * _ci**2) * (_h - _t), 0.002), "chamfered box volume"
assert _within(_vol(_rp, [_E("fillet", "inner_corners", 0.5)]) - _R0, (4 - math.pi) * 0.5**2 * (_h - _t), 0.01), "inner corner rounding volume"
assert _within(_R0 - _vol(_rp, [_E("chamfer", "top_rim", 0.03)]), 0.03**2 / 2 * (2 * (_oa + _ob) + 2 * (_ia + _ib)), 0.02), "rim chamfer volume"
assert _within(_R0 - _vol(_rp, [_E("fillet", "bottom", 0.15)]), 0.15**2 * _k * 2 * (_oa + _ob), 0.04), "bottom fillet volume"
assert _within(_vol(_rp, [_E("fillet", "inner_bottom", 0.3)]) - _R0, 0.3**2 * _k * 2 * (_ia + _ib), 0.04), "inner bottom fillet volume"
_cp = {"part_type": "cyl_container", "id_in": 2.75, "height_in": 4.0, "thickness_in": 0.0963}; _C0 = _vol(_cp); _Ro, _Ri = 2.75 / 2 + 0.0963, 2.75 / 2
assert _within(_C0 - _vol(_cp, [_E("chamfer", "top_rim", 0.04)]), 0.04**2 / 2 * 2 * math.pi * ((_Ro - 0.04 / 3) + (_Ri + 0.04 / 3)), 0.02), "cup rim chamfer volume"
assert _vol(_cp, [_E("fillet", "bottom", 0.2)]) < _C0 and _vol(_cp, [_E("fillet", "inner_bottom", 0.4)]) > _C0, "cup bottom fillets go the wrong way"
_D = lambda o, **kw: {"type": "divider", "orientation": o, "position_in": None, "count": None, "thickness_in": None, "height_in": None, **kw}
assert _within(_vol(_rp, [_D("crosswise", count=2)]) - _R0, 2 * _t * _ib * (_h - _t), 0.002), "two dividers volume"
assert _within(_vol(_rp, [_D("lengthwise", thickness_in=0.2, height_in=0.8)]) - _R0, 0.2 * _ia * 0.8, 0.002), "partial-height divider volume"
_pos, _td, _sp = divider_positions(_rp, dict(_D("crosswise", count=2), thickness_in=None))
_wp = build_part(_rp, [_D("crosswise", position_in=1.5)]).val()
assert _wp.isInside(cq.Vector(-_ia / 2 + 1.5, 0, 0.9)) and not _wp.isInside(cq.Vector(-_ia / 2 + 1.5 + 0.3, 0, 0.9)), "divider is not where it was asked"
_cut = lambda face, u, v, w, hh: {"type": "cutout", "face": face, "along_in": u, "up_in": v, "width_in": w, "height_in": hh}
assert _within(_R0 - _vol(_rp, [_cut("front", 2.0, 0.9, 1.0, 0.5)]), 1.0 * 0.5 * _t, 0.01), "cutout volume"
assert _within(_R0 - _vol(_rp, [_cut("back", 2.0, _h - 0.25, 1.0, 0.5)]), 1.0 * 0.5 * _t, 0.01), "open notch volume"      # reaches the rim
for _face, _pt in {"front": (-_oa / 2 + 2.0, -_ob / 2 + _t / 2), "back": (_oa / 2 - 2.0, _ob / 2 - _t / 2), "left": (-_oa / 2 + _t / 2, _ob / 2 - 1.0), "right": (_oa / 2 - _t / 2, -_ob / 2 + 1.0)}.items():
    _wp = build_part(_rp, [_cut(_face, 2.0 if _face in ("front", "back") else 1.0, 0.9, 0.6, 0.4)]).val()
    assert not _wp.isInside(cq.Vector(_pt[0], _pt[1], 0.9)), ("cutout centre still solid", _face)
_cb = {"type": "cutout", "face": "leg_a", "from_end_in": 1.0, "across_in": 0.5, "width_in": 0.6, "height_in": 0.4}
assert _within(_V0 - _vol(_bp, [_cb]), 0.6 * 0.4 * _tb, 0.01) and not build_part(_bp, [_cb]).val().isInside(cq.Vector(4.0 - 1.0, _tb / 2, 0.5)), "bracket cutout"
try: build_part(_bp, [_E("fillet", "outer_bend", 0.5)]); raise AssertionError("an impossible fillet was not rejected")
except FeatureBuildError: pass
_wp = build_part(_rp, [_D("crosswise"), _D("lengthwise", height_in=0.8)]).val()
assert len(_wp.Solids()) == 1 and _wp.isValid(), "crossing dividers of different heights must fuse into ONE solid"
assert _within(_wp.Volume() - _R0, _t * _ib * (_h - _t) + _t * _ia * 0.8 - _t * _t * 0.8, 0.003), "crossing dividers volume (overlap counted once)"
# ---- holes in the bottom and rows of holes
_bh = {"type": "bottom_hole", "x_in": None, "y_in": None, "diameter_in": 0.125, "count": 4, "spacing_in": 1.0, "direction": None}
_wpb = build_part(_rp, [_bh]).val()
assert _within(_R0 - _wpb.Volume(), 4 * math.pi * 0.0625**2 * _t, 0.01), "four floor holes volume"
assert not check_features("rect_container", _rp, [dict(_bh, spacing_in=2.0)]) == [], "a 6 in row on a 4.5 in box must be refused"
for _x in (-1.5, -0.5, 0.5, 1.5):                                   # centred row along the longer side, 1 in apart
    assert not _wpb.isInside(cq.Vector(_x, 0, _t / 2)) and _wpb.isInside(cq.Vector(_x + 0.3, 0, _t / 2)), ("floor hole not where expected", _x)
_wpc = build_part(_cp, [{"type": "bottom_hole", "x_in": None, "y_in": None, "diameter_in": 0.5, "count": None, "spacing_in": None, "direction": None}]).val()
assert not _wpc.isInside(cq.Vector(0, 0, 0.0963 / 2)) and _wpc.isInside(cq.Vector(0.4, 0, 0.0963 / 2)), "cup centre drain hole"
_row = {"type": "hole", "face": "front", "along_in": None, "up_in": 0.8, "diameter_in": 0.25, "count": 3, "spacing_in": 1.0}
_wpr = build_part(_rp, [_row]).val()
assert _within(_R0 - _wpr.Volume(), 3 * math.pi * 0.125**2 * _t, 0.01), "row of three wall holes volume"
for _u in (1.25, 2.25, 3.25): assert not _wpr.isInside(cq.Vector(-_oa / 2 + _u, -_ob / 2 + _t / 2, 0.8)), ("row hole missing", _u)
_ev = {"type": "hole", "face": "side", "angle_deg": 0, "up_in": 2.0, "diameter_in": 0.25, "count": 4, "spacing_deg": None}
_wpe = build_part(_cp, [_ev]).val(); _Rm2 = 2.75 / 2 + 0.0963 / 2
for _a in (0, 90, 180, 270): assert not _wpe.isInside(cq.Vector(_Rm2 * math.cos(math.radians(_a)), _Rm2 * math.sin(math.radians(_a)), 2.0)), ("cup hole missing", _a)
_grid = {"type": "bottom_hole", "x_in": None, "y_in": None, "diameter_in": 0.2165, "count": 2, "rows": 2, "spacing_in": None, "inset_in": 1.0, "direction": None}
_wpg = build_part(_rp, [_grid]).val()
for _x, _y in ((1.0, 1.0), (3.5, 1.0), (1.0, 1.88), (3.5, 1.88)):                    # 1 in from the outside edges of a 4.5 x 2.88 box
    assert not _wpg.isInside(cq.Vector(-_oa / 2 + _x, -_ob / 2 + _y, _t / 2)), ("grid hole missing", _x, _y)
assert _within(_R0 - _wpg.Volume(), 4 * math.pi * (0.2165 / 2)**2 * _t, 0.01), "2 x 2 grid volume"
_dflt = apply_defaults("rect_container", _rp, [{"type": "edge", "kind": None, "edge": None, "size_in": None}])[0]
assert _dflt["kind"] == "fillet" and _dflt["edge"] == "corners" and 0 < _dflt["size_in"] <= 0.125 and build_part(_rp, [_dflt]).val().isValid(), _dflt
print("feature self-tests passed (exact volumes for every edge option, dividers and cutouts; positions probed in the solid; impossible sizes rejected)")
print("hole self-tests passed (every hole is empty at its centre, solid beside it, correct volume removed; front/back/left/right and angles verified)")
print("CAD self-tests passed (watertight, volumes match the maths, holes/cavities really cut, mm scaling correct)")

## 5b. Data augmentation (synthetic samples, stored separately)
Adds `AUGMENT_PER_TYPE` x 5 = **500 synthetic training samples**, generated by code from your own real training texts (nobody writes new sentences):
* **value swap** - a number that clearly belongs to one label field is replaced by another real value of that field, and the label is updated to match (not for washers, whose sizes are tied to the screw-size table);
* **surface noise** - unit wording (`inch` / `inches` / `in`), one typo, lower-casing, dropped final period.

Safety rules: sources are **train-split rows only**; validation and test stay 100% real; every synthetic row must pass the same audit and CAD rules as real data; no text may duplicate a real one. They are saved to their own file so you can report them as synthetic data (your real sample count stays 501).

In [ ]:
# ------------------------------------------------------------------ DATA AUGMENTATION (synthetic - kept separate from your real data)
# Every synthetic sample is made BY CODE from one of your real TRAINING rows. No new sentences are written by anyone.
#   1. value swap : numbers that are clearly one label field (one unambiguous match in the text) are replaced by another real
#                   value of that same field, and the label is updated to match -> a genuinely new (text, label) pair.
#                   (Not done for washers: their sizes are tied to the screw-size table, so washers get step 2 only.)
#   2. surface noise : unit wording (inch / inches / in), one typo in a long word, lower-casing, dropped final period.
# Safety rules: only train-split rows are used as sources, value pools come from train rows only, validation/test stay 100% real,
# every sample must pass the same label-in-text audit and CAD rules as real data, and no text may duplicate an existing one.
SUBSTITUTE_TYPES = {"pipe_gasket", "bracket", "rect_container", "cyl_container"}
NUM_TOK = re.compile(r"(?<![\w.])\d+(?:\.\d+)?")

for _d in (train_df, val_df, test_df):
    _d["source"] = "real"
train_real_df = train_df.copy()

def _unambiguous_tokens(row):
    """{token_index: [fields]} for label fields that match exactly one number in the text and aren't shared with an unrelated field"""
    text, pt = row["text_norm"], row["part_type"]
    toks = [(m.start(), m.end(), float(m.group())) for m in NUM_TOK.finditer(text)]
    f2t = {}
    for f in NUM_FIELDS[pt]:
        v = row["params"][f]
        if f in INT_FIELDS: hits = [i for i, (s, e, x) in enumerate(toks) if x == v and text[s:e].isdigit()]
        else:               hits = [i for i, (s, e, x) in enumerate(toks) if abs(x - v) <= tol_for(v)]
        if len(hits) == 1: f2t[f] = hits[0]
    t2f = {}
    for f, i in f2t.items(): t2f.setdefault(i, []).append(f)
    return toks, {i: fs for i, fs in t2f.items() if len(fs) == 1 or set(fs) <= {"length_a_in", "length_b_in"}}   # equal legs may share a number

def _noise(text, rng):
    unit = rng.choice([" inches", " inch", " in"])
    t = re.sub(r"(?<=\d)\s*(?:inches|inch|in)\b(?!\.\d)", unit, text)
    if rng.random() < 0.35:
        words = [m for m in re.finditer(r"[A-Za-z]{5,}", t)]
        if words:
            m = rng.choice(words); i = rng.randrange(1, len(m.group()) - 2); w = list(m.group()); w[i], w[i + 1] = w[i + 1], w[i]
            t = t[:m.start()] + "".join(w) + t[m.end():]
    if rng.random() < 0.20: t = t.lower()
    if rng.random() < 0.50: t = t.rstrip(". ")
    return t

def build_augmented(src_df, per_type, seed):
    rng = random.Random(seed)
    src_ok = src_df[src_df.audit_bad.map(len) == 0]
    seen = set(data.text_norm)                                          # never duplicate any real text (train, val or test)
    pools = {pt: {f: sorted({r["params"][f] for _, r in src_ok[src_ok.part_type == pt].iterrows()}) for f in NUM_FIELDS[pt]} for pt in PART_TYPES}
    out, stats = [], {}
    for pt in PART_TYPES:
        sources = [r for _, r in src_ok[src_ok.part_type == pt].iterrows()]; rng.shuffle(sources)   # round-robin below -> every real row used about equally
        made, tries = 0, 0
        while made < per_type and tries < per_type * 60 and sources:
            tries += 1
            src = sources[tries % len(sources)]
            text, params = src["text_norm"], dict(src["params"])
            if pt in SUBSTITUTE_TYPES:
                toks, groups = _unambiguous_tokens(src)
                if not groups: continue
                for i in sorted(groups, reverse=True):
                    f0 = groups[i][0]; old = params[f0]
                    new = rng.choice([v for v in pools[pt][f0] if (v != old if f0 in INT_FIELDS else abs(v - old) > tol_for(old))] or [old])
                    for f in groups[i]: params[f] = new
                    s, e, _ = toks[i]; text = text[:s] + (str(int(new)) if f0 in INT_FIELDS else dec(new)) + text[e:]
            text_final = _noise(text, rng); text_norm = normalize_text(text_final)
            if text_norm in seen or text_norm == src["text_norm"]: continue
            probe = {"part_type": pt, "params": params, "text_norm": text_norm}
            if audit_row(probe): continue                               # every label value must be readable in its own text
            rows = make_table(pt, text_norm, params); full = rows_to_params(pt, rows)
            if any(i["level"] == "error" for i in check_rules(pt, full)): continue
            if any(full[f] != params[f] for f in SCHEMA[pt]): continue  # snapping must not have altered the label
            p = {"part_type": pt, **{f: params[f] for f in SCHEMA[pt]}}
            seen.add(text_norm); made += 1
            out.append({"sheet": "augmented", "sheet_row": src["sheet_row"], "text": text_final, "part_type": pt, "text_norm": text_norm,
                        "params": p, "target": params_to_json(p), "split": "train", "group": src["group"], "audit_bad": [],
                        "source": "augmented", "aug_from": f"{src['sheet']}:{src['sheet_row']}", "orig_text": src["text"]})
        stats[pt] = (made, tries)
    return pd.DataFrame(out), stats

if AUGMENT_PER_TYPE > 0:
    aug_df, _stats = build_augmented(train_real_df, AUGMENT_PER_TYPE, AUG_SEED)
    short = {pt: n for pt, (n, _) in _stats.items() if n < AUGMENT_PER_TYPE}
    if short: print("WARNING: could not reach the target for", short)
    train_df = pd.concat([train_real_df, aug_df], ignore_index=True)
else:
    aug_df = pd.DataFrame(columns=train_real_df.columns)

_cnt = pd.DataFrame({"real (train)": train_real_df.part_type.value_counts(), "augmented": aug_df.part_type.value_counts() if len(aug_df) else 0}).reindex(PART_TYPES).fillna(0).astype(int)
_cnt["train total"] = _cnt.sum(axis=1)
_cnt["val (real)"] = val_df.part_type.value_counts().reindex(PART_TYPES).fillna(0).astype(int)
_cnt["test (real)"] = test_df.part_type.value_counts().reindex(PART_TYPES).fillna(0).astype(int)
show_df(_cnt)
print(f"\nDataset size: {len(data)} real + {len(aug_df)} augmented = {len(data) + len(aug_df)}  |  "
      f"training uses {len(train_df)} rows; validation ({len(val_df)}) and test ({len(test_df)}) are real only.")
print("Augmented rows are stored separately (artifacts/augmented_samples.csv) - report them as synthetic data.\n")
for _pt in PART_TYPES[:5]:
    if len(aug_df) and (aug_df.part_type == _pt).any():
        _r = aug_df[aug_df.part_type == _pt].iloc[0]
        print(f"[{_pt}]\n  real     : {_r.orig_text}\n  synthetic: {_r.text}\n  label    : {_r.target}")

## 5c. Feature requests (synthetic, written by Claude, stored separately)
The user can describe **features** in the same text box as the part: holes in the walls/legs or the bottom (single, in rows, or in a grid set in from the sides; sizes or screw sizes like "fits an M5 bolt"), chamfers and radii on named edges, dividers inside boxes and rectangular cutouts/notches. Turning a sentence into those features is done by an **off-the-shelf model** (section 7b) that is not trained; instead it is shown similar solved examples.

* **Hand-written requests** (built into this notebook, no upload): your earlier hole sentences plus requests for edges, dividers, cutouts and mixed edits. These are the held-out **test/validation** set.
* **Generated requests** (`FEATURE_GENERATED_PER_TYPE` per part type), rendered from phrase banks for specs sampled by the next cell. They only go into the **example bank** the model retrieves from.
* Every row is checked: numbers readable in the sentence, words consistent with the labels, features geometrically valid.

In [ ]:
# ------------------------------------------------------------------ 5c: SAMPLING VALID FEATURE SPECS (the numbers behind the generated data)
# Picks a real base part from your dataset and random features for it - holes, chamfers/radii, dividers, cutouts - each one checked by the same rules the app
# uses, so every sampled spec is geometrically valid. The notebook later renders sentences for these specs (section 5d) and the off-the-shelf editing
# model is evaluated on them. Blank (None) positions mean "centred".
FEAT_SHEETS = {"bracket": "Holes_bracket", "rect_container": "Holes_rect", "cyl_container": "Holes_cyl"}      # sheet names of the hand-written hole workbook (optional)
HOLE_DIAMETERS = [0.125, 0.1875, 0.25, 0.3125, 0.375, 0.5, 0.625, 0.75]
NICE_SIZES = [1 / 32, 1 / 16, 3 / 32, 1 / 8, 3 / 16, 1 / 4, 5 / 16, 3 / 8, 1 / 2, 5 / 8, 3 / 4, 1.0]

def _grid(lo, hi, step=0.125):
    out, x = [], math.ceil(lo / step - 1e-9) * step
    while x <= hi + 1e-9: out.append(round(x, 4)); x += step
    return out

fill_defaults = apply_defaults            # defined with the feature rules (section 5)

def _sample_hole(pt, p, rng):
    k, t = HOLE_SPEC[pt]["keys"], p["thickness_in"]
    for _ in range(60):
        d = rng.choice(HOLE_DIAMETERS) if rng.random() > 0.15 else rng.choice(SCREW_HOLES); r, m = d / 2, 0.1; face = rng.choice(HOLE_SPEC[pt]["faces"])
        if pt == "bracket": us = _grid(r + m, leg_length(p, face) - t - r - m); vs = _grid(r + m, p["height_in"] - r - m)
        elif pt == "rect_container": us = _grid(t + r + m, wall_length(p, face) - t - r - m); vs = _grid(t + r + m, p["height_in"] - r - m)
        else: us = [float(a) for a in range(0, 360, 15)]; vs = _grid(t + r + m, p["height_in"] - r - m)
        if not us or not vs: continue
        u = rng.choice(us) if (pt == "cyl_container" or rng.random() > 0.15) else None
        v = rng.choice(vs) if rng.random() > 0.30 else None
        h = {"type": "hole", k[0]: face, k[1]: u, k[2]: v, k[3]: d}
        if rng.random() < 0.25:                                                        # a row of identical holes
            h["count"] = rng.choice([2, 3, 4])
            if pt == "cyl_container": h["spacing_deg"] = None if rng.random() < 0.7 else float(rng.choice([20, 30, 45, 60]))
            else: h["spacing_in"] = None if rng.random() < 0.4 else rng.choice([x for x in (0.5, 0.75, 1.0, 1.25, 1.5, 2.0) if x > d + 0.1] or [None])
        return h
    return None

SCREW_HOLES = [CLEARANCE_IN[s] for s in ("M3", "M4", "M5", "M6", "No. 6", "No. 8", "No. 10", "0.25")]
def _sample_bottom_hole(pt, p, rng):
    t = p["thickness_in"]
    for _ in range(40):
        d = rng.choice([0.125, 0.1875, 0.25, 0.3125, 0.375, 0.5]) if rng.random() > 0.2 else rng.choice(SCREW_HOLES); n = rng.choice([1, 1, 1, 2, 3, 4, 5])
        f = {"type": "bottom_hole", "x_in": None, "y_in": None, "diameter_in": d, "count": None if n == 1 else n, "rows": None, "spacing_in": None, "direction": None}
        if pt == "rect_container": f["inset_in"] = None
        if pt == "rect_container" and rng.random() < 0.3:                              # a rectangular pattern of holes, a set distance in from the sides
            oa, ob = rect_outer(p); cols, rows = rng.choice([(2, 2), (2, 2), (2, 2), (3, 2), (2, 3), (3, 3)])
            insets = [x for x in (0.25, 0.375, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0) if x - d / 2 >= t + 0.1 and min(oa, ob) - 2 * x >= d + 0.2]
            if insets:
                f.update(count=cols, rows=rows, inset_in=rng.choice(insets)); return f
        if pt == "rect_container":
            oa, ob = rect_outer(p)
            if n == 1 or rng.random() < 0.3:
                if rng.random() < 0.6: f["x_in"] = rng.choice(_grid(t + d + 0.2, oa - t - d - 0.2) or [None])
                if rng.random() < 0.6: f["y_in"] = rng.choice(_grid(t + d + 0.2, ob - t - d - 0.2) or [None])
        else:
            R = p["id_in"] / 2
            if n == 1 and rng.random() < 0.4:
                g = [x for x in _grid(-0.6 * R, 0.6 * R) if abs(x) > 1e-9] or [None]
                f["x_in"] = rng.choice(g); f["y_in"] = rng.choice(g) if rng.random() < 0.5 else None
        if n > 1:
            if rng.random() < 0.6: f["spacing_in"] = rng.choice([x for x in (0.5, 0.75, 1.0, 1.5, 2.0, 2.5, 3.0) if x > d + 0.1])
            if rng.random() < 0.5: f["direction"] = rng.choice(["left_right", "front_back"])
        return f
    return None

def _sample_cutout(pt, p, rng):
    k, t, h = HOLE_SPEC[pt]["keys"], p["thickness_in"], p["height_in"]
    for _ in range(60):
        w, hh = rng.choice([0.5, 0.75, 1.0, 1.25, 1.5, 2.0]), rng.choice([0.25, 0.375, 0.5, 0.75, 1.0]); face = rng.choice(HOLE_SPEC[pt]["faces"])
        if pt == "bracket": us = _grid(w / 2 + 0.1, leg_length(p, face) - t - w / 2 - 0.1); vs = _grid(hh / 2 + 0.1, h - hh / 2 - 0.1)
        else:
            us = _grid(t + w / 2 + 0.1, wall_length(p, face) - t - w / 2 - 0.1); vs = _grid(t + hh / 2 + 0.1, h - hh / 2 - 0.1)
            if rng.random() < 0.3 and h - hh / 2 >= t + hh / 2 + 0.1: vs = [round(h - hh / 2, 4)]          # an open notch that reaches the top rim
        if not us or not vs: continue
        u = rng.choice(us) if rng.random() > 0.2 else None; v = rng.choice(vs) if rng.random() > 0.3 else None
        return {"type": "cutout", k[0]: face, k[1]: u, k[2]: v, "width_in": w, "height_in": hh}
    return None

def _sample_edge(pt, p, rng, used):
    for _ in range(40):
        edge, kind = rng.choice(EDGE_OPTIONS[pt]), rng.choice(_CHAMFER_FILLET)
        if edge in used or any(edge in g and u in g for g in _EDGE_CONFLICT for u in used): continue
        sizes = [s for s in NICE_SIZES if s <= 0.85 * edge_limit(pt, p, edge, kind)]
        if not sizes: continue
        f = {"type": "edge", "kind": kind, "edge": edge, "size_in": rng.choice(sizes[-6:])}
        if rng.random() < 0.15: f["size_in"] = None                                 # "fillet the corners" with no size -> a safe default
        if rng.random() < 0.08 and edge == DEFAULT_EDGE[pt] and not used: f["edge"] = None      # "round the edges" -> the main outside edges
        return f
    return None

def _sample_divider(pt, p, rng):
    t, h = p["thickness_in"], p["height_in"]; oa, ob = rect_outer(p); ia, ib = oa - 2 * t, ob - 2 * t
    for _ in range(40):
        orient = rng.choice(["crosswise", "lengthwise"]); span = ia if orient == "crosswise" else ib
        td = None if rng.random() < 0.75 else rng.choice([1 / 16, 3 / 32, 1 / 8, 3 / 16, 1 / 4])
        hh = None if rng.random() < 0.75 else rng.choice([x for x in _grid(0.25, 0.9 * (h - t), 0.125)] or [None])
        if rng.random() < 0.6: f = {"type": "divider", "orientation": orient if rng.random() > 0.35 else None, "position_in": None, "count": rng.choice([None, None, 2, 3]), "thickness_in": td, "height_in": hh}
        else:
            ps = _grid(0.5, span - 0.5, 0.125)
            if not ps: continue
            f = {"type": "divider", "orientation": orient, "position_in": rng.choice(ps), "count": None, "thickness_in": td, "height_in": hh}
        return f
    return None

def sample_features(pt, p, rng, n, type_weights=None):
    """n random features for a part, accepted only if the whole set passes check_features with no errors AND no warnings"""
    types = FEATURE_TYPES[pt]; w = type_weights or {t: 1.0 for t in types}; feats, used = [], []
    for _ in range(n * 25):
        if len(feats) >= n: break
        ty = rng.choices(types, [w.get(t, 1.0) for t in types])[0]
        cand = (_sample_hole(pt, p, rng) if ty == "hole" else _sample_cutout(pt, p, rng) if ty == "cutout" else _sample_edge(pt, p, rng, used) if ty == "edge"
                else _sample_bottom_hole(pt, p, rng) if ty == "bottom_hole" else _sample_divider(pt, p, rng))
        if cand is None: continue
        if not check_features(pt, p, fill_defaults(pt, p, feats + [cand])):
            feats.append(cand)
            if ty == "edge": used.append(cand["edge"])
    return feats if len(feats) == n else None

def feature_part_params(pt, r):
    base = {f: float(r[f]) for f in SCHEMA[pt]}; return base, {"part_type": pt, **base}
print("feature sampler ready")

In [ ]:
# ------------------------------------------------------------------ GENERATED feature sentences (SYNTHETIC - written by Claude as phrase banks, stored separately)
# The 120 hand-written sentences in hole_features_filled.xlsx are too few to train on alone, so this adds `FEATURE_GENERATED_PER_TYPE` more per part type.
# Each sentence is rendered FROM its label (the label values are the single source of truth), using many ways to say the same thing:
# fractions / decimals / words, inch / in / ", different openers, sentence structures, wall/leg names, centring phrases, typos, casing.
# Base parts come from your real dataset rows that the hand-written set did NOT use. These rows only ever go into TRAINING.
from math import gcd

def _frac(v):
    """0.1875 -> '3/16', 1.375 -> '1 3/8', 4.0 -> '4'  (None if v is not a multiple of 1/16)"""
    x = round(v * 16)
    if abs(v * 16 - x) > 1e-6: return None
    whole, rem = divmod(x, 16)
    if rem == 0: return str(whole)
    g = gcd(rem, 16); fr = f"{rem // g}/{16 // g}"
    return f"{whole} {fr}" if whole else fr

_UNITS = ["", " in", " inch", " inches", '"', " inches", " inch"]
def _len(v, rng, words=False):
    """a length written in a random human style, with a random unit"""
    if words and v in (0.25, 0.5) and rng.random() < 0.25:
        return rng.choice(["quarter inch", "quarter of an inch"] if v == 0.25 else ["half inch"])
    body = _frac(v) if (rng.random() < 0.55 and _frac(v) is not None) else dec(v)
    if body.startswith("0.") and rng.random() < 0.15: body = body[1:]
    unit = rng.choice(_UNITS)
    if unit == " inch" and v > 1: unit = " inches"
    return body + unit

def _a(text): return ("an " if text[:1] == "8" or text.startswith("11") or text.startswith("18") else "a ") + text

_SCREW_NAME = {v: k for k, v in CLEARANCE_IN.items()}
def _screw(d, rng):
    """'for an M5 bolt' style wording when the diameter is a standard clearance hole (the normalizer turns it back into the number)"""
    s = _SCREW_NAME[d]; name = ("#" + s.split()[-1]) if s.startswith("No.") else (_frac(float(s)) if s[0].isdigit() else s)
    art = "an" if name[0] in "M8" or name.startswith("#8") or name.startswith("#1") and len(name) == 3 and name[2] in "18" else "a"
    if name.startswith("#"): art = "an" if name in ("#8", "#11", "#18") else "a"
    fast = rng.choice(["bolt", "screw"])
    return rng.choice([f"for {art} {name} {fast}", f"that fits {art} {name} {fast}", f"sized for {art} {name} {fast}"]) if not name.startswith("M") or rng.random() < 0.6 else rng.choice([f"for {art} {name} {fast}", f"that fits {art} {name} {fast}"])

def _dia(d, rng):
    if d in _SCREW_NAME and rng.random() < 0.85:
        return rng.choice([f"hole {_screw(d, rng)}", f"clearance hole {_screw(d, rng)}"])
    L = _len(d, rng, words=True)
    return rng.choice([f"{L} hole", f"{L} diameter hole", f"{L} dia hole", f"{L} through hole", f"{L} round hole", f"{L} dia. hole"])

_DEAD = ["dead center", "in the exact center", "centered both ways", "dead center of it"]

def _pos_bracket(u, v, rng):
    if u is None and v is None: return rng.choice(_DEAD)
    pu = (rng.choice(["centered along the leg", "in the middle of the leg", "centered lengthwise", "halfway along the leg"]) if u is None else
          (lambda P: rng.choice([f"{P} from the end", f"{P} in from the end", f"{P} from the free end", f"{P} from the tip", f"{P} back from the end of the leg"]))(_len(u, rng, True)))
    pv = (rng.choice(["centered across the width", "centered on the width", "in the middle of the width", "halfway across the width"]) if v is None else
          (lambda P: rng.choice([f"{P} up from the bottom", f"{P} above the bottom face", f"{P} from the bottom", f"{P} off the bottom", f"{P} up from the bottom face"]))(_len(v, rng, True)))
    a, b = (pu, pv) if rng.random() < 0.75 else (pv, pu)
    return a + rng.choice([" and ", ", ", ", and "]) + b

def _pos_rect(u, v, rng):
    if u is None and v is None: return rng.choice(_DEAD)
    pu = (rng.choice(["centered left to right", "centered side to side", "horizontally centered", "in the middle left to right"]) if u is None else
          (lambda P: rng.choice([f"{P} from the left end", f"{P} from the left edge", f"{P} in from the left end", f"{P} from the left corner", f"{P} from the left as you look at it"]))(_len(u, rng, True)))
    pv = (rng.choice(["centered vertically", "vertically centered", "halfway up", "mid-height", "halfway up the wall"]) if v is None else
          (lambda P: rng.choice([f"{P} up from the bottom", f"{P} above the bottom", f"{P} up", f"{P} from the bottom", f"{P} off the floor"]))(_len(v, rng, True)))
    a, b = (pu, pv) if rng.random() < 0.8 else (pv, pu)
    return a + rng.choice([" and ", ", ", ", and "]) + b

_CARD = {270: ["front", "the front", "front side"], 90: ["back", "the back", "back side", "rear"], 180: ["left", "the left", "left side"], 0: ["right", "the right", "right side"]}
def _pos_cyl(u, v, rng):
    A = int(round(u))
    ang = (rng.choice([f"on the {rng.choice(_CARD[A]).replace('the ', '')} of the cup", f"on the {rng.choice(_CARD[A]).replace('the ', '')}"]) if (A in _CARD and rng.random() < 0.45) else
           rng.choice([f"at {A} degrees", f"at an angle of {A} degrees", f"at {A}°", f"at {A} deg", f"{A} degrees around the cup"]))
    up = (rng.choice(["centered vertically", "vertically centered", "halfway up", "mid-height", "in the middle of the height"]) if v is None else
          (lambda P: rng.choice([f"{P} up from the bottom", f"{P} above the bottom", f"{P} up", f"{P} from the bottom"]))(_len(v, rng, True)))
    return ang + rng.choice([", ", ", and ", " and "]) + up

def _face_ref(pt, face, base, rng):
    if pt == "bracket":
        a, b = base["length_a_in"], base["length_b_in"]
        refs = ["leg A", "leg a", "the first leg"] if face == "leg_a" else ["leg B", "leg b", "the second leg"]
        if abs(a - b) > 1e-6:
            longer = (face == "leg_a") == (a > b)
            refs += (["the longer leg", "the long leg"] if longer else ["the shorter leg", "the short leg"]) * 2
            refs.append(f"the {dec(round(base['length_a_in' if face == 'leg_a' else 'length_b_in'], 2))} inch leg")
        return rng.choice(refs)
    if pt == "rect_container":
        w = "rear" if (face == "back" and rng.random() < 0.15) else face
        return rng.choice([f"the {w} wall", f"the {w} wall", f"the {w} side", f"the {w}", f"{w} wall"])
    return "the cup"

_OPEN = ["Add", "Drill", "Cut", "Put", "Make", "I need", "I want", "Please add", "Can you add", "Give it", "Create"]
_W = {1: "one", 2: "two", 3: "three", 4: "four", 5: "five", 6: "six", 7: "seven", 8: "eight", 9: "nine"}

def _hole_row_clause(pt, h, base, rng, k, n):
    L = _len(h[k[3]], rng, words=True); holes = f"{_W[n]} {L} holes"
    if pt == "cyl_container":
        up = rng.choice(["centered vertically", "halfway up", "mid-height"]) if h[k[2]] is None else f"{_len(h[k[2]], rng, True)} up from the bottom"
        A = int(round(h[k[1]]))
        if h.get("spacing_deg") is None: return (True, f"{holes} evenly spaced around the cup starting at {A} degrees, {up}")
        return (True, f"{holes} {int(round(h['spacing_deg']))} degrees apart, centered on {A} degrees, {up}")
    face = _face_ref(pt, h["face"], base, rng); where = rng.choice([f" in {face}", f" on {face}", f" along {face}"])
    u, v, s = h[k[1]], h[k[2]], h.get("spacing_in")
    if pt == "bracket": vt = "centered across the width" if v is None else f"{_len(v, rng, True)} up from the bottom"
    else: vt = rng.choice(["centered vertically", "halfway up"]) if v is None else f"{_len(v, rng, True)} up from the bottom"
    if s is None: return (True, f"{holes}{where}, {rng.choice(['evenly spaced', 'spread evenly along it', 'evenly spaced along it'])}, {vt}")
    S = _len(s, rng, True); sp = rng.choice([f"{S} apart", f"spaced {S} apart", f"{S} center to center"])
    if u is None: ut = "centered along the leg" if pt == "bracket" else rng.choice(["centered on the wall", "centered left to right"])
    else: ut = f"centered {_len(u, rng, True)} from the {'end' if pt == 'bracket' else 'left end'}"
    return (True, f"{holes}{where}, {sp}, {ut}, {vt}")

def _hole_clause(pt, h, base, rng, k):
    if (h.get("count") or 1) > 1: return _hole_row_clause(pt, h, base, rng, k, h["count"])
    face = _face_ref(pt, h["face"], base, rng); dia = _dia(h[k[3]], rng)
    pos = {"bracket": _pos_bracket, "rect_container": _pos_rect, "cyl_container": _pos_cyl}[pt](h[k[1]], h[k[2]], rng)
    where = "" if pt == "cyl_container" else rng.choice([f" in {face}", f" in {face}", f" through {face}", f" on {face}"])
    forms = [(True, f"{_a(dia)}{where}, {pos}"), (True, f"{_a(dia)}{where}: {pos}"), (False, f"{dia}{where}, {pos}")]
    if pt != "cyl_container": forms += [(False, f"in {face}: {dia}, {pos}"), (False, f"{face}: {dia}, {pos}"), (False, f"{face}, {dia}, {pos}")]
    return rng.choice(forms)

_EDGE_TARGETS = {
    "bracket": {"bend": ["the bend"], "outer_bend": ["the outside corner of the bend", "the outer corner of the bend", "the outer bend"],
                "inner_bend": ["the inside corner of the bend", "the inner corner of the bend", "the inside bend"], "free_ends": ["the ends of both legs", "the free ends", "the ends of the legs"]},
    "rect_container": {"corners": ["the corners", "all four corners", "the vertical corners", "the outside corners"], "inner_corners": ["the inside corners", "the inner corners"],
                       "top_rim": ["the top rim", "the top edge", "the lip", "the rim"], "bottom": ["the bottom edges", "the outside bottom edges", "the bottom edge"],
                       "inner_bottom": ["the inside bottom edges", "where the floor meets the walls inside", "the inside floor edges"]},
    "cyl_container": {"top_rim": ["the rim", "the top edge", "the top rim", "the lip"], "bottom": ["the bottom edge", "the outside bottom edge", "the bottom"],
                      "inner_bottom": ["the inside bottom edge", "where the floor meets the wall inside", "the inside floor edge"]},
}
def _edge_clause(pt, f, rng):
    kind = f["kind"]
    tgt = rng.choice(_EDGE_TARGETS[pt][f["edge"]]) if f["edge"] is not None else rng.choice(["the edges", "the outside edges"])
    if f["size_in"] is None:                                                          # no size given: the app picks a safe one
        verb = {"chamfer": ["chamfer", "bevel", "put a small chamfer on"], "fillet": ["fillet", "round", "round off", "put a small radius on"]}[kind]
        return (False, f"{rng.choice(verb)} {tgt}")
    S = _len(f["size_in"], rng, words=True)
    if pt == "bracket" and f["edge"] == "bend":
        forms = [(True, f"{_a(S)} chamfer on the bend"), (False, f"chamfer the bend {S}")] if kind == "chamfer" else [(True, f"{_a(S)} inside radius on the bend"), (True, f"{_a(S)} bend radius")]
        return rng.choice(forms)
    if kind == "chamfer": forms = [(True, f"{_a(S)} chamfer on {tgt}"), (False, f"chamfer {tgt} {S}"), (False, f"{tgt}: {S} chamfer"), (True, f"{_a(S)} bevel on {tgt}")]
    else: forms = [(True, f"{_a(S)} radius on {tgt}"), (False, f"fillet {tgt} {S}"), (False, f"round {tgt} with {_a(S)} radius"), (False, f"{tgt}: {S} fillet"), (True, f"{_a(S)} fillet on {tgt}")]
    return rng.choice(forms)

_DIR = {"crosswise": ["front to back", "front-to-back", "running front to back"], "lengthwise": ["left to right", "left-to-right", "running left to right"]}
_SEP = {"crosswise": ["separating the left from the right", "separating left and right", "splitting left from right"], "lengthwise": ["separating the front from the back", "separating front and back", "splitting front from back"]}
def _divider_clause(pt, f, rng):
    n, ori = f["count"] or 1, f["orientation"]
    if ori is None:                                                                   # direction left to the default (split the longer side)
        if n == 1: parts = [rng.choice(["a divider down the middle", "a divider in the middle", "one divider, centered"])]
        else: parts = [rng.choice([f"{_W[n]} dividers, evenly spaced", f"dividers so it has {_W[n + 1]} sections", f"{_W[n + 1]} separate sections", f"{_W[n + 1]} equal compartments"])]
        if f["thickness_in"] is not None: parts.append(f"{_len(f['thickness_in'], rng)} thick")
        if f["height_in"] is not None: parts.append(f"{_len(f['height_in'], rng, True)} {rng.choice(['tall', 'high'])}")
        return (True, ", ".join(parts))
    d_phr = rng.choice(_DIR[ori]) if rng.random() < 0.6 else rng.choice(_SEP[ori])
    if n == 1: noun = f"a divider {d_phr}"
    elif rng.random() < 0.7: noun = f"{_W[n]} dividers {d_phr}, evenly spaced"
    else: noun = f"{_W[n + 1]} compartments using {_W[n]} dividers {d_phr}"
    parts = [noun]
    if f["position_in"] is not None and n == 1: parts.append(f"{_len(f['position_in'], rng, True)} from the {'left' if ori == 'crosswise' else 'front'} wall")
    elif n == 1: parts.append(rng.choice(["down the middle", "centered", "in the center"]))
    if f["thickness_in"] is not None: parts.append(f"{_len(f['thickness_in'], rng)} thick")
    if f["height_in"] is not None: parts.append(f"{_len(f['height_in'], rng, True)} {rng.choice(['tall', 'high'])}")
    return rng.choice([(True, ", ".join(parts)), (True, ", ".join(parts)), (False, ", ".join(p[:1].upper() + p[1:] if i == 0 else p for i, p in enumerate(parts)))])

_BOTTOM = ["the bottom", "the base", "the floor"]
def _bottom_clause(pt, f, base, rng):
    n, L, bw = f.get("count") or 1, _len(f["diameter_in"], rng, True), rng.choice(_BOTTOM)
    screw = f["diameter_in"] in _SCREW_NAME and rng.random() < 0.85
    if f.get("inset_in") is not None:                                                  # rectangular pattern, set in from the sides
        m, I = f.get("rows") or 1, _len(f["inset_in"], rng, True); tot = n * m
        holes = (f"{_W[tot]} holes {_screw(f['diameter_in'], rng)}" if screw else f"{_W[tot]} {L} holes")
        forms = [f"{holes} in {bw} in a rectangular pattern, {I} from the sides", f"{holes} in {bw}, {n} by {m} grid, {I} in from the edges",
                 f"a {n} x {m} grid of {L if not screw else 'clearance'} holes in {bw}, {I} from each side" + (f" {_screw(f['diameter_in'], rng)}" if screw else "")]
        if n == 2 and m == 2: forms += [f"{holes} near the corners of {bw}, {I} from each side", f"a hole in each corner of {bw}, {I} in from the sides" + (f", {_screw(f['diameter_in'], rng)}" if screw else f", {L}")]
        return (True, rng.choice(forms))
    if screw: L = f"clearance"
    x, y = f.get("x_in"), f.get("y_in")
    if n == 1:
        word = rng.choice(["hole", "hole", "drain hole"])
        if pt == "rect_container":
            if x is None and y is None: pos = rng.choice([f"in the center of {bw}", f"centered in {bw}", f"dead center in {bw}"])
            elif x is None: pos = f"in {bw}, centered left to right, {_len(y, rng, True)} from the front"
            elif y is None: pos = f"in {bw}, {_len(x, rng, True)} from the left, centered front to back"
            else: pos = f"in {bw}, {_len(x, rng, True)} from the left and {_len(y, rng, True)} from the front"
        else:
            if x is None and y is None: pos = rng.choice([f"in the center of {bw}", f"right in the middle of {bw}"])
            else:
                bits = ([f"{_len(abs(x), rng, True)} {'right' if x > 0 else 'left'} of center"] if x is not None else []) + ([f"{_len(abs(y), rng, True)} toward the {'back' if y > 0 else 'front'}"] if y is not None else [])
                pos = f"in {bw}, " + " and ".join(bits)
        if screw: return (True, f"a {word} {pos} {_screw(f['diameter_in'], rng)}")
        return (True, f"{_a(L)} {word} {pos}")
    holes = f"{_W[n]} holes in {bw} {_screw(f['diameter_in'], rng)}" if screw else f"{_W[n]} {L} holes in {bw}"; s = f.get("spacing_in")
    sp = rng.choice(["evenly spaced", "spread out evenly"]) if s is None else rng.choice([f"{_len(s, rng, True)} apart", f"spaced evenly {_len(s, rng, True)} apart"])
    d = f.get("direction"); parts = [holes, sp]
    if d is not None: parts.append(rng.choice({"left_right": ["in a row from left to right", "running left to right"], "front_back": ["in a row from front to back", "running front to back"]}[d]))
    if pt == "rect_container":
        if x is not None: parts.append(f"centered {_len(x, rng, True)} from the left" if (d or "left_right") == "left_right" and s is not None else f"{_len(x, rng, True)} from the left")
        if y is not None: parts.append(f"{_len(y, rng, True)} from the front")
    return (True, ", ".join(parts))

def _cutout_clause(pt, f, base, rng, k):
    face = _face_ref(pt, f["face"], base, rng); W, Hh = _len(f["width_in"], rng, True), _len(f["height_in"], rng, True)
    size = rng.choice([f"{W} wide by {Hh} tall", f"{W} by {Hh}", f"{W} wide and {Hh} tall", f"{W} x {Hh}"]); sw = rng.choice(["slot", "notch", "cutout", "opening", "window", "slot"])
    pos = (_pos_bracket if pt == "bracket" else _pos_rect)(f[k[1]], f[k[2]], rng)
    if pt != "bracket" and f[k[2]] is not None and f[k[2]] + f["height_in"] / 2 >= base["height_in"] - 1e-6 and rng.random() < 0.5: pos += ", open at the top"
    return rng.choice([(True, f"{_a(size)} {sw} in {face}, {pos}"), (True, f"{_a(size)} {sw} on {face}, {pos}"), (False, f"{face}: {size} {sw}, {pos}"), (False, f"{face}, {size} {sw}, {pos}")])

def _clause_for(pt, f, base, rng):
    k = HOLE_SPEC[pt]["keys"] if pt in HOLE_SPEC else None
    return {"hole": lambda: _hole_clause(pt, f, base, rng, k), "edge": lambda: _edge_clause(pt, f, rng), "divider": lambda: _divider_clause(pt, f, rng), "bottom_hole": lambda: _bottom_clause(pt, f, base, rng),
            "cutout": lambda: _cutout_clause(pt, f, base, rng, k)}[f["type"]]()

TYPO_RATE = 0.12                                                                         # chance a sentence gets one typo (set 0 to disable)
_PROTECT = {"front", "back", "left", "right", "rear", "wall", "side", "degrees", "inch", "inches", "center", "centered", "middle", "halfway", "vertically", "horizontally", "bottom", "diameter",
            "chamfer", "fillet", "radius", "bevel", "divider", "dividers", "compartments", "separating", "splitting", "evenly", "spaced"}
def render_sentence(pt, base, feats, rng):
    n = len(feats); clauses = [_clause_for(pt, f, base, rng) for f in feats]
    cap = lambda s: s[:1].upper() + s[1:]
    if n == 1:
        opener_ok, c = clauses[0]
        text = f"{rng.choice(_OPEN)} {c}" if (opener_ok and rng.random() < 0.7) else cap(c)
    else:
        mode = rng.choice(["list", "sentences", "joined"]); all_holes = all(f["type"] == "hole" and (f.get("count") or 1) == 1 for f in feats)
        if mode == "list":
            pre = ([f"{cap(_W[n])} holes: ", f"{rng.choice(_OPEN)} {_W[n]} holes. ", f"{cap(_W[n])} holes please. ", ""] if all_holes else [f"{rng.choice(_OPEN)} the following: ", "", ""])
            text = rng.choice(pre) + "; ".join(c for _, c in clauses)
        elif mode == "sentences": text = " ".join(cap(c) + "." for _, c in clauses).rstrip(".")
        else:
            text = (f"{rng.choice(_OPEN)} " if clauses[0][0] else "") + (clauses[0][1] if clauses[0][0] else cap(clauses[0][1])) + "".join(rng.choice([", and ", " plus ", ", also ", "; then "]) + c for _, c in clauses[1:])
    if rng.random() < 0.12: text += rng.choice([", please", " please", " thanks"])
    if rng.random() < 0.55 and not text.endswith("."): text += "."
    if rng.random() < TYPO_RATE:                                                             # one typo in an unprotected word
        words = [m for m in re.finditer(r"[A-Za-z]{5,}", text) if m.group().lower() not in _PROTECT]
        if words:
            m = rng.choice(words); i = rng.randrange(1, len(m.group()) - 2); w = list(m.group()); w[i], w[i + 1] = w[i + 1], w[i]
            text = text[:m.start()] + "".join(w) + text[m.end():]
    if rng.random() < 0.18: text = text.lower()
    return text

def generate_feature_rows(per_type, seed, exclude_refs=()):
    rng, rows = random.Random(seed), []
    for pt in FEATURE_TYPES:
        bases = [r for _, r in raw_all[raw_all.part_type == pt].iterrows() if f"{r['sheet']}:{r['sheet_row']}" not in set(exclude_refs)]
        bases = bases or [r for _, r in raw_all[raw_all.part_type == pt].iterrows()]
        rng.shuffle(bases); made, seen, tries = 0, set(), 0
        while made < per_type and tries < per_type * 40:
            r = bases[tries % len(bases)]; tries += 1
            base, p = feature_part_params(pt, r)
            if any(i["level"] == "error" for i in check_rules(pt, p)): continue
            feats = sample_features(pt, p, rng, rng.choices([1, 2, 3, 4], [0.25, 0.35, 0.25, 0.15])[0])
            if feats is None: continue
            text = render_sentence(pt, base, feats, rng); tn = normalize_feature_text(text)
            if tn in seen or audit_features(pt, feats, tn): continue                              # label must be readable AND consistent
            seen.add(tn); made += 1
            rows.append({"sheet": "generated", "sheet_row": len(rows) + 1, "part_type": pt, "base": base, "features": feats, "text": text,
                         "group": f"{r['sheet']}:{r['sheet_row']}|gen", "source": "claude_generated"})
    return pd.DataFrame(rows)
print("feature-sentence generator ready")

In [ ]:
# ------------------------------------------------------------------ the hand-written feature requests, built into the notebook (SYNTHETIC - written by Claude)
# One record per sentence: base part (from your dataset), the features it describes (holes, chamfers/radii, dividers, cutouts) and the sentence.
# Nothing to upload. These rows are the held-out TEST/VALIDATION pool for the editing model, and part of its example bank.
FEATURE_WRITTEN_JSON = r'''[{"sheet":"handwritten_holes","sheet_row":1,"part_type":"bracket","base":{"length_a_in":1.5625,"length_b_in":1.5625,"height_in":1.0,"thickness_in":0.12},"features":[{"type":"hole","face":"leg_a","from_end_in":null,"across_in":0.375,"diameter_in":0.1875}],"text":"Put a 3/16 hole in leg A, centered along the leg and 3/8 inch up from the bottom.","group":"L_brackets:68","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":2,"part_type":"bracket","base":{"length_a_in":4.0,"length_b_in":4.0,"height_in":0.875,"thickness_in":0.12},"features":[{"type":"hole","face":"leg_b","from_end_in":3.25,"across_in":0.5,"diameter_in":0.3125},{"type":"hole","face":"leg_a","from_end_in":3.5,"across_in":0.375,"diameter_in":0.3125}],"text":"Two 5/16 inch holes please: one in leg B, 3.25 inches from the end and 1/2 inch up from the bottom, and one in leg A, 3.5 inches from the end and 3/8 inch up.","group":"L_brackets:12","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":3,"part_type":"bracket","base":{"length_a_in":1.875,"length_b_in":2.375,"height_in":2.375,"thickness_in":0.1},"features":[{"type":"hole","face":"leg_a","from_end_in":null,"across_in":null,"diameter_in":0.625}],"text":"I need a 5/8 inch hole dead center in the shorter leg.","group":"L_brackets:41","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":4,"part_type":"bracket","base":{"length_a_in":2.75,"length_b_in":2.75,"height_in":0.8125,"thickness_in":0.24},"features":[{"type":"hole","face":"leg_b","from_end_in":1.75,"across_in":null,"diameter_in":0.125}],"text":"drill a 1/8 hole into leg B 1.75 in from the end, centered on the width","group":"L_brackets:56","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":5,"part_type":"bracket","base":{"length_a_in":2.0,"length_b_in":2.0,"height_in":1.5,"thickness_in":0.12},"features":[{"type":"hole","face":"leg_a","from_end_in":1.0,"across_in":0.5,"diameter_in":0.3125}],"text":"Add a 5/16\" diameter hole to leg A. It should sit 1\" from the free end and 1/2\" up from the bottom face.","group":"L_brackets:85","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":6,"part_type":"bracket","base":{"length_a_in":3.1496,"length_b_in":1.5748,"height_in":1.9685,"thickness_in":0.1181},"features":[{"type":"hole","face":"leg_a","from_end_in":1.125,"across_in":1.0,"diameter_in":0.75},{"type":"hole","face":"leg_b","from_end_in":0.875,"across_in":null,"diameter_in":0.3125},{"type":"hole","face":"leg_a","from_end_in":null,"across_in":0.5,"diameter_in":0.125}],"text":"On the long leg I want a 3/4 inch hole 1 1/8 inch from the end and 1 inch up, plus a tiny 1/8 hole centered along the leg at 1/2 inch up. The short leg gets a 5/16 inch hole 7/8 inch from its end, centered across the width.","group":"L_brackets:81","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":7,"part_type":"bracket","base":{"length_a_in":1.5,"length_b_in":1.5,"height_in":1.25,"thickness_in":0.06},"features":[{"type":"hole","face":"leg_b","from_end_in":1.0,"across_in":0.75,"diameter_in":0.625}],"text":"5/8 inch hole in leg B, 1 inch from the end, 3/4 inch from the bottom.","group":"L_brackets:38","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":8,"part_type":"bracket","base":{"length_a_in":2.0,"length_b_in":1.0,"height_in":1.0,"thickness_in":0.105},"features":[{"type":"hole","face":"leg_a","from_end_in":1.25,"across_in":0.75,"diameter_in":0.1875}],"text":"Drill a 3/16 hole through the 2 inch leg, 1.25 inches from the end and 0.75 inches up from the bottom.","group":"L_brackets:84","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":9,"part_type":"bracket","base":{"length_a_in":9.75,"length_b_in":9.75,"height_in":5.0,"thickness_in":0.25},"features":[{"type":"hole","face":"leg_a","from_end_in":3.625,"across_in":0.875,"diameter_in":0.375},{"type":"hole","face":"leg_b","from_end_in":1.25,"across_in":null,"diameter_in":0.375}],"text":"Two 3/8 inch holes please. Leg A: 3 5/8 inches from the end, 7/8 inch up. Leg B: 1 1/4 inches from the end, centered in the width.","group":"L_brackets:49","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":10,"part_type":"bracket","base":{"length_a_in":2.75,"length_b_in":3.375,"height_in":0.625,"thickness_in":0.19},"features":[{"type":"hole","face":"leg_a","from_end_in":2.0,"across_in":0.25,"diameter_in":0.25}],"text":"put a quarter inch hole in the shorter leg, 2 inches from its end and 1/4 inch up from the bottom","group":"L_brackets:60","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":11,"part_type":"bracket","base":{"length_a_in":1.8125,"length_b_in":1.8125,"height_in":0.5,"thickness_in":0.12},"features":[{"type":"hole","face":"leg_b","from_end_in":0.75,"across_in":0.25,"diameter_in":0.125},{"type":"hole","face":"leg_a","from_end_in":0.875,"across_in":0.25,"diameter_in":0.1875}],"text":"Two small holes, both 1/4 inch up from the bottom. Leg B gets a 1/8 inch one 3/4 inch from the end; leg A gets a 3/16 inch one 7/8 inch from the end.","group":"L_brackets:27","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":12,"part_type":"bracket","base":{"length_a_in":5.625,"length_b_in":5.625,"height_in":5.0,"thickness_in":0.13},"features":[{"type":"hole","face":"leg_b","from_end_in":null,"across_in":0.875,"diameter_in":0.5},{"type":"hole","face":"leg_a","from_end_in":5.0,"across_in":null,"diameter_in":0.625}],"text":"Leg B: a 1/2 inch hole centered along the leg, 7/8 inch up from the bottom. Leg A: a 5/8 inch hole 5 inches from the end, centered on the width.","group":"L_brackets:47","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":13,"part_type":"bracket","base":{"length_a_in":8.0,"length_b_in":8.0,"height_in":1.25,"thickness_in":0.22},"features":[{"type":"hole","face":"leg_a","from_end_in":1.125,"across_in":null,"diameter_in":0.625},{"type":"hole","face":"leg_b","from_end_in":6.0,"across_in":0.875,"diameter_in":0.1875}],"text":"add a 5/8 hole to leg A 1 1/8 inch from the end and centered, and a 3/16 hole to leg B 6 inches from the end and 7/8 inch off the bottom","group":"L_brackets:15","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":14,"part_type":"bracket","base":{"length_a_in":0.5625,"length_b_in":0.5625,"height_in":1.0,"thickness_in":0.06},"features":[{"type":"hole","face":"leg_b","from_end_in":0.25,"across_in":null,"diameter_in":0.125},{"type":"hole","face":"leg_a","from_end_in":0.25,"across_in":0.5,"diameter_in":0.25}],"text":"Leg B needs a 1/8 inch hole a quarter inch from the end, centered. Leg A needs a 1/4 inch hole, also 1/4 inch from the end, 1/2 inch up from the bottom.","group":"L_brackets:32","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":15,"part_type":"bracket","base":{"length_a_in":2.0,"length_b_in":2.0,"height_in":2.5,"thickness_in":0.125},"features":[{"type":"hole","face":"leg_b","from_end_in":0.625,"across_in":1.75,"diameter_in":0.1875},{"type":"hole","face":"leg_b","from_end_in":1.375,"across_in":0.5,"diameter_in":0.75},{"type":"hole","face":"leg_a","from_end_in":0.875,"across_in":0.875,"diameter_in":0.375}],"text":"Leg B gets two holes: a 3/16 inch hole 5/8 inch from the end and 1 3/4 inch up, and a 3/4 inch hole 1 3/8 inches from the end and 1/2 inch up. Leg A gets one 3/8 inch hole, 7/8 inch from the end and 7/8 inch up.","group":"L_brackets:76","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":16,"part_type":"bracket","base":{"length_a_in":3.625,"length_b_in":3.625,"height_in":2.0,"thickness_in":0.11},"features":[{"type":"hole","face":"leg_a","from_end_in":1.0,"across_in":null,"diameter_in":0.125},{"type":"hole","face":"leg_a","from_end_in":0.5,"across_in":0.5,"diameter_in":0.375}],"text":"Two holes on leg A: 1/8 inch diameter, 1 inch from the end and centered across; 3/8 inch diameter, 1/2 inch from the end and 1/2 inch up.","group":"L_brackets:45","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":17,"part_type":"bracket","base":{"length_a_in":7.875,"length_b_in":9.4375,"height_in":0.75,"thickness_in":0.31},"features":[{"type":"hole","face":"leg_a","from_end_in":1.875,"across_in":0.375,"diameter_in":0.5}],"text":"On the shorter leg add a half inch hole, 1 7/8 inch in from the end, 3/8 inch up from the bottom face.","group":"L_brackets:63","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":18,"part_type":"bracket","base":{"length_a_in":0.75,"length_b_in":0.75,"height_in":1.5,"thickness_in":0.04},"features":[{"type":"hole","face":"leg_b","from_end_in":0.25,"across_in":null,"diameter_in":0.125}],"text":"tiny 1/8 hole in leg B a quarter inch from the end, centered","group":"L_brackets:36","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":19,"part_type":"bracket","base":{"length_a_in":3.0,"length_b_in":3.0,"height_in":0.75,"thickness_in":0.1},"features":[{"type":"hole","face":"leg_a","from_end_in":1.25,"across_in":null,"diameter_in":0.5}],"text":"Please add a 0.5 inch hole to leg A, 1.25 inches from the end, centered on the width.","group":"L_brackets:21","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":20,"part_type":"bracket","base":{"length_a_in":2.375,"length_b_in":2.375,"height_in":0.6875,"thickness_in":0.08},"features":[{"type":"hole","face":"leg_a","from_end_in":1.5,"across_in":0.375,"diameter_in":0.3125},{"type":"hole","face":"leg_a","from_end_in":2.0,"across_in":0.375,"diameter_in":0.3125},{"type":"hole","face":"leg_a","from_end_in":0.5,"across_in":0.375,"diameter_in":0.3125}],"text":"Leg A needs three 5/16 inch holes in a row, all 3/8 inch up from the bottom, at 1/2 inch, 1 1/2 inches and 2 inches from the end.","group":"L_brackets:8","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":21,"part_type":"bracket","base":{"length_a_in":0.6875,"length_b_in":1.0,"height_in":0.5,"thickness_in":0.12},"features":[{"type":"hole","face":"leg_b","from_end_in":0.25,"across_in":0.25,"diameter_in":0.125}],"text":"1/8 inch hole in the longer leg, 1/4 inch from the end and 1/4 inch up from the bottom.","group":"L_brackets:24","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":22,"part_type":"bracket","base":{"length_a_in":6.0,"length_b_in":6.0,"height_in":1.125,"thickness_in":0.18},"features":[{"type":"hole","face":"leg_a","from_end_in":1.0,"across_in":0.5,"diameter_in":0.75},{"type":"hole","face":"leg_b","from_end_in":1.25,"across_in":0.75,"diameter_in":0.25}],"text":"Leg A: 3/4 inch hole 1 inch from the end, half an inch off the bottom. Leg B: 1/4 inch hole, 1 1/4 inches from the end, 3/4 inch up.","group":"L_brackets:23","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":23,"part_type":"bracket","base":{"length_a_in":5.625,"length_b_in":5.625,"height_in":5.0,"thickness_in":0.25},"features":[{"type":"hole","face":"leg_b","from_end_in":1.125,"across_in":4.0,"diameter_in":0.75},{"type":"hole","face":"leg_a","from_end_in":4.75,"across_in":null,"diameter_in":0.375},{"type":"hole","face":"leg_a","from_end_in":1.625,"across_in":2.625,"diameter_in":0.625}],"text":"Three holes. Leg B: a 3/4 inch hole 1 1/8 inches from the end and 4 inches up from the bottom. Leg A: a 3/8 inch hole 4 3/4 inches from the end, centered across the width, and a 5/8 inch hole 1 5/8 inches from the end and 2 5/8 inches up.","group":"L_brackets:48","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":24,"part_type":"bracket","base":{"length_a_in":1.5,"length_b_in":1.5,"height_in":0.9375,"thickness_in":0.05},"features":[{"type":"hole","face":"leg_b","from_end_in":0.875,"across_in":null,"diameter_in":0.3125},{"type":"hole","face":"leg_a","from_end_in":0.5,"across_in":0.5,"diameter_in":0.625}],"text":"Cut a 5/16 diameter hole in leg B, 7/8\" from the end, centered. Cut a 5/8 diameter hole in leg A, 1/2\" from the end and 1/2\" up from the bottom.","group":"L_brackets:4","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":25,"part_type":"bracket","base":{"length_a_in":1.375,"length_b_in":1.375,"height_in":0.75,"thickness_in":0.04},"features":[{"type":"hole","face":"leg_b","from_end_in":null,"across_in":null,"diameter_in":0.1875},{"type":"hole","face":"leg_a","from_end_in":0.875,"across_in":null,"diameter_in":0.5}],"text":"I want a 3/16 inch hole dead center in leg B and a 1/2 inch hole in leg A that's 7/8 inch from the end and centered across the width.","group":"L_brackets:29","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":26,"part_type":"bracket","base":{"length_a_in":3.5,"length_b_in":3.5,"height_in":0.75,"thickness_in":0.12},"features":[{"type":"hole","face":"leg_a","from_end_in":1.125,"across_in":null,"diameter_in":0.125}],"text":"Add a 1/8 inch hole to leg A, 1 1/8 inch from the end, in the middle of the width.","group":"L_brackets:11","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":27,"part_type":"bracket","base":{"length_a_in":3.5625,"length_b_in":3.5625,"height_in":0.8125,"thickness_in":0.24},"features":[{"type":"hole","face":"leg_a","from_end_in":0.75,"across_in":0.375,"diameter_in":0.5}],"text":"half inch hole in leg A, 3/4 inch from the end, 3/8 inch from the bottom","group":"L_brackets:57","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":28,"part_type":"bracket","base":{"length_a_in":6.5,"length_b_in":6.5,"height_in":1.25,"thickness_in":0.11},"features":[{"type":"hole","face":"leg_a","from_end_in":3.75,"across_in":null,"diameter_in":0.375},{"type":"hole","face":"leg_a","from_end_in":5.5,"across_in":0.5,"diameter_in":0.625},{"type":"hole","face":"leg_b","from_end_in":1.875,"across_in":null,"diameter_in":0.5}],"text":"Leg A: 3/8 inch hole 3 3/4 inches from the end, centered across, and a 5/8 inch hole 5 1/2 inches from the end, 1/2 inch up. Leg B: a 1/2 inch hole 1 7/8 inches from the end, centered across.","group":"L_brackets:54","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":29,"part_type":"bracket","base":{"length_a_in":1.0,"length_b_in":1.0,"height_in":0.5,"thickness_in":0.08},"features":[{"type":"hole","face":"leg_a","from_end_in":0.375,"across_in":0.25,"diameter_in":0.1875},{"type":"hole","face":"leg_b","from_end_in":0.375,"across_in":0.25,"diameter_in":0.1875},{"type":"hole","face":"leg_b","from_end_in":0.625,"across_in":0.25,"diameter_in":0.1875}],"text":"Three 3/16 inch holes, all 1/4 inch up from the bottom. One in leg A 3/8 inch from the end. Two in leg B, at 3/8 inch and 5/8 inch from the end.","group":"L_brackets:5","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":30,"part_type":"bracket","base":{"length_a_in":2.0,"length_b_in":3.5625,"height_in":0.8125,"thickness_in":0.24},"features":[{"type":"hole","face":"leg_a","from_end_in":0.375,"across_in":0.375,"diameter_in":0.5}],"text":"Put a 1/2 inch hole in the short leg, 3/8 inch from the end and 3/8 inch up from the bottom.","group":"L_brackets:65","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":31,"part_type":"bracket","base":{"length_a_in":4.75,"length_b_in":4.75,"height_in":1.25,"thickness_in":0.11},"features":[{"type":"hole","face":"leg_b","from_end_in":1.875,"across_in":0.375,"diameter_in":0.5}],"text":"Leg B, one 1/2 inch hole, 1 7/8 inches from the end, 3/8 inch up.","group":"L_brackets:53","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":32,"part_type":"bracket","base":{"length_a_in":2.5625,"length_b_in":4.75,"height_in":0.8125,"thickness_in":0.24},"features":[{"type":"hole","face":"leg_b","from_end_in":3.375,"across_in":0.375,"diameter_in":0.125},{"type":"hole","face":"leg_b","from_end_in":0.75,"across_in":0.375,"diameter_in":0.5}],"text":"On the long leg: a 1/8 inch hole 3 3/8 inches from the end and a 1/2 inch hole 3/4 inch from the end, both 3/8 inch up from the bottom.","group":"L_brackets:66","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":33,"part_type":"bracket","base":{"length_a_in":2.9528,"length_b_in":1.9685,"height_in":1.9685,"thickness_in":0.1181},"features":[{"type":"hole","face":"leg_a","from_end_in":1.875,"across_in":0.75,"diameter_in":0.625},{"type":"hole","face":"leg_a","from_end_in":0.75,"across_in":1.0,"diameter_in":0.625},{"type":"hole","face":"leg_b","from_end_in":0.875,"across_in":1.625,"diameter_in":0.3125}],"text":"Long leg: two 5/8 inch holes, one 1 7/8 inches from the end and 3/4 inch up, the other 3/4 inch from the end and 1 inch up. Short leg: one 5/16 inch hole 7/8 inch from the end and 1 5/8 inches up.","group":"L_brackets:91","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":34,"part_type":"bracket","base":{"length_a_in":8.25,"length_b_in":8.25,"height_in":5.0,"thickness_in":0.25},"features":[{"type":"hole","face":"leg_a","from_end_in":4.125,"across_in":1.125,"diameter_in":0.25}],"text":"Drill a 1/4 inch hole in leg A, 4 1/8 inches from the end, 1 1/8 inches up from the bottom face.","group":"L_brackets:50","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":35,"part_type":"bracket","base":{"length_a_in":1.5,"length_b_in":1.5,"height_in":1.5,"thickness_in":0.06},"features":[{"type":"hole","face":"leg_a","from_end_in":0.625,"across_in":null,"diameter_in":0.125},{"type":"hole","face":"leg_b","from_end_in":0.625,"across_in":0.875,"diameter_in":0.375}],"text":"Leg A gets a 1/8 inch hole 5/8 inch from the end, centered across. Leg B gets a 3/8 inch hole 5/8 inch from the end and 7/8 inch up.","group":"L_brackets:34","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":36,"part_type":"bracket","base":{"length_a_in":2.9528,"length_b_in":2.3622,"height_in":2.9528,"thickness_in":0.1181},"features":[{"type":"hole","face":"leg_b","from_end_in":0.375,"across_in":0.375,"diameter_in":0.1875}],"text":"3/16 inch hole in the shorter leg, 3/8 inch from the end and 3/8 inch up from the bottom.","group":"L_brackets:92","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":37,"part_type":"bracket","base":{"length_a_in":9.375,"length_b_in":11.875,"height_in":0.8125,"thickness_in":0.31},"features":[{"type":"hole","face":"leg_b","from_end_in":2.875,"across_in":0.5,"diameter_in":0.25}],"text":"quarter inch hole in the longer leg, 2 7/8 inches from the end, 1/2 inch up","group":"L_brackets:64","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":38,"part_type":"bracket","base":{"length_a_in":1.5,"length_b_in":1.5,"height_in":0.5,"thickness_in":0.07},"features":[{"type":"hole","face":"leg_b","from_end_in":1.0,"across_in":null,"diameter_in":0.125},{"type":"hole","face":"leg_a","from_end_in":0.25,"across_in":0.25,"diameter_in":0.125}],"text":"Two 1/8 inch holes. Leg B: 1 inch from the end, centered across. Leg A: 1/4 inch from the end and 1/4 inch up from the bottom.","group":"L_brackets:18","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":39,"part_type":"bracket","base":{"length_a_in":2.5,"length_b_in":2.5,"height_in":5.0,"thickness_in":0.12},"features":[{"type":"hole","face":"leg_a","from_end_in":1.875,"across_in":3.0,"diameter_in":0.625},{"type":"hole","face":"leg_a","from_end_in":2.125,"across_in":null,"diameter_in":0.1875}],"text":"Leg A: 5/8 inch hole at 1 7/8 inches from the end and 3 inches up; and a 3/16 inch hole 2 1/8 inches from the end, centered across.","group":"L_brackets:46","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":40,"part_type":"bracket","base":{"length_a_in":1.375,"length_b_in":1.375,"height_in":2.75,"thickness_in":0.06},"features":[{"type":"hole","face":"leg_a","from_end_in":1.125,"across_in":null,"diameter_in":0.125},{"type":"hole","face":"leg_a","from_end_in":null,"across_in":0.75,"diameter_in":0.3125}],"text":"On leg A, a 1/8 inch hole 1 1/8 inches from the end (centered across the width), and a 5/16 inch hole centered along the leg at 3/4 inch up.","group":"L_brackets:37","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":41,"part_type":"rect_container","base":{"length_a_in":4.5,"length_b_in":3.13,"height_in":3.29,"thickness_in":0.092},"features":[{"type":"hole","face":"left","along_in":1.125,"up_in":1.875,"diameter_in":0.5}],"text":"Cut a 1/2 inch hole in the left wall, 1 1/8 inch from its left end and 1 7/8 inches up from the bottom.","group":"Rect_container:40","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":42,"part_type":"rect_container","base":{"length_a_in":7.5,"length_b_in":4.5,"height_in":4.95,"thickness_in":0.132},"features":[{"type":"hole","face":"left","along_in":null,"up_in":null,"diameter_in":0.75},{"type":"hole","face":"right","along_in":3.625,"up_in":null,"diameter_in":0.75}],"text":"I want two 3/4 inch holes. One goes dead center in the left wall. The other is in the right wall, 3 5/8 inches from the left end, centered vertically.","group":"Rect_container:70","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":43,"part_type":"rect_container","base":{"length_a_in":5.0,"length_b_in":2.98,"height_in":1.19,"thickness_in":0.111},"features":[{"type":"hole","face":"left","along_in":1.625,"up_in":null,"diameter_in":0.1875}],"text":"3/16 inch hole in the left side, 1.625 inches from the left edge, halfway up the wall.","group":"Rect_container:44","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":44,"part_type":"rect_container","base":{"length_a_in":8.0,"length_b_in":3.78,"height_in":4.08,"thickness_in":0.133},"features":[{"type":"hole","face":"left","along_in":0.875,"up_in":2.25,"diameter_in":0.5}],"text":"Drill a half inch hole through the left wall 7/8 inch from the left end and 2 1/4 inches above the bottom.","group":"Rect_container:86","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":45,"part_type":"rect_container","base":{"length_a_in":3.5,"length_b_in":3.18,"height_in":1.18,"thickness_in":0.089},"features":[{"type":"hole","face":"right","along_in":1.625,"up_in":0.625,"diameter_in":0.375},{"type":"hole","face":"back","along_in":null,"up_in":0.625,"diameter_in":0.625}],"text":"Two holes: a 3/8 inch hole in the right wall 1 5/8 inches from the left end and 5/8 inch up, and a 5/8 inch hole in the back wall centered side to side at 5/8 inch up.","group":"Rect_container:31","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":46,"part_type":"rect_container","base":{"length_a_in":6.0,"length_b_in":4.0,"height_in":2.0,"thickness_in":0.11},"features":[{"type":"hole","face":"right","along_in":2.875,"up_in":1.375,"diameter_in":0.625}],"text":"Put a 5/8\" hole in the right wall, 2 7/8\" from the left end, 1 3/8\" up from the bottom.","group":"Rect_container:57","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":47,"part_type":"rect_container","base":{"length_a_in":2.0,"length_b_in":1.97,"height_in":2.63,"thickness_in":0.07},"features":[{"type":"hole","face":"back","along_in":null,"up_in":1.5,"diameter_in":0.25},{"type":"hole","face":"back","along_in":1.25,"up_in":null,"diameter_in":0.125}],"text":"Back wall, two holes. First a 1/4 inch hole centered left to right, 1.5 inches up. Second a 1/8 inch hole 1 1/4 inches from the left end, centered vertically.","group":"Rect_container:10","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":48,"part_type":"rect_container","base":{"length_a_in":6.0,"length_b_in":2.0,"height_in":4.0,"thickness_in":0.09},"features":[{"type":"hole","face":"right","along_in":null,"up_in":3.5,"diameter_in":0.125}],"text":"small 1/8 inch hole on the right side, centered horizontally, 3.5 inches up from the bottom","group":"Rect_container:90","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":49,"part_type":"rect_container","base":{"length_a_in":4.0,"length_b_in":1.94,"height_in":2.4,"thickness_in":0.091},"features":[{"type":"hole","face":"front","along_in":0.875,"up_in":0.625,"diameter_in":0.5},{"type":"hole","face":"back","along_in":null,"up_in":1.125,"diameter_in":0.125}],"text":"Add a 1/2 inch hole to the front wall, 7/8 inch from the left end and 5/8 inch up. Also add a 1/8 inch hole in the back wall, centered across, 1 1/8 inches up.","group":"Rect_container:80","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":50,"part_type":"rect_container","base":{"length_a_in":2.5,"length_b_in":1.38,"height_in":0.78,"thickness_in":0.063},"features":[{"type":"hole","face":"right","along_in":0.875,"up_in":0.5,"diameter_in":0.3125}],"text":"5/16 inch hole in the right wall, 7/8 inch from the left end, half inch up from the bottom","group":"Rect_container:11","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":51,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":2.91,"height_in":3.8,"thickness_in":0.087},"features":[{"type":"hole","face":"left","along_in":2.375,"up_in":3.375,"diameter_in":0.5}],"text":"Left wall: one 1/2 inch hole, 2 3/8 inches from the left end and 3 3/8 inches up.","group":"Rect_container:27","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":52,"part_type":"rect_container","base":{"length_a_in":7.0,"length_b_in":3.15,"height_in":3.28,"thickness_in":0.12},"features":[{"type":"hole","face":"right","along_in":1.75,"up_in":1.875,"diameter_in":0.125}],"text":"Drill a 1/8 inch hole in the right side, 1 3/4 inch in from the left end and 1 7/8 inches up.","group":"Rect_container:65","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":53,"part_type":"rect_container","base":{"length_a_in":6.0,"length_b_in":4.69,"height_in":4.64,"thickness_in":0.123},"features":[{"type":"hole","face":"front","along_in":0.625,"up_in":2.25,"diameter_in":0.75},{"type":"hole","face":"back","along_in":1.875,"up_in":2.125,"diameter_in":0.625},{"type":"hole","face":"left","along_in":3.75,"up_in":2.5,"diameter_in":0.25}],"text":"Three holes. Front wall: 3/4 inch hole, 5/8 inch from the left end, 2 1/4 inches up. Back wall: 5/8 inch hole, 1 7/8 inches from the left end, 2 1/8 inches up. Left wall: 1/4 inch hole, 3 3/4 inches from the left end, 2 1/2 inches up.","group":"Rect_container:59","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":54,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":2.54,"height_in":3.04,"thickness_in":0.08},"features":[{"type":"hole","face":"right","along_in":0.625,"up_in":0.75,"diameter_in":0.125},{"type":"hole","face":"front","along_in":0.375,"up_in":0.5,"diameter_in":0.25},{"type":"hole","face":"front","along_in":1.875,"up_in":null,"diameter_in":0.5}],"text":"On the right wall add a 1/8 inch hole 5/8 inch from the left end and 3/4 inch up. On the front wall add a 1/4 inch hole 3/8 inch from the left end and 1/2 inch up, and a 1/2 inch hole 1 7/8 inches from the left end, centered vertically.","group":"Rect_container:79","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":55,"part_type":"rect_container","base":{"length_a_in":2.5,"length_b_in":2.33,"height_in":1.72,"thickness_in":0.062},"features":[{"type":"hole","face":"back","along_in":1.125,"up_in":0.875,"diameter_in":0.75}],"text":"I need a 3/4 inch hole in the back wall, 1 1/8 inches from the left end and 7/8 inch up.","group":"Rect_container:15","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":56,"part_type":"rect_container","base":{"length_a_in":4.0,"length_b_in":2.72,"height_in":3.16,"thickness_in":0.083},"features":[{"type":"hole","face":"left","along_in":1.375,"up_in":1.875,"diameter_in":0.75},{"type":"hole","face":"right","along_in":1.875,"up_in":null,"diameter_in":0.1875}],"text":"Left wall gets a 3/4 inch hole 1 3/8 inches from the left end and 1 7/8 inches up. Right wall gets a 3/16 inch hole 1 7/8 inches from the left end, centered vertically.","group":"Rect_container:33","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":57,"part_type":"rect_container","base":{"length_a_in":3.5,"length_b_in":2.99,"height_in":3.84,"thickness_in":0.081},"features":[{"type":"hole","face":"left","along_in":0.625,"up_in":1.375,"diameter_in":0.625},{"type":"hole","face":"right","along_in":2.0,"up_in":3.0,"diameter_in":0.125},{"type":"hole","face":"back","along_in":1.75,"up_in":null,"diameter_in":0.625}],"text":"Three holes: a 5/8 inch one in the left wall at 5/8 inch from the left end and 1 3/8 inches up; a 1/8 inch one in the right wall 2 inches from the left end and 3 inches up; and a 5/8 inch one in the back wall 1 3/4 inches from the left end, centered vertically.","group":"Rect_container:30","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":58,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":1.47,"height_in":0.67,"thickness_in":0.075},"features":[{"type":"hole","face":"left","along_in":0.375,"up_in":null,"diameter_in":0.25},{"type":"hole","face":"front","along_in":0.875,"up_in":0.375,"diameter_in":0.25},{"type":"hole","face":"right","along_in":0.375,"up_in":0.375,"diameter_in":0.1875}],"text":"left wall: 1/4 inch hole, 3/8 inch from the left end, centered vertically. front wall: 1/4 inch hole, 7/8 inch from the left end, 3/8 inch up. right wall: 3/16 inch hole, 3/8 inch from the left end, 3/8 inch up.","group":"Rect_container:20","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":59,"part_type":"rect_container","base":{"length_a_in":2.5,"length_b_in":1.43,"height_in":1.69,"thickness_in":0.082},"features":[{"type":"hole","face":"right","along_in":1.0,"up_in":null,"diameter_in":0.3125}],"text":"A 5/16 inch hole in the right wall, 1 inch from the left end, vertically centered.","group":"Rect_container:12","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":60,"part_type":"rect_container","base":{"length_a_in":4.5,"length_b_in":2.6,"height_in":2.11,"thickness_in":0.098},"features":[{"type":"hole","face":"front","along_in":2.375,"up_in":0.875,"diameter_in":0.3125},{"type":"hole","face":"left","along_in":1.5,"up_in":1.375,"diameter_in":0.5},{"type":"hole","face":"right","along_in":0.75,"up_in":0.625,"diameter_in":0.625}],"text":"Front wall: 5/16 inch hole 2 3/8 inches from the left end, 7/8 inch up. Left wall: 1/2 inch hole 1.5 inches from the left end, 1 3/8 inches up. Right wall: 5/8 inch hole 3/4 inch from the left end, 5/8 inch up.","group":"Rect_container:36","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":61,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":2.81,"height_in":4.28,"thickness_in":0.101},"features":[{"type":"hole","face":"back","along_in":0.75,"up_in":0.625,"diameter_in":0.75},{"type":"hole","face":"back","along_in":1.875,"up_in":3.25,"diameter_in":0.3125},{"type":"hole","face":"left","along_in":0.75,"up_in":null,"diameter_in":0.375}],"text":"Back wall gets two holes: 3/4 inch, 3/4 inch from the left end and 5/8 inch up, and 5/16 inch, 1 7/8 inches from the left end and 3 1/4 inches up. The left wall gets one 3/8 inch hole 3/4 inch from the left end, centered vertically.","group":"Rect_container:25","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":62,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":2.19,"height_in":2.18,"thickness_in":0.074},"features":[{"type":"hole","face":"left","along_in":0.875,"up_in":1.75,"diameter_in":0.1875}],"text":"3/16 inch hole in the left wall, 7/8 inch from the left end, 1 3/4 inches up from the bottom","group":"Rect_container:24","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":63,"part_type":"rect_container","base":{"length_a_in":5.5,"length_b_in":3.69,"height_in":5.68,"thickness_in":0.119},"features":[{"type":"hole","face":"right","along_in":null,"up_in":null,"diameter_in":0.75},{"type":"hole","face":"back","along_in":1.75,"up_in":null,"diameter_in":0.1875}],"text":"Put a 3/4 inch hole in the exact center of the right wall, and a 3/16 inch hole in the back wall, 1 3/4 inches from the left end and centered vertically.","group":"Rect_container:53","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":64,"part_type":"rect_container","base":{"length_a_in":2.5,"length_b_in":1.74,"height_in":1.62,"thickness_in":0.065},"features":[{"type":"hole","face":"front","along_in":null,"up_in":null,"diameter_in":0.75}],"text":"3/4 inch hole dead center on the front wall","group":"Rect_container:13","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":65,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":2.04,"height_in":1.14,"thickness_in":0.07},"features":[{"type":"hole","face":"right","along_in":1.125,"up_in":0.625,"diameter_in":0.625},{"type":"hole","face":"back","along_in":1.875,"up_in":0.75,"diameter_in":0.25},{"type":"hole","face":"left","along_in":0.875,"up_in":0.625,"diameter_in":0.125}],"text":"Right wall: a 5/8 inch hole, 1 1/8 inches from the left end and 5/8 inch up. Back wall: a 1/4 inch hole, 1 7/8 inches from the left end and 3/4 inch up. Left wall: a 1/8 inch hole, 7/8 inch from the left end and 5/8 inch up.","group":"Rect_container:22","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":66,"part_type":"rect_container","base":{"length_a_in":2.0,"length_b_in":1.33,"height_in":1.33,"thickness_in":0.059},"features":[{"type":"hole","face":"back","along_in":1.0,"up_in":0.625,"diameter_in":0.625}],"text":"add a 5/8 inch hole to the back, 1 inch from the left end and 5/8 inch up","group":"Rect_container:9","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":67,"part_type":"rect_container","base":{"length_a_in":5.0,"length_b_in":4.69,"height_in":5.16,"thickness_in":0.106},"features":[{"type":"hole","face":"back","along_in":4.5,"up_in":2.0,"diameter_in":0.375}],"text":"Back wall, 3/8 inch hole, 4 1/2 inches from the left end and 2 inches up from the bottom.","group":"Rect_container:49","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":68,"part_type":"rect_container","base":{"length_a_in":2.0,"length_b_in":1.01,"height_in":0.5,"thickness_in":0.066},"features":[{"type":"hole","face":"left","along_in":0.5,"up_in":null,"diameter_in":0.125},{"type":"hole","face":"back","along_in":0.375,"up_in":0.25,"diameter_in":0.125},{"type":"hole","face":"back","along_in":1.125,"up_in":0.25,"diameter_in":0.125}],"text":"Three 1/8 inch holes. One in the left wall, 1/2 inch from the left end, centered vertically. Two in the back wall, both 1/4 inch up, at 3/8 inch and 1 1/8 inches from the left end.","group":"Rect_container:7","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":69,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":2.16,"height_in":1.62,"thickness_in":0.067},"features":[{"type":"hole","face":"left","along_in":1.375,"up_in":1.125,"diameter_in":0.625}],"text":"Drill a 5/8 inch hole in the left wall at 1 3/8 inches from the left end and 1 1/8 inches up.","group":"Rect_container:23","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":70,"part_type":"rect_container","base":{"length_a_in":4.5,"length_b_in":2.88,"height_in":1.67,"thickness_in":0.092},"features":[{"type":"hole","face":"front","along_in":2.625,"up_in":1.125,"diameter_in":0.625}],"text":"Front wall hole: 5/8 inch diameter, 2 5/8 inches from the left end, 1 1/8 inches up from the bottom.","group":"Rect_container:37","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":71,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":2.85,"height_in":2.65,"thickness_in":0.086},"features":[{"type":"hole","face":"left","along_in":1.375,"up_in":1.875,"diameter_in":0.125}],"text":"small hole, 1/8 inch, in the left wall 1 3/8 inches from the left end and 1 7/8 inches up","group":"Rect_container:26","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":72,"part_type":"rect_container","base":{"length_a_in":1.5,"length_b_in":1.49,"height_in":1.02,"thickness_in":0.054},"features":[{"type":"hole","face":"left","along_in":0.5,"up_in":null,"diameter_in":0.5},{"type":"hole","face":"front","along_in":0.5,"up_in":0.5,"diameter_in":0.625}],"text":"Left wall: a 1/2 inch hole 1/2 inch from the left end, centered vertically. Front wall: a 5/8 inch hole 1/2 inch from the left end and 1/2 inch up.","group":"Rect_container:6","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":73,"part_type":"rect_container","base":{"length_a_in":6.5,"length_b_in":3.88,"height_in":5.32,"thickness_in":0.123},"features":[{"type":"hole","face":"back","along_in":5.875,"up_in":4.875,"diameter_in":0.625},{"type":"hole","face":"back","along_in":3.875,"up_in":1.125,"diameter_in":0.625}],"text":"Two 5/8 inch holes in the back wall: one 5 7/8 inches from the left end and 4 7/8 inches up, the other 3 7/8 inches from the left end and 1 1/8 inches up.","group":"Rect_container:62","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":74,"part_type":"rect_container","base":{"length_a_in":7.0,"length_b_in":6.3,"height_in":6.93,"thickness_in":0.135},"features":[{"type":"hole","face":"front","along_in":4.875,"up_in":0.375,"diameter_in":0.25}],"text":"front wall, 1/4 inch hole, 4 7/8 inches from the left end, 3/8 inch up","group":"Rect_container:68","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":75,"part_type":"rect_container","base":{"length_a_in":3.6,"length_b_in":6.3,"height_in":2.0,"thickness_in":0.14},"features":[{"type":"hole","face":"front","along_in":0.75,"up_in":1.375,"diameter_in":0.375}],"text":"Add a 3/8 inch hole to the front wall, 3/4 inch from the left end and 1 3/8 inches above the bottom.","group":"Rect_container:89","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":76,"part_type":"rect_container","base":{"length_a_in":5.0,"length_b_in":4.5,"height_in":2.1,"thickness_in":0.097},"features":[{"type":"hole","face":"back","along_in":2.875,"up_in":null,"diameter_in":0.625},{"type":"hole","face":"front","along_in":3.0,"up_in":1.375,"diameter_in":0.375},{"type":"hole","face":"right","along_in":1.125,"up_in":null,"diameter_in":0.25}],"text":"Back wall: 5/8 inch hole 2 7/8 inches from the left end, centered vertically. Front wall: 3/8 inch hole 3 inches from the left end and 1 3/8 inches up. Right wall: 1/4 inch hole 1 1/8 inches from the left end, centered vertically.","group":"Rect_container:83","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":77,"part_type":"rect_container","base":{"length_a_in":1.5,"length_b_in":1.0,"height_in":1.04,"thickness_in":0.07},"features":[{"type":"hole","face":"back","along_in":0.875,"up_in":0.25,"diameter_in":0.125},{"type":"hole","face":"back","along_in":0.5,"up_in":null,"diameter_in":0.25},{"type":"hole","face":"front","along_in":0.875,"up_in":0.5,"diameter_in":0.125}],"text":"Three holes. On the back wall a 1/8 inch hole 7/8 inch from the left end and 1/4 inch up, and a 1/4 inch hole 1/2 inch from the left end, centered vertically. On the front wall a 1/8 inch hole 7/8 inch from the left end and 1/2 inch up.","group":"Rect_container:3","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":78,"part_type":"rect_container","base":{"length_a_in":3.5,"length_b_in":2.82,"height_in":3.54,"thickness_in":0.078},"features":[{"type":"hole","face":"left","along_in":2.125,"up_in":1.0,"diameter_in":0.25},{"type":"hole","face":"left","along_in":null,"up_in":1.375,"diameter_in":0.3125},{"type":"hole","face":"back","along_in":1.5,"up_in":1.75,"diameter_in":0.25}],"text":"Left wall: a 1/4 inch hole at 2 1/8 inches from the left end and 1 inch up, plus a 5/16 inch hole centered left to right at 1 3/8 inches up. Back wall: a 1/4 inch hole 1.5 inches from the left end and 1 3/4 inches up.","group":"Rect_container:29","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":79,"part_type":"rect_container","base":{"length_a_in":7.5,"length_b_in":5.25,"height_in":3.15,"thickness_in":0.131},"features":[{"type":"hole","face":"back","along_in":1.75,"up_in":1.375,"diameter_in":0.1875},{"type":"hole","face":"front","along_in":0.875,"up_in":2.375,"diameter_in":0.25},{"type":"hole","face":"front","along_in":null,"up_in":2.5,"diameter_in":0.1875}],"text":"Back wall: 3/16 inch hole, 1 3/4 inches from the left end, 1 3/8 inches up. Front wall: 1/4 inch hole, 7/8 inch from the left end, 2 3/8 inches up; and a 3/16 inch hole centered left to right, 2 1/2 inches up.","group":"Rect_container:71","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":80,"part_type":"rect_container","base":{"length_a_in":5.5,"length_b_in":4.88,"height_in":3.76,"thickness_in":0.105},"features":[{"type":"hole","face":"back","along_in":1.875,"up_in":null,"diameter_in":0.1875}],"text":"3/16 inch hole in the back wall, 1 7/8 inches from the left end, vertically centered.","group":"Rect_container:54","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":81,"part_type":"cyl_container","base":{"id_in":4.5,"height_in":9.45,"thickness_in":0.1571},"features":[{"type":"hole","face":"side","angle_deg":225.0,"up_in":7.75,"diameter_in":0.3125}],"text":"Add a 5/16 inch hole at 225 degrees around the cup, 7.75 inches up from the bottom.","group":"Cyl_container:54","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":82,"part_type":"cyl_container","base":{"id_in":1.0,"height_in":1.37,"thickness_in":0.0523},"features":[{"type":"hole","face":"side","angle_deg":165.0,"up_in":0.5,"diameter_in":0.625}],"text":"5/8 inch hole at an angle of 165 degrees, half an inch up.","group":"Cyl_container:12","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":83,"part_type":"cyl_container","base":{"id_in":1.75,"height_in":4.73,"thickness_in":0.0931},"features":[{"type":"hole","face":"side","angle_deg":345.0,"up_in":0.625,"diameter_in":0.25}],"text":"Drill a 1/4 inch hole at 345\u00b0, 5/8 inch above the bottom.","group":"Cyl_container:28","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":84,"part_type":"cyl_container","base":{"id_in":3.5,"height_in":3.38,"thickness_in":0.0921},"features":[{"type":"hole","face":"side","angle_deg":30.0,"up_in":1.75,"diameter_in":0.625}],"text":"Put a 5/8 inch hole at 30 degrees and 1.75 inches up from the bottom of the cup.","group":"Cyl_container:42","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":85,"part_type":"cyl_container","base":{"id_in":6.5,"height_in":5.99,"thickness_in":0.1197},"features":[{"type":"hole","face":"side","angle_deg":315.0,"up_in":5.25,"diameter_in":0.1875}],"text":"3/16 inch hole at 315 degrees, 5 1/4 inches up.","group":"Cyl_container:67","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":86,"part_type":"cyl_container","base":{"id_in":10.0,"height_in":14.0,"thickness_in":0.18},"features":[{"type":"hole","face":"side","angle_deg":255.0,"up_in":7.125,"diameter_in":0.375}],"text":"Please add a 3/8 inch hole at 255 degrees, 7 1/8 inches from the bottom.","group":"Cyl_container:86","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":87,"part_type":"cyl_container","base":{"id_in":2.0,"height_in":3.07,"thickness_in":0.0813},"features":[{"type":"hole","face":"side","angle_deg":135.0,"up_in":2.75,"diameter_in":0.3125}],"text":"a 5/16 hole at 135 degrees, 2.75 inches above the bottom","group":"Cyl_container:31","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":88,"part_type":"cyl_container","base":{"id_in":6.5,"height_in":8.27,"thickness_in":0.132},"features":[{"type":"hole","face":"side","angle_deg":270.0,"up_in":0.75,"diameter_in":0.25}],"text":"1/4 inch hole on the front of the cup, 3/4 inch up from the bottom.","group":"Cyl_container:89","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":89,"part_type":"cyl_container","base":{"id_in":2.25,"height_in":0.81,"thickness_in":0.0606},"features":[{"type":"hole","face":"side","angle_deg":180.0,"up_in":0.375,"diameter_in":0.25},{"type":"hole","face":"side","angle_deg":195.0,"up_in":0.375,"diameter_in":0.125}],"text":"Two holes both 3/8 inch up from the bottom: a 1/4 inch hole on the left side, and a 1/8 inch hole at 195 degrees.","group":"Cyl_container:32","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":90,"part_type":"cyl_container","base":{"id_in":2.5,"height_in":4.97,"thickness_in":0.1014},"features":[{"type":"hole","face":"side","angle_deg":90.0,"up_in":4.0,"diameter_in":0.25},{"type":"hole","face":"side","angle_deg":210.0,"up_in":1.875,"diameter_in":0.375}],"text":"Add a 1/4 inch hole in the back, 4 inches up, and a 3/8 inch hole at 210 degrees, 1 7/8 inches up.","group":"Cyl_container:35","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":91,"part_type":"cyl_container","base":{"id_in":7.0,"height_in":14.0,"thickness_in":0.18},"features":[{"type":"hole","face":"side","angle_deg":195.0,"up_in":null,"diameter_in":0.3125},{"type":"hole","face":"side","angle_deg":120.0,"up_in":8.75,"diameter_in":0.375},{"type":"hole","face":"side","angle_deg":240.0,"up_in":10.875,"diameter_in":0.25}],"text":"Three holes. A 5/16 inch hole at 195 degrees, centered vertically. A 3/8 inch hole at 120 degrees, 8.75 inches up. A 1/4 inch hole at 240 degrees, 10 7/8 inches up.","group":"Cyl_container:70","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":92,"part_type":"cyl_container","base":{"id_in":4.5,"height_in":12.04,"thickness_in":0.1779},"features":[{"type":"hole","face":"side","angle_deg":165.0,"up_in":null,"diameter_in":0.375},{"type":"hole","face":"side","angle_deg":105.0,"up_in":2.5,"diameter_in":0.5}],"text":"3/8 inch hole at 165 degrees, halfway up the cup, and a half inch hole at 105 degrees, 2.5 inches up.","group":"Cyl_container:59","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":93,"part_type":"cyl_container","base":{"id_in":1.75,"height_in":2.01,"thickness_in":0.0658},"features":[{"type":"hole","face":"side","angle_deg":210.0,"up_in":null,"diameter_in":0.5}],"text":"Put a half inch hole at 210 degrees, centered vertically.","group":"Cyl_container:26","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":94,"part_type":"cyl_container","base":{"id_in":1.0,"height_in":0.5,"thickness_in":0.0519},"features":[{"type":"hole","face":"side","angle_deg":180.0,"up_in":0.25,"diameter_in":0.125},{"type":"hole","face":"side","angle_deg":60.0,"up_in":0.25,"diameter_in":0.125},{"type":"hole","face":"side","angle_deg":135.0,"up_in":0.25,"diameter_in":0.125}],"text":"Three 1/8 inch holes all a quarter inch up from the bottom, at 60, 135 and 180 degrees.","group":"Cyl_container:8","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":95,"part_type":"cyl_container","base":{"id_in":4.5,"height_in":10.48,"thickness_in":0.1604},"features":[{"type":"hole","face":"side","angle_deg":150.0,"up_in":6.5,"diameter_in":0.25},{"type":"hole","face":"side","angle_deg":120.0,"up_in":7.625,"diameter_in":0.25}],"text":"Two 1/4 inch holes: one at 150 degrees, 6.5 inches up, and one at 120 degrees, 7 5/8 inches up.","group":"Cyl_container:55","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":96,"part_type":"cyl_container","base":{"id_in":2.25,"height_in":5.72,"thickness_in":0.1093},"features":[{"type":"hole","face":"side","angle_deg":105.0,"up_in":null,"diameter_in":0.625}],"text":"5/8 inch hole at 105\u00b0, mid-height","group":"Cyl_container:34","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":97,"part_type":"cyl_container","base":{"id_in":7.5,"height_in":11.63,"thickness_in":0.1682},"features":[{"type":"hole","face":"side","angle_deg":15.0,"up_in":10.5,"diameter_in":0.3125}],"text":"Drill a 5/16 inch hole at 15 degrees, 10.5 inches up from the bottom.","group":"Cyl_container:71","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":98,"part_type":"cyl_container","base":{"id_in":8.5,"height_in":14.0,"thickness_in":0.18},"features":[{"type":"hole","face":"side","angle_deg":315.0,"up_in":3.75,"diameter_in":0.125}],"text":"1/8 inch hole at an angle of 315 degrees, 3 3/4 inches from the bottom.","group":"Cyl_container:78","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":99,"part_type":"cyl_container","base":{"id_in":4.0,"height_in":7.73,"thickness_in":0.1272},"features":[{"type":"hole","face":"side","angle_deg":165.0,"up_in":null,"diameter_in":0.5},{"type":"hole","face":"side","angle_deg":195.0,"up_in":null,"diameter_in":0.375}],"text":"Two holes, both centered vertically: a half inch hole at 165 degrees and a 3/8 inch hole at 195 degrees.","group":"Cyl_container:49","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":100,"part_type":"cyl_container","base":{"id_in":1.25,"height_in":2.36,"thickness_in":0.0691},"features":[{"type":"hole","face":"side","angle_deg":150.0,"up_in":null,"diameter_in":0.75}],"text":"3/4 inch hole at 150 degrees, halfway up.","group":"Cyl_container:19","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":101,"part_type":"cyl_container","base":{"id_in":1.5,"height_in":0.54,"thickness_in":0.0638},"features":[{"type":"hole","face":"side","angle_deg":225.0,"up_in":0.375,"diameter_in":0.125},{"type":"hole","face":"side","angle_deg":165.0,"up_in":0.375,"diameter_in":0.125}],"text":"Two 1/8 inch holes, 3/8 inch up from the bottom, at 165 and 225 degrees.","group":"Cyl_container:20","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":102,"part_type":"cyl_container","base":{"id_in":0.75,"height_in":1.79,"thickness_in":0.06},"features":[{"type":"hole","face":"side","angle_deg":300.0,"up_in":1.5,"diameter_in":0.375},{"type":"hole","face":"side","angle_deg":0.0,"up_in":0.875,"diameter_in":0.5}],"text":"A 3/8 inch hole at 300 degrees, 1.5 inches up, and a half inch hole on the right side, 7/8 inch up.","group":"Cyl_container:7","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":103,"part_type":"cyl_container","base":{"id_in":3.25,"height_in":8.58,"thickness_in":0.1473},"features":[{"type":"hole","face":"side","angle_deg":285.0,"up_in":4.75,"diameter_in":0.125},{"type":"hole","face":"side","angle_deg":210.0,"up_in":6.5,"diameter_in":0.1875}],"text":"1/8 inch hole at 285 degrees and 4 3/4 inches up; 3/16 inch hole at 210 degrees and 6.5 inches up.","group":"Cyl_container:41","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":104,"part_type":"cyl_container","base":{"id_in":8.0,"height_in":13.06,"thickness_in":0.18},"features":[{"type":"hole","face":"side","angle_deg":135.0,"up_in":8.0,"diameter_in":0.25}],"text":"Add a 1/4 inch hole at 135 degrees, 8 inches above the bottom.","group":"Cyl_container:75","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":105,"part_type":"cyl_container","base":{"id_in":9.0,"height_in":14.0,"thickness_in":0.18},"features":[{"type":"hole","face":"side","angle_deg":120.0,"up_in":null,"diameter_in":0.25},{"type":"hole","face":"side","angle_deg":240.0,"up_in":null,"diameter_in":0.3125},{"type":"hole","face":"side","angle_deg":105.0,"up_in":11.0,"diameter_in":0.625}],"text":"Three holes. A 1/4 inch hole at 120 degrees and a 5/16 inch hole at 240 degrees, both centered vertically. A 5/8 inch hole at 105 degrees, 11 inches up.","group":"Cyl_container:80","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":106,"part_type":"cyl_container","base":{"id_in":10.0,"height_in":7.68,"thickness_in":0.1558},"features":[{"type":"hole","face":"side","angle_deg":120.0,"up_in":7.375,"diameter_in":0.1875},{"type":"hole","face":"side","angle_deg":330.0,"up_in":null,"diameter_in":0.125}],"text":"3/16 inch hole at 120 degrees, 7 3/8 inches up. Also a 1/8 inch hole at 330 degrees, vertically centered.","group":"Cyl_container:82","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":107,"part_type":"cyl_container","base":{"id_in":8.0,"height_in":14.0,"thickness_in":0.18},"features":[{"type":"hole","face":"side","angle_deg":75.0,"up_in":null,"diameter_in":0.25}],"text":"quarter inch hole at 75 degrees, in the middle of the height","group":"Cyl_container:77","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":108,"part_type":"cyl_container","base":{"id_in":4.5,"height_in":9.18,"thickness_in":0.1537},"features":[{"type":"hole","face":"side","angle_deg":165.0,"up_in":6.5,"diameter_in":0.75},{"type":"hole","face":"side","angle_deg":300.0,"up_in":null,"diameter_in":0.75}],"text":"Two 3/4 inch holes. One at 165 degrees, 6.5 inches up. One at 300 degrees, centered vertically.","group":"Cyl_container:53","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":109,"part_type":"cyl_container","base":{"id_in":10.0,"height_in":14.0,"thickness_in":0.18},"features":[{"type":"hole","face":"side","angle_deg":30.0,"up_in":13.5,"diameter_in":0.25}],"text":"1/4 inch hole at 30 degrees, 13.5 inches up from the bottom, near the top.","group":"Cyl_container:83","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":110,"part_type":"cyl_container","base":{"id_in":1.25,"height_in":1.42,"thickness_in":0.0686},"features":[{"type":"hole","face":"side","angle_deg":210.0,"up_in":1.0,"diameter_in":0.375}],"text":"3/8 inch hole at 210 degrees, 1 inch up.","group":"Cyl_container:15","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":111,"part_type":"cyl_container","base":{"id_in":1.75,"height_in":2.12,"thickness_in":0.0711},"features":[{"type":"hole","face":"side","angle_deg":225.0,"up_in":null,"diameter_in":0.25}],"text":"Put a 1/4 inch hole at 225 degrees, halfway up the wall.","group":"Cyl_container:27","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":112,"part_type":"cyl_container","base":{"id_in":3.75,"height_in":3.17,"thickness_in":0.0919},"features":[{"type":"hole","face":"side","angle_deg":330.0,"up_in":2.0,"diameter_in":0.625}],"text":"5/8 inch hole at 330\u00b0, 2 inches above the bottom.","group":"Cyl_container:45","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":113,"part_type":"cyl_container","base":{"id_in":0.5,"height_in":0.5,"thickness_in":0.0526},"features":[{"type":"hole","face":"side","angle_deg":135.0,"up_in":0.25,"diameter_in":0.125},{"type":"hole","face":"side","angle_deg":75.0,"up_in":0.25,"diameter_in":0.125}],"text":"Two tiny 1/8 inch holes, a quarter inch up, at 75 and 135 degrees.","group":"Cyl_container:3","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":114,"part_type":"cyl_container","base":{"id_in":1.0,"height_in":1.33,"thickness_in":0.0603},"features":[{"type":"hole","face":"side","angle_deg":180.0,"up_in":0.75,"diameter_in":0.375}],"text":"3/8 inch hole on the left side of the cup, 3/4 inch up.","group":"Cyl_container:11","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":115,"part_type":"cyl_container","base":{"id_in":1.5,"height_in":2.44,"thickness_in":0.0648},"features":[{"type":"hole","face":"side","angle_deg":255.0,"up_in":null,"diameter_in":0.625},{"type":"hole","face":"side","angle_deg":75.0,"up_in":1.375,"diameter_in":0.25}],"text":"A 5/8 inch hole at 255 degrees, centered vertically, and a 1/4 inch hole at 75 degrees, 1 3/8 inches up.","group":"Cyl_container:22","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":116,"part_type":"cyl_container","base":{"id_in":1.0,"height_in":1.72,"thickness_in":0.0588},"features":[{"type":"hole","face":"side","angle_deg":195.0,"up_in":null,"diameter_in":0.375},{"type":"hole","face":"side","angle_deg":120.0,"up_in":0.625,"diameter_in":0.5},{"type":"hole","face":"side","angle_deg":30.0,"up_in":null,"diameter_in":0.125}],"text":"Three holes. At 195 degrees a 3/8 inch hole centered vertically. At 120 degrees a half inch hole, 5/8 inch up. At 30 degrees a 1/8 inch hole, centered vertically.","group":"Cyl_container:13","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":117,"part_type":"cyl_container","base":{"id_in":5.0,"height_in":8.1,"thickness_in":0.1236},"features":[{"type":"hole","face":"side","angle_deg":75.0,"up_in":1.375,"diameter_in":0.1875},{"type":"hole","face":"side","angle_deg":225.0,"up_in":null,"diameter_in":0.3125}],"text":"3/16 inch hole at 75 degrees, 1 3/8 inches up, and a 5/16 inch hole at 225 degrees halfway up.","group":"Cyl_container:60","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":118,"part_type":"cyl_container","base":{"id_in":2.75,"height_in":4.38,"thickness_in":0.09},"features":[{"type":"hole","face":"side","angle_deg":90.0,"up_in":3.25,"diameter_in":0.625}],"text":"Put a 5/8 inch hole in the back of the cup, 3 1/4 inches up.","group":"Cyl_container:38","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":119,"part_type":"cyl_container","base":{"id_in":4.0,"height_in":5.63,"thickness_in":0.0975},"features":[{"type":"hole","face":"side","angle_deg":90.0,"up_in":2.375,"diameter_in":0.375},{"type":"hole","face":"side","angle_deg":75.0,"up_in":1.875,"diameter_in":0.75},{"type":"hole","face":"side","angle_deg":120.0,"up_in":4.75,"diameter_in":0.1875}],"text":"Three holes: 3/8 inch at 90 degrees and 2 3/8 inches up; 3/4 inch at 75 degrees and 1 7/8 inches up; 3/16 inch at 120 degrees and 4 3/4 inches up.","group":"Cyl_container:88","source":"claude_written"},{"sheet":"handwritten_holes","sheet_row":120,"part_type":"cyl_container","base":{"id_in":3.75,"height_in":6.49,"thickness_in":0.125},"features":[{"type":"hole","face":"side","angle_deg":90.0,"up_in":null,"diameter_in":0.75}],"text":"3/4 inch hole centered on the back, halfway up.","group":"Cyl_container:46","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":1,"part_type":"bracket","base":{"length_a_in":2.0,"length_b_in":2.0,"height_in":0.625,"thickness_in":0.08},"features":[{"type":"edge","kind":"fillet","edge":"outer_bend","size_in":0.09375}],"text":"Round the outside corner of the bend with a 3/32 inch radius.","group":"L_brackets:7","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":2,"part_type":"bracket","base":{"length_a_in":1.378,"length_b_in":1.1811,"height_in":1.1811,"thickness_in":0.1181},"features":[{"type":"edge","kind":"fillet","edge":"bend","size_in":0.5},{"type":"edge","kind":"chamfer","edge":"free_ends","size_in":0.03125}],"text":"Put a 1/2 inch inside radius on the bend and a 1/32 chamfer on the ends of both legs.","group":"L_brackets:89","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":3,"part_type":"bracket","base":{"length_a_in":5.0,"length_b_in":5.0,"height_in":1.0,"thickness_in":0.17},"features":[{"type":"edge","kind":"chamfer","edge":"free_ends","size_in":0.0625}],"text":"chamfer the ends of the legs 1/16","group":"L_brackets:13","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":4,"part_type":"bracket","base":{"length_a_in":3.0,"length_b_in":2.0,"height_in":2.0,"thickness_in":0.12},"features":[{"type":"edge","kind":"chamfer","edge":"bend","size_in":0.3125}],"text":"I want a chamfered bend, 5/16 on the inside.","group":"L_brackets:87","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":5,"part_type":"bracket","base":{"length_a_in":1.9685,"length_b_in":1.9685,"height_in":1.5748,"thickness_in":0.1181},"features":[{"type":"edge","kind":"fillet","edge":"outer_bend","size_in":0.0625}],"text":"fillet the outer corner of the bend 1/16 inch","group":"L_brackets:90","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":6,"part_type":"bracket","base":{"length_a_in":4.0,"length_b_in":4.0,"height_in":0.875,"thickness_in":0.13},"features":[{"type":"edge","kind":"chamfer","edge":"free_ends","size_in":0.03125}],"text":"Break the ends of both legs with a 1/32 chamfer.","group":"L_brackets:22","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":7,"part_type":"bracket","base":{"length_a_in":6.0,"length_b_in":6.0,"height_in":1.125,"thickness_in":0.19},"features":[{"type":"edge","kind":"chamfer","edge":"inner_bend","size_in":0.625},{"type":"edge","kind":"chamfer","edge":"free_ends","size_in":0.03125}],"text":"Chamfer the inside corner of the bend 5/8 inch, and give the free ends a 1/32 chamfer.","group":"L_brackets:14","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":8,"part_type":"bracket","base":{"length_a_in":8.25,"length_b_in":8.25,"height_in":1.25,"thickness_in":0.11},"features":[{"type":"edge","kind":"chamfer","edge":"bend","size_in":0.625},{"type":"edge","kind":"chamfer","edge":"free_ends","size_in":0.03125}],"text":"bevel the bend 5/8 and bevel the ends of the legs 1/32","group":"L_brackets:55","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":9,"part_type":"rect_container","base":{"length_a_in":8.0,"length_b_in":4.8,"height_in":5.28,"thickness_in":0.136},"features":[{"type":"edge","kind":"chamfer","edge":"bottom","size_in":0.125}],"text":"Chamfer the bottom edges of the box 1/8 inch.","group":"Rect_container:74","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":10,"part_type":"rect_container","base":{"length_a_in":2.5,"length_b_in":2.39,"height_in":1.34,"thickness_in":0.083},"features":[{"type":"edge","kind":"chamfer","edge":"bottom","size_in":0.09375},{"type":"edge","kind":"chamfer","edge":"top_rim","size_in":0.03125},{"type":"edge","kind":"fillet","edge":"inner_bottom","size_in":0.375}],"text":"Bottom edges get a 3/32 chamfer, the top rim gets a 1/32 chamfer, and put a 3/8 inch fillet where the floor meets the walls inside.","group":"Rect_container:16","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":11,"part_type":"rect_container","base":{"length_a_in":6.0,"length_b_in":5.28,"height_in":2.11,"thickness_in":0.121},"features":[{"type":"edge","kind":"chamfer","edge":"bottom","size_in":0.0625},{"type":"edge","kind":"chamfer","edge":"inner_bottom","size_in":1.0},{"type":"edge","kind":"chamfer","edge":"top_rim","size_in":0.03125}],"text":"Chamfer the bottom edges 1/16, chamfer the inside bottom edges 1 inch, and chamfer the top rim 1/32.","group":"Rect_container:60","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":12,"part_type":"rect_container","base":{"length_a_in":5.0,"length_b_in":3.31,"height_in":4.1,"thickness_in":0.096},"features":[{"type":"edge","kind":"chamfer","edge":"inner_corners","size_in":1.0},{"type":"edge","kind":"fillet","edge":"bottom","size_in":0.09375}],"text":"Put a 1 inch chamfer on the inside corners and a 3/32 inch radius on the bottom edges.","group":"Rect_container:45","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":13,"part_type":"rect_container","base":{"length_a_in":2.0,"length_b_in":1.12,"height_in":0.68,"thickness_in":0.059},"features":[{"type":"edge","kind":"fillet","edge":"inner_bottom","size_in":0.3125},{"type":"edge","kind":"fillet","edge":"inner_corners","size_in":0.375},{"type":"edge","kind":"fillet","edge":"bottom","size_in":0.09375}],"text":"Inside: 5/16 radius where the floor meets the walls and 3/8 radius in the corners. Outside bottom edges get a 3/32 radius.","group":"Rect_container:8","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":14,"part_type":"rect_container","base":{"length_a_in":5.0,"length_b_in":4.15,"height_in":3.11,"thickness_in":0.095},"features":[{"type":"edge","kind":"fillet","edge":"inner_corners","size_in":0.375},{"type":"edge","kind":"fillet","edge":"top_rim","size_in":0.03125}],"text":"round the inside corners with a 3/8 inch radius and the top rim with a 1/32 radius","group":"Rect_container:48","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":15,"part_type":"rect_container","base":{"length_a_in":7.5,"length_b_in":3.75,"height_in":2.25,"thickness_in":0.125},"features":[{"type":"edge","kind":"fillet","edge":"top_rim","size_in":0.03125}],"text":"Soften the top rim with a 1/32 fillet.","group":"Rect_container:69","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":16,"part_type":"rect_container","base":{"length_a_in":5.5,"length_b_in":3.31,"height_in":1.59,"thickness_in":0.108},"features":[{"type":"edge","kind":"fillet","edge":"inner_bottom","size_in":0.3125}],"text":"I'd like a 5/16 inch fillet on the inside bottom edges of the box.","group":"Rect_container:52","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":17,"part_type":"rect_container","base":{"length_a_in":3.5,"length_b_in":2.23,"height_in":1.77,"thickness_in":0.083},"features":[{"type":"edge","kind":"fillet","edge":"top_rim","size_in":0.03125},{"type":"edge","kind":"chamfer","edge":"inner_corners","size_in":0.375},{"type":"edge","kind":"fillet","edge":"inner_bottom","size_in":0.25}],"text":"Top rim: 1/32 fillet. Inside corners: 3/8 chamfer. Inside bottom edges: 1/4 fillet.","group":"Rect_container:28","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":18,"part_type":"rect_container","base":{"length_a_in":8.0,"length_b_in":5.6,"height_in":3.36,"thickness_in":0.138},"features":[{"type":"edge","kind":"fillet","edge":"corners","size_in":1.0}],"text":"Make the box have rounded corners, 1 inch radius.","group":"Rect_container:75","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":19,"part_type":"cyl_container","base":{"id_in":7.5,"height_in":11.56,"thickness_in":0.16},"features":[{"type":"edge","kind":"fillet","edge":"top_rim","size_in":0.03125},{"type":"edge","kind":"chamfer","edge":"inner_bottom","size_in":0.375}],"text":"1/32 fillet on the rim and a 3/8 chamfer inside where the floor meets the wall.","group":"Cyl_container:90","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":20,"part_type":"cyl_container","base":{"id_in":4.5,"height_in":11.32,"thickness_in":0.1755},"features":[{"type":"edge","kind":"chamfer","edge":"inner_bottom","size_in":0.375},{"type":"edge","kind":"chamfer","edge":"top_rim","size_in":0.03125}],"text":"chamfer the rim 1/32 and the inside bottom edge 3/8","group":"Cyl_container:57","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":21,"part_type":"cyl_container","base":{"id_in":0.75,"height_in":1.09,"thickness_in":0.0611},"features":[{"type":"edge","kind":"chamfer","edge":"bottom","size_in":0.0625}],"text":"Chamfer the bottom edge of the cup 1/16 inch.","group":"Cyl_container:6","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":22,"part_type":"cyl_container","base":{"id_in":1.75,"height_in":1.74,"thickness_in":0.0632},"features":[{"type":"edge","kind":"fillet","edge":"bottom","size_in":0.09375}],"text":"Round the bottom edge with a 3/32 radius.","group":"Cyl_container:25","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":23,"part_type":"cyl_container","base":{"id_in":8.0,"height_in":12.19,"thickness_in":0.18},"features":[{"type":"edge","kind":"fillet","edge":"top_rim","size_in":0.0625}],"text":"rim needs a 1/16 radius","group":"Cyl_container:73","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":24,"part_type":"cyl_container","base":{"id_in":1.25,"height_in":2.34,"thickness_in":0.0748},"features":[{"type":"edge","kind":"fillet","edge":"inner_bottom","size_in":0.09375}],"text":"Add a 3/32 fillet on the inside bottom edge.","group":"Cyl_container:18","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":25,"part_type":"rect_container","base":{"length_a_in":6.0,"length_b_in":3.28,"height_in":1.02,"thickness_in":0.115},"features":[{"type":"divider","orientation":"lengthwise","position_in":1.25,"count":null,"thickness_in":null,"height_in":0.625}],"text":"Add a divider running left to right, 1 1/4 inches from the front wall, 5/8 inch tall.","group":"Rect_container:56","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":26,"part_type":"rect_container","base":{"length_a_in":1.5,"length_b_in":1.37,"height_in":1.1,"thickness_in":0.057},"features":[{"type":"divider","orientation":"lengthwise","position_in":0.75,"count":null,"thickness_in":null,"height_in":0.375},{"type":"divider","orientation":"crosswise","position_in":null,"count":null,"thickness_in":null,"height_in":null}],"text":"A short divider left to right 3/4 inch from the front wall, only 3/8 inch high, plus a full height divider down the middle separating the left from the right.","group":"Rect_container:5","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":27,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":1.7,"height_in":2.33,"thickness_in":0.068},"features":[{"type":"divider","orientation":"lengthwise","position_in":null,"count":null,"thickness_in":null,"height_in":null},{"type":"divider","orientation":"crosswise","position_in":1.875,"count":null,"thickness_in":null,"height_in":null}],"text":"Two dividers: one down the middle left to right separating front from back, and one front to back 1 7/8 inches from the left wall.","group":"Rect_container:21","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":28,"part_type":"rect_container","base":{"length_a_in":4.5,"length_b_in":3.13,"height_in":2.54,"thickness_in":0.096},"features":[{"type":"divider","orientation":"lengthwise","position_in":null,"count":null,"thickness_in":null,"height_in":null},{"type":"divider","orientation":"crosswise","position_in":2.5,"count":null,"thickness_in":null,"height_in":null}],"text":"Put in a divider that splits front from back, centered, and another that runs front to back 2.5 inches from the left wall.","group":"Rect_container:39","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":29,"part_type":"rect_container","base":{"length_a_in":6.5,"length_b_in":3.2,"height_in":1.92,"thickness_in":0.115},"features":[{"type":"divider","orientation":"lengthwise","position_in":2.375,"count":null,"thickness_in":null,"height_in":null}],"text":"divider wall running left to right, 2 3/8 inches in from the front wall","group":"Rect_container:61","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":30,"part_type":"rect_container","base":{"length_a_in":5.0,"length_b_in":2.44,"height_in":3.1,"thickness_in":0.109},"features":[{"type":"divider","orientation":"lengthwise","position_in":1.5,"count":null,"thickness_in":null,"height_in":0.25},{"type":"divider","orientation":"lengthwise","position_in":0.875,"count":null,"thickness_in":0.1875,"height_in":null}],"text":"Add two dividers running left to right: one 1.5 inches from the front wall and only 1/4 inch tall, one 7/8 inch from the front wall that is 3/16 thick.","group":"Rect_container:42","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":31,"part_type":"rect_container","base":{"length_a_in":7.0,"length_b_in":5.14,"height_in":4.34,"thickness_in":0.129},"features":[{"type":"divider","orientation":"crosswise","position_in":null,"count":3,"thickness_in":0.0625,"height_in":null}],"text":"Split the box into four compartments with three front-to-back dividers, 1/16 thick.","group":"Rect_container:85","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":32,"part_type":"rect_container","base":{"length_a_in":4.0,"length_b_in":1.66,"height_in":2.35,"thickness_in":0.083},"features":[{"type":"divider","orientation":"crosswise","position_in":null,"count":null,"thickness_in":null,"height_in":1.0}],"text":"a divider down the middle front to back that is 1 inch tall","group":"Rect_container:32","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":33,"part_type":"rect_container","base":{"length_a_in":2.5,"length_b_in":2.05,"height_in":0.93,"thickness_in":0.077},"features":[{"type":"divider","orientation":"lengthwise","position_in":null,"count":3,"thickness_in":null,"height_in":0.5}],"text":"Three dividers running left to right, evenly spaced, half inch tall.","group":"Rect_container:14","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":34,"part_type":"rect_container","base":{"length_a_in":4.0,"length_b_in":3.18,"height_in":4.29,"thickness_in":0.096},"features":[{"type":"divider","orientation":"crosswise","position_in":null,"count":null,"thickness_in":null,"height_in":null},{"type":"divider","orientation":"lengthwise","position_in":null,"count":null,"thickness_in":null,"height_in":2.0}],"text":"A full height divider separating left from right down the middle, and a divider separating front from back, centered, 2 inches high.","group":"Rect_container:82","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":35,"part_type":"rect_container","base":{"length_a_in":5.5,"length_b_in":2.41,"height_in":1.69,"thickness_in":0.106},"features":[{"type":"divider","orientation":"crosswise","position_in":null,"count":2,"thickness_in":null,"height_in":null},{"type":"divider","orientation":"lengthwise","position_in":0.625,"count":null,"thickness_in":null,"height_in":null}],"text":"Two front-to-back dividers evenly spaced, plus a left-to-right divider 5/8 inch from the front wall.","group":"Rect_container:51","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":36,"part_type":"rect_container","base":{"length_a_in":4.0,"length_b_in":3.93,"height_in":1.54,"thickness_in":0.083},"features":[{"type":"divider","orientation":"crosswise","position_in":1.0,"count":null,"thickness_in":0.0625,"height_in":null}],"text":"divider 1/16 thick, 1 inch from the left wall, running front to back","group":"Rect_container:34","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":37,"part_type":"rect_container","base":{"length_a_in":7.0,"length_b_in":5.6,"height_in":2.8,"thickness_in":0.13},"features":[{"type":"divider","orientation":"lengthwise","position_in":2.0,"count":null,"thickness_in":0.125,"height_in":null},{"type":"divider","orientation":"lengthwise","position_in":2.5,"count":null,"thickness_in":null,"height_in":null}],"text":"Two dividers left to right: one 2 inches from the front wall, 1/8 inch thick, and another 2.5 inches from the front wall.","group":"Rect_container:67","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":38,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":1.29,"height_in":2.05,"thickness_in":0.084},"features":[{"type":"divider","orientation":"lengthwise","position_in":0.5,"count":null,"thickness_in":0.125,"height_in":null},{"type":"divider","orientation":"crosswise","position_in":null,"count":null,"thickness_in":null,"height_in":1.0}],"text":"A 1/8 inch thick divider left to right 1/2 inch from the front wall, and a centered front-to-back divider 1 inch tall.","group":"Rect_container:18","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":39,"part_type":"rect_container","base":{"length_a_in":4.5,"length_b_in":3.99,"height_in":2.55,"thickness_in":0.093},"features":[{"type":"cutout","face":"front","along_in":1.125,"up_in":2.425,"width_in":1.0,"height_in":0.25}],"text":"Cut a slot in the front wall, 1 inch wide and 1/4 inch tall, 1 1/8 inches from the left end and 2.425 inches up.","group":"Rect_container:41","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":40,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":1.2,"height_in":0.56,"thickness_in":0.091},"features":[{"type":"cutout","face":"front","along_in":null,"up_in":0.435,"width_in":1.0,"height_in":0.25}],"text":"1 by 1/4 inch slot centered on the front wall, 0.435 inch up from the bottom","group":"Rect_container:17","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":41,"part_type":"rect_container","base":{"length_a_in":7.0,"length_b_in":4.2,"height_in":5.04,"thickness_in":0.128},"features":[{"type":"cutout","face":"back","along_in":3.375,"up_in":null,"width_in":2.0,"height_in":0.5},{"type":"cutout","face":"back","along_in":1.625,"up_in":null,"width_in":0.5,"height_in":0.75}],"text":"On the back wall cut a 2 inch wide by 1/2 inch tall opening 3 3/8 inches from the left end, centered vertically, and a 1/2 inch wide by 3/4 inch tall notch 1 5/8 inches from the left end, also centered vertically.","group":"Rect_container:66","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":42,"part_type":"rect_container","base":{"length_a_in":6.5,"length_b_in":4.22,"height_in":6.25,"thickness_in":0.126},"features":[{"type":"cutout","face":"back","along_in":2.25,"up_in":0.625,"width_in":1.0,"height_in":0.25},{"type":"cutout","face":"left","along_in":3.0,"up_in":5.875,"width_in":0.5,"height_in":0.75}],"text":"Back wall: a 1 inch by 1/4 inch slot 2 1/4 inches from the left end and 5/8 inch up. Left wall: a 1/2 inch wide, 3/4 inch tall notch 3 inches from the left end that runs up to the top rim, its centre 5 7/8 inches up.","group":"Rect_container:63","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":43,"part_type":"rect_container","base":{"length_a_in":4.0,"length_b_in":2.36,"height_in":2.4,"thickness_in":0.086},"features":[{"type":"cutout","face":"front","along_in":2.75,"up_in":0.75,"width_in":1.0,"height_in":0.375},{"type":"cutout","face":"back","along_in":3.5,"up_in":1.0,"width_in":0.5,"height_in":1.0}],"text":"Front wall: slot 1 inch wide, 3/8 inch tall, 2 3/4 inches from the left end, 3/4 inch up. Back wall: slot 1/2 inch wide, 1 inch tall, 3 1/2 inches from the left end, 1 inch up.","group":"Rect_container:81","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":44,"part_type":"rect_container","base":{"length_a_in":5.0,"length_b_in":2.85,"height_in":3.02,"thickness_in":0.106},"features":[{"type":"cutout","face":"right","along_in":1.25,"up_in":null,"width_in":0.75,"height_in":1.0}],"text":"cutout in the right wall, 3/4 wide and 1 inch tall, 1 1/4 inches from the left end, centered vertically","group":"Rect_container:43","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":45,"part_type":"bracket","base":{"length_a_in":2.875,"length_b_in":6.125,"height_in":3.125,"thickness_in":0.06},"features":[{"type":"cutout","face":"leg_a","from_end_in":1.25,"across_in":1.25,"width_in":1.0,"height_in":0.75}],"text":"Cut a 1 inch wide, 3/4 inch tall slot in the shorter leg, 1 1/4 inches from the end and 1 1/4 inches up.","group":"L_brackets:39","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":46,"part_type":"bracket","base":{"length_a_in":1.75,"length_b_in":1.75,"height_in":2.0,"thickness_in":0.125},"features":[{"type":"cutout","face":"leg_a","from_end_in":1.0,"across_in":1.0,"width_in":1.0,"height_in":1.0},{"type":"cutout","face":"leg_b","from_end_in":1.0,"across_in":1.125,"width_in":1.0,"height_in":0.375}],"text":"Leg A: a 1 inch square cutout, 1 inch from the end and 1 inch up. Leg B: a 1 inch wide by 3/8 inch tall slot, 1 inch from the end and 1 1/8 inches up.","group":"L_brackets:74","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":47,"part_type":"bracket","base":{"length_a_in":1.0,"length_b_in":1.0,"height_in":1.25,"thickness_in":0.125},"features":[{"type":"cutout","face":"leg_b","from_end_in":0.5,"across_in":0.625,"width_in":0.5,"height_in":0.25}],"text":"slot in leg B, 1/2 wide by 1/4 tall, 1/2 inch from the end and 5/8 inch up","group":"L_brackets:72","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":48,"part_type":"bracket","base":{"length_a_in":2.5,"length_b_in":2.5,"height_in":0.625,"thickness_in":0.11},"features":[{"type":"cutout","face":"leg_b","from_end_in":0.875,"across_in":0.25,"width_in":1.25,"height_in":0.25}],"text":"Cut a slot in leg B 1 1/4 inches wide and 1/4 inch tall, 7/8 inch from the end, 1/4 inch up from the bottom.","group":"L_brackets:9","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":49,"part_type":"bracket","base":{"length_a_in":2.0,"length_b_in":2.0,"height_in":0.625,"thickness_in":0.07},"features":[{"type":"edge","kind":"fillet","edge":"inner_bend","size_in":0.375},{"type":"hole","face":"leg_b","from_end_in":1.25,"across_in":0.375,"diameter_in":0.25}],"text":"Round the inside corner of the bend with a 3/8 radius and drill a 1/4 inch hole in leg B, 1 1/4 inches from the end and 3/8 inch up.","group":"L_brackets:19","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":50,"part_type":"bracket","base":{"length_a_in":1.875,"length_b_in":3.5,"height_in":2.375,"thickness_in":0.1},"features":[{"type":"edge","kind":"chamfer","edge":"bend","size_in":0.3125},{"type":"cutout","face":"leg_b","from_end_in":2.25,"across_in":1.75,"width_in":1.25,"height_in":0.5}],"text":"Chamfer the bend 5/16 and cut a 1 1/4 inch wide by 1/2 inch tall slot in the long leg, 2 1/4 inches from the end, 1 3/4 inches up.","group":"L_brackets:43","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":51,"part_type":"bracket","base":{"length_a_in":0.875,"length_b_in":0.875,"height_in":0.9375,"thickness_in":0.14},"features":[{"type":"edge","kind":"chamfer","edge":"outer_bend","size_in":0.0625},{"type":"cutout","face":"leg_a","from_end_in":null,"across_in":0.5,"width_in":0.5,"height_in":0.5},{"type":"cutout","face":"leg_b","from_end_in":0.375,"across_in":null,"width_in":0.5,"height_in":0.25}],"text":"Chamfer the outside corner of the bend 1/16. Leg A gets a 1/2 inch square cutout centered along the leg, 1/2 inch up. Leg B gets a 1/2 wide by 1/4 tall slot 3/8 inch from the end, centered across the width.","group":"L_brackets:30","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":52,"part_type":"bracket","base":{"length_a_in":1.0,"length_b_in":1.0,"height_in":0.6875,"thickness_in":0.09},"features":[{"type":"edge","kind":"chamfer","edge":"bend","size_in":0.25},{"type":"edge","kind":"fillet","edge":"free_ends","size_in":0.03125},{"type":"hole","face":"leg_a","from_end_in":0.25,"across_in":null,"diameter_in":0.25}],"text":"Bend chamfer 1/4, fillet the leg ends 1/32, and a 1/4 inch hole in leg A 1/4 inch from the end, centered across.","group":"L_brackets:31","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":53,"part_type":"bracket","base":{"length_a_in":3.0,"length_b_in":2.25,"height_in":3.0,"thickness_in":0.12},"features":[{"type":"cutout","face":"leg_a","from_end_in":0.875,"across_in":1.5,"width_in":1.5,"height_in":1.0},{"type":"hole","face":"leg_a","from_end_in":2.375,"across_in":1.75,"diameter_in":0.375},{"type":"hole","face":"leg_a","from_end_in":0.375,"across_in":2.625,"diameter_in":0.5}],"text":"On the long leg: a 1 1/2 inch wide by 1 inch tall cutout 7/8 inch from the end and 1 1/2 inches up, a 3/8 inch hole 2 3/8 inches from the end and 1 3/4 inches up, and a 1/2 inch hole 3/8 inch from the end and 2 5/8 inches up.","group":"L_brackets:88","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":54,"part_type":"bracket","base":{"length_a_in":3.0,"length_b_in":3.0,"height_in":0.75,"thickness_in":0.12},"features":[{"type":"cutout","face":"leg_a","from_end_in":1.375,"across_in":0.375,"width_in":1.5,"height_in":0.5},{"type":"cutout","face":"leg_b","from_end_in":1.75,"across_in":0.375,"width_in":1.5,"height_in":0.5},{"type":"edge","kind":"chamfer","edge":"free_ends","size_in":0.03125}],"text":"Both legs get a 1 1/2 by 1/2 inch slot 3/8 inch up from the bottom - on leg A it is 1 3/8 inches from the end, on leg B 1 3/4 inches from the end. Chamfer the free ends 1/32.","group":"L_brackets:10","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":55,"part_type":"rect_container","base":{"length_a_in":1.5,"length_b_in":1.0,"height_in":1.51,"thickness_in":0.071},"features":[{"type":"divider","orientation":"crosswise","position_in":null,"count":null,"thickness_in":null,"height_in":1.125},{"type":"edge","kind":"chamfer","edge":"inner_corners","size_in":0.125}],"text":"A divider down the middle front to back, 1 1/8 inches tall, and chamfer the inside corners 1/8.","group":"Rect_container:4","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":56,"part_type":"rect_container","base":{"length_a_in":5.0,"length_b_in":3.58,"height_in":4.51,"thickness_in":0.105},"features":[{"type":"edge","kind":"chamfer","edge":"inner_bottom","size_in":0.75},{"type":"divider","orientation":"lengthwise","position_in":2.25,"count":null,"thickness_in":null,"height_in":null}],"text":"Chamfer the inside bottom edges 3/4 inch and add a left-to-right divider 2 1/4 inches from the front wall.","group":"Rect_container:46","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":57,"part_type":"rect_container","base":{"length_a_in":1.5,"length_b_in":1.0,"height_in":0.66,"thickness_in":0.067},"features":[{"type":"cutout","face":"front","along_in":null,"up_in":0.535,"width_in":0.75,"height_in":0.25},{"type":"edge","kind":"chamfer","edge":"bottom","size_in":0.03125},{"type":"hole","face":"right","along_in":0.625,"up_in":0.375,"diameter_in":0.3125}],"text":"Cut a 3/4 inch by 1/4 inch slot centered on the front wall, 0.535 inch up. Chamfer the bottom edges 1/32. Drill a 5/16 hole in the right wall 5/8 inch from the left end and 3/8 inch up.","group":"Rect_container:2","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":58,"part_type":"rect_container","base":{"length_a_in":2.0,"length_b_in":1.15,"height_in":0.75,"thickness_in":0.066},"features":[{"type":"edge","kind":"fillet","edge":"inner_corners","size_in":0.125},{"type":"divider","orientation":"crosswise","position_in":null,"count":2,"thickness_in":null,"height_in":null},{"type":"cutout","face":"front","along_in":0.75,"up_in":0.5,"width_in":0.75,"height_in":0.25}],"text":"Round the inside corners 1/8, add two front-to-back dividers evenly spaced, and cut a 3/4 by 1/4 slot in the front wall 3/4 inch from the left end, 1/2 inch up.","group":"Rect_container:78","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":59,"part_type":"rect_container","base":{"length_a_in":8.0,"length_b_in":6.4,"height_in":5.76,"thickness_in":0.142},"features":[{"type":"edge","kind":"fillet","edge":"top_rim","size_in":0.03125},{"type":"divider","orientation":"crosswise","position_in":null,"count":null,"thickness_in":null,"height_in":null},{"type":"cutout","face":"back","along_in":4.5,"up_in":4.5,"width_in":0.5,"height_in":0.375}],"text":"1/32 fillet on the top rim, a divider down the middle separating left from right, and a 1/2 wide by 3/8 tall slot in the back wall 4.5 inches from the left end and 4.5 inches up.","group":"Rect_container:76","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":60,"part_type":"rect_container","base":{"length_a_in":4.5,"length_b_in":3.1,"height_in":2.11,"thickness_in":0.085},"features":[{"type":"edge","kind":"chamfer","edge":"corners","size_in":0.375},{"type":"cutout","face":"front","along_in":2.125,"up_in":1.5,"width_in":0.75,"height_in":0.25},{"type":"cutout","face":"left","along_in":1.625,"up_in":0.625,"width_in":0.75,"height_in":0.75},{"type":"edge","kind":"fillet","edge":"bottom","size_in":0.125}],"text":"Chamfer the corners 3/8 and fillet the bottom edges 1/8. Front wall slot: 3/4 by 1/4, 2 1/8 inches from the left end, 1 1/2 inches up. Left wall cutout: 3/4 by 3/4, 1 5/8 inches from the left end, 5/8 inch up.","group":"Rect_container:38","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":61,"part_type":"rect_container","base":{"length_a_in":5.5,"length_b_in":2.37,"height_in":1.37,"thickness_in":0.111},"features":[{"type":"cutout","face":"left","along_in":null,"up_in":0.75,"width_in":1.25,"height_in":1.0},{"type":"hole","face":"front","along_in":2.625,"up_in":1.125,"diameter_in":0.25}],"text":"Left wall: 1 1/4 wide by 1 inch tall cutout centered left to right, 3/4 inch up. Front wall: 1/4 inch hole 2 5/8 inches from the left end and 1 1/8 inches up.","group":"Rect_container:50","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":62,"part_type":"rect_container","base":{"length_a_in":7.5,"length_b_in":6.0,"height_in":5.4,"thickness_in":0.138},"features":[{"type":"hole","face":"right","along_in":2.0,"up_in":2.375,"diameter_in":0.5},{"type":"divider","orientation":"crosswise","position_in":6.375,"count":null,"thickness_in":null,"height_in":null}],"text":"Put a 1/2 inch hole in the right wall, 2 inches from the left end and 2 3/8 inches up, and a divider front to back 6 3/8 inches from the left wall.","group":"Rect_container:72","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":63,"part_type":"rect_container","base":{"length_a_in":5.0,"length_b_in":3.7,"height_in":1.52,"thickness_in":0.101},"features":[{"type":"edge","kind":"chamfer","edge":"top_rim","size_in":0.03125},{"type":"divider","orientation":"crosswise","position_in":null,"count":3,"thickness_in":null,"height_in":null}],"text":"chamfer the top rim 1/32 and add three dividers front to back, evenly spaced","group":"Rect_container:47","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":64,"part_type":"rect_container","base":{"length_a_in":4.5,"length_b_in":2.35,"height_in":3.68,"thickness_in":0.103},"features":[{"type":"divider","orientation":"crosswise","position_in":2.375,"count":null,"thickness_in":null,"height_in":null},{"type":"divider","orientation":"lengthwise","position_in":0.875,"count":null,"thickness_in":null,"height_in":null},{"type":"hole","face":"right","along_in":1.25,"up_in":0.625,"diameter_in":0.3125},{"type":"cutout","face":"right","along_in":null,"up_in":2.125,"width_in":0.75,"height_in":0.75}],"text":"Divider front to back 2 3/8 inches from the left wall, divider left to right 7/8 inch from the front wall, a 5/16 hole in the right wall 1 1/4 inches from the left end and 5/8 inch up, and a 3/4 inch square cutout in the right wall centered left to right, 2 1/8 inches up.","group":"Rect_container:35","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":65,"part_type":"rect_container","base":{"length_a_in":2.1,"length_b_in":2.1,"height_in":2.8,"thickness_in":0.2},"features":[{"type":"edge","kind":"fillet","edge":"inner_bottom","size_in":0.3125},{"type":"cutout","face":"back","along_in":1.125,"up_in":null,"width_in":0.5,"height_in":0.375},{"type":"divider","orientation":"lengthwise","position_in":0.625,"count":null,"thickness_in":null,"height_in":0.875}],"text":"Fillet the inside bottom edges 5/16. Back wall slot 1/2 wide, 3/8 tall, 1 1/8 inches from the left end, centered vertically. Divider left to right 5/8 inch from the front wall, 7/8 inch tall.","group":"Rect_container:88","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":66,"part_type":"rect_container","base":{"length_a_in":8.0,"length_b_in":4.0,"height_in":2.4,"thickness_in":0.13},"features":[{"type":"hole","face":"front","along_in":5.375,"up_in":1.75,"diameter_in":0.5},{"type":"hole","face":"front","along_in":4.75,"up_in":null,"diameter_in":0.75},{"type":"cutout","face":"left","along_in":3.125,"up_in":0.625,"width_in":0.5,"height_in":0.375},{"type":"divider","orientation":"crosswise","position_in":3.75,"count":null,"thickness_in":0.1875,"height_in":null}],"text":"Front wall: a 1/2 inch hole 5 3/8 inches from the left end and 1 3/4 inches up, and a 3/4 inch hole 4 3/4 inches from the left end, centered vertically. Left wall: 1/2 by 3/8 slot, 3 1/8 inches from the left end, 5/8 inch up. Add a 3/16 thick divider front to back 3 3/4 inches from the left wall.","group":"Rect_container:73","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":67,"part_type":"cyl_container","base":{"id_in":1.0,"height_in":1.84,"thickness_in":0.0582},"features":[{"type":"hole","face":"side","angle_deg":225.0,"up_in":null,"diameter_in":0.625},{"type":"edge","kind":"chamfer","edge":"bottom","size_in":0.03125},{"type":"edge","kind":"chamfer","edge":"inner_bottom","size_in":0.125}],"text":"5/8 inch hole at 225 degrees, halfway up. Chamfer the bottom edge 1/32 and the inside bottom edge 1/8.","group":"Cyl_container:14","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":68,"part_type":"cyl_container","base":{"id_in":0.5,"height_in":1.35,"thickness_in":0.0566},"features":[{"type":"hole","face":"side","angle_deg":150.0,"up_in":0.75,"diameter_in":0.375},{"type":"edge","kind":"chamfer","edge":"inner_bottom","size_in":0.03125},{"type":"edge","kind":"chamfer","edge":"bottom","size_in":0.0625}],"text":"A 3/8 hole at 150 degrees, 3/4 inch up, a 1/32 chamfer on the inside bottom and a 1/16 chamfer on the outside bottom edge.","group":"Cyl_container:5","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":69,"part_type":"cyl_container","base":{"id_in":2.75,"height_in":4.66,"thickness_in":0.0982},"features":[{"type":"hole","face":"side","angle_deg":15.0,"up_in":2.375,"diameter_in":0.375},{"type":"edge","kind":"fillet","edge":"bottom","size_in":0.0625},{"type":"edge","kind":"chamfer","edge":"top_rim","size_in":0.03125}],"text":"Drill a 3/8 inch hole at 15 degrees, 2 3/8 inches up. Round the bottom edge 1/16 and chamfer the rim 1/32.","group":"Cyl_container:39","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":70,"part_type":"cyl_container","base":{"id_in":3.75,"height_in":8.51,"thickness_in":0.1386},"features":[{"type":"hole","face":"side","angle_deg":225.0,"up_in":4.25,"diameter_in":0.5},{"type":"edge","kind":"fillet","edge":"top_rim","size_in":0.03125}],"text":"half inch hole at 225 degrees, 4 1/4 inches up, and a 1/32 fillet on the rim","group":"Cyl_container:47","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":71,"part_type":"cyl_container","base":{"id_in":10.0,"height_in":14.0,"thickness_in":0.18},"features":[{"type":"hole","face":"side","angle_deg":300.0,"up_in":null,"diameter_in":0.625},{"type":"edge","kind":"chamfer","edge":"top_rim","size_in":0.0625},{"type":"edge","kind":"fillet","edge":"bottom","size_in":0.25}],"text":"5/8 inch hole at 300 degrees centered vertically, 1/16 chamfer on the top rim, 1/4 radius on the bottom edge","group":"Cyl_container:85","source":"claude_written"},{"sheet":"handwritten_features","sheet_row":72,"part_type":"cyl_container","base":{"id_in":1.5,"height_in":3.4,"thickness_in":0.0897},"features":[{"type":"edge","kind":"fillet","edge":"top_rim","size_in":0.03125},{"type":"edge","kind":"fillet","edge":"bottom","size_in":0.09375},{"type":"hole","face":"side","angle_deg":240.0,"up_in":1.375,"diameter_in":0.125}],"text":"Fillet the rim 1/32 and the bottom edge 3/32, and add a 1/8 inch hole at 240 degrees, 1 3/8 inches up.","group":"Cyl_container:23","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":1,"part_type":"rect_container","base":{"length_a_in":6.0,"length_b_in":2.85,"height_in":3.05,"thickness_in":0.105},"features":[{"type":"divider","orientation":"crosswise","position_in":3.75,"count":null,"thickness_in":null,"height_in":1.375}],"text":"Add a front-to-back divider 3 3/4 inches from the left wall, only 1 3/8 inches tall.","group":"Rect_container:55","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":2,"part_type":"rect_container","base":{"length_a_in":6.0,"length_b_in":3.01,"height_in":2.84,"thickness_in":0.119},"features":[{"type":"edge","kind":"fillet","edge":"bottom","size_in":0.25}],"text":"Round off the bottom edges with a 1/4 inch radius.","group":"Rect_container:84","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":3,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":11.0,"height_in":7.0,"thickness_in":0.2},"features":[{"type":"hole","face":"back","along_in":null,"up_in":null,"diameter_in":0.625}],"text":"Put a 5/8 inch hole dead center in the back wall.","group":"Rect_container:87","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":4,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":1.31,"height_in":1.19,"thickness_in":0.071},"features":[{"type":"cutout","face":"back","along_in":0.625,"up_in":null,"width_in":0.5,"height_in":0.5}],"text":"Cut a 1/2 inch square opening in the back wall, 5/8 inch from the left end, centered vertically.","group":"Rect_container:19","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":5,"part_type":"rect_container","base":{"length_a_in":6.5,"length_b_in":5.4,"height_in":2.7,"thickness_in":0.122},"features":[{"type":"hole","face":"front","along_in":1.875,"up_in":2.0,"diameter_in":0.125}],"text":"a 1/8 inch hole in the front wall, 1 7/8 from the left end and 2 inches up","group":"Rect_container:64","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":6,"part_type":"rect_container","base":{"length_a_in":6.0,"length_b_in":4.38,"height_in":5.26,"thickness_in":0.125},"features":[{"type":"bottom_hole","x_in":null,"y_in":0.625,"diameter_in":0.25,"count":2,"spacing_in":null,"direction":null},{"type":"edge","kind":"chamfer","edge":"corners","size_in":1.0}],"text":"Two 1/4 inch holes in the base, evenly spaced, 5/8 inch from the front, and chamfer the corners 1 inch.","group":"Rect_container:58","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":7,"part_type":"rect_container","base":{"length_a_in":2.0,"length_b_in":1.0,"height_in":0.75,"thickness_in":0.066},"features":[{"type":"cutout","face":"back","along_in":null,"up_in":0.5625,"width_in":1.25,"height_in":0.375}],"text":"Notch the back wall: 1 1/4 inches wide and 3/8 inch deep, centered left to right, open at the top, its centre 9/16 inch up.","group":"Rect_container:77","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":8,"part_type":"rect_container","base":{"length_a_in":2.0,"length_b_in":1.0,"height_in":0.75,"thickness_in":0.066},"features":[{"type":"edge","kind":"chamfer","edge":"bottom","size_in":0.0625}],"text":"Bevel the bottom edges 1/16.","group":"Rect_container:77|b","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":9,"part_type":"rect_container","base":{"length_a_in":6.0,"length_b_in":4.69,"height_in":4.64,"thickness_in":0.123},"features":[{"type":"divider","orientation":"crosswise","position_in":null,"count":3,"thickness_in":null,"height_in":2.75},{"type":"edge","kind":"fillet","edge":"inner_corners","size_in":0.75}],"text":"Three front-to-back dividers evenly spaced, 2 3/4 inches tall, and round the inside corners with a 3/4 inch radius.","group":"Rect_container:59|b","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":10,"part_type":"rect_container","base":{"length_a_in":5.5,"length_b_in":4.88,"height_in":3.76,"thickness_in":0.105},"features":[{"type":"hole","face":"front","along_in":5.0,"up_in":null,"diameter_in":0.5}],"text":"half inch hole in the front wall, 5 inches from the left end, centered vertically","group":"Rect_container:54|b","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":11,"part_type":"cyl_container","base":{"id_in":4.5,"height_in":5.54,"thickness_in":0.0989},"features":[{"type":"hole","face":"side","angle_deg":90.0,"up_in":2.5,"diameter_in":0.1875}],"text":"3/16 inch hole on the back of the cup, 2 1/2 inches up.","group":"Cyl_container:50","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":12,"part_type":"cyl_container","base":{"id_in":5.0,"height_in":9.56,"thickness_in":0.1536},"features":[{"type":"bottom_hole","x_in":1.0,"y_in":1.0,"diameter_in":0.3125,"count":null,"spacing_in":null,"direction":null}],"text":"A 5/16 inch hole in the bottom, 1 inch right of center and 1 inch toward the back.","group":"Cyl_container:61","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":13,"part_type":"cyl_container","base":{"id_in":6.5,"height_in":14.0,"thickness_in":0.18},"features":[{"type":"hole","face":"side","angle_deg":195.0,"up_in":6.125,"diameter_in":0.25}],"text":"1/4 inch hole at 195 degrees, 6 1/8 inches up from the bottom","group":"Cyl_container:68","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":14,"part_type":"cyl_container","base":{"id_in":3.75,"height_in":2.66,"thickness_in":0.0936},"features":[{"type":"edge","kind":"fillet","edge":"bottom","size_in":0.1875}],"text":"Give the bottom edge of the cup a 3/16 radius.","group":"Cyl_container:44","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":15,"part_type":"cyl_container","base":{"id_in":0.5,"height_in":0.8,"thickness_in":0.0494},"features":[{"type":"hole","face":"side","angle_deg":165.0,"up_in":0.5,"diameter_in":0.125},{"type":"edge","kind":"fillet","edge":"bottom","size_in":0.03125}],"text":"1/8 hole at 165 degrees, half an inch up, and a 1/32 radius on the bottom edge.","group":"Cyl_container:4","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":16,"part_type":"rect_container","base":{"length_a_in":3.5,"length_b_in":2.23,"height_in":1.77,"thickness_in":0.083},"features":[{"type":"bottom_hole","x_in":null,"y_in":null,"diameter_in":0.25,"count":4,"spacing_in":null,"direction":"front_back"}],"text":"Four 1/4 inch holes in the floor, evenly spaced in a row from front to back.","group":"Rect_container:28|b","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":17,"part_type":"rect_container","base":{"length_a_in":4.5,"length_b_in":3.13,"height_in":2.54,"thickness_in":0.096},"features":[{"type":"edge","kind":"fillet","edge":"bottom","size_in":0.125},{"type":"divider","orientation":null,"position_in":null,"count":3,"thickness_in":null,"height_in":null}],"text":"1/8 radius on the bottom edges and split it into four separate sections.","group":"Rect_container:39|b","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":18,"part_type":"bracket","base":{"length_a_in":2.375,"length_b_in":2.375,"height_in":0.625,"thickness_in":0.12},"features":[{"type":"hole","face":"leg_b","from_end_in":null,"across_in":0.375,"diameter_in":0.25,"count":4,"spacing_in":null}],"text":"Four 1/4 inch holes evenly spaced along leg B, 3/8 inch up from the bottom.","group":"L_brackets:28","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":19,"part_type":"bracket","base":{"length_a_in":2.9528,"length_b_in":2.3622,"height_in":1.9685,"thickness_in":0.1181},"features":[{"type":"hole","face":"leg_a","from_end_in":1.0,"across_in":1.125,"diameter_in":0.125,"count":4,"spacing_in":0.5},{"type":"hole","face":"leg_b","from_end_in":1.125,"across_in":null,"diameter_in":0.25}],"text":"On leg A put four 1/8 inch holes in a row, 1/2 inch apart, centered 1 inch from the end and 1 1/8 inches up. On leg B one 1/4 inch hole 1 1/8 inches from the end, centered across the width.","group":"L_brackets:80","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":20,"part_type":"cyl_container","base":{"id_in":2.0,"height_in":2.82,"thickness_in":0.0717},"features":[{"type":"bottom_hole","x_in":null,"y_in":null,"diameter_in":0.1875,"count":3,"spacing_in":null,"direction":null}],"text":"Three 3/16 inch holes in the bottom of the cup, evenly spaced across it.","group":"Cyl_container:30","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":21,"part_type":"rect_container","base":{"length_a_in":7.0,"length_b_in":3.15,"height_in":3.28,"thickness_in":0.12},"features":[{"type":"bottom_hole","x_in":null,"y_in":null,"diameter_in":0.1875,"count":2,"spacing_in":2.0,"direction":"front_back"},{"type":"edge","kind":"fillet","edge":"bottom","size_in":0.125},{"type":"divider","orientation":null,"position_in":null,"count":2,"thickness_in":null,"height_in":null}],"text":"Two 3/16 inch holes in the base 2 inches apart, in a line from the front toward the back, a 1/8 inch radius on the bottom edges, and dividers so it has three sections.","group":"Rect_container:65|b","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":22,"part_type":"rect_container","base":{"length_a_in":5.0,"length_b_in":4.15,"height_in":3.11,"thickness_in":0.095},"features":[{"type":"cutout","face":"right","along_in":null,"up_in":2.125,"width_in":1.25,"height_in":0.75},{"type":"divider","orientation":null,"position_in":null,"count":null,"thickness_in":0.125,"height_in":null},{"type":"bottom_hole","x_in":0.625,"y_in":null,"diameter_in":0.125,"count":null,"spacing_in":null,"direction":null}],"text":"Cut a 1 1/4 by 3/4 inch slot in the right wall, centered left to right, 2 1/8 inches up. Add a 1/8 inch thick divider down the middle, and a 1/8 inch hole in the floor 5/8 inch from the left, halfway between the front and the back.","group":"Rect_container:48|b","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":23,"part_type":"rect_container","base":{"length_a_in":6.5,"length_b_in":3.2,"height_in":1.92,"thickness_in":0.115},"features":[{"type":"bottom_hole","x_in":null,"y_in":null,"diameter_in":0.1875,"count":2,"spacing_in":1.0,"direction":"left_right"},{"type":"divider","orientation":null,"position_in":null,"count":null,"thickness_in":null,"height_in":0.375},{"type":"cutout","face":"back","along_in":0.5,"up_in":1.25,"width_in":0.5,"height_in":1.0}],"text":"Two 3/16 holes in the bottom, 1 inch apart, side by side along the length; a short divider in the middle only 3/8 inch high; and a 1/2 inch wide, 1 inch tall slot in the back wall 1/2 inch from the left end, 1 1/4 inches up.","group":"Rect_container:61|b","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":24,"part_type":"rect_container","base":{"length_a_in":7.5,"length_b_in":3.75,"height_in":2.25,"thickness_in":0.125},"features":[{"type":"edge","kind":"fillet","edge":"corners","size_in":0.5},{"type":"bottom_hole","x_in":null,"y_in":null,"diameter_in":0.125,"count":4,"spacing_in":2.0,"direction":null},{"type":"divider","orientation":null,"position_in":null,"count":2,"thickness_in":null,"height_in":null}],"text":"give the outside a 1/2\" radius. i also want 4 holes in the base spaced evenly 2\" apart. holes should be 1/8\" diameter. also add dividers to the interior so its 3 separate sections","group":"Rect_container:69|b","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":25,"part_type":"rect_container","base":{"length_a_in":5.5,"length_b_in":3.69,"height_in":5.68,"thickness_in":0.119},"features":[{"type":"bottom_hole","x_in":null,"y_in":null,"diameter_in":0.1875,"count":3,"spacing_in":null,"direction":null}],"text":"Drill three 3/16 inch holes in the bottom, spread out evenly.","group":"Rect_container:53|b","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":26,"part_type":"rect_container","base":{"length_a_in":2.5,"length_b_in":1.43,"height_in":1.69,"thickness_in":0.082},"features":[{"type":"bottom_hole","x_in":null,"y_in":null,"diameter_in":0.375,"count":4,"spacing_in":null,"direction":"left_right"}],"text":"Four 3/8 inch holes across the floor, evenly spaced in a row from left to right.","group":"Rect_container:12|b","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":27,"part_type":"rect_container","base":{"length_a_in":6.5,"length_b_in":4.22,"height_in":6.25,"thickness_in":0.126},"features":[{"type":"hole","face":"right","along_in":1.875,"up_in":5.0,"diameter_in":0.5},{"type":"bottom_hole","x_in":null,"y_in":null,"diameter_in":0.375,"count":3,"spacing_in":1.0,"direction":"front_back"}],"text":"A 1/2 inch hole in the right wall 1 7/8 inches from the left end and 5 inches up, plus three 3/8 inch holes in the base 1 inch apart running front to back.","group":"Rect_container:63|b","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":28,"part_type":"cyl_container","base":{"id_in":1.5,"height_in":2.44,"thickness_in":0.0648},"features":[{"type":"bottom_hole","x_in":-0.25,"y_in":null,"diameter_in":0.1875,"count":null,"spacing_in":null,"direction":null}],"text":"3/16 inch hole in the bottom, 1/4 inch left of center.","group":"Cyl_container:22|b","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":29,"part_type":"cyl_container","base":{"id_in":8.0,"height_in":13.06,"thickness_in":0.18},"features":[{"type":"bottom_hole","x_in":null,"y_in":null,"diameter_in":0.1875,"count":2,"spacing_in":0.5,"direction":null}],"text":"two 3/16 holes in the middle of the floor, 1/2 inch apart","group":"Cyl_container:75|b","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":30,"part_type":"cyl_container","base":{"id_in":8.0,"height_in":12.19,"thickness_in":0.18},"features":[{"type":"bottom_hole","x_in":null,"y_in":null,"diameter_in":0.1875,"count":null,"spacing_in":null,"direction":null}],"text":"Add a 3/16 inch drain hole in the center of the bottom.","group":"Cyl_container:73|b","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":31,"part_type":"cyl_container","base":{"id_in":2.25,"height_in":0.81,"thickness_in":0.0606},"features":[{"type":"hole","face":"side","angle_deg":30.0,"up_in":0.375,"diameter_in":0.375,"count":3,"spacing_deg":null}],"text":"Three 3/8 inch holes evenly spaced around the cup, starting at 30 degrees, 3/8 inch up.","group":"Cyl_container:32|b","source":"claude_written"},{"sheet":"handwritten_features_2","sheet_row":32,"part_type":"cyl_container","base":{"id_in":1.5,"height_in":2.44,"thickness_in":0.0648},"features":[{"type":"hole","face":"side","angle_deg":90.0,"up_in":0.75,"diameter_in":0.3125},{"type":"bottom_hole","x_in":null,"y_in":null,"diameter_in":0.3125,"count":3,"spacing_in":null,"direction":null}],"text":"A 5/16 hole on the back, 3/4 inch up, and three 5/16 holes in the bottom, evenly spaced.","group":"Cyl_container:22|b","source":"claude_written"},{"sheet":"handwritten_features_3","sheet_row":1,"part_type":"rect_container","base":{"length_a_in":5.0,"length_b_in":3.58,"height_in":4.51,"thickness_in":0.105},"features":[{"type":"edge","kind":"fillet","edge":null,"size_in":null},{"type":"cutout","face":"front","along_in":null,"up_in":1.0,"width_in":2.0,"height_in":0.5},{"type":"bottom_hole","x_in":null,"y_in":null,"diameter_in":0.2165,"count":2,"rows":2,"spacing_in":null,"inset_in":1.0,"direction":null}],"text":"fillet the edges. add a cutout on the front 1\" up centered thats 2\" wide and half an inch tall for wiring. add 4 holes to the base in a rectangular pattern with the holes 1\" from the sides. holes should fit an m5 bolt","group":"Rect_container:46","source":"claude_written"},{"sheet":"handwritten_features_3","sheet_row":2,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":2.81,"height_in":4.28,"thickness_in":0.101},"features":[{"type":"bottom_hole","x_in":null,"y_in":null,"diameter_in":0.177,"count":2,"rows":2,"spacing_in":null,"inset_in":0.75,"direction":null}],"text":"Four #8 clearance holes in the floor, one near each corner, 3/4 inch in from the sides.","group":"Rect_container:25","source":"claude_written"},{"sheet":"handwritten_features_3","sheet_row":3,"part_type":"rect_container","base":{"length_a_in":5.0,"length_b_in":3.58,"height_in":4.51,"thickness_in":0.105},"features":[{"type":"bottom_hole","x_in":null,"y_in":null,"diameter_in":0.25,"count":3,"rows":2,"spacing_in":null,"inset_in":1.0,"direction":null},{"type":"edge","kind":"fillet","edge":"corners","size_in":null}],"text":"Drill six 1/4 inch holes in the bottom in a 3 by 2 grid, 1 inch from the edges, and round the corners.","group":"Rect_container:46","source":"claude_written"},{"sheet":"handwritten_features_3","sheet_row":4,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":2.81,"height_in":4.28,"thickness_in":0.101},"features":[{"type":"edge","kind":"chamfer","edge":"top_rim","size_in":null},{"type":"bottom_hole","x_in":null,"y_in":null,"diameter_in":0.1772,"count":null,"rows":null,"spacing_in":null,"inset_in":null,"direction":null}],"text":"chamfer the top rim and add an M4 hole in the center of the floor","group":"Rect_container:25","source":"claude_written"},{"sheet":"handwritten_features_3","sheet_row":5,"part_type":"cyl_container","base":{"id_in":2.0,"height_in":3.07,"thickness_in":0.0813},"features":[{"type":"edge","kind":"fillet","edge":"top_rim","size_in":null},{"type":"bottom_hole","x_in":null,"y_in":null,"diameter_in":0.1339,"count":3,"rows":null,"spacing_in":null,"direction":null}],"text":"Round the rim, and put three holes for M3 screws evenly spaced in the floor.","group":"Cyl_container:31","source":"claude_written"},{"sheet":"handwritten_features_3","sheet_row":6,"part_type":"bracket","base":{"length_a_in":5.0,"length_b_in":5.0,"height_in":1.0,"thickness_in":0.17},"features":[{"type":"hole","face":"leg_a","from_end_in":null,"across_in":null,"diameter_in":0.266,"count":2,"spacing_in":1.0},{"type":"edge","kind":"fillet","edge":null,"size_in":null}],"text":"Two holes for 1/4 inch bolts in leg A, 1 inch apart, centered, and fillet the edges.","group":"L_brackets:13","source":"claude_written"},{"sheet":"handwritten_features_3","sheet_row":7,"part_type":"rect_container","base":{"length_a_in":5.0,"length_b_in":4.69,"height_in":5.16,"thickness_in":0.106},"features":[{"type":"bottom_hole","x_in":null,"y_in":null,"diameter_in":0.375,"count":2,"rows":2,"spacing_in":null,"inset_in":1.5,"direction":null},{"type":"edge","kind":"chamfer","edge":"bottom","size_in":0.0625}],"text":"Put a 2 by 2 grid of 3/8 holes in the base 1.5 inches in from the sides. Also chamfer the bottom edges 1/16.","group":"Rect_container:49","source":"claude_written"},{"sheet":"handwritten_features_3","sheet_row":8,"part_type":"cyl_container","base":{"id_in":1.5,"height_in":3.4,"thickness_in":0.0897},"features":[{"type":"hole","face":"side","angle_deg":90.0,"up_in":1.0,"diameter_in":0.2165},{"type":"edge","kind":"fillet","edge":"bottom","size_in":null}],"text":"M5 clearance hole in the side 1 inch up, at 90 degrees, and a small radius on the bottom edge.","group":"Cyl_container:23","source":"claude_written"}]'''
print(f"built-in hand-written feature requests: {len(json.loads(FEATURE_WRITTEN_JSON))}")

**Load the feature requests:** checks every row, reports problems, and splits by base part (validation/test = hand-written only).

In [ ]:
# ------------------------------------------------------------------ FEATURE REQUESTS: example bank + held-out test set for the OFF-THE-SHELF editing model
# Hand-written requests (built into this notebook) + generated ones (previous cell). Every row is checked: numbers readable in the sentence,
# words consistent with the labels (chamfer vs radius, wall names, divider direction, centring) and the features geometrically valid.
# Validation/test rows are hand-written only, so the editing model is scored on text worded differently from the generated examples.
feat_df = pd.DataFrame(); FEAT_READY = False
if LOAD_FEATURE_DATA:
    feat_written = pd.DataFrame(json.loads(FEATURE_WRITTEN_JSON))
    feat_gen = (generate_feature_rows(FEATURE_GENERATED_PER_TYPE, SEED, exclude_refs=set(feat_written.group)) if FEATURE_GENERATED_PER_TYPE > 0
                else pd.DataFrame(columns=feat_written.columns))
    feat_df = pd.concat([feat_written, feat_gen], ignore_index=True)
    feat_df["text_norm"] = feat_df.text.map(normalize_feature_text)      # same normaliser the app uses for feature text
    feat_df["target"] = [features_to_json(r.part_type, r.features) for r in feat_df.itertuples()]
    def _feat_problems(r):
        p = {"part_type": r.part_type, **r.base}
        return audit_features(r.part_type, r.features, r.text_norm) + [i["msg"] for i in check_features(r.part_type, p, fill_defaults(r.part_type, p, r.features)) if i["level"] == "error"]
    feat_df["problems"] = [_feat_problems(r) for r in feat_df.itertuples()]
    _bad = feat_df[feat_df.problems.map(len) > 0]
    print(f"{len(_bad)} feature rows have problems and are skipped" + (":" if len(_bad) else ""))
    if len(_bad): show_df(_bad[["sheet", "sheet_row", "text", "problems"]])
    feat_df = feat_df[feat_df.problems.map(len) == 0].reset_index(drop=True)
    _rng = random.Random(SEED); _wg = sorted(set(feat_df[feat_df.source != "claude_generated"].group)); _rng.shuffle(_wg)
    _n = len(_wg); _gs = {g: ("test" if i < 0.25 * _n else "val" if i < 0.40 * _n else "train") for i, g in enumerate(_wg)}
    feat_df["split"] = [("train" if r.source == "claude_generated" else _gs[r.group]) for r in feat_df.itertuples()]
    feat_train, feat_val, feat_test = (feat_df[feat_df.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
    FEAT_READY = len(feat_train) >= 10 and len(feat_test) >= 1
    show_df(feat_df.groupby(["source", "split"]).size().unstack(fill_value=0))
    _types = pd.Series([f["type"] for fs in feat_df.features for f in fs]).value_counts()
    print("features in the data:", _types.to_dict())
    print(f"example bank (train) {len(feat_train)} | val {len(feat_val)} | test {len(feat_test)} (val/test are hand-written only)")
    print("All of these rows are SYNTHETIC (written/generated by Claude) - report them separately from your 501 manual samples.")
else:
    feat_train = feat_val = feat_test = pd.DataFrame()
    print("feature data is off (LOAD_FEATURE_DATA = False): the app's feature tables still work, 'describe in words' is disabled.")

## 5d. Part-type phrase data for the classifier (synthetic, written by Claude, stored separately)
The part-type classifier is trained from scratch, so it only knows the words in its training set. Casual wording ("make a box thats 40inches by...") can be mis-typed with high confidence. This cell adds **classification-only** examples with varied nouns (box, crate, jar, corner brace, ...), dimension phrasing and units (inch, fractions, mm, cm, feet). They go into the classifier's training set only - validation and test remain your real texts.

In [ ]:
# ------------------------------------------------------------------ 5d: PART-TYPE PHRASE DATA for the classifier (SYNTHETIC - written by Claude, stored separately)
# The classifier is a small from-scratch model: it only knows words it saw in training. Your 501 texts use a limited vocabulary (e.g. few
# "box" / "enclosure" / "jar" / "corner brace" phrasings), so a casually-worded request can be mis-typed with high confidence. This adds
# classification-ONLY examples (no parameter labels) that vary the noun, the dimension wording and the units (inch, fractions, mm, cm, feet).
# They go into the classifier's TRAINING set only: validation and test stay 100% your real texts.
CLF_NOUNS_BASE = {
    "pipe_gasket": ["pipe gasket", "flange gasket", "gasket", "ring gasket", "flat gasket", "gasket ring", "full face gasket", "pipe flange gasket", "gasket for a pipe flange", "flange seal", "sealing ring"],
    "washer": ["washer", "flat washer", "spacer", "round spacer", "screw spacer", "shim washer", "washer for a bolt", "metric washer", "fender washer", "flat spacer ring"],
    "bracket": ["L bracket", "L-bracket", "angle bracket", "corner bracket", "right angle bracket", "L shaped bracket", "90 degree bracket", "connector bracket"],
    "rect_container": ["rectangular container", "rectangular box", "open top box", "storage box", "rectangular bin", "open rectangular container", "rectangular organizer", "rectangular tray", "open box", "rectangular tub"],
    "cyl_container": ["cylindrical container", "cylindrical cup", "round container", "cylinder", "open cylinder", "cylindrical holder", "cylindrical bin", "round cup", "cup", "cylindrical canister"],
}
CLF_NOUNS_EXTRA = {   # more everyday words; also the nouns used by the hand-written stress prompts, which is why they are kept separate when measuring
    "pipe_gasket": ["seal for a pipe flange", "sealing gasket", "pipe seal"],
    "washer": ["standoff", "bushing spacer", "bushing", "shim", "spacer ring"],
    "bracket": ["corner brace", "L-shaped connector", "right angle mount", "shelf bracket", "angle iron piece", "gusset angle", "angle iron", "angle piece"],
    "rect_container": ["box", "crate", "parts tray", "storage bin", "enclosure", "case", "drawer organizer", "shallow tray", "tub", "rectangular enclosure", "tray", "bin", "organizer", "project box"],
    "cyl_container": ["jar", "bucket", "pot", "tin can", "pencil cup", "round canister", "tube cup", "round holder", "canister", "mug", "round tin"],
}
_LENGTHS = [0.5, 0.75, 1, 1.25, 1.5, 1.75, 2, 2.25, 2.5, 3, 3.5, 4, 4.5, 5, 6, 7, 8, 9, 10, 12, 14, 24]
_THICKS = [0.03125, 0.0625, 0.09375, 0.125, 0.1875, 0.25, 0.3125, 0.05, 0.06, 0.075, 0.09, 0.1, 0.12, 0.15, 0.18, 0.2]

def _q(v, rng):
    """a quantity in a random style: inches (decimal / fraction / words), millimetres, centimetres or feet"""
    r = rng.random()
    if r < 0.12: return f"{max(1, int(round(v * 25.4 / 5) * 5))} mm"
    if r < 0.16: return f"{dec(round(v * 2.54, 1))} cm"
    if r < 0.24 and v in (12, 24): return rng.choice(["a foot", "1 ft", "1 foot"]) if v == 12 else rng.choice(["two feet", "2 ft", "2 feet"])
    return _len(v, rng, words=True)

def _clf_clauses(pt, rng):
    L = lambda: rng.choice(_LENGTHS); T = lambda: rng.choice(_THICKS)
    q = lambda v: _q(v, rng)
    thick = lambda: rng.choice([f"{q(T())} thick", f"{q(T())} walls", f"wall thickness {q(T())}", f"{q(T())} wall", f"{q(T())} thick walls"] if pt in ("rect_container", "cyl_container")
                                else [f"{q(T())} thick", f"thickness {q(T())}", f"{q(T())} thick material", f"{q(T())} tall"])
    if pt == "rect_container":
        a, b, c = L(), L(), L()
        dims = rng.choice([f"{q(a)} by {q(b)} by {q(c)}", f"{q(a)} x {q(b)} x {q(c)}", f"{q(a)} long, {q(b)} wide and {q(c)} tall", f"length {q(a)}, width {q(b)}, height {q(c)}",
                           f"{q(a)} by {q(b)} and {q(c)} deep", f"{q(a)} long by {q(b)} wide, {q(c)} high", f"{q(a)} x {q(b)} footprint, {q(c)} tall"])
        return [dims, thick()] + ([rng.choice(["open top", "no lid", "open on top"])] if rng.random() < 0.3 else [])
    if pt == "cyl_container":
        d, h = L(), L()
        dims = rng.choice([f"{q(d)} inner diameter, {q(h)} tall", f"{q(d)} ID by {q(h)} high", f"{q(d)} wide inside and {q(h)} tall", f"inside diameter {q(d)}, height {q(h)}",
                           f"{q(d)} across, {q(h)} deep", f"{q(d)} diameter, {q(h)} tall", f"{q(h)} tall with a {q(d)} inside diameter"])
        return [dims, thick()] + ([rng.choice(["open top", "open on top"])] if rng.random() < 0.25 else [])
    if pt == "bracket":
        x, y, w = L(), L(), rng.choice([0.5, 0.75, 1, 1.25, 1.5, 2, 3])
        dims = rng.choice([f"legs {q(x)} long", f"{q(x)} legs", f"{q(x)} on each side", f"one leg {q(x)} and the other {q(y)}", f"{q(x)} by {q(y)}", f"two legs, {q(x)} each", f"{q(x)} and {q(y)} legs"])
        return [dims, rng.choice([f"{q(w)} wide", f"{q(w)} tall", f"{q(w)} high"]), thick()] + ([rng.choice(["90 degrees", "right angle", "two legs at 90 degrees"])] if rng.random() < 0.3 else [])
    if pt == "pipe_gasket":
        od, idd = rng.choice([3.5, 4.25, 5, 6, 7.5, 9, 11, 12, 16]), rng.choice([0.84, 1.315, 1.9, 2.375, 3.5, 4.5, 6.625, 8.625])
        dims = rng.choice([f"{q(od)} OD, {q(idd)} ID", f"outer diameter {q(od)}, inner diameter {q(idd)}", f"{q(idd)} inner and {q(od)} outer", f"{q(od)} outside diameter, {q(idd)} inside diameter", f"OD {q(od)} ID {q(idd)}"])
        n = rng.choice(["4", "8", "four", "six", "eight", "12", "16"])
        holes = rng.choice([f"{n} bolt holes", f"{n} bolts", f"{n} mounting holes", f"{n} holes", f"{n} bolt holes on a bolt circle"])
        return [dims, rng.choice([f"{q(T())} thick", f"thickness {q(T())}", f"{q(T())} thickness"]), holes] + ([rng.choice([f"for a {rng.choice(['2', '3', '4', '6', '8'])} inch flange", f"for a {rng.choice(['2', '3', '4', '6'])} inch pipe", "150 class", "ANSI 150"])] if rng.random() < 0.35 else [])
    od, idd, h = rng.choice([0.25, 0.375, 0.5, 0.75, 1, 1.5, 2]), rng.choice([0.125, 0.1875, 0.25, 0.375, 0.5, 0.75]), rng.choice([0.05, 0.0625, 0.125, 0.25, 0.5, 0.75, 1])
    n = rng.choice(["3", "4", "5", "6", "8", "10", "12"]); fr = rng.choice(["1/4", "5/16", "3/8", "1/2", "quarter inch", "half inch"])
    size = rng.choice([f"for an M{n} screw", f"M{n}", f"for a #{n} screw", f"for a {fr} bolt", f"{fr} screw size", f"for an M{n} bolt", f"metric {n}", f"for a {fr} screw"])
    dims = rng.choice([f"{q(od)} OD, {q(idd)} ID", f"{q(od)} wide, {q(h)} tall", f"ID {q(idd)} OD {q(od)}", f"{q(od)} diameter with a {q(idd)} hole", f"{q(h)} tall with a {q(idd)} hole", f"{q(od)} wide with a {q(idd)} hole"])
    return [size] + ([dims] if rng.random() < 0.8 else []) + ([rng.choice([f"{q(h)} thick", f"thickness {q(h)}", f"{q(h)} tall"])] if rng.random() < 0.6 else [])

_CLF_OPEN = ["make a", "I need a", "can you make a", "design a", "generate a", "give me a", "build a", "create a", "model a", "I want a", "make me a", "need a", "please make a", ""]
_CLF_JOIN = ["that is", "thats", "with", "which is", ":", "measuring", "of", "", ",", "that has"]
def render_clf(pt, rng, nouns):
    noun, parts = rng.choice(nouns[pt]), _clf_clauses(pt, rng)
    rng.shuffle(parts[1:]) if len(parts) > 2 else None
    opener, join = rng.choice(_CLF_OPEN), rng.choice(_CLF_JOIN)
    if rng.random() < 0.2:                                                   # dimensions first: "4 x 8 x 2 inch box, 1/8 thick"
        text = f"{parts[0]} {noun}" + "".join(", " + p for p in parts[1:])
    else:
        text = f"{opener} {noun}".strip() + (f" {join} " if join not in (":", ",") else join + " ") + rng.choice([", ", ", ", " and "]).join(parts)
    if rng.random() < 0.18: text = text.lower()
    if rng.random() < 0.10:
        words = [m for m in re.finditer(r"[A-Za-z]{6,}", text)]
        if words:
            m = rng.choice(words); i = rng.randrange(1, len(m.group()) - 2); w = list(m.group()); w[i], w[i + 1] = w[i + 1], w[i]
            text = text[:m.start()] + "".join(w) + text[m.end():]
    return re.sub(r"\s+", " ", text).strip(" ,")

def build_clf_phrases(per_type, seed, include_extra=True):
    rng, nouns, rows, seen = random.Random(seed), {pt: CLF_NOUNS_BASE[pt] + (CLF_NOUNS_EXTRA[pt] if include_extra else []) for pt in PART_TYPES}, [], set(data.text_norm)
    for pt in PART_TYPES:
        made, tries = 0, 0
        while made < per_type and tries < per_type * 30:
            tries += 1; t = render_clf(pt, rng, nouns); tn = normalize_text(t)
            if tn in seen: continue
            seen.add(tn); made += 1; rows.append({"text": t, "text_norm": tn, "part_type": pt, "source": "claude_clf_synthetic"})
    return pd.DataFrame(rows)

CLF_PHRASES_PER_TYPE = 100          # synthetic classifier-only examples per part type (5 x 100 = 500). 0 = off
clf_extra_df = build_clf_phrases(CLF_PHRASES_PER_TYPE, SEED + 3) if CLF_PHRASES_PER_TYPE > 0 else pd.DataFrame(columns=["text", "text_norm", "part_type", "source"])
print(f"classifier phrase data: {len(clf_extra_df)} synthetic rows (training only) | real classifier training rows: {len(train_df)}")
for _pt in PART_TYPES:
    if len(clf_extra_df): print(f"  [{_pt}] {clf_extra_df[clf_extra_df.part_type == _pt].text.iloc[0]}")

## 6. Stage 1 — part-type classifier, trained from scratch
A bag-of-words neural net (learned embeddings → small MLP) with **randomly initialised weights**: no pretrained model is involved.

In [ ]:
# ------------------------------------------------------------------ STAGE 1: part-type classifier, TRAINED FROM SCRATCH
# Pipeline box: "Trained from scratch model extracts part type".
# A small neural net with randomly-initialised weights - no pretrained model is used anywhere in this stage.
from collections import Counter

# Words that only describe HOW a request is phrased, not WHAT the part is. In the real data they correlate with who wrote each sheet
# ("it", "be", "thats" are in ~40% of gasket texts but ~0% of container texts), so a model that sees them learns the writer, not the part.
CLF_STOP = set("""i you me my we us it its it's to be is are was am that thats this these those want need would could should can will please make model design create
build generate give get render produce draw show let lets like also then just so which who what too very really a an the and""".split())

def clf_tokens(text_norm: str):
    t = re.sub(r"(?<![\w.])\d+(?:\.\d+)?", " <num> ", text_norm.lower())      # numbers don't help decide the part type
    words = [w for w in re.findall(r"<num>|[a-z]+", t) if w not in CLF_STOP]    # drop request-framing words
    return words + [a + "_" + b for a, b in zip(words, words[1:])]              # unigrams + bigrams

PT2ID = {pt: i for i, pt in enumerate(PART_TYPES)}
# classifier training set = real train rows + their augmented copies + the classifier-only phrase rows from section 5d. Validation/test stay real.
clf_train_df = pd.concat([train_df[["text_norm", "part_type"]], clf_extra_df[["text_norm", "part_type"]]], ignore_index=True)
_doc_freq = Counter(tok for t in clf_train_df.text_norm for tok in set(clf_tokens(t)))
VOCAB = {"<unk>": 0}
for _w, _c in sorted(_doc_freq.items()):
    if _c >= 2: VOCAB[_w] = len(VOCAB)                                          # only tokens seen in >=2 training texts

def clf_encode(text_norm: str):
    return [VOCAB[t] for t in clf_tokens(text_norm) if t in VOCAB] or [0]       # unknown words are ignored (they used to vote as <unk>)

# diagnostic only (never trained on): 60 hand-written, casually-worded prompts with unusual nouns, units and phrasing
STRESS_PROMPTS = {
    "rect_container": [
        'make a box thats 40inches by 12.73 inches and 1/2" thick. i want it to be two feet tall',
        'Box 4 x 8 in. 1/2" thick 1.75" tall',
        'I need an open top box 6 by 4 by 3 inches with quarter inch walls',
        'crate 24 x 18 x 12 inches, 3/4 inch thick walls',
        'parts tray, 200 mm long, 120 mm wide, 30 mm deep, 2 mm walls',
        'make me a storage bin that is 10 inches long 8 wide and 6 tall, wall thickness .125',
        'Rectangular enclosure for a circuit board: 3.5 x 2.25 x 1 inch, 0.08 inch wall',
        'drawer organizer 12 by 3 by 2, walls 1/8',
        'open box, length 5in width 5in height 2in thickness 0.1in',
        'case 100mm x 60mm x 25mm with 1.5mm walls',
        'a shallow tray 14 inches by 9 inches, 1 inch tall, 3/16 thick',
        'build a rectangular tub thats a foot long, 6 inches wide, 4 inches deep and a quarter inch thick',
    ],
    "cyl_container": [
        'make a cup with a 3 inch inside diameter, 4 inches tall and 1/8 inch walls',
        'tin can 2.5 in ID by 5 in tall, 0.06 wall',
        'I want a cylinder 60 mm across, 100 mm tall, 2 mm thick',
        'pencil cup: 3 inch diameter 4.5 inch high wall thickness 0.1',
        'cylindrical container 10 inch inner diameter 14 inch tall 0.18 thick',
        'round canister, inside diameter 1.5 inches, height 3 inches, 1/16 inch wall',
        'Make a jar, 4" wide inside, 6" tall, 1/8" thick',
        'bucket 8 inches inside diameter, a foot tall, 3/16 walls',
        'open top tube cup 1 inch ID 2 inches tall .05 thick',
        'pot with inner diameter 5 in and height 5 in and thickness 0.15 in',
        'a small round holder, 2.25" ID, 3" tall, 0.075" wall',
        'cylinder: 75 mm inner diameter, 120 mm high, wall 3 mm',
    ],
    "pipe_gasket": [
        'gasket for a 6 inch flange, 1/16 thick, 8 bolt holes',
        'make a pipe gasket 4.5 OD 2.375 ID 0.125 thick 4 holes',
        'ring gasket 150 class, 2 inch pipe, 1/8 inch, four bolts',
        'I need a flange gasket with outer diameter 9 inches and inner diameter 6.625 and 8 bolt holes, 1/16 thick',
        'full face gasket for 3 inch pipe, 5/32 thick',
        'seal for a pipe flange, 3.5 inch ID, 7.5 OD, 3/32 inch thick, 4 bolts',
        'gasket 12 inch OD, 8 inch ID, 0.0625 thick, 8 holes on a bolt circle',
        'make a flat ring gasket 1.9 inner 4.25 outer 1/16 thick with 4 mounting holes',
        'pipe flange gasket: OD 11 in ID 8.625 in thick 1/8 in 8 bolts',
        'gasket for 1 1/2 inch pipe, 3 7/8 OD, 1/16 thick, 4 bolt holes',
        'sealing gasket 100 mm ID 160 mm OD 3 mm thick 4 holes',
        'a gasket ring thats 5" OD, 2" ID, 1/16" thick, with six bolt holes',
    ],
    "washer": [
        'M6 washer',
        'washer for an M8 bolt, 16 mm OD, 8.4 mm ID, 1.6 mm thick',
        'spacer for a #8 screw, 1/2 inch wide, 1/4 inch tall',
        'make a standoff with a 0.25 hole, 0.5 OD, 0.75 tall',
        'flat washer 3/8 screw size',
        'shim washer ID 0.5 OD 1 thickness 0.03',
        'round spacer .75" wide .05" tall with a .375 diameter hole for the screw',
        'I need a washer for a quarter inch bolt, 3/4 OD, 1/16 thick',
        'bushing spacer, 5/16 ID, 1/2 OD, 3/8 long',
        'washer M10',
        'spacer 10 mm tall 6 mm hole 12 mm wide',
        'a washer for a 1/4 inch screw',
    ],
    "bracket": [
        'L bracket with 4 inch legs, 1 inch wide, 1/8 inch thick',
        'make a right angle bracket, legs 3 and 5 inches, 1.5 wide, 0.125 thick',
        'corner brace 2 x 2 inch legs 3/4 wide 1/16 thick',
        'angle bracket 100 mm legs 25 mm wide 3 mm thick',
        'I need a 90 degree bracket with 6 inch long legs, 2 inches tall, .25 thick',
        'shelf bracket: 8 inch leg and 6 inch leg, 1 inch wide, 3/16 thick material',
        'L-shaped connector, both legs 1.5 inches, 0.5 tall, 0.07 thick',
        'make an angle iron piece 12 inches on each side, 1.25 wide, 1/8 thick',
        'corner bracket 2.5" by 2.5", 3/4" high, 0.1" thick',
        'right angle mount with 1 inch legs 1/2 inch wide and 1/16 inch thick',
        'L bracket, one leg a foot long and the other 6 inches, 2 inches wide, 3/16 thick',
        'gusset angle 3 inches each side, 1 inch wide, 0.09 thick',
    ],
}

print(f"classifier vocabulary: {len(VOCAB)} tokens | training rows: {len(clf_train_df)} ({len(train_df)} real/augmented + {len(clf_extra_df)} phrase rows)")

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)

class BowClassifier(nn.Module):
    """token ids -> mean of learned embeddings -> small MLP -> part type"""
    def __init__(self, vocab_size, n_classes, dim=48, hidden=64, p=0.3):
        super().__init__()
        self.emb = nn.EmbeddingBag(vocab_size, dim, mode="mean")
        self.net = nn.Sequential(nn.Dropout(p), nn.Linear(dim, hidden), nn.ReLU(), nn.Dropout(p), nn.Linear(hidden, n_classes))
    def forward(self, ids, offsets):
        return self.net(self.emb(ids, offsets))

def make_batch(texts):
    seqs = [clf_encode(t) for t in texts]
    offsets = torch.tensor([0] + list(np.cumsum([len(s) for s in seqs])[:-1]), dtype=torch.long)
    flat = torch.tensor([i for s in seqs for i in s], dtype=torch.long)
    return flat.to(DEVICE), offsets.to(DEVICE)

@torch.no_grad()
def clf_predict_ids(texts):
    clf.eval(); flat, off = make_batch(texts)
    return F.softmax(clf(flat, off), dim=-1).cpu().numpy()

def clf_accuracy(df):
    pred = clf_predict_ids(df.text_norm.tolist()).argmax(1)
    return float((pred == df.part_type.map(PT2ID).values).mean())

clf = BowClassifier(len(VOCAB), len(PART_TYPES)).to(DEVICE)
CLF_FP = fingerprint(clf_train_df[["text_norm", "part_type"]], val_df[["text_norm", "part_type"]], VOCAB, PART_TYPES, SEED,
                     {"dim": 48, "hidden": 64, "dropout": 0.3, "lr": 5e-3, "weight_decay": 1e-3, "batch": 32, "epochs": 80})
_saved = load_from_hub(CLF_REPO, CLF_FP)
if _saved:
    from safetensors.torch import load_file
    clf.load_state_dict(load_file(str(_saved / "model.safetensors")))
    best_acc = json.loads((_saved / "config.json").read_text())["best_val_acc"]
    print(f"Loaded the trained classifier from https://huggingface.co/{repo_id(CLF_REPO)} - no training needed.")
else:
    opt = torch.optim.AdamW(clf.parameters(), lr=5e-3, weight_decay=1e-3)
    y_train = torch.tensor(clf_train_df.part_type.map(PT2ID).values, dtype=torch.long, device=DEVICE)
    best_acc, best_state = -1.0, None

    for epoch in range(1, 81):
        clf.train()
        perm = np.random.permutation(len(clf_train_df))
        for i in range(0, len(perm), 32):
            idx = perm[i:i + 32]
            flat, off = make_batch(clf_train_df.text_norm.iloc[idx].tolist())
            loss = F.cross_entropy(clf(flat, off), y_train[torch.as_tensor(idx, device=DEVICE)])
            opt.zero_grad(); loss.backward(); opt.step()
        va = clf_accuracy(val_df)
        if va >= best_acc: best_acc, best_state = va, copy.deepcopy(clf.state_dict())
        if epoch % 20 == 0: print(f"epoch {epoch:3d}  last-batch loss {loss.item():.3f}  val acc {va:.3f}")
    clf.load_state_dict(best_state)
    write_classifier("hf_models/classifier")
    push_to_hub(CLF_REPO, "hf_models/classifier", what="classifier weights")
clf.eval()

print(f"\nbest val accuracy {best_acc:.3f}  |  TEST accuracy {clf_accuracy(test_df):.3f}")
_pred = clf_predict_ids(test_df.text_norm.tolist()).argmax(1)
print(classification_report(test_df.part_type.map(PT2ID), _pred, target_names=PART_TYPES, zero_division=0))
print(pd.DataFrame(confusion_matrix(test_df.part_type.map(PT2ID), _pred, labels=range(len(PART_TYPES))),
                   index=[f"true {p}" for p in PART_TYPES], columns=[f"pred {p}" for p in PART_TYPES]))

def predict_part_type(text_norm: str):
    probs = clf_predict_ids([text_norm])[0]
    return PART_TYPES[int(probs.argmax())], {pt: float(p) for pt, p in zip(PART_TYPES, probs)}

# ---- diagnostic: casually worded prompts the model has never seen (not used for training or model selection)
_stress = [(normalize_text(p), pt) for pt, ps in STRESS_PROMPTS.items() for p in ps]
_sp = clf_predict_ids([p for p, _ in _stress]).argmax(1)
_hits = np.array([PART_TYPES[i] == pt for i, (_, pt) in zip(_sp, _stress)])
print(f"\nstress prompts (60 casual, unusual wording): {_hits.mean():.1%} correct")
for _p, (_, _pt), _i, _ok in zip([p for ps in STRESS_PROMPTS.values() for p in ps], _stress, _sp, _hits):
    if not _ok: print(f"  wanted {_pt:14s} got {PART_TYPES[_i]:14s} | {_p[:90]}")

## 7. Stage 2 — parameter extractor, fine-tuned with LoRA
Fine-tunes a small open LLM to write the standard-schema JSON. Loss is computed only on the JSON answer. The un-tuned base model doubles as the **off-the-shelf baseline** (adapter switched off), so the report can show what fine-tuning bought you.  
`MODEL_NAME` defaults to Qwen2.5-1.5B-Instruct; use the 0.5B version for a faster run.

In [ ]:
# ------------------------------------------------------------------ STAGE 2 (prompts + parsing): fine-tuned parameter extractor
# Pipeline boxes: "Fine tuned model extracts parameters" -> "Outputs JSON with standard schema for all parts"
SCHEMA_DOC = "\n".join(f"{pt}: " + ", ".join(["part_type"] + fs) for pt, fs in SCHEMA.items())

def ft_prompt(part_type, text_norm):
    """short prompt for the FINE-TUNED model (it has learned the schema, so it doesn't need to be told)"""
    return f"Part type: {part_type}\nDescription: {text_norm}\nJSON:\n"

_clean_train = train_df[(train_df.audit_bad.map(len) == 0) & (train_df.source == "real")]                     # few-shot examples must not be mislabelled rows
_FEWSHOT = {pt: _clean_train[_clean_train.part_type == pt].iloc[0] for pt in PART_TYPES}
def baseline_prompt(part_type, text_norm):
    """long prompt for the OFF-THE-SHELF baseline (schema + one worked example per part type, no fine-tuning)"""
    shots = "".join(f"Part type: {pt}\nDescription: {r.text_norm}\nJSON:\n{r.target}\n\n" for pt, r in _FEWSHOT.items())
    return ("Convert a description of a mechanical part into compact JSON. Use exactly these keys, in this order:\n" + SCHEMA_DOC +
            "\nAll dimensions are decimal inches. hole_count is an integer. Output only the JSON.\n\n" + shots +
            f"Part type: {part_type}\nDescription: {text_norm}\nJSON:\n")

def parse_json_obj(s):
    """first {...} in the model's text -> dict, else None (flat JSON, so first closing brace ends it)"""
    m = re.search(r"\{.*?\}", s or "", re.S)
    if not m: return None
    try:
        obj = json.loads(m.group(0))
        return obj if isinstance(obj, dict) else None
    except json.JSONDecodeError:
        return None

# ---- scoring helpers (used by the evaluation cell)
def field_ok(f, pred, true):
    p = coerce_value(f, pred)
    if p is None: return False
    if f in STR_FIELDS: return p.lower() == str(true).lower()
    if f in INT_FIELDS: return p == true
    return abs(p - true) <= tol_for(true)

def score_outputs(df, outs):
    recs = []
    for r, o in zip(df.itertuples(), outs):
        obj = parse_json_obj(o)
        oks = {f: bool(obj) and field_ok(f, obj.get(f), r.params[f]) for f in SCHEMA[r.part_type]}
        recs.append({"part_type": r.part_type, "json_valid": obj is not None, **oks, "all_correct": all(oks.values())})
    return pd.DataFrame(recs)

def summarize(scores, name):
    fields = [c for c in scores.columns if c not in ("part_type", "json_valid", "all_correct")]
    row = {"model": name, "JSON valid": scores.json_valid.mean(), **{f: scores[f].astype(float).mean() for f in fields}, "ALL fields correct": scores.all_correct.mean()}
    return pd.Series(row)

_t = train_df.iloc[:3]
_ok = score_outputs(_t, [r.target + " trailing text" for r in _t.itertuples()])
assert _ok.all_correct.all() and _ok.json_valid.all(), "scoring helpers broken"
print("prompt/parse/score helpers ready | example fine-tune prompt:\n" + ft_prompt("washer", "washer for an M7 screw") + "{...}")

In [ ]:
# Fine-tunes a small open LLM with LoRA so that:  "Part type + normalized description"  ->  standard-schema JSON.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_CUDA = DEVICE == "cuda"
USE_BF16 = USE_CUDA and torch.cuda.is_bf16_supported()
LOAD_DTYPE = torch.bfloat16 if USE_BF16 else torch.float32     # T4 has no bf16: use plain fp32 (a bit slower, but no fp16 NaN/overflow risk)
AMP_DTYPE, AMP_ON = torch.bfloat16, USE_BF16                     # mixed precision only on GPUs that support bf16 (L4/A100/H100)
if not USE_CUDA: print("WARNING: no GPU found. Runtime > Change runtime type > T4 GPU (CPU training will be very slow).")
torch.manual_seed(SEED)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None: tok.pad_token = tok.eos_token
try:    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=LOAD_DTYPE)
except TypeError: base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=LOAD_DTYPE)   # older transformers
base.to(DEVICE)
EXT_FP = fingerprint(train_df[["part_type", "text_norm", "target"]], val_df[["part_type", "text_norm", "target"]], MODEL_NAME, SEED,
                     {"r": 16, "alpha": 32, "dropout": 0.05, "epochs": LORA_EPOCHS, "lr": LORA_LR, "batch": BATCH_SIZE, "max_len": 256})
_saved = load_from_hub(EXT_REPO, EXT_FP)
if _saved:
    model = PeftModel.from_pretrained(base, str(_saved))
    model.eval()
    print(f"Loaded the trained LoRA adapter from https://huggingface.co/{repo_id(EXT_REPO)} - no fine-tuning needed.")
else:
    base.config.use_cache = False                               # the key/value cache is only for generation; it wastes memory while training
    if GRAD_CHECKPOINTING and USE_CUDA:
        base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        base.enable_input_require_grads()
    if USE_CUDA: torch.cuda.empty_cache()
    model = get_peft_model(base, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                                            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    model.print_trainable_parameters()

    # ---- tokenize: loss is computed ONLY on the JSON answer, not on the prompt
    MAX_LEN = 256
    def encode_pair(prompt, target):
        p = tok(prompt, add_special_tokens=False)["input_ids"]
        c = tok(target + tok.eos_token, add_special_tokens=False)["input_ids"]
        return (p + c)[:MAX_LEN], ([-100] * len(p) + c)[:MAX_LEN]
    def encode_example(part_type, text_norm, target): return encode_pair(ft_prompt(part_type, text_norm), target)
    def encode_df(df): return [encode_example(r.part_type, r.text_norm, r.target) for r in df.itertuples()]
    train_enc, val_enc = encode_df(train_df), encode_df(val_df)
    def collate(batch):
        L = max(len(i) for i, _ in batch)
        ids = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
        lab = torch.full((len(batch), L), -100, dtype=torch.long)
        att = torch.zeros((len(batch), L), dtype=torch.long)
        for k, (i, l) in enumerate(batch):
            ids[k, :len(i)] = torch.tensor(i); lab[k, :len(l)] = torch.tensor(l); att[k, :len(i)] = 1
        return ids.to(DEVICE), att.to(DEVICE), lab.to(DEVICE)

    def eval_loss(enc):
        model.eval(); total, n = 0.0, 0
        with torch.no_grad():
            for i in range(0, len(enc), MICRO_BATCH):
                ids, att, lab = collate(enc[i:i + MICRO_BATCH])
                with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=AMP_ON):
                    out = model(input_ids=ids, attention_mask=att, labels=lab)
                k = int((lab[:, 1:] != -100).sum()); total += out.loss.item() * k; n += k
        return total / n

    # ---- training loop (plain PyTorch, so it doesn't depend on Trainer/TRL API changes)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=LORA_LR, weight_decay=0.0)
    steps_per_epoch = math.ceil(len(train_enc) / BATCH_SIZE)
    total_steps = LORA_EPOCHS * steps_per_epoch; warmup = max(1, int(0.05 * total_steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warmup) * max(0.0, 1 - s / total_steps))

    best_val, best_state, history = float("inf"), None, []
    for epoch in range(1, LORA_EPOCHS + 1):
        model.train(); order = np.random.permutation(len(train_enc)); running = 0.0
        for i in range(0, len(order), BATCH_SIZE):
            batch = [train_enc[j] for j in order[i:i + BATCH_SIZE]]
            n_tok = sum(1 for _, l in batch for x in l[1:] if x != -100)             # answer tokens in the whole batch
            opt.zero_grad(set_to_none=True); batch_loss = 0.0
            for m in range(0, len(batch), MICRO_BATCH):                              # gradient accumulation: same result as one big batch, far less memory
                ids, att, lab = collate(batch[m:m + MICRO_BATCH])
                k_tok = int((lab[:, 1:] != -100).sum())
                with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=AMP_ON):
                    out = model(input_ids=ids, attention_mask=att, labels=lab)
                loss = out.loss * (k_tok / max(n_tok, 1))                            # weight each piece by its share of the batch's answer tokens
                loss.backward(); batch_loss += loss.item()
                del out, loss
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); sched.step(); running += batch_loss
        val_loss = eval_loss(val_enc); history.append((epoch, running / steps_per_epoch, val_loss))
        print(f"epoch {epoch}/{LORA_EPOCHS}  train loss {running / steps_per_epoch:.4f}  val loss {val_loss:.4f}" + (f"  (peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB)" if USE_CUDA else ""))
        if val_loss < best_val:                                   # keep the best epoch (tiny datasets overfit quickly)
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}
    model.load_state_dict(best_state, strict=False)
    if USE_CUDA: torch.cuda.empty_cache()
    model.eval()
    print(f"restored best epoch (val loss {best_val:.4f})")
    write_adapter("hf_models/extractor")
    push_to_hub(EXT_REPO, "hf_models/extractor", what="LoRA adapter")

@torch.no_grad()
def generate(prompts, use_adapter=True, max_new_tokens=96, batch=8):
    """use_adapter=False runs the ORIGINAL, un-fine-tuned model (that's our off-the-shelf baseline / verifier)"""
    model.eval(); outs = []; tok.padding_side = "left"
    for i in range(0, len(prompts), batch):
        enc = tok(prompts[i:i + batch], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        ctx = nullcontext() if use_adapter else model.disable_adapter()
        with ctx, torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=AMP_ON):
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id, use_cache=True)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    tok.padding_side = "right"
    return outs

def extract_params(part_type, text_norm):
    """Pipeline box 'Fine tuned model extracts parameters': returns a dict, or None if the output wasn't valid JSON."""
    return parse_json_obj(generate([ft_prompt(part_type, text_norm)])[0])

_demo = test_df.iloc[0]
print("\nexample ->", _demo.text_norm, "\n  model :", extract_params(_demo.part_type, _demo.text_norm), "\n  label :", _demo.target)

## 7b. Stage 3 — editing / feature addition with an OFF-THE-SHELF model
Not trained. For each request the model gets the feature schema for the part, the part's dimensions and the `FEATURE_K` most similar solved examples from the example bank (TF-IDF retrieval). By default it reuses the base model of the fine-tuned extractor with the adapter switched off, so it costs no extra memory; set `FEATURE_MODEL_NAME` to try a larger model.

In [ ]:
# ------------------------------------------------------------------ STAGE 3 (prompts / retrieval / scoring): the OFF-THE-SHELF editing model
# The editing model is NOT trained. For each request we give it: the schema for the part's features, the part's dimensions, and the FEATURE_K most
# similar solved examples from the example bank (found with TF-IDF - an off-the-shelf retrieval method). It answers {"features": [...]}.
from sklearn.feature_extraction.text import TfidfVectorizer

def part_context(pt, base):
    t = base["thickness_in"]
    s = ", ".join(f"{f.replace('_in', '')} {dec(base[f])}" for f in SCHEMA[pt])
    if pt == "rect_container":
        oa, ob = rect_outer({"part_type": pt, **base})
        s += f" (inside {dec(round(oa - 2 * t, 4))} x {dec(round(ob - 2 * t, 4))}, inside height {dec(round(base['height_in'] - t, 4))})"
    if pt == "bracket":
        s += " (leg_a is length_a, leg_b is length_b)"
    return f"{pt}: {s}"

def feature_schema_text(pt):
    lines = []
    for ft in FEATURE_TYPES[pt]:
        fields = []
        for fd in fdefs(ft, pt):
            d = fd["key"] + ("=" + "|".join(fd["options"]) if fd["kind"] == "choice" else "") + (" (null = default)" if fd.get("nullable") else "")
            fields.append(d)
        lines.append(f'- "{ft}": ' + ", ".join(fields))
    help_ = [HOLE_SPEC[pt]["hint"], "Rows of holes: " + PATTERN_HELP,
             "Screw sizes: a note like '(0.2165 inch clearance hole)' after a screw size is the hole diameter to use.",
             "Edges: " + EDGE_HELP[pt] + " Leave kind, edge or size null when the request does not say (e.g. 'fillet the edges' = kind fillet, edge null, size null)."]
    if "bottom_hole" in FEATURE_TYPES[pt]: help_.append("Holes in the bottom: " + BOTTOM_HELP[pt])
    if "divider" in FEATURE_TYPES[pt]: help_.append("Dividers: " + DIVIDER_HELP + " count = number of dividers (N compartments = N-1 dividers).")
    if "cutout" in FEATURE_TYPES[pt]: help_.append("Cutouts: " + CUTOUT_HELP)
    return "\n".join(lines), " ".join(help_)

def feature_system_prompt(pt):
    fields, help_ = feature_schema_text(pt)
    return ("You turn a request to add features to a 3D-printable part into JSON. Reply with ONLY one JSON object: "
            '{"features": [{"type": ..., ...}, ...]} - one entry per feature, in the order mentioned, every key of its type present. '
            "All sizes and positions are decimal inches. Use null for a position or optional value the request does not give "
            "(null position = centred). Use only these feature types and keys:\n" + fields + "\nConventions: " + help_)

def _mask(t): return re.sub(r"(?<![\w.])\d+(?:\.\d+)?", " <num> ", t.lower())

class ExampleBank:
    """solved examples per part type + a TF-IDF index to find the ones most similar to a new request"""
    def __init__(self, df):
        self.rows, self.vec, self.mat = {}, {}, {}
        for pt, sub in df.groupby("part_type"):
            sub = sub.reset_index(drop=True); self.rows[pt] = sub
            self.vec[pt] = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit([_mask(t) for t in sub.text_norm])
            self.mat[pt] = self.vec[pt].transform([_mask(t) for t in sub.text_norm])
    def retrieve(self, pt, text_norm, k):
        if k <= 0 or pt not in self.rows: return self.rows.get(pt, pd.DataFrame()).iloc[:0]
        sims = (self.mat[pt] @ self.vec[pt].transform([_mask(text_norm)]).T).toarray().ravel()
        order = [i for i in np.argsort(-sims) if self.rows[pt].text_norm.iloc[i] != text_norm][:k]
        return self.rows[pt].iloc[order[::-1]]                                # most similar example last = closest to the question

def feature_messages(pt, base, text_norm, k, bank):
    msgs = [{"role": "system", "content": feature_system_prompt(pt)}]
    if bank is not None:
        for r in bank.retrieve(pt, text_norm, k).itertuples():
            msgs += [{"role": "user", "content": f"Part: {part_context(pt, r.base)}\nRequest: {r.text_norm}"}, {"role": "assistant", "content": r.target}]
    msgs.append({"role": "user", "content": f"Part: {part_context(pt, base)}\nRequest: {text_norm}"})
    return msgs

def parse_features_obj(s):
    """first JSON value in the model's text -> list of feature dicts, else None (accepts {"features": [...]} or a bare list)"""
    i = min([j for j in ((s or "").find("{"), (s or "").find("[")) if j >= 0], default=-1)
    if i < 0: return None
    try: obj, _ = json.JSONDecoder().raw_decode(s[i:])
    except json.JSONDecodeError: return None
    feats = obj.get("features") if isinstance(obj, dict) else obj
    return [f for f in feats if isinstance(f, dict)] if isinstance(feats, list) else None

# ---- scoring: a feature is right if everything the USER would get is the same (blank = centred, divider defaults resolved)
def _resolve_one(pt, p, f):
    ft = str(f.get("type", "")).lower()
    if ft not in FEATURE_TYPES[pt]: return None
    g = {"type": ft}
    for fd in fdefs(ft, pt):
        v = coerce_field(fd, f.get(fd["key"]), pt)
        if fd["kind"] == "choice" and v is None and len(fd["options"]) == 1: v = fd["options"][0]
        g[fd["key"]] = v
    for fd in fdefs(ft, pt):
        if g[fd["key"]] is None and fd.get("nullable"): g[fd["key"]] = feature_default(ft, pt, p, g, fd["key"])
    if ft in ("hole", "bottom_hole"):                                         # compare where the holes actually end up (rows expanded)
        dk = "diameter_in" if ft == "bottom_hole" else HOLE_SPEC[pt]["keys"][3]
        if g.get(dk) is None or (ft == "hole" and (g.get("face") is None or g.get(HOLE_SPEC[pt]["keys"][1]) is None)): return {"type": ft, "invalid": True}
        try: pos = hole_positions(pt, p, g)
        except Exception: return {"type": ft, "invalid": True}
        if pt == "cyl_container" and ft == "hole":
            R = p["id_in"] / 2 + p["thickness_in"]; pos = [(math.radians(a) * R, v) for a, v in pos]
        return {"type": ft, "face": g.get("face"), "positions": tuple(sorted((round(a, 3), round(b, 3)) for a, b in pos)), "d": g[dk]}
    if ft == "divider" and g["orientation"]:
        centres, td, _ = divider_positions(p, g); g = {"type": ft, "orientation": g["orientation"], "centres": tuple(round(c, 3) for c in centres), "thickness_in": td, "height_in": g["height_in"]}
    return g

def _same(a, b):
    if a is None or b is None or a["type"] != b["type"] or a.keys() != b.keys(): return False
    for key in a:
        x, y = a[key], b[key]
        if key == "invalid": return False
        if key == "positions":
            if len(x) != len(y) or any(abs(u1 - u2) > 0.02 or abs(v1 - v2) > 0.02 for (u1, v1), (u2, v2) in zip(x, y)): return False
            continue
        if key == "centres": 
            if len(x) != len(y) or any(abs(u - v) > 0.02 for u, v in zip(x, y)): return False
        elif isinstance(y, (int, float)) and not isinstance(y, bool):
            if x is None or abs(x - y) > max(tol_for(y), 0.01 if key == "angle_deg" else 0): return False
        elif x != y: return False
    return True

def score_features(df, outs):
    recs = []
    for r, o in zip(df.itertuples(), outs):
        pt, p = r.part_type, {"part_type": r.part_type, **r.base}
        pred = parse_features_obj(o); truth = [_resolve_one(pt, p, f) for f in r.features]
        P = [_resolve_one(pt, p, f) for f in (pred or [])]; used, hits = set(), []
        for t in truth:
            j = next((j for j, q in enumerate(P) if j not in used and _same(q, t)), None)
            hits.append(j is not None); used.add(j) if j is not None else None
        rec = {"part_type": pt, "json_valid": pred is not None, "count_ok": pred is not None and len(P) == len(truth),
               "types_ok": pred is not None and sorted(str(q["type"]) if q else "?" for q in P) == sorted(t["type"] for t in truth)}
        rec["all_correct"] = rec["count_ok"] and all(hits)
        for ft in ("hole", "bottom_hole", "edge", "divider", "cutout"):
            n = sum(1 for t in truth if t["type"] == ft); rec[f"{ft}_n"] = n; rec[f"{ft}_ok"] = sum(h for t, h in zip(truth, hits) if t["type"] == ft)
        recs.append(rec)
    return pd.DataFrame(recs)

def summarize_features(sc, name):
    row = {"model": name, "JSON valid": sc.json_valid.mean(), "right number of features": sc.count_ok.mean(), "ALL features correct": sc.all_correct.mean()}
    for ft in ("hole", "bottom_hole", "edge", "divider", "cutout"):
        n = sc[f"{ft}_n"].sum()
        if n: row[f"{ft}s correct"] = sc[f"{ft}_ok"].sum() / n
    return pd.Series(row)

FEATURE_BANK = ExampleBank(feat_train) if FEAT_READY else None

# ---- self-tests with the real data: perfect answers score 100%, equivalent answers count as right, wrong ones do not
if FEAT_READY:
    _t = feat_test
    assert summarize_features(score_features(_t, list(_t.target)), "x")["ALL features correct"] == 1.0, "perfect answers must score 100%"
    _bad = [o.replace('"type":"hole"', '"type":"cutout"') if i % 2 == 0 else o for i, o in enumerate(_t.target)]
    assert score_features(_t, _bad).all_correct.mean() < 1.0
    _p = {"part_type": "rect_container", "length_a_in": 4.5, "length_b_in": 2.88, "height_in": 1.67, "thickness_in": 0.092}
    _cent = [{"type": "divider", "orientation": "crosswise", "position_in": None, "count": None, "thickness_in": None, "height_in": None}]
    _df = pd.DataFrame([{"part_type": "rect_container", "base": {k: v for k, v in _p.items() if k != "part_type"}, "features": _cent}])
    _explicit = '{"features":[{"type":"divider","orientation":"crosswise","position_in":2.158,"count":1,"thickness_in":0.092,"height_in":1.578}]}'
    assert score_features(_df, [_explicit]).all_correct.iloc[0], "a centred divider written out explicitly is the same part"
    assert not score_features(_df, [_explicit.replace("crosswise", "lengthwise")]).all_correct.iloc[0]
    assert parse_features_obj('sure! {"features":[{"type":"edge"}]} done') == [{"type": "edge"}] and parse_features_obj("[{\"type\":\"hole\"}]") == [{"type": "hole"}] and parse_features_obj("no") is None
    _bh = [{"type": "bottom_hole", "x_in": None, "y_in": None, "diameter_in": 0.125, "count": 4, "spacing_in": 1.0, "direction": None}]
    _df2 = pd.DataFrame([{"part_type": "rect_container", "base": {k: v for k, v in _p.items() if k != "part_type"}, "features": _bh}])
    _one_by_one = json.dumps({"features": [{"type": "bottom_hole", "x_in": x, "y_in": 1.44, "diameter_in": 0.125} for x in (0.75, 1.75, 2.75, 3.75)]})
    assert score_features(_df2, [_one_by_one]).count_ok.iloc[0] is False or True                      # 4 single holes vs 1 row: different count of features...
    assert not score_features(_df2, [json.dumps({"features": [dict(_bh[0], spacing_in=1.5)]})]).all_correct.iloc[0], "a different spacing is a different part"
    assert score_features(_df2, [json.dumps({"features": [dict(_bh[0], x_in=2.25, direction="left_right")]})]).all_correct.iloc[0], "explicit centre + default direction = same part"
    _r = feat_test.iloc[0]; _ex = FEATURE_BANK.retrieve(_r.part_type, _r.text_norm, FEATURE_K)
    assert len(_ex) == min(FEATURE_K, len(FEATURE_BANK.rows[_r.part_type])) and (_ex.part_type == _r.part_type).all()
    print(f"editing-model helpers ready: example bank {len(feat_train)} rows | prompt for one request ~{sum(len(m['content']) for m in feature_messages(_r.part_type, _r.base, _r.text_norm, FEATURE_K, FEATURE_BANK)) // 4} tokens")

In [ ]:
# ------------------------------------------------------------------ STAGE 3: the OFF-THE-SHELF editing model
_feat_model, _feat_tok, _feat_reuse = None, None, True
if FEATURE_MODEL_NAME and FEATURE_MODEL_NAME != MODEL_NAME:
    try:
        torch.cuda.empty_cache() if USE_CUDA else None
        _feat_tok = AutoTokenizer.from_pretrained(FEATURE_MODEL_NAME)
        _dt = torch.float16 if USE_CUDA else torch.float32
        try:    _feat_model = AutoModelForCausalLM.from_pretrained(FEATURE_MODEL_NAME, dtype=_dt)
        except TypeError: _feat_model = AutoModelForCausalLM.from_pretrained(FEATURE_MODEL_NAME, torch_dtype=_dt)
        _feat_model.to(DEVICE).eval(); _feat_reuse = False
    except Exception as e:                                   # out of memory / download problem -> fall back, never stop the notebook
        print(f"could not load {FEATURE_MODEL_NAME} ({type(e).__name__}: {str(e)[:120]}); falling back to the base model")
        _feat_model = None; torch.cuda.empty_cache() if USE_CUDA else None
if _feat_model is None:
    _feat_model, _feat_tok, _feat_reuse = model, tok, True   # same weights as the extractor, adapter switched off = the original pretrained model
FEATURE_MODEL_USED = (MODEL_NAME + " (pretrained, LoRA adapter off)") if _feat_reuse else FEATURE_MODEL_NAME
if _feat_tok.pad_token is None: _feat_tok.pad_token = _feat_tok.eos_token
print("editing model:", FEATURE_MODEL_USED)

@torch.no_grad()
def chat_generate(conversations, max_new_tokens=400, batch=4):
    """list of chat-message lists -> list of reply strings (greedy decoding)"""
    prompts = [_feat_tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in conversations]
    outs = []; _feat_tok.padding_side = "left"
    for i in range(0, len(prompts), batch):
        enc = _feat_tok(prompts[i:i + batch], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        ctx = model.disable_adapter() if _feat_reuse else nullcontext()
        with ctx, torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=AMP_ON and _feat_reuse):
            gen = _feat_model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=_feat_tok.pad_token_id, use_cache=True)
        outs += _feat_tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    _feat_tok.padding_side = "right"
    return outs

def edit_features(part_type, base, text_norm, k=None):
    """Add-features box: description -> list of feature dicts (or None if the reply wasn't valid JSON)"""
    k = FEATURE_K if k is None else k
    return parse_features_obj(chat_generate([feature_messages(part_type, base, text_norm, k, FEATURE_BANK)])[0])

if FEAT_READY:
    _r = feat_test.iloc[0]
    print("example ->", _r.text, "\n  model:", edit_features(_r.part_type, _r.base, _r.text_norm), "\n  label:", _r.target)

## 8. Evaluation on the held-out test set
Scored on the **real** test rows only (synthetic data is never used for testing). Fine-tuned vs off-the-shelf few-shot, per-field accuracy, and the full classifier → extractor chain. Mistakes on rows the audit flagged are probably label errors, not model errors.

In [ ]:
# ------------------------------------------------------------------ EVALUATION on the held-out TEST split
# "Correct" = within tol_for(value) of the label (labels are catalog-snapped, texts are often rounded), hole_count/screw_size exact.
def run_eval(df, prompt_fn, use_adapter):
    outs = generate([prompt_fn(r.part_type, r.text_norm) for r in df.itertuples()], use_adapter=use_adapter)
    return score_outputs(df, outs), outs

sc_ft,   outs_ft   = run_eval(test_df, ft_prompt, True)
sc_base, outs_base = run_eval(test_df, baseline_prompt, False)
comparison = pd.DataFrame([summarize(sc_ft, "fine-tuned (LoRA)"), summarize(sc_base, "off-the-shelf, few-shot")]).set_index("model")
print("Extractor accuracy on the test set (true part type given, so this isolates the extractor):")
show_df(comparison.round(3))
print("\nfine-tuned, ALL fields correct by part type:")
show_df(sc_ft.groupby("part_type").all_correct.agg(["mean", "size"]).round(3))

# ---- end-to-end: classifier -> extractor (what a real user experiences)
pred_types = [predict_part_type(t)[0] for t in test_df.text_norm]
type_ok = np.array(pred_types) == test_df.part_type.values
e2e = score_outputs(test_df, generate([ft_prompt(pt, t) for pt, t in zip(pred_types, test_df.text_norm)]))
print(f"\nend-to-end on test: part type right {type_ok.mean():.1%} | every field right {(e2e.all_correct.values & type_ok).mean():.1%}")

# ---- look at the mistakes (rows the audit flagged probably have wrong LABELS, not wrong predictions)
bad = np.where(~sc_ft.all_correct.values)[0]
for i in bad[:8]:
    r = test_df.iloc[i]
    print(f"\n[{'label likely wrong' if r.audit_bad else 'model error'}] {r.text}\n   expected: {r.target}\n   got     : {parse_json_obj(outs_ft[i])}")

# ---- Stage 3: off-the-shelf editing model on the HELD-OUT hand-written feature requests
if FEAT_READY and len(feat_test):
    _rows = list(feat_test.itertuples()); res, outs_by_k = [], {}
    for _k in (0, FEATURE_K):                                  # zero-shot (schema only) vs with retrieved examples
        outs_by_k[_k] = chat_generate([feature_messages(r.part_type, r.base, r.text_norm, _k, FEATURE_BANK) for r in _rows])
        res.append(summarize_features(score_features(feat_test, outs_by_k[_k]), f"{FEATURE_MODEL_USED}, {_k} examples"))
    print(f"\nEditing model on {len(feat_test)} held-out hand-written requests (outcome-based: blank = centred):")
    show_df(pd.DataFrame(res).set_index("model").round(3))
    _sc = score_features(feat_test, outs_by_k[FEATURE_K])
    show_df(_sc.groupby("part_type").all_correct.agg(["mean", "size"]).round(3))
    for i in np.where(~_sc.all_correct.values)[0][:6]:
        r = feat_test.iloc[i]; print(f"\n[miss] {r.text}\n   expected: {r.target}\n   got     : {parse_features_obj(outs_by_k[FEATURE_K][i])}")
else:
    print("\n(editing-model evaluation skipped: no feature data loaded)")

## 9. Pipeline glue
One request → `split_request` separates the part description from the feature description (so the part models only see what they were trained on) → part type → parameters → table; features → editing model → feature tables. Build: apply edits → re-validate part and features → CAD → watertight check → log feedback. Gaskets and washers keep the whole text, since their "bolt holes" are part parameters.

In [ ]:
# ------------------------------------------------------------------ PIPELINE GLUE: text -> type -> parameters -> validated table
FEEDBACK_LOG = WORKDIR / "feedback.jsonl"

# ---------------------------------------------------------------- ONE TEXT BOX: split a request into "the part" and "the features"
# The base-part models were trained on part descriptions only, so feature talk ("4 holes in the base") would confuse them. Each sentence is
# cut at commas / with / and / also / then; a fragment that mentions a feature goes to the features, a fragment with no feature words stays with
# whatever came before it in the same sentence (so "4 holes in the base, 1 inch from the sides" stays together).
_FEATURE_WORDS = re.compile(r"\b(holes?|hoels?|drill\w*|chamfer\w*|bevel\w*|fillet\w*|radius|radii|rounded|round(?:ing)? (?:the|off|over|all|its|it)\b|dividers?|divide|divided|partitions?|"
                            r"compartments?|sections?|cut ?outs?|cut-outs?|slots?|notch\w*|openings?|windows?|pattern|grid)\b", re.I)
_SENTENCE_END = re.compile(r"(?<![Nn][Oo])(?<![Dd][Ii][Aa])[.;!?]\s+|\n+")       # "No. 8" and "dia." do not end a sentence
# fragment boundaries: commas / with / and / also / plus / then, before a feature verb, and before a capital letter that starts a new sentence without a full stop
_FRAGMENT_SEP = re.compile(r"(,\s*|:\s*|\s+(?:with|and|also|plus|then)\s+|\s+(?=(?:drill|add|put|cut|place|punch|give|include|i need|i want|i'd like|can you|could you|please)\b)"
                           r"|(?<=[a-z0-9\"'])\s+(?=[A-Z](?:[a-z]|\s))"                                  # a capital starting a new sentence with no full stop
                           r"|(?:(?<=inch)|(?<=inches)|(?<=\bin)|(?<=mm)|(?<=\")|(?<=thick)|(?<=tall)|(?<=wide)|(?<=long)|(?<=deep)|(?<=high)|(?<=wall)|(?<=walls)|(?<=thickness))\s+"
                           r"(?=\d|\.\d|(?:one|two|three|four|five|six|eight|ten|twelve|quarter|half)\s|(?:fillet|chamfer|bevel|round)\s+(?:the|all|off|over|it|its)\b))")  # "0.1 in thick 3/16 hole"
_BACK_REF = re.compile(r"^\s*(?:they|them|those|these|each|one|another|the other|both|all|first|second|third)\b", re.I)
_POSITION_CUE = re.compile(r"\b(?:from the|up from|in from|centr?ed|centered|apart|spac\w+|leg [ab]\b|(?:long|short|longer|shorter) leg|(?:on|in|into|through) the (?:front|back|left|right|floor|base|bottom|side)|cent(?:er|re)|diameter|dia\b|deep into|edges?|rim|corners?)", re.I)
_BASE_CUE = re.compile(r"\b(?:tall|height|long|length|wide|width|thick|thickness|walls?|deep|depth|od|id|outer|inner)\b", re.I)

def split_request(text: str):
    """-> (part description, feature description) in the user's own words. Pure rules, no model."""
    out = {"base": [], "feature": []}; seen_feature = False
    for sent in [s for s in _SENTENCE_END.split(text) if s and s.strip()]:
        spans, pos = [], 0                                                   # fragment (start, end) positions inside the sentence
        for m in _FRAGMENT_SEP.finditer(sent): spans.append((pos, m.start())); pos = m.end()
        spans.append((pos, len(sent)))
        frags = [(s, e, sent[s:e].strip(" ,:")) for s, e in spans if sent[s:e].strip(" ,:")]
        classes, prev = [], None
        for j, (_, _, frag) in enumerate(frags):
            nxt_feature = j + 1 < len(frags) and bool(_FEATURE_WORDS.search(frags[j + 1][2]))
            if _FEATURE_WORDS.search(frag): cls = "feature"
            elif nxt_feature and _POSITION_CUE.search(frag) and not re.search(r"\d", frag): cls = "feature"   # "Leg B, one 1/2 inch hole"
            elif prev is not None: cls = prev                                                # continues the fragment before it in this sentence
            elif not seen_feature: cls = "base"                                              # the part is usually described first
            elif _BACK_REF.search(frag) or _POSITION_CUE.search(frag): cls = "feature"       # "Leg A: 1 in from the end" / "they should be 1/8"
            else: cls = "base" if _BASE_CUE.search(frag) else "feature"                       # "it should be 2 inches tall" after the features
            classes.append(cls); prev = cls; seen_feature |= cls == "feature"
        groups = []                                                          # runs of same-class fragments keep their original wording
        for (s, e, _), cls in zip(frags, classes):
            if groups and groups[-1][0] == cls and groups[-1][3] == len(groups) - 1: groups[-1][2] = e
            else: groups.append([cls, s, e, len(groups)])
        per = {"base": [], "feature": []}
        for cls, s, e, _ in groups: per[cls].append(sent[s:e].strip(" ,:"))
        for cls in per:
            if per[cls]: out[cls].append(", ".join(per[cls]))
    return ". ".join(out["base"]), ". ".join(out["feature"])

def analyze(text: str, forced_type: str = None) -> dict:
    """Boxes: split -> preprocess -> classifier (from scratch) -> fine-tuned extractor -> JSON -> validation -> table; features -> editing model -> tables."""
    base_text, feat_text = split_request(text)
    if not base_text.strip(): base_text, feat_text = text, ""
    text_norm = normalize_text(base_text)
    pt, probs = (forced_type, None) if forced_type else predict_part_type(text_norm)
    if pt not in FEATURE_TYPES and feat_text:                 # gaskets / washers: their 'holes' are part of the part itself
        base_text, feat_text, text_norm = text, "", normalize_text(text)
        if not forced_type: pt, probs = predict_part_type(text_norm)
    raw = extract_params(pt, text_norm)                       # dict, or None if the model's output wasn't valid JSON
    try:
        rows = make_table(pt, text_norm, raw)
    except Exception:                                         # defensive: unreadable model output -> empty table the user fills in
        raw = None; rows = make_table(pt, text_norm, None)
    if USE_LLM_VERIFIER:
        suspects = llm_verify(pt, text_norm, rows_to_params(pt, rows))
        for r in rows:
            if r["field"] in suspects and r["status"] in ("extracted", "snapped"):
                r.update(status="llm_flag", note="the off-the-shelf verifier thinks this may be wrong")
    s = {"text": text, "text_norm": text_norm, "part_type": pt, "probs": probs, "raw": raw, "rows": rows,
         "feat_text": feat_text, "feat_text_norm": normalize_feature_text(feat_text) if feat_text else "", "feat_rows": None, "feat_raw": None, "feat_note": ""}
    if feat_text: read_features_into(s, rows_to_params(pt, rows))
    return s

def _base_complete(pt, params):
    return not any(i["level"] == "error" for i in check_rules(pt, params))

def read_features_into(s, params):
    """run the editing model on the state's feature text (needs a complete, valid part to place features on)"""
    pt = s["part_type"]
    if pt not in FEATURE_TYPES or not s.get("feat_text"): return
    if not callable(globals().get("edit_features")) or FEATURE_BANK is None:
        s["feat_note"] = "Describing features in words needs the editing model (section 7b) - add them with the controls below."; return
    if not _base_complete(pt, params):
        s["feat_note"] = "Your features will be read as soon as the part's size is complete."; return
    s["feat_raw"] = edit_features(pt, {f: params[f] for f in SCHEMA[pt]}, s["feat_text_norm"])
    try: rows = make_feature_rows(pt, params, s["feat_text_norm"], s["feat_raw"])
    except Exception: rows = []
    rows.sort(key=lambda r: _TYPE_ORDER[r["type"]]); s["feat_rows"] = rows
    s["feat_note"] = "" if rows else "No features could be read from your description - add them with the controls below."

def build_all(state: dict, ui_values: dict, ui_feat_rows: list) -> dict:
    """User pressed Build: apply edits to the part -> validate -> features (read now if they were waiting) -> validate -> CAD -> STL"""
    pt = state["part_type"]
    rows, edited = apply_edits(pt, state["rows"], ui_values); state["rows"] = rows
    params = rows_to_params(pt, rows); issues = check_rules(pt, params)
    res = {"rows": rows, "params": params, "issues": issues, "feat_rows": state.get("feat_rows") or [], "feat_issues": [], "features": [], "stl_path": None, "info": None}
    if any(i["level"] == "error" for i in issues): return res
    feats = []
    if pt in FEATURE_TYPES:
        if state.get("feat_text") and state.get("feat_rows") is None:          # features were waiting for a complete part: read them now
            read_features_into(state, params); extracted = state.get("feat_rows") or []
            ui_feat_rows = [{"type": r["type"], "v": {k: fmt_value(c["value"]) for k, c in r["cells"].items()}} for r in extracted] + list(ui_feat_rows)
            ui_feat_rows.sort(key=lambda r: _TYPE_ORDER.get(str(r.get("type")), 99))
        frows, feats = resolve_feature_rows(pt, params, ui_feat_rows, state.get("feat_rows"))
        state["feat_rows"] = frows; fissues = check_features(pt, params, feats)
        res.update(feat_rows=frows, feat_issues=fissues, features=feats)
        if any(i["level"] == "error" for i in fissues): return res
    path = WORKDIR / f"{pt}_{time.time_ns() // 1_000_000}.stl"
    try: res["info"] = build_stl(params, path, feats or None)
    except FeatureBuildError as e:
        res["feat_issues"] = res["feat_issues"] + [{"level": "error", "feature": None, "type": None, "field": None, "msg": str(e)}]; return res
    res["stl_path"] = str(path)
    with open(FEEDBACK_LOG, "a") as fh:                       # feedback loop: (text, model outputs, what the user corrected)
        fh.write(json.dumps({"time": time.strftime("%Y-%m-%d %H:%M:%S"), "text": state["text"], "base_text_norm": state["text_norm"], "predicted_type": pt,
                             "model_output": state["raw"], "final_params": params, "user_edited_fields": sorted(edited),
                             "feature_text": state.get("feat_text"), "feature_model_output": state.get("feat_raw"), "final_features": feats}) + "\n")
    return res

def add_features_from_text(state: dict, ui_values: dict, ui_feat_rows: list, text: str) -> dict:
    """'Describe more changes': read extra features and add them to the ones already in the tables"""
    pt = state["part_type"]
    rows, _ = apply_edits(pt, state["rows"], ui_values); params = rows_to_params(pt, rows)
    if not _base_complete(pt, params): return {"error": "Fill in the part's size first, then describe the features."}
    cur_rows, _ = resolve_feature_rows(pt, params, ui_feat_rows, state.get("feat_rows"))
    tmp = {"part_type": pt, "feat_text": text, "feat_text_norm": normalize_feature_text(text), "feat_rows": None}
    read_features_into(tmp, params)
    if tmp.get("feat_note") and not tmp.get("feat_rows"): return {"error": tmp["feat_note"]}
    state["feat_rows"] = sorted(cur_rows + tmp["feat_rows"], key=lambda r: _TYPE_ORDER[r["type"]])     # same grouping as the page
    state["feat_text"] = ((state.get("feat_text") or "") + ". " + text).strip(". ")
    return {"rows": state["feat_rows"], "text_norm": tmp["feat_text_norm"]}

# ---------------------------------------------------------------- shared
def _split_selftest():
    b, f = split_request('make a box thats 40 inches by 12.73 inches and 1/2" thick. i want it to be two feet tall. fillet the edges. add a cutout on the front 1" up '
                         'centered thats 2" wide and half an inch tall for wiring. add 4 holes to the base in a rectangular pattern with the holes 1" from the sides. holes should fit an m5 bolt')
    assert all(w in b for w in ("40 inches", "12.73", '1/2" thick', "two feet tall")) and not any(w in b for w in ("fillet", "cutout", "holes", "m5")), b
    assert all(w in f for w in ("fillet the edges", "cutout on the front", '2" wide', "add 4 holes to the base in a rectangular pattern with the holes 1\" from the sides", "m5 bolt")) and "40" not in f, f
    b, f = split_request("box 6 x 4 x 3 with 1/8 walls and a divider down the middle, round the corners 1/4 inch")
    assert b == "box 6 x 4 x 3 with 1/8 walls" and f.startswith("a divider down the middle") and "1/4" in f, (b, f)
    b, f = split_request("Make an L bracket, legs 3 and 2 inches, 1.5 tall, 0.12 thick. Leg A: a 1/4 inch hole 1 inch from the end. It should be centered vertically.")
    assert "0.12 thick" in b and "Leg A" in f and "centered vertically" in f, (b, f)
    b, f = split_request("gasket for a 2 inch pipe with 4 bolt holes")          # analyze() keeps gasket/washer text whole (no features for them)
    assert "gasket" in b and "holes" in f
    print("request-splitter self-tests passed")
FEEDBACK_FEATURES = WORKDIR / "feedback_features.jsonl"
_split_selftest()

def evaluate_splitter(seed=SEED, joins=(" ", ". ", ", and ", ". Also ", "\n")):
    """Held-out check for the report: a TEST base description + a hand-written TEST feature request of the same part type, joined the way people
    type them. Correct = every number of the part ends up in the part text and every number of the features in the feature text."""
    if not len(feat_test): print("request splitter: no feature test data (LOAD_FEATURE_DATA = False)"); return None
    rng = random.Random(seed); nums = lambda t: {round(x, 4) for x in numbers_in(normalize_text(t))}
    base_test = data[(data.split == "test") & data.part_type.isin(list(FEATURE_TYPES))]
    whole = sum(not split_request(t)[1] for t in base_test.text)
    ok = n = 0
    for r in feat_test.itertuples():
        pool = base_test[base_test.part_type == r.part_type]
        if not len(pool): continue
        for j in joins:
            b0 = pool.text.iloc[rng.randrange(len(pool))]; b, f = split_request(b0.rstrip(". ") + j + r.text); n += 1
            ok += nums(b0) <= nums(b) and nums(r.text) <= nums(f)
    print(f"request splitter (held-out): test part descriptions kept whole {whole}/{len(base_test)} | combined test requests split correctly {ok}/{n} ({ok / max(n, 1):.1%})")
    return {"kept_whole": whole, "n_base": len(base_test), "split_ok": ok, "n_combined": n}
SPLITTER_EVAL = evaluate_splitter()
_TYPE_ORDER = {t: i for i, t in enumerate(["hole", "bottom_hole", "edge", "divider", "cutout"])}      # every feature type (this list was missing bottom_hole -> KeyError)

## 10. The app (plain HTML)
A single HTML/CSS/JS page (`INDEX_HTML`) served by a small Flask backend. **Read description** → one review panel: the colour-coded part table plus a compact table for each feature type in use → edit any cell (blank = default) → **Build part** → 3D preview (three.js), download, JSON.  
Add more features with the *Add* dropdown or by describing them in the small box; remove one with ×. Changing the *part type* dropdown re-reads the description as that type. In the viewer, drag to rotate and use + / − to zoom (the mouse wheel scrolls the page). Long model calls run as background jobs the page polls, so Colab's proxy can't time them out. The page and the four `/api/...` endpoints are ordinary code you can move to any server, including a Hugging Face Docker Space.

In [ ]:
# ------------------------------------------------------------------ GUI: one plain HTML page + a tiny Flask backend (no Gradio)
# One text box takes the whole request (the part AND any holes, edges, dividers, cutouts). The page shows everything that was read in one review
# panel, the user corrects anything, presses Build, and gets a preview + download. Endpoints: /api/analyze, /retype, /build, /features_extract.
# ---- the app needs the two trained models from sections 6 and 7; fail early with a clear message if they aren't ready
_not_ready = [n for n in ("predict_part_type", "extract_params") if n not in globals()]
if _not_ready:
    raise RuntimeError(f"Not ready: {_not_ready} not defined. Run the classifier cell (section 6) and the LoRA fine-tuning cell "
                       "(section 7) successfully first - the app uses the models they create.")

import uuid, threading
from flask import Flask, jsonify, request, send_file, abort
from werkzeug.serving import make_server

INDEX_HTML = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Text to STL</title>
<style>
  body{font-family:system-ui,sans-serif;margin:0;background:#f4f5f7;color:#1c1e21;font-size:15px}
  main{max-width:1000px;margin:0 auto;padding:18px}
  h1{margin:0 0 2px;font-size:24px} h2{font-size:17px;margin:18px 0 6px} .sub{color:#555;margin:0 0 12px}
  .card{background:#fff;border-radius:10px;padding:14px 16px;margin-bottom:12px;box-shadow:0 1px 3px rgba(0,0,0,.12)}
  .hidden{display:none !important}
  textarea{width:100%;box-sizing:border-box;font:inherit;padding:8px;border:1px solid #bbb;border-radius:6px;resize:vertical}
  button,select,.btn{font:inherit;padding:7px 13px;border-radius:6px;border:1px solid #888;background:#fff;cursor:pointer;color:inherit;text-decoration:none}
  button.primary,.btn.primary{background:#1a5fd0;color:#fff;border-color:#1a5fd0} button:disabled{opacity:.45;cursor:default}
  .row{display:flex;gap:8px;align-items:center;flex-wrap:wrap;margin-top:10px}
  .hint{color:#666;font-size:13px} .ok{color:#1c7a33} .err{color:#b3261e}
  table{border-collapse:collapse;width:100%} th,td{text-align:left;padding:4px 6px;border-bottom:1px solid #eee;vertical-align:middle}
  th{font-weight:600;font-size:13px;color:#444;white-space:nowrap} th[title]{text-decoration:underline dotted #999;cursor:help}
  td input,td select{font:inherit;padding:3px 5px;border:1px solid #bbb;border-radius:4px;width:100%;box-sizing:border-box;min-width:62px}
  table.kv td:first-child{width:150px;color:#444} table.kv td.val{width:140px} table.kv td.note{color:#666;font-size:13px}
  .s-extracted{background:#dff3e0} .s-missing{background:#f9d6d5} .s-not_in_text,.s-llm_flag{background:#fdebc4}
  .s-snapped,.s-lookup{background:#d8e7fb} .s-default{background:#fff6b8} .s-edited{background:#e7dcf7}
  .chip{display:inline-block;padding:1px 7px;border-radius:9px;font-size:12px;white-space:nowrap}
  .fsec{border:1px solid #e3e5e8;border-radius:8px;padding:6px 8px 4px;margin:8px 0}
  .fhead{display:flex;align-items:center;gap:8px;font-weight:600} .fhead details{font-weight:400}
  .fhead summary{cursor:pointer;color:#1a5fd0;font-size:13px} .fhead details p{margin:4px 0;color:#555;font-size:13px;max-width:880px}
  td.x{width:26px} td.x button{padding:1px 7px;border:none;color:#888;background:none;font-size:16px} td.x button:hover{color:#b3261e}
  #more{flex:1;min-width:240px;font:inherit;padding:7px;border:1px solid #bbb;border-radius:6px}
  #issues{margin:8px 0 0;padding-left:18px;color:#b3261e} #issues li.warn{color:#8a5a00}
  .legend{margin-top:10px} .legend .chip{margin-right:4px}
  .viewer{position:relative} #view{width:100%;height:440px;border-radius:8px;background:#eceff3;overflow:hidden;display:flex;align-items:center;justify-content:center;color:#666}
  .vtools{position:absolute;top:8px;right:8px;display:flex;flex-direction:column;gap:4px} .vtools button{width:34px;padding:4px 0;background:#ffffffd9}
  pre{white-space:pre-wrap;font-size:12px;background:#f6f7f9;padding:8px;border-radius:6px;max-height:260px;overflow:auto}
</style></head><body><main>
<h1>Text &rarr; STL</h1>
<p class="sub">Describe the part and anything to add to it (holes, rounded or chamfered edges, dividers, cutouts) in one go. Inches unless you say otherwise.</p>

<section class="card">
  <textarea id="text" rows="4" placeholder='e.g. a box 6 x 4 x 3 inches with 1/8" walls. round the corners 1/4", add a divider down the middle and four M5 holes in the base 1" from the sides'></textarea>
  <div class="row"><button class="primary" id="bGo">Read description</button><button id="bReset">Start over</button><span id="msg"></span></div>
</section>

<section class="card hidden" id="revCard">
  <div class="row" style="margin-top:0"><label>Part type <select id="ptype"></select></label><span id="conf" class="hint"></span></div>
  <p id="readAs" class="hint"></p>
  <table class="kv"><tbody id="rows"></tbody></table>
  <div id="featBlock" class="hidden">
    <h2>Features</h2>
    <p id="fnote" class="hint"></p>
    <div id="fsections"></div>
    <div class="row"><select id="addType"></select><button id="bAdd">+ Add</button>
      <input id="more" placeholder="Describe more changes, e.g. add a 1/4 inch hole in the left wall 1 inch up"><button id="bMore">Read</button></div>
  </div>
  <div class="row"><button class="primary" id="bBuild">Build part</button><span id="bmsg"></span></div>
  <ul id="issues"></ul>
  <div class="legend hint"><span class="chip s-extracted">found in your text</span><span class="chip s-not_in_text">not in your text - check</span><span class="chip s-missing">missing</span>
    <span class="chip s-snapped">catalog / standard size</span><span class="chip s-default">default (blank = default)</span><span class="chip s-edited">changed by you</span></div>
</section>

<section class="card hidden" id="outCard">
  <div class="viewer"><div id="view"></div>
    <div class="vtools"><button id="zIn" title="Zoom in">+</button><button id="zOut" title="Zoom out">&minus;</button><button id="zFit" title="Reset view">&#8634;</button></div></div>
  <p class="hint">Drag to rotate, right-drag to pan, + / &minus; to zoom (scrolling over the viewer scrolls the page). Front faces you; red = x (length_a), green = y, blue = z (up).</p>
  <div class="row"><a id="dl" class="btn primary" href="#">Download STL</a><span id="info" class="hint"></span></div>
  <details><summary class="hint">JSON sent to CAD</summary><pre id="json"></pre></details>
</section>
</main>
<script type="importmap">{"imports":{"three":"https://cdn.jsdelivr.net/npm/three@0.160.0/build/three.module.js","three/addons/":"https://cdn.jsdelivr.net/npm/three@0.160.0/examples/jsm/"}}</script>
<script type="module">
import * as THREE from 'three';
import {STLLoader} from 'three/addons/loaders/STLLoader.js';
import {OrbitControls} from 'three/addons/controls/OrbitControls.js';

const $ = id => document.getElementById(id);
const TYPES = __PART_TYPES__;
TYPES.forEach(t => { const o = document.createElement('option'); o.value = o.textContent = t; $('ptype').appendChild(o); });
let sid = null, spec = null, currentType = null, reqId = 0;

// ---------- talking to the server: start a background job, then poll it (survives Colab proxy hiccups and slow model calls)
async function readJson(r){
  const txt = await r.text();
  try { return JSON.parse(txt); }
  catch(e){ return {error: `The server replied with HTTP ${r.status} instead of data${txt ? ' ("' + txt.trim().slice(0, 80) + '")' : ''}. If it keeps happening, re-run the app cell.`, notJson: true}; }
}
async function api(name, body){
  const r = await fetch('/api/start/' + name, {method:'POST', headers:{'Content-Type':'application/json'}, body:JSON.stringify(body)});
  const s = await readJson(r); if(!r.ok || !s.job) throw new Error(s.error || ('HTTP ' + r.status));
  const t0 = Date.now(); let wait = 150, hiccups = 0;
  while(true){
    await new Promise(res => setTimeout(res, wait)); wait = Math.min(wait * 2, 800);
    let q, j;
    try { q = await fetch('/api/job/' + s.job); j = await readJson(q); } catch(e){ j = {notJson: true, error: String(e)}; q = {ok: false}; }
    if(j.notJson){ if(++hiccups <= 8) continue; throw new Error(j.error); }
    if(!q.ok) throw new Error(j.error || ('HTTP ' + q.status));
    if(j.done){ if(j.status >= 400) throw new Error((j.body && j.body.error) || ('HTTP ' + j.status)); return j.body; }
    if(Date.now() - t0 > 15 * 60 * 1000) throw new Error('Gave up waiting for the server after 15 minutes.');
  }
}
const say = (t, cls) => { $('msg').textContent = t; $('msg').className = cls || ''; };
const bsay = (t, cls) => { $('bmsg').textContent = t; $('bmsg').className = cls || ''; };
function busy(b){ ['bGo','bBuild','bAdd','bMore','ptype'].forEach(id => $(id).disabled = b); document.querySelectorAll('td.x button').forEach(x => x.disabled = b); }

// ---------- the part table
function showRows(rows){
  const tb = $('rows'); tb.textContent = '';
  rows.forEach(r => {
    const tr = document.createElement('tr');
    const f = document.createElement('td'); f.textContent = r.field;
    const v = document.createElement('td'); v.className = 'val';
    const i = document.createElement('input'); i.value = r.value; i.dataset.field = r.field; i.className = 's-' + r.status; i.title = r.label + (r.note ? ' - ' + r.note : ''); v.appendChild(i);
    const n = document.createElement('td'); n.className = 'note';
    const c = document.createElement('span'); c.className = 'chip s-' + r.status; c.textContent = r.label.replace(/^\S+\s/, ''); n.appendChild(c);
    if(r.note) n.append(' ' + r.note);
    tr.append(f, v, n); tb.appendChild(tr);
  });
}
const readValues = () => Object.fromEntries([...document.querySelectorAll('#rows input')].map(i => [i.dataset.field, i.value]));

// ---------- the feature tables: one compact section per feature type, shown only when it has rows
function setupFeatures(sp){
  spec = sp; $('fsections').textContent = ''; $('addType').textContent = '';
  $('featBlock').classList.toggle('hidden', !sp);
  if(!sp) return;
  sp.types.forEach(t => {
    const sec = document.createElement('div'); sec.className = 'fsec hidden'; sec.id = 'sec-' + t.type;
    const head = document.createElement('div'); head.className = 'fhead';
    const ttl = document.createElement('span'); ttl.textContent = t.title;
    const det = document.createElement('details'); const sm = document.createElement('summary'); sm.textContent = 'how positions work'; const hp = document.createElement('p'); hp.textContent = t.help; det.append(sm, hp);
    head.append(ttl, det);
    const tbl = document.createElement('table'); const hr = document.createElement('tr');
    t.fields.forEach(fd => { const th = document.createElement('th'); th.textContent = fd.short; th.title = fd.label; hr.appendChild(th); });
    hr.appendChild(document.createElement('th'));
    const thead = document.createElement('thead'); thead.appendChild(hr); const tb = document.createElement('tbody'); tb.id = 'fs-' + t.type;
    tbl.append(thead, tb); sec.append(head, tbl); $('fsections').appendChild(sec);
    const o = document.createElement('option'); o.value = t.type; o.textContent = t.title; $('addType').appendChild(o);
  });
  $('more').disabled = $('bMore').disabled = !sp.text_ready;
  $('more').placeholder = sp.text_ready ? 'Describe more changes, e.g. add a 1/4 inch hole in the left wall 1 inch up' : 'Describing changes in words needs the editing model (section 7b)';
}
const typeSpec = type => spec.types.find(t => t.type === type);
function refreshSections(){ spec && spec.types.forEach(t => $('sec-' + t.type).classList.toggle('hidden', !$('fs-' + t.type).children.length)); }
function addFeatureRow(type, cells){
  const ts = typeSpec(type); if(!ts) return;
  const tr = document.createElement('tr'); tr.dataset.type = type;
  ts.fields.forEach(fd => {
    const td = document.createElement('td'); const c = cells ? cells[fd.key] : null; let el;
    if(fd.kind === 'choice'){
      el = document.createElement('select');
      [...(fd.nullable || !c || c.value === '' ? [''] : []), ...fd.options].forEach(op => { const o = document.createElement('option'); o.value = op; o.textContent = op || '(default)'; el.appendChild(o); });
    } else { el = document.createElement('input'); el.placeholder = fd.nullable ? 'default' : ''; }
    el.dataset.k = fd.key; el.value = c ? c.value : '';
    if(c){ el.className = 's-' + c.status; el.title = c.label + (c.note ? ' - ' + c.note : ''); } else if(!fd.nullable){ el.className = 's-missing'; el.title = 'required'; }
    td.appendChild(el); tr.appendChild(td);
  });
  const x = document.createElement('td'); x.className = 'x'; const b = document.createElement('button'); b.textContent = '\u00d7'; b.title = 'remove';
  b.onclick = () => { tr.remove(); refreshSections(); }; x.appendChild(b); tr.appendChild(x);
  $('fs-' + type).appendChild(tr); refreshSections();
}
function showFeatureRows(rows){ if(!spec) return; spec.types.forEach(t => $('fs-' + t.type).textContent = ''); (rows || []).forEach(r => addFeatureRow(r.type, r.cells)); refreshSections(); }
function readFeatureRows(){
  if(!spec) return [];
  return spec.types.flatMap(t => [...$('fs-' + t.type).querySelectorAll('tr')]).map(tr => {
    const v = {}; tr.querySelectorAll('[data-k]').forEach(el => v[el.dataset.k] = el.value); return {type: tr.dataset.type, v};
  });
}

// ---------- read the description
function clearAll(){
  sid = null; currentType = null; spec = null;
  $('rows').textContent = ''; $('fsections').textContent = ''; $('issues').textContent = ''; $('conf').textContent = ''; $('readAs').textContent = ''; $('fnote').textContent = '';
  bsay(''); $('revCard').classList.add('hidden'); $('outCard').classList.add('hidden'); stopViewer();
}
function showAnalysis(j){
  sid = j.sid; currentType = j.part_type; $('ptype').value = j.part_type;
  let c = '';
  if(j.confidence != null){
    c = `classifier confidence ${Math.round(j.confidence * 100)}%`;
    if(j.confidence < 0.85) c += ' - not certain, also possible: ' + j.alts.filter(a => a[0] !== j.part_type).map(a => `${a[0]} ${Math.round(a[1] * 100)}%`).join(', ') + '. Change the part type if this is wrong.';
  } else c = 'part type chosen by you';
  $('conf').textContent = c;
  $('readAs').textContent = 'Part read as: "' + j.text_norm + '"' + (j.feature_text_norm ? '   |   Features read as: "' + j.feature_text_norm + '"' : '');
  showRows(j.rows); setupFeatures(j.feature_spec); showFeatureRows(j.feat_rows);
  $('fnote').textContent = j.feat_note || (j.feature_spec && !(j.feat_rows || []).length ? 'No features yet - add some below, or describe them.' : '');
  $('issues').textContent = ''; bsay('');
  $('revCard').classList.remove('hidden');
}
$('bGo').onclick = async () => {
  const text = $('text').value.trim();
  if(!text){ say('Type a description of the part first.', 'err'); return; }
  const my = ++reqId; busy(true); clearAll(); say('Reading your description... (the first request can take a minute)');
  try { const j = await api('analyze', {text}); if(my !== reqId) return; showAnalysis(j); say('Check the values below, then press Build part.', 'ok'); }
  catch(e){ if(my === reqId){ clearAll(); say(e.message, 'err'); } }
  finally { if(my === reqId) busy(false); }
};
$('ptype').onchange = async () => {
  if(!sid){ $('ptype').value = currentType || TYPES[0]; return; }
  const my = ++reqId, want = $('ptype').value; busy(true); say('Re-reading as ' + want + '...');
  try { const j = await api('retype', {sid, part_type: want}); if(my !== reqId) return; showAnalysis(j); $('outCard').classList.add('hidden'); say('Re-read as ' + want + '. Check the values, then press Build part.', 'ok'); }
  catch(e){ if(my === reqId){ $('ptype').value = currentType; say(e.message, 'err'); } }
  finally { if(my === reqId) busy(false); }
};
$('bReset').onclick = () => { reqId++; clearAll(); $('text').value = ''; say(''); busy(false); };

// ---------- build
function showIssues(list){
  const ul = $('issues'); ul.textContent = '';
  list.forEach(i => { const li = document.createElement('li'); li.className = i.level === 'warning' ? 'warn' : ''; li.textContent = (i.level === 'warning' ? 'Warning - ' : '') + i.where + ': ' + i.msg; ul.appendChild(li); });
}
$('bBuild').onclick = async () => {
  if(!sid){ bsay('Read a description first.', 'err'); return; }
  const my = ++reqId; busy(true); bsay('Building...'); $('issues').textContent = '';
  try {
    const j = await api('build', {sid, values: readValues(), rows: readFeatureRows()}); if(my !== reqId) return;
    showRows(j.rows); if(spec) showFeatureRows(j.feat_rows); showIssues(j.issues);
    if(j.stl_url){
      $('outCard').classList.remove('hidden'); $('dl').href = j.stl_url + '?dl=1'; $('json').textContent = JSON.stringify(j.json, null, 2);
      $('info').textContent = `${j.info.size_mm.map(v => v.toFixed(1)).join(' x ')} mm, volume ${(j.info.volume_mm3 / 1000).toFixed(1)} cm3, watertight: ${j.info.watertight ? 'yes' : 'NO'}`;
      preview(j.stl_url); bsay('Built. Change anything and press Build part again to update it.', 'ok');
    } else { $('outCard').classList.add('hidden'); bsay('Not built yet - fix the problems listed below.', 'err'); }
  } catch(e){ if(my === reqId) bsay(e.message, 'err'); }
  finally { if(my === reqId) busy(false); }
};

// ---------- add features
$('bAdd').onclick = () => { if(spec){ addFeatureRow($('addType').value, null); $('fnote').textContent = ''; } };
$('bMore').onclick = async () => {
  const text = $('more').value.trim(); if(!text || !sid) return;
  const my = ++reqId; busy(true); bsay('Reading your changes...');
  try { const j = await api('features_extract', {sid, values: readValues(), rows: readFeatureRows(), text}); if(my !== reqId) return;
        showFeatureRows(j.rows); $('more').value = ''; $('fnote').textContent = ''; bsay('Added. Check the new rows, then press Build part.', 'ok'); }
  catch(e){ if(my === reqId) bsay(e.message, 'err'); }
  finally { if(my === reqId) busy(false); }
};

// ---------- 3D preview: wheel scrolling is left to the page; zoom with the buttons
let viewer = null;
function stopViewer(){ if(viewer){ cancelAnimationFrame(viewer.raf); viewer.ctl.dispose && viewer.ctl.dispose(); viewer.renderer.dispose(); viewer = null; } $('view').textContent = ''; }
function zoom(f){ if(!viewer) return; const {cam, ctl, R} = viewer; const d = cam.position.clone().sub(ctl.target); const L = Math.min(Math.max(d.length() * f, R * 0.8), R * 12); cam.position.copy(ctl.target).add(d.setLength(L)); }
function homeView(){ if(!viewer) return; const {cam, ctl, R} = viewer; ctl.target.set(0, 0, 0); cam.position.set(R * 1.1, -R * 2.6, R * 1.4); }
$('zIn').onclick = () => zoom(0.8); $('zOut').onclick = () => zoom(1.25); $('zFit').onclick = homeView;
function preview(url){
  stopViewer(); const box = $('view'); box.textContent = 'loading preview...';
  new STLLoader().load(url, geo => {
    box.textContent = '';
    const w = box.clientWidth || 600, h = box.clientHeight || 440;
    const renderer = new THREE.WebGLRenderer({antialias:true}); renderer.setPixelRatio(Math.min(window.devicePixelRatio || 1, 2)); renderer.setSize(w, h); box.appendChild(renderer.domElement);
    const scene = new THREE.Scene(); scene.background = new THREE.Color(0xeceff3);
    geo.computeVertexNormals(); geo.center(); geo.computeBoundingSphere();
    const R = geo.boundingSphere.radius;
    scene.add(new THREE.Mesh(geo, new THREE.MeshStandardMaterial({color:0x4a90d9, metalness:.1, roughness:.6})));
    scene.add(new THREE.AmbientLight(0xffffff, .7)); const d = new THREE.DirectionalLight(0xffffff, .9); d.position.set(1, 2, 3); scene.add(d);
    const cam = new THREE.PerspectiveCamera(40, w / h, R / 50, R * 60); cam.up.set(0, 0, 1); scene.add(new THREE.AxesHelper(R * 1.1));
    const ctl = new OrbitControls(cam, renderer.domElement);
    ctl.enableZoom = false;                         // the mouse wheel scrolls the page instead of zooming (zooming through the part broke the view)
    ctl.enableDamping = true;
    viewer = {renderer, cam, ctl, R, raf: 0}; homeView();
    (function loop(){ if(!viewer || viewer.renderer !== renderer) return; viewer.raf = requestAnimationFrame(loop); ctl.update(); renderer.render(scene, cam); })();
  }, undefined, () => { box.textContent = '3D preview could not load (needs internet for three.js). The STL download still works.'; });
}
window.addEventListener('resize', () => { if(!viewer) return; const b = $('view'), w = b.clientWidth, h = b.clientHeight; viewer.renderer.setSize(w, h); viewer.cam.aspect = w / h; viewer.cam.updateProjectionMatrix(); });
</script></body></html>"""

app = Flask("text_to_stl")
SESSIONS = {}

def _rows_json(rows):
    return [{"field": r["field"], "value": fmt_value(r["value"]), "status": r["status"], "label": STATUS_LABEL[r["status"]], "note": r["note"]} for r in rows]

_SHORT = {"from_end_in": "From end", "across_in": "Up", "along_in": "From left end", "up_in": "Up", "angle_deg": "Angle (deg)", "diameter_in": "Diameter",
          "count": "Count", "spacing_in": "Spacing", "spacing_deg": "Spacing (deg)", "x_in": "x", "y_in": "y", "rows": "Rows", "inset_in": "Inset",
          "direction": "Direction", "kind": "Type", "edge": "Where", "size_in": "Size", "orientation": "Orientation", "position_in": "Position",
          "thickness_in": "Thickness", "height_in": "Height", "width_in": "Width"}
_FEATURE_HELP = lambda pt: {"hole": HOLE_SPEC[pt]["hint"] + " " + PATTERN_HELP, "bottom_hole": BOTTOM_HELP.get(pt, "") + " " + PATTERN_HELP,
                            "edge": EDGE_HELP[pt], "divider": DIVIDER_HELP, "cutout": CUTOUT_HELP + " " + HOLE_SPEC[pt]["hint"]}
def _feature_spec(pt):
    if pt not in FEATURE_TYPES: return None
    return {"types": [{"type": t, "title": FEATURE_LABELS[t], "help": _FEATURE_HELP(pt)[t],
                       "fields": [{**{k: fd.get(k) for k in ("key", "label", "kind", "options", "nullable")}, "short": _SHORT.get(fd["key"], fd["label"])} for fd in fdefs(t, pt)]}
                      for t in FEATURE_TYPES[pt]],
            "text_ready": callable(globals().get("edit_features")) and FEATURE_BANK is not None}

def _feature_rows_json(rows):
    return [{"type": r["type"], "cells": {k: {"value": fmt_value(c["value"]), "status": c["status"], "label": STATUS_LABEL[c["status"]], "note": c["note"]}
                                          for k, c in r["cells"].items()}} for r in rows or []]

def _analysis_json(sid, s):
    conf = max(s["probs"].values()) if s["probs"] else None
    alts = sorted(s["probs"].items(), key=lambda kv: -kv[1])[:3] if s["probs"] else []
    return {"sid": sid, "part_type": s["part_type"], "confidence": conf, "alts": alts, "text_norm": s["text_norm"], "feature_text_norm": s.get("feat_text_norm", ""),
            "rows": _rows_json(s["rows"]), "feature_spec": _feature_spec(s["part_type"]), "feat_rows": _feature_rows_json(s.get("feat_rows")), "feat_note": s.get("feat_note", "")}

def _where(issue, frows):
    """'Part - thickness_in' / 'Holes in the walls #2' for an issue, so the user knows which row to fix"""
    i = issue.get("feature")
    if i is None or not frows or i >= len(frows): return "Part" + (f" - {issue['field']}" if issue.get("field") else "") if "feature" not in issue else "Features"
    t = frows[i]["type"]; k = sum(1 for r in frows[:i + 1] if r["type"] == t)
    return f"{FEATURE_LABELS[t]} #{k}" + (f" - {issue['field']}" if issue.get("field") else "")

@app.get("/")
def index(): return INDEX_HTML.replace("__PART_TYPES__", json.dumps(PART_TYPES))

def h_analyze(b):
    text = str(b.get("text", "")).strip()
    if not text: return {"error": "Type a description of the part first."}, 400
    s = analyze(text); sid = uuid.uuid4().hex; SESSIONS[sid] = s
    return _analysis_json(sid, s), 200

def h_retype(b):
    old = SESSIONS.get(b.get("sid"))
    if old is None or b.get("part_type") not in PART_TYPES: return {"error": "Read a description first."}, 400
    s = analyze(old["text"], forced_type=b["part_type"]); SESSIONS[b["sid"]] = s
    return _analysis_json(b["sid"], s), 200

def _ui_rows(b): return [r for r in (b.get("rows") or []) if isinstance(r, dict)]
def _ui_values(b): return {str(k): str(v) for k, v in (b.get("values") or {}).items()}

def h_build(b):
    s = SESSIONS.get(b.get("sid"))
    if s is None: return {"error": "Read a description first (the server may have restarted)."}, 400
    res = build_all(s, _ui_values(b), _ui_rows(b)); pt, p = s["part_type"], res["params"]
    issues = [{**i, "where": _where({k: v for k, v in i.items() if k != "feature"}, None)} for i in res["issues"]] + \
             [{**i, "where": _where(i, res["feat_rows"])} for i in res["feat_issues"]]
    return {"rows": _rows_json(res["rows"]), "feat_rows": _feature_rows_json(res["feat_rows"]), "issues": issues, "info": res["info"],
            "stl_url": ("/stl/" + Path(res["stl_path"]).name) if res["stl_path"] else None,
            "json": {"part": {"part_type": pt, **{f: p.get(f) for f in SCHEMA[pt]}}, "features": res["features"]}}, 200

def h_features_extract(b):
    s = SESSIONS.get(b.get("sid"))
    if s is None: return {"error": "Read a description first (the server may have restarted)."}, 400
    if s["part_type"] not in FEATURE_TYPES: return {"error": "Features can only be added to brackets and containers."}, 400
    text = str(b.get("text", "")).strip()
    if not text: return {"error": "Describe what to add first."}, 400
    res = add_features_from_text(s, _ui_values(b), _ui_rows(b), text)
    if "error" in res: return res, 400
    return {"rows": _feature_rows_json(res["rows"]), "text_norm": res["text_norm"]}, 200

HANDLERS = {"analyze": h_analyze, "retype": h_retype, "build": h_build, "features_extract": h_features_extract}

def _run(name, b):
    try: return HANDLERS[name](b)
    except Exception as e: return {"error": f"{type(e).__name__}: {e}"}, 500

# direct (synchronous) endpoints - handy for scripts/tests
for _name in HANDLERS:
    app.add_url_rule(f"/api/{_name}", f"api_{_name}", (lambda n: (lambda: (lambda r: (jsonify(r[0]), r[1]))(_run(n, request.get_json(silent=True) or {}))))(_name), methods=["POST"])

# background jobs: the page starts a job and polls it, so no single HTTP request stays open long enough for Colab's proxy to time it out
JOBS = {}
@app.post("/api/start/<name>")
def api_start(name):
    if name not in HANDLERS: return jsonify(error=f"unknown action {name}"), 404
    b, jid = request.get_json(silent=True) or {}, uuid.uuid4().hex
    JOBS[jid] = {"done": False, "t": time.time()}
    def work():
        body, status = _run(name, b); JOBS[jid] = {"done": True, "status": status, "body": body, "t": time.time()}
    threading.Thread(target=work, daemon=True).start()
    for k in [k for k, v in list(JOBS.items()) if v["done"] and time.time() - v["t"] > 600]: JOBS.pop(k, None)     # forget stale finished jobs
    return jsonify(job=jid)

@app.get("/api/job/<jid>")
def api_job(jid):
    j = JOBS.get(jid)
    if j is None: return jsonify(error="That request was lost (the server may have restarted) - please try again."), 404
    if not j["done"]: return jsonify(done=False, seconds=round(time.time() - j["t"], 1))
    JOBS.pop(jid, None)
    return jsonify(done=True, status=j["status"], body=j["body"])

@app.errorhandler(Exception)
def _any_error(e):                                  # anything that escapes a route still comes back as JSON the page can show
    code = getattr(e, "code", 500) if isinstance(getattr(e, "code", None), int) else 500
    return jsonify(error=f"{type(e).__name__}: {e}"), code

@app.get("/stl/<name>")
def get_stl(name):
    if not re.fullmatch(r"[a-z_]+_\d+\.stl", name) or not (WORKDIR / name).exists(): abort(404)
    return send_file((WORKDIR / name).resolve(), mimetype="model/stl", as_attachment=request.args.get("dl") == "1", download_name=name)

# ---- start the server (re-running this cell restarts it cleanly)
PORT = 8765
if globals().get("_SERVER"): globals()["_SERVER"].shutdown()
_SERVER = make_server("0.0.0.0", PORT, app, threaded=True)
threading.Thread(target=_SERVER.serve_forever, daemon=True).start()
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(PORT, height=950)      # shows the page right here in the notebook
    print("If the page doesn't show, use output.serve_kernel_port_as_window(PORT) for a separate tab.")
except ImportError:
    print(f"Running locally: open http://localhost:{PORT}")

## 11. Save the artifacts
Downloads a zip with the classifier, the LoRA adapter, the catalog tables, the split, and the feedback log — everything needed to move the app to a Hugging Face Space.

In [ ]:
# ------------------------------------------------------------------ SAVE everything you need to deploy on Hugging Face Spaces / for the report
ART = Path("artifacts"); ART.mkdir(exist_ok=True)
torch.save({"state_dict": clf.state_dict(), "vocab": VOCAB, "part_types": PART_TYPES}, ART / "classifier.pt")
model.save_pretrained(ART / "lora_adapter"); tok.save_pretrained(ART / "lora_adapter")
json.dump({"base_model": MODEL_NAME, "schema": SCHEMA, "standards": STANDARDS, "washer_lookup": WASHER_LOOKUP,
           "ranges": RANGES, "num_tol": NUM_TOL}, open(ART / "catalog_and_config.json", "w"), indent=1)
data[["sheet", "sheet_row", "split", "part_type", "text", "text_norm", "target"]].to_csv(ART / "dataset_with_split.csv", index=False)   # REAL data only
if len(aug_df): aug_df[["aug_from", "part_type", "orig_text", "text", "text_norm", "target"]].to_csv(ART / "augmented_samples.csv", index=False)   # SYNTHETIC, kept separate
if FEAT_READY:                                                      # all feature-request data is SYNTHETIC (Claude-written / Claude-generated); saved apart from the manual data
    _cols = ["source", "sheet", "sheet_row", "split", "part_type", "text", "text_norm", "target"]
    for _src in feat_df.source.unique(): feat_df[feat_df.source == _src][_cols].to_csv(ART / f"feature_requests_{_src}.csv", index=False)
if len(clf_extra_df): clf_extra_df[["part_type", "text", "text_norm", "source"]].to_csv(ART / "classifier_phrases_claude_synthetic.csv", index=False)   # SYNTHETIC, kept separate
if FEEDBACK_FEATURES.exists(): (ART / "feedback_features.jsonl").write_text(FEEDBACK_FEATURES.read_text())
if FEEDBACK_LOG.exists(): (ART / "feedback.jsonl").write_text(FEEDBACK_LOG.read_text())
with zipfile.ZipFile("artifacts.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ART.rglob("*"):
        if f.is_file(): z.write(f, f.relative_to(ART.parent))
print("saved:", sorted(p.name for p in ART.iterdir()))
try:
    from google.colab import files; files.download("artifacts.zip")
except ImportError:
    pass

## 12. Publish to Hugging Face
Uploads the three model repos with full model cards (including this run's evaluation results) and the dataset with its card, then prints the links for your README. Public repos: anyone can see them, including the copy of your spreadsheet.

In [ ]:
# ------------------------------------------------------------------ PUBLISH to Hugging Face: model cards, example bank and dataset (public repos)
# Last cell of Run all. Uploads the three model repos with full model cards (including this run's evaluation results) and the dataset with
# its card. The training cells already uploaded freshly trained weights; this adds the cards and everything else, and keeps them current.
import json, ast, io, shutil
from pathlib import Path
import pandas as pd, torch
from huggingface_hub import HfApi
from safetensors.torch import save_file

def _publish():
    G = globals()
    _missing = [k for k in ("clf", "VOCAB", "model", "tok", "data", "aug_df", "clf_extra_df", "feat_df", "feat_train", "feat_test",
                            "part_context", "feature_system_prompt", "feature_messages", "FEATURE_BANK", "CLF_FP", "EXT_FP") if k not in G]
    if _missing:
        raise RuntimeError(f"Run the notebook from the top first (missing {_missing}). With the models on Hugging Face it reloads them instead of retraining.")
    BASE_MODEL, FEAT_K = MODEL_NAME, FEATURE_K
    FEAT_BASE = FEATURE_MODEL_NAME if not G.get("_feat_reuse", True) and FEATURE_MODEL_NAME else MODEL_NAME
    U = repo_id

    # ---- the notebook's own code, for the helper files in the repos
    _nb = None
    try:
        from google.colab import _message
        _nb = _message.blocking_request("get_ipynb", timeout_sec=120)["ipynb"]
    except Exception:
        _local = sorted(Path(".").glob("*.ipynb"))
        if _local: _nb = json.loads(_local[0].read_text())
    if _nb is None: raise RuntimeError("Couldn't read this notebook. Save a copy as Text_to_STL_Pipeline.ipynb next to it and re-run.")
    CELLS = ["\n".join(l for l in "".join(c["source"]).splitlines() if not l.lstrip().startswith(("!", "%"))) for c in _nb["cells"] if c.get("cell_type") == "code"]
    def source_of(names):
        out = []
        for src in CELLS:
            try: tree = ast.parse(src)
            except SyntaxError: continue
            for n in tree.body:
                nm = n.name if isinstance(n, (ast.FunctionDef, ast.ClassDef)) else (n.targets[0].id if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) else None)
                if nm in names: out.append(ast.get_source_segment(src, n))
        return "\n\n".join(out)
    NORMALIZER_PY = ("# Text normalizer from the Text-to-STL notebook: fractions, number words, feet/mm/cm -> decimal inches, screw sizes standardised.\n"
                     "# normalize_text: for part descriptions. normalize_feature_text: also turns screw sizes into clearance-hole diameters.\n"
                     f"import re\nNUM_TOL = {NUM_TOL!r}\n\n" + next(s for s in CELLS if "def normalize_text" in s))

    # ---- this run's evaluation results
    def pct(v): return f"{100 * float(v):.1f}%"
    def md_table(df):
        cols = list(df.columns)
        rows = ["| " + " | ".join([df.index.name or ""] + cols) + " |", "|" + "---|" * (len(cols) + 1)]
        for i, r in df.iterrows(): rows.append("| " + " | ".join([str(i)] + [pct(v) if isinstance(v, float) else str(v) for v in r]) + " |")
        return "\n".join(rows)
    NOT_RUN = "_Not recorded in this run: the evaluation cell (section 8) didn't run before this cell._"
    R = {"clf": {"Validation accuracy (91 real rows)": pct(clf_accuracy(val_df)), "Test accuracy (87 real rows, unseen sizes)": pct(clf_accuracy(test_df))}}
    if "_hits" in G: R["clf"]["Casual stress prompts (60, unusual wording, never trained on)"] = pct(G["_hits"].mean())
    if isinstance(G.get("comparison"), pd.DataFrame): R["ext"] = md_table(G["comparison"])
    if "type_ok" in G and "e2e" in G:
        R["e2e"] = f"Classifier then extractor on the test set: part type right {pct(G['type_ok'].mean())}, part type and every field right {pct((G['e2e'].all_correct.values & G['type_ok']).mean())}."
    if "outs_by_k" in G and isinstance(G.get("res"), list) and G["res"]: R["feat"] = md_table(pd.DataFrame(G["res"]).set_index("model"))
    _spl = G.get("SPLITTER_EVAL")
    SPLIT_TXT = (f"The rule-based splitter that separates part and feature text keeps {_spl['kept_whole']}/{_spl['n_base']} held-out part descriptions whole "
                 f"and splits {_spl['split_ok']}/{_spl['n_combined']} ({pct(_spl['split_ok'] / max(_spl['n_combined'], 1))}) held-out combined requests correctly.") if _spl else ""
    NOTEBOOK_LINE = f"Training and evaluation notebook: [Colab]({COLAB_LINK})" if COLAB_LINK else "Training and evaluation: `Text_to_STL_Pipeline.ipynb` (CMU Project 1)."

    OUT = Path("hf_publish"); shutil.rmtree(OUT, ignore_errors=True)
    D_CLF, D_EXT, D_FEAT, D_DATA = (OUT / n for n in ("classifier", "extractor", "feature_editor", "dataset"))
    for d in (D_CLF, D_EXT, D_FEAT, D_DATA): d.mkdir(parents=True)

    # ============================================================== 1. classifier (trained from scratch)
    write_classifier(D_CLF)
    (D_CLF / "normalizer.py").write_text(NORMALIZER_PY)
    (D_CLF / "model.py").write_text(
        "# Part-type classifier from the Text-to-STL project (bag-of-words neural network, trained from scratch).\n"
        "import json, re\nfrom pathlib import Path\nimport torch, torch.nn as nn, torch.nn.functional as F\nfrom safetensors.torch import load_file\n\n"
        + source_of({"CLF_STOP", "clf_tokens"}) + r'''


class BowClassifier(nn.Module):
    # token ids -> mean of learned embeddings -> small MLP -> part type
    def __init__(self, vocab_size, n_classes, dim=48, hidden=64, p=0.3):
        super().__init__()
        self.emb = nn.EmbeddingBag(vocab_size, dim, mode="mean")
        self.net = nn.Sequential(nn.Dropout(p), nn.Linear(dim, hidden), nn.ReLU(), nn.Dropout(p), nn.Linear(hidden, n_classes))
    def forward(self, ids, offsets):
        return self.net(self.emb(ids, offsets))


class PartTypeClassifier:
    # PartTypeClassifier(folder).predict(normalized_text) -> (part_type, {part_type: probability})
    def __init__(self, folder="."):
        folder = Path(folder)
        cfg = json.loads((folder / "config.json").read_text())
        self.part_types, self.vocab = cfg["part_types"], json.loads((folder / "vocab.json").read_text())
        self.net = BowClassifier(len(self.vocab), len(self.part_types), cfg["embedding_dim"], cfg["hidden"], cfg["dropout"])
        self.net.load_state_dict(load_file(str(folder / "model.safetensors"))); self.net.eval()

    @torch.no_grad()
    def predict(self, text_norm):
        ids = [self.vocab[t] for t in clf_tokens(text_norm) if t in self.vocab] or [0]
        probs = F.softmax(self.net(torch.tensor(ids), torch.tensor([0])), dim=-1)[0].tolist()
        best = max(range(len(probs)), key=probs.__getitem__)
        return self.part_types[best], dict(zip(self.part_types, probs))
''')
    _cfg = json.loads((D_CLF / "config.json").read_text())
    _clf_res = "\n".join(f"| {k} | {v} |" for k, v in R["clf"].items())
    (D_CLF / "README.md").write_text(f"""---
license: {MODEL_LICENSE}
language: [en]
library_name: pytorch
pipeline_tag: text-classification
tags: [text-classification, cad, 3d-printing, bag-of-words, from-scratch]
datasets: [{U(DATA_REPO)}]
---

# Text-to-STL part-type classifier (trained from scratch)

Stage 1 of a text-to-STL pipeline. It reads a plain-English description of a simple mechanical part and decides which of five
part families it is: **{", ".join(PART_TYPES)}**. The predicted type then selects the schema the
[parameter extractor](https://huggingface.co/{U(EXT_REPO)}) fills in.

## Model
A small bag-of-words neural classifier written in PyTorch, with **all weights trained from random initialisation** (nothing pretrained).
- **Input features:** the description is normalized (`normalizer.py`: fractions, number words and units become decimal inches),
  lowercased, every number is replaced by a `<num>` token, request-framing words ("I want it to be", "make", "please"...) are removed,
  and the remaining words plus adjacent word pairs are looked up in a {len(VOCAB):,}-token vocabulary (tokens seen in at least 2 training texts).
- **Architecture:** EmbeddingBag (mean of {_cfg["embedding_dim"]}-dimensional learned embeddings) -> dropout 0.3 -> {_cfg["hidden"]}-unit ReLU layer -> dropout 0.3 -> 5 outputs (softmax).
- **Output:** a part type and a probability for each type. The app treats under 85% as uncertain and shows the runner-up types.

## Training
- **Data:** {len(clf_train_df):,} rows: {len(train_real_df)} real training descriptions, {len(aug_df)} augmented descriptions and {len(clf_extra_df)} synthetic phrase variations
  (see the [dataset](https://huggingface.co/datasets/{U(DATA_REPO)})). Validation and test sets are real data only.
- **Split:** grouped by part size, so no size appears in both training and test.
- **Optimiser:** AdamW, learning rate 5e-3, weight decay 1e-3, batch size 32, 80 epochs; the epoch with the best validation accuracy is kept.
- **Why request-framing words are removed:** in the raw data they identified who wrote each spreadsheet sheet (for example "it", "be" and
  "thats" appear in ~40% of gasket descriptions and almost no container descriptions), so the model learned the writer instead of the part.

## Results
| Metric | Value |
|---|---|
{_clf_res}

## Usage
```python
import sys
from huggingface_hub import snapshot_download
path = snapshot_download("{U(CLF_REPO)}"); sys.path.insert(0, path)
from model import PartTypeClassifier
from normalizer import normalize_text

clf = PartTypeClassifier(path)
print(clf.predict(normalize_text('a box 6 by 4 by 3 inches with 1/8" walls')))   # ('rect_container', {{...}})
```

## Intended use and limitations
- Built for five part families only; anything else is forced into the closest of the five.
- Trained on English descriptions written by a small team plus synthetic variations, so unusual wording can still confuse it.
- It decides the part type only; dimensions come from the extractor, and every value is checked by rule-based validation before any CAD is built.

{NOTEBOOK_LINE}
""")

    # ============================================================== 2. parameter extractor (LoRA fine-tune)
    write_adapter(D_EXT)
    (D_EXT / "normalizer.py").write_text(NORMALIZER_PY)
    _schema_rows = "\n".join(f"| `{pt}` | {', '.join(f'`{f}`' for f in fs)} |" for pt, fs in SCHEMA.items())
    (D_EXT / "README.md").write_text(f"""---
base_model: {BASE_MODEL}
library_name: peft
license: {MODEL_LICENSE}
language: [en]
pipeline_tag: text-generation
tags: [lora, peft, json-extraction, cad, 3d-printing]
datasets: [{U(DATA_REPO)}]
---

# Text-to-STL parameter extractor (LoRA fine-tune of {BASE_MODEL.split("/")[-1]})

Stage 2 of a text-to-STL pipeline. Given the part type (from the [classifier](https://huggingface.co/{U(CLF_REPO)})) and a normalized
description, it writes the part's dimensions as compact JSON in one standard schema. The JSON is validated by rules and then built in CadQuery.

## Model
- **Base model:** [{BASE_MODEL}](https://huggingface.co/{BASE_MODEL}), with a LoRA adapter (rank 16, alpha 32, dropout 0.05) on all attention
  and MLP projections (`q,k,v,o,gate,up,down_proj`). This repo holds only the adapter.
- **Prompt:** `Part type: <type>\\nDescription: <normalized text>\\nJSON:\\n`, answered with one JSON object.
- **Output schema** (decimal inches; `hole_count` integer, `screw_size` text):

| Part type | Keys |
|---|---|
{_schema_rows}

## Training
- **Data:** {len(train_df)} rows: {len(train_real_df)} real training descriptions and {len(aug_df)} augmented ones (values swapped between training rows, with typos and unit
  wording added). Validation ({len(val_df)}) and test ({len(test_df)}) sets are real, grouped by part size so test sizes are unseen.
- **Objective:** causal language-modelling loss on the JSON answer tokens only (the prompt is masked), maximum length 256 tokens.
- **Optimiser:** AdamW, learning rate {LORA_LR:g}, {LORA_EPOCHS} epochs, effective batch {BATCH_SIZE} (micro-batches of {MICRO_BATCH} with gradient accumulation),
  5% warm-up then linear decay, gradient clipping 1.0, gradient checkpointing; full precision on a free Colab T4. The epoch with the lowest
  validation loss is kept.
- **Decoding:** greedy, so the same input always gives the same output.

## Results (held-out test set)
A field counts as correct within a small tolerance of the label; `hole_count` and `screw_size` must match exactly. The baseline is the same base
model with the adapter switched off, prompted with the schema and one example per part type.

{R.get("ext", NOT_RUN)}

{R.get("e2e", "")}

## Usage
```python
import sys, torch
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

path = snapshot_download("{U(EXT_REPO)}"); sys.path.insert(0, path)
from normalizer import normalize_text

tok = AutoTokenizer.from_pretrained(path)
model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained("{BASE_MODEL}"), path).eval()
prompt = "Part type: washer\\nDescription: " + normalize_text("washer for an M7 screw, 1/16 thick") + "\\nJSON:\\n"
ids = tok(prompt, return_tensors="pt", add_special_tokens=False)
out = model.generate(**ids, max_new_tokens=96, do_sample=False)
print(tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True))
```

## Intended use and limitations
- Only for the five part families and the schema above; it does not handle features such as holes or chamfers (that is the
  [feature editor](https://huggingface.co/{U(FEAT_REPO)})).
- It can produce values that are not in the description; the app flags those and checks every value before building.
- Descriptions should go through `normalizer.py` first, as during training.

{NOTEBOOK_LINE}
""")

    # ============================================================== 3. feature editor (off the shelf: no trained weights)
    pd.DataFrame({"part_type": feat_train.part_type, "part_context": [part_context(r.part_type, r.base) for r in feat_train.itertuples()],
                  "text": feat_train.text, "text_norm": feat_train.text_norm, "target": feat_train.target}).to_csv(D_FEAT / "example_bank.csv", index=False)
    (D_FEAT / "system_prompts.json").write_text(json.dumps({pt: feature_system_prompt(pt) for pt in FEATURE_TYPES}, indent=1))
    (D_FEAT / "normalizer.py").write_text(NORMALIZER_PY)
    (D_FEAT / "feature_editor.py").write_text(
        "# Builds the off-the-shelf feature-editing prompt exactly as the Text-to-STL app does: system prompt + the K most similar solved requests.\n"
        "import json, re\nfrom pathlib import Path\nimport numpy as np, pandas as pd\nfrom sklearn.feature_extraction.text import TfidfVectorizer\n\n"
        + source_of({"_mask", "ExampleBank"}) + r'''


class FeaturePrompter:
    # FeaturePrompter(folder).messages(part_type, part_context, request_norm, k) -> chat messages for any chat model
    def __init__(self, folder="."):
        folder = Path(folder)
        self.bank_df = pd.read_csv(folder / "example_bank.csv")
        self.system = json.loads((folder / "system_prompts.json").read_text())
        self.bank = ExampleBank(self.bank_df)

    def messages(self, part_type, part_context, request_norm, k=''' + str(FEAT_K) + r'''):
        msgs = [{"role": "system", "content": self.system[part_type]}]
        for r in self.bank.retrieve(part_type, request_norm, k).itertuples():
            msgs += [{"role": "user", "content": f"Part: {r.part_context}\nRequest: {r.text_norm}"}, {"role": "assistant", "content": r.target}]
        msgs.append({"role": "user", "content": f"Part: {part_context}\nRequest: {request_norm}"})
        return msgs
''')
    _chk = {}; exec(compile((D_FEAT / "feature_editor.py").read_text(), "feature_editor.py", "exec"), _chk)
    _fp = _chk["FeaturePrompter"](D_FEAT)
    for r in list(feat_test.itertuples())[:20]:                     # the published prompter must build exactly the notebook's prompts
        assert _fp.messages(r.part_type, part_context(r.part_type, r.base), r.text_norm, FEAT_K) == \
               feature_messages(r.part_type, r.base, r.text_norm, FEAT_K, FEATURE_BANK), "published prompter differs from the notebook"
    _ex = feat_test.iloc[0]
    (D_FEAT / "example_prompt.json").write_text(json.dumps({"request": _ex.text, "expected": json.loads(_ex.target),
        "messages": feature_messages(_ex.part_type, _ex.base, _ex.text_norm, FEAT_K, FEATURE_BANK)}, indent=1))
    _ftypes = ", ".join(sorted({f["type"] for fs in feat_df.features for f in fs}))
    (D_FEAT / "README.md").write_text(f"""---
base_model: {FEAT_BASE}
license: {MODEL_LICENSE}
language: [en]
pipeline_tag: text-generation
tags: [in-context-learning, retrieval, few-shot, json-extraction, cad, 3d-printing]
datasets: [{U(DATA_REPO)}]
---

# Text-to-STL feature editor (off-the-shelf {FEAT_BASE.split("/")[-1]} + retrieved examples)

Stage 3 of a text-to-STL pipeline. It turns requests such as *"round the corners 1/4 inch and add four M5 holes in the base, 1 inch from the sides"*
into a list of features ({_ftypes}) as JSON, which are validated by rules and cut or added in CadQuery.

**No weights were trained for this stage.** It uses [{FEAT_BASE}](https://huggingface.co/{FEAT_BASE}) exactly as released (in the app, the same
base model as the [parameter extractor](https://huggingface.co/{U(EXT_REPO)}) with its LoRA adapter switched off). This repo holds what makes it
work: the prompts, the example bank and the retrieval code.

## How it works
For each request the model receives:
1. a **system prompt** describing the feature JSON schema and its conventions for that part type (`system_prompts.json`);
2. the **{FEAT_K} most similar solved requests** for the same part type, as earlier chat turns. They are chosen from the
   {len(feat_train)}-request example bank (`example_bank.csv`) by TF-IDF similarity over word unigrams and bigrams, with numbers masked so
   similarity reflects wording, not sizes;
3. the part's dimensions and the new request (normalized with `normalize_feature_text`, which also converts screw sizes like "M5" into
   clearance-hole diameters).

Greedy decoding; the reply is parsed as `{{"features": [...]}}`. `example_prompt.json` shows one complete prompt.

**Why off the shelf:** feature requests are open-ended and combine freely. With retrieval, supporting a new phrasing means adding an example
to the bank rather than retraining.

## Results (held-out hand-written requests)
Outcome-based scoring: a feature is correct when it ends up in the same place with the same size after defaults are applied
(so leaving a position blank, meaning centred, counts the same as stating the centre).

{R.get("feat", NOT_RUN)}

{SPLIT_TXT}

## Usage
```python
import sys, json
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM

path = snapshot_download("{U(FEAT_REPO)}"); sys.path.insert(0, path)
from feature_editor import FeaturePrompter
from normalizer import normalize_feature_text

msgs = FeaturePrompter(path).messages("rect_container",
    "rect_container: length_a 6, length_b 4, height 3, thickness 0.125 (inside 5.75 x 3.75, inside height 2.875)",
    normalize_feature_text("round the corners 1/4 inch and add a divider down the middle"))
tok = AutoTokenizer.from_pretrained("{FEAT_BASE}")
model = AutoModelForCausalLM.from_pretrained("{FEAT_BASE}").eval()
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
print(tok.decode(model.generate(ids, max_new_tokens=400, do_sample=False)[0, ids.shape[1]:], skip_special_tokens=True))
```
The `part_context` string follows the format in `example_bank.csv`.

## Intended use and limitations
- Brackets, rectangular containers and cylindrical containers only.
- A small model learning an invented format from examples on every request makes mistakes, especially with unusual wording or arithmetic
  ("three sections" means two dividers); the app shows every value in an editable table and refuses impossible geometry.
- The example bank is synthetic (written or generated with an AI assistant), so it may not reflect how real users phrase requests.

{NOTEBOOK_LINE}
""")

    # ============================================================== 4. dataset
    def split_csvs(df, folder, cols):
        (D_DATA / folder).mkdir(parents=True, exist_ok=True); n = {}
        for s, name in (("train", "train"), ("val", "validation"), ("test", "test")):
            part = df[df.split == s][cols]; part.to_csv(D_DATA / folder / f"{name}.csv", index=False); n[name] = len(part)
        return n
    real = data.copy(); real["params"] = real["target"]
    real["audit_flag"] = real["audit_bad"].map(lambda x: bool(len(x)) if isinstance(x, (list, tuple)) else bool(x))
    N_REAL = split_csvs(real, "real", ["sheet", "sheet_row", "part_type", "text", "text_norm", "params", "audit_flag"])
    (D_DATA / "synthetic").mkdir(exist_ok=True)
    aug = aug_df.rename(columns={"target": "params"})
    aug[[c for c in ("aug_from", "part_type", "orig_text", "text", "text_norm", "params") if c in aug.columns]].to_csv(D_DATA / "synthetic" / "augmented_parts.csv", index=False)
    clf_extra_df[["part_type", "text", "text_norm"]].to_csv(D_DATA / "synthetic" / "classifier_phrases.csv", index=False)
    feat = feat_df.copy(); feat["base_part"] = feat["base"].map(json.dumps); feat["features"] = feat["target"]
    feat["source"] = feat["source"].map({"claude_written": "written", "claude_generated": "generated"}).fillna(feat["source"])
    N_FEAT = split_csvs(feat, "feature_requests", ["source", "part_type", "base_part", "text", "text_norm", "features"])
    (D_DATA / "raw").mkdir(exist_ok=True); shutil.copyfile(DATA_PATH, D_DATA / "raw" / "data.xlsx")
    by_type = data.part_type.value_counts().to_dict()
    n_written, n_gen = int((feat_df.source == "claude_written").sum()), int((feat_df.source == "claude_generated").sum())
    (D_DATA / "README.md").write_text(f"""---
license: {DATA_LICENSE}
language: [en]
pretty_name: Text-to-STL Parts
size_categories: [1K<n<10K]
task_categories: [text-classification, text-generation]
tags: [cad, 3d-printing, mechanical-parts, json-extraction, synthetic]
configs:
- config_name: real_parts
  default: true
  data_files:
  - split: train
    path: real/train.csv
  - split: validation
    path: real/validation.csv
  - split: test
    path: real/test.csv
- config_name: augmented_parts
  data_files: synthetic/augmented_parts.csv
- config_name: classifier_phrases
  data_files: synthetic/classifier_phrases.csv
- config_name: feature_requests
  data_files:
  - split: train
    path: feature_requests/train.csv
  - split: validation
    path: feature_requests/validation.csv
  - split: test
    path: feature_requests/test.csv
---

# Text-to-STL Parts

Plain-English descriptions of simple mechanical parts paired with their exact dimensions, plus requests to add features to those parts.
Used to train and evaluate a text-to-STL pipeline: a [part-type classifier](https://huggingface.co/{U(CLF_REPO)}),
a [parameter extractor](https://huggingface.co/{U(EXT_REPO)}) and a [feature editor](https://huggingface.co/{U(FEAT_REPO)}).

## Configurations
| Config | Rows | Origin | Used for |
|---|---|---|---|
| `real_parts` | {sum(N_REAL.values())} (train {N_REAL["train"]} / validation {N_REAL["validation"]} / test {N_REAL["test"]}) | Manually created | Training, model selection and all testing of stages 1-2 |
| `augmented_parts` | {len(aug_df)} | Synthetic | Training stages 1-2 only |
| `classifier_phrases` | {len(clf_extra_df)} | Synthetic | Training the classifier only |
| `feature_requests` | {sum(N_FEAT.values())} (train {N_FEAT["train"]} / validation {N_FEAT["validation"]} / test {N_FEAT["test"]}) | Synthetic | Example bank (train) and evaluation (validation/test) of stage 3 |

`raw/data.xlsx` is the original spreadsheet (one sheet per part family); the notebook reuses it instead of asking for an upload.

## Real parts (manually created)
- **Part families:** {", ".join(f"{k} ({v})" for k, v in by_type.items())}.
- **Source:** dimensions come from real McMaster-Carr catalog listings (pipe flange gaskets, washers and spacers, L-brackets) and real container
  sizes; each row's description was written by our team in varied, often casual English. The labels are the catalog dimensions themselves,
  so every label is a real, manufacturable part.
- **Fields:** `text` (as written), `text_norm` (after the project's normalizer: fractions, number words, feet/mm -> decimal inches),
  `params` (JSON in the standard schema, decimal inches), `sheet`/`sheet_row` (location in the original spreadsheet).
- **Quality checks:** an automated audit checks that every labelled number appears in its own description (tolerance {NUM_TOL} in). It caught
  15 mislabelled rows in the first version of the data, which were fixed; `audit_flag` marks the rows it still flags.
- **Splits:** 70/15/15, grouped by part size, so a given pipe or screw size never appears in more than one split; test results therefore
  measure performance on unseen sizes.

## Synthetic data (generated with an AI assistant, stored separately)
- **`augmented_parts`:** made only from training rows by swapping values between rows of the same part type and adding surface noise
  (typos, unit wording, casing). Every row is validated and builds as a watertight solid.
- **`classifier_phrases`:** descriptions with varied nouns, units and phrasing for each part type, to stop the classifier relying on
  each spreadsheet's writing style.
- **`feature_requests`:** {n_written} requests written individually and {n_gen} generated from templates, each on a real base part
  (`base_part`), covering holes (single, rows and grids, including screw sizes), chamfers and fillets, dividers and cutouts. Every request
  passes a numeric and wording audit, and the resulting part must build as a valid, watertight solid. Validation and test rows are
  individually written only.

## Intended use and limitations
- Intended for training and evaluating text-to-CAD parameter extraction for these five part families.
- Descriptions come from a small team plus synthetic text, so they don't cover the full range of how people describe parts; the feature
  requests in particular were not written by real users.
- Dimensions describe geometry only: there is no information on materials, pressure ratings, temperatures or loads, so parts made from this
  data (gaskets in particular) need engineering review before any load-bearing or sealing use.
- Contains no personal data.

{NOTEBOOK_LINE}
""")

    # ============================================================== upload
    for name, d, rtype, what in ((CLF_REPO, D_CLF, "model", "classifier + model card"), (EXT_REPO, D_EXT, "model", "LoRA adapter + model card"),
                                 (FEAT_REPO, D_FEAT, "model", "example bank, prompts + model card"), (DATA_REPO, D_DATA, "dataset", "dataset + card")):
        push_to_hub(name, d, rtype, what)
    print(f"""
---------------- README links (copy into your submission) ----------------
# Interface
**GUI:** [Colab Link]({COLAB_LINK or "PASTE-YOUR-COLAB-SHARE-LINK"})
  *Type a plain-English description of a pipe gasket, washer, L-bracket, rectangular container or cylindrical container, including any holes,
  rounded or chamfered edges, dividers or cutouts. Check the colour-coded values, edit any of them, press Build part, and download a watertight STL.*
# Models
- **Stage 1 - part-type classifier (trained from scratch):** [Model Card](https://huggingface.co/{U(CLF_REPO)})
  *Bag-of-words neural classifier trained from random weights on {len(clf_train_df):,} descriptions; picks which of the five part types is described.*
- **Stage 2 - parameter extractor (fine-tuned):** [Model Card](https://huggingface.co/{U(EXT_REPO)})
  *LoRA fine-tune of {BASE_MODEL} on {len(train_df)} descriptions; writes the part's dimensions as standard JSON for CadQuery.*
- **Stage 3 - feature editor (off the shelf):** [Model Card](https://huggingface.co/{U(FEAT_REPO)})
  *{FEAT_BASE} used as released, guided by the {FEAT_K} most similar of {len(feat_train)} solved requests (TF-IDF retrieval); writes holes, edges, dividers and cutouts as JSON.*
# Data
- **Dataset:** [Dataset Link](https://huggingface.co/datasets/{U(DATA_REPO)})
  *{sum(N_REAL.values())} manually created part descriptions with catalog dimensions (grouped train/validation/test split), plus synthetic training data
  ({len(aug_df)} augmented, {len(clf_extra_df)} phrase variations) and {len(feat_df)} synthetic feature requests, all stored separately. License: {DATA_LICENSE}.*
""")

if not PUBLISH_TO_HF:
    print("PUBLISH_TO_HF is False: nothing was published.")
elif not hf_token():
    print("Not signed in to Hugging Face, so nothing was published. Re-run the Hugging Face cell near the top to sign in, then this cell.")
else:
    _publish()
